### Mandi Price Data Exploration

#### This notebook explores historical mandi price data for Wheat, Rice and Maize across Uttar Pradesh, Punjab and Haryana.

#### The objective is to create yearly price indicators and price-volatility features that can later be merged with production and rainfall data.

In [1]:
import pandas as pd
import numpy as np
import re
import csv

In [2]:
file_path = "../../data/raw/prices/wheat_up_2021_01.csv"

In [3]:
def parse_agmarknet_price_csv(
    file_path,
    state,
    commodity
):
    records = []

    current_market = None

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as file:

        reader = csv.reader(file)

        for row in reader:

            if not row:
                continue

            first_value = row[0].strip()

            if first_value.startswith("Market Name :"):

                current_market = (
                    first_value
                    .split(":", 1)[1]
                    .strip()
                )

                continue

            if first_value == "Arrival Date":
                continue

            if first_value.startswith(
                "Date Wise Prices for Specified Commodity"
            ):
                continue

            if first_value.startswith("Commodity :"):
                continue

            if (
                current_market
                and
                re.fullmatch(
                    r"\d{2}/\d{2}/\d{4}",
                    first_value
                )
            ):

                row += [""] * (6 - len(row))

                records.append(
                    {
                        "State": state,
                        "Commodity": commodity,
                        "Market": current_market,
                        "Arrival_Date": row[0].strip(),
                        "Arrivals_MT": row[1].strip(),
                        "Variety": row[2].strip(),
                        "Min_Price": row[3].strip(),
                        "Max_Price": row[4].strip(),
                        "Modal_Price": row[5].strip()
                    }
                )

    price_data = pd.DataFrame(records)

    if not price_data.empty:

        price_data["Arrival_Date"] = pd.to_datetime(
            price_data["Arrival_Date"],
            format="%d/%m/%Y",
            errors="coerce"
        )

        numeric_columns = [
            "Arrivals_MT",
            "Min_Price",
            "Max_Price",
            "Modal_Price"
        ]

        for column in numeric_columns:

            price_data[column] = pd.to_numeric(
                price_data[column],
                errors="coerce"
            )

    return price_data

In [4]:
price_raw = parse_agmarknet_price_csv(
    file_path,
    state="Uttar Pradesh",
    commodity="Wheat"
)

In [5]:
price_raw.head()

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price
0,Uttar Pradesh,Wheat,Achalda APMC,2021-01-01,6.0,Dara,1590.0,1610.0,1600.0
1,Uttar Pradesh,Wheat,Achalda APMC,2021-01-02,5.0,Dara,1560.0,1600.0,1580.0
2,Uttar Pradesh,Wheat,Achalda APMC,2021-01-04,6.0,Dara,1560.0,1600.0,1580.0
3,Uttar Pradesh,Wheat,Achalda APMC,2021-01-05,5.0,Dara,1560.0,1600.0,1580.0
4,Uttar Pradesh,Wheat,Achalda APMC,2021-01-06,6.0,Dara,1600.0,1700.0,1680.0


In [6]:
price_raw.shape

(3517, 9)

In [7]:
price_raw.columns.tolist()

['State',
 'Commodity',
 'Market',
 'Arrival_Date',
 'Arrivals_MT',
 'Variety',
 'Min_Price',
 'Max_Price',
 'Modal_Price']

In [8]:
price_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3517 entries, 0 to 3516
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   State         3517 non-null   object        
 1   Commodity     3517 non-null   object        
 2   Market        3517 non-null   object        
 3   Arrival_Date  3517 non-null   datetime64[ns]
 4   Arrivals_MT   3517 non-null   float64       
 5   Variety       3517 non-null   object        
 6   Min_Price     3509 non-null   float64       
 7   Max_Price     3510 non-null   float64       
 8   Modal_Price   3517 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(4)
memory usage: 247.4+ KB


In [9]:
# Validating the date range

print(
    "Start:",price_raw["Arrival_Date"].min()
)

print(
    "End:",price_raw["Arrival_Date"].max()
)



Start: 2021-01-01 00:00:00
End: 2021-01-31 00:00:00


In [10]:
# Checking how many markets are parsed 

price_raw["Market"].nunique()

159

In [11]:
price_raw["Market"].unique()[:20]

array(['Achalda APMC', 'Achnera APMC', 'Agra APMC', 'Ait APMC',
       'Ajuha APMC', 'Aliganj APMC', 'Aligarh APMC', 'Amroha APMC',
       'Anandnagar APMC', 'Atarra APMC', 'Atrauli APMC', 'Auraiya APMC',
       'Ayodhya APMC', 'Azamgarh APMC', 'Baberu APMC', 'Babrala APMC',
       'Badayoun APMC', 'Bahraich APMC', 'Ballia APMC', 'Balrampur APMC'],
      dtype=object)

In [12]:
# Missing value 

price_raw.isnull().sum()

State           0
Commodity       0
Market          0
Arrival_Date    0
Arrivals_MT     0
Variety         0
Min_Price       8
Max_Price       7
Modal_Price     0
dtype: int64

In [13]:
price_raw["Modal_Price"].describe()

count    3517.000000
mean     1698.987489
std        71.494682
min      1450.000000
25%      1660.000000
50%      1705.000000
75%      1735.000000
max      2420.000000
Name: Modal_Price, dtype: float64

In [14]:
# Checking for any strange prices

price_raw[
price_raw["Modal_Price"]<=0 ]

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price


In [15]:
price_raw[
price_raw["Min_Price"]>price_raw["Max_Price"]]

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price


In [16]:
# Calculating the volatility calculation

average_price = (
    price_raw["Modal_Price"].mean()
)

price_std = (
    price_raw["Modal_Price"].std()
)

price_volatility_pct = (
    price_std/average_price)*100

In [17]:
print(
    "Average Modal Price:",
    round(average_price, 2)
)

print(
    "Price Standard Deviation:",
    round(price_std, 2)
)

print(
    "Price Volatility %:",
    round(price_volatility_pct, 2)
)

Average Modal Price: 1698.99
Price Standard Deviation: 71.49
Price Volatility %: 4.21


In [18]:
market_reference = pd.read_csv(
    "../../data/raw/prices/agmarknet_current_market_reference.csv"
)

In [19]:
market_reference.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price
0,Andhra Pradesh,Prakasam,Santhamaguluru APMC,Paddy(Common),B P T,FAQ,21/09/2026,2500.0,2580.0,2550.0
1,Andhra Pradesh,Prakasam,Maddipadu APMC,Paddy(Common),B P T,FAQ,21/09/2026,2800.0,2800.0,2800.0
2,Keralam,Kozhikode(Calicut),Mukkom Market,Pineapple,Other,Large,21/09/2026,8000.0,9000.0,8500.0
3,Keralam,Kozhikode(Calicut),Mukkom Market,Coconut,Other,FAQ,21/09/2026,5800.0,6100.0,6000.0
4,Keralam,Kozhikode(Calicut),Mukkom Market,Pumpkin,Pumpkin,FAQ,21/09/2026,1800.0,2000.0,1900.0


In [20]:
market_reference.columns.tolist()

['State',
 'District',
 'Market',
 'Commodity',
 'Variety',
 'Grade',
 'Arrival_Date',
 'Min_x0020_Price',
 'Max_x0020_Price',
 'Modal_x0020_Price']

In [21]:
market_reference = market_reference[
["State" , "District" , "Market"]].copy()

In [22]:
# Removing the duplicates

market_reference = (
    market_reference.drop_duplicates().reset_index(drop=True))

In [23]:
market_reference.shape

(898, 3)

In [24]:
# Making a helper function 

def normalise_string_name(name):

    name = str(name).strip().lower()

    name = re.sub(
    r"\bapmc\b","",name)

    name = re.sub(
    r"s+/"," ",name)

    return name.strip()

In [25]:
# Applying on both datasets

price_raw["Market_Key"]=(
    price_raw["Market"].apply(normalise_string_name))


#Reference dataset

market_reference["Market_Key"]=(
    market_reference["Market"].apply(normalise_string_name))

In [26]:
price_raw.head()

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price,Market_Key
0,Uttar Pradesh,Wheat,Achalda APMC,2021-01-01,6.0,Dara,1590.0,1610.0,1600.0,achalda
1,Uttar Pradesh,Wheat,Achalda APMC,2021-01-02,5.0,Dara,1560.0,1600.0,1580.0,achalda
2,Uttar Pradesh,Wheat,Achalda APMC,2021-01-04,6.0,Dara,1560.0,1600.0,1580.0,achalda
3,Uttar Pradesh,Wheat,Achalda APMC,2021-01-05,5.0,Dara,1560.0,1600.0,1580.0,achalda
4,Uttar Pradesh,Wheat,Achalda APMC,2021-01-06,6.0,Dara,1600.0,1700.0,1680.0,achalda


In [27]:
market_reference.head()

,State,District,Market,Market_Key
0,Andhra Pradesh,Prakasam,Santhamaguluru APMC,santhamaguluru
1,Andhra Pradesh,Prakasam,Maddipadu APMC,maddipadu
2,Keralam,Kozhikode(Calicut),Mukkom Market,mukkom market
3,Telangana,Ranga Reddy,"Saroornagar,RBZ","saroornagar,rbz"
4,Telangana,Hyderabad,RYTHU BAZAR FALAKNUMA,rythu bazar falaknuma


In [28]:
# Checking the mapping table(Currently working with up wheat)

up_market_reference = market_reference[
    market_reference["State"]=="Uttar Pradesh"].copy()
    
    

In [29]:
up_market_reference.head()

,State,District,Market,Market_Key
131,Uttar Pradesh,Etah,Aliganj APMC,aliganj
134,Uttar Pradesh,Jhansi,Jhansi APMC,jhansi
135,Uttar Pradesh,Kasganj,Kasganj APMC,kasganj
136,Uttar Pradesh,Aligarh,Khair APMC,khair
138,Uttar Pradesh,Hamirpur,Maudaha APMC,maudaha


In [30]:
# Checking for ambiguous markets

market_ambiguity = (
    up_market_reference.groupby('Market_Key')["District"].nunique().reset_index(name="District_count"))

In [31]:
market_ambiguity [
   market_ambiguity['District_count']>1]

,Market_Key,District_count


In [32]:
# Creating clean Market -> District table 

market_to_districts = (
    up_market_reference[
    ["Market_Key" , "District"]
    ].drop_duplicates())
    
    

In [33]:
market_to_districts.head()

,Market_Key,District
131,aliganj,Etah
134,jhansi,Jhansi
135,kasganj,Kasganj
136,khair,Aligarh
138,maudaha,Hamirpur


In [34]:
# Merging district into historical prices 

price_with_district = price_raw.merge(
    market_to_districts,
    on="Market_Key",
    how="left")
    

In [35]:
price_with_district.head()

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price,Market_Key,District
0,Uttar Pradesh,Wheat,Achalda APMC,2021-01-01,6.0,Dara,1590.0,1610.0,1600.0,achalda,NaN
1,Uttar Pradesh,Wheat,Achalda APMC,2021-01-02,5.0,Dara,1560.0,1600.0,1580.0,achalda,NaN
2,Uttar Pradesh,Wheat,Achalda APMC,2021-01-04,6.0,Dara,1560.0,1600.0,1580.0,achalda,NaN
3,Uttar Pradesh,Wheat,Achalda APMC,2021-01-05,5.0,Dara,1560.0,1600.0,1580.0,achalda,NaN
4,Uttar Pradesh,Wheat,Achalda APMC,2021-01-06,6.0,Dara,1600.0,1700.0,1680.0,achalda,NaN


In [36]:
# Check how many markets matched

price_with_district["District"].isnull().sum()

np.int64(1110)

In [37]:
unmatched_markets = (
    price_with_district[
        price_with_district["District"].isnull()
    ][["Market", "Market_Key"]]
    .drop_duplicates()
)

print("Unmatched rows:", price_with_district["District"].isnull().sum())
print("Unmatched unique markets:", unmatched_markets.shape[0])

unmatched_markets.head(30)

Unmatched rows: 1110
Unmatched unique markets: 53


,Market,Market_Key
0,Achalda APMC,achalda
23,Achnera APMC,achnera
335,Baberu APMC,baberu
434,Ballia APMC,ballia
458,Balrampur APMC,balrampur
483,Banda APMC,banda
530,Bansi APMC,bansi
561,Barhaj APMC,barhaj
669,Bharuasumerpur APMC,bharuasumerpur
700,Bijnaur APMC,bijnaur


In [38]:
total_markets = price_raw["Market_Key"].nunique()

matched_markets = (
    price_with_district
    .dropna(subset=["District"])["Market_Key"]
    .nunique()
)

print("Total markets:", total_markets)
print("Matched markets:", matched_markets)

print(
    "Market match %:",
    round(matched_markets / total_markets * 100, 2)
)

Total markets: 159
Matched markets: 106
Market match %: 66.67


In [39]:
# Creating a clean list of 53 markets

unmatched_summary = (
    price_with_district[
        price_with_district["District"].isnull()
    ]
    .groupby(
        ["Market", "Market_Key"]
    )
    .agg(
        Price_Rows=("Arrival_Date", "size"),
        First_Date=("Arrival_Date", "min"),
        Last_Date=("Arrival_Date", "max")
    )
    .reset_index()
    .sort_values(
        "Price_Rows",
        ascending=False
    )
)

unmatched_summary

,Market,Market_Key,Price_Rows,First_Date,Last_Date
21,Gadaura APMC,gadaura,30,2021-01-01,2021-01-31
31,Khaga APMC,khaga,29,2021-01-01,2021-01-31
16,Dhanaura APMC,dhanaura,27,2021-01-01,2021-01-31
20,Firozabad APMC,firozabad,27,2021-01-01,2021-01-31
17,Etah APMC,etah,27,2021-01-01,2021-01-31
26,Haldaur APMC,haldaur,27,2021-01-01,2021-01-27
36,Mahoba APMC,mahoba,27,2021-01-01,2021-01-31
35,Maharajganj APMC,maharajganj,26,2021-01-01,2021-01-30
30,Karvi APMC,karvi,26,2021-01-01,2021-01-31
8,Bharuasumerpur APMC,bharuasumerpur,26,2021-01-01,2021-01-31


In [40]:
unmatched_summary.to_csv(
    "../../data/processed/unmatched_up_markets.csv",
    index=False
)

In [41]:
print("Unique unmatched markets:", len(unmatched_summary))

print(
    "Price rows affected:",
    unmatched_summary["Price_Rows"].sum()
)

Unique unmatched markets: 53
Price rows affected: 1110


In [42]:
# Loadint the unmatches_summary file

unmatched_markets = pd.read_csv(
    "../../data/processed/unmatched_up_markets.csv"
)

unmatched_markets.head()

,Market,Market_Key,Price_Rows,First_Date,Last_Date
0,Gadaura APMC,gadaura,30,2021-01-01,2021-01-31
1,Khaga APMC,khaga,29,2021-01-01,2021-01-31
2,Dhanaura APMC,dhanaura,27,2021-01-01,2021-01-31
3,Firozabad APMC,firozabad,27,2021-01-01,2021-01-31
4,Etah APMC,etah,27,2021-01-01,2021-01-31


In [43]:
def clean_market_for_search(market_name):

    market_name = str(market_name)

    market_name = re.sub(
        r"\bAPMC\b",
        "",
        market_name,
        flags=re.IGNORECASE
    )

    return market_name.strip()

In [44]:
unmatched_markets["Search_Name"] = (
    unmatched_markets["Market"]
    .apply(clean_market_for_search)
)

In [45]:
# Importing the geocoder

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


In [46]:
geolocator = Nominatim(
    user_agent="crop-yield-risk-dashboard-market-mapping",
    timeout=10
)

geocode_market = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False
)

In [47]:
# Testing one market first

test_location = geocode_market(
    "Khaga, Uttar Pradesh, India",
    addressdetails=True,
    exactly_one=True
)

test_location

Location(Khaga, Fatehpur, Uttar Pradesh, India, (25.7196316, 81.1058945, 0.0))

In [48]:
test_location.raw["address"]

{'county': 'Khaga',
 'state_district': 'Fatehpur',
 'state': 'Uttar Pradesh',
 'ISO3166-2-lvl4': 'IN-UP',
 'country': 'India',
 'country_code': 'in'}

In [49]:
#  Creating a function that extract possible disctict

def find_market_district(market_name):

    search_name = clean_market_for_search(
        market_name
    )

    query = (
        f"{search_name}, "
        f"Uttar Pradesh, India"
    )

    try:

        location = geocode_market(
            query,
            addressdetails=True,
            exactly_one=True,
            timeout=10
        )

        if location is None:
            return pd.Series(
                [None, None]
            )

        address = location.raw.get(
            "address",
            {}
        )

        district = (
            address.get("state_district")
            or address.get("county")
            or address.get("district")
            or address.get("city_district")
        )

        return pd.Series(
            [
                district,
                location.address
            ]
        )

    except Exception as e:

        print(
            f"Could not geocode {market_name}: {e}"
        )

        return pd.Series(
            [None, None]
        )

In [50]:
find_market_district(
    "Khaga APMC"
)

0                                 Fatehpur
1    Khaga, Fatehpur, Uttar Pradesh, India
dtype: object

In [51]:
# Running for all ummatched list

unmatched_markets[
    [
        "Suggested_District",
        "Geocoded_Address"
    ]
] = unmatched_markets[
    "Market"
].apply(
    find_market_district
)

In [52]:
unmatched_markets[
    [
        "Market",
        "Suggested_District",
        "Geocoded_Address"
    ]
]

,Market,Suggested_District,Geocoded_Address
0,Gadaura APMC,None,None
1,Khaga APMC,Fatehpur,"Khaga, Fatehpur, Uttar Pradesh, India"
2,Dhanaura APMC,Amroha,"Dhanaura, Amroha, Uttar Pradesh, India"
3,Firozabad APMC,Firozabad,"Firozabad, Uttar Pradesh, India"
4,Etah APMC,Etah,"Etah, Uttar Pradesh, India"
5,Haldaur APMC,Bijnor,"Haldaur, Bijnor, Uttar Pradesh, 246726, India"
6,Mahoba APMC,Mahoba,"Mahoba, Uttar Pradesh, India"
7,Maharajganj APMC,Raebareli,"Maharajganj, Raebareli, Uttar Pradesh, India"
8,Karvi APMC,None,None
9,Bharuasumerpur APMC,None,None


In [53]:
unmatched_markets.to_csv(
    "../../data/processed/"
    "unmatched_up_markets_with_candidates.csv",
    index=False
)

In [54]:
# How many failed
print(
    "Still unmatched:",
    unmatched_markets[
        "Suggested_District"
    ].isnull().sum()
)

Still unmatched: 18


In [55]:
unmatched_markets[
    unmatched_markets[
        "Suggested_District"
    ].isnull()
][
    ["Market", "Price_Rows"]
]

,Market,Price_Rows
0,Gadaura APMC,30
8,Karvi APMC,26
9,Bharuasumerpur APMC,26
13,Mohamadabad APMC,26
15,Chandoli APMC,25
23,Khairagarh APMC,25
26,Gurusarai APMC,24
27,Sikandraraau APMC,24
28,Paliakala APMC,24
29,Chorichora APMC,24


In [56]:
# Clean district name
def clean_district_name(name):

    if pd.isna(name):
        return None

    name = str(name).strip()

    name = re.sub(
        r"\s+District$",
        "",
        name,
        flags=re.IGNORECASE
    )

    return name.strip()

In [57]:
unmatched_markets[
    "Suggested_District_Clean"
] = (
    unmatched_markets[
        "Suggested_District"
    ]
    .apply(clean_district_name)
)

In [58]:
unmatched_markets[
    [
        "Market",
        "Suggested_District",
        "Suggested_District_Clean"
    ]
].head(20)

,Market,Suggested_District,Suggested_District_Clean
0,Gadaura APMC,None,None
1,Khaga APMC,Fatehpur,Fatehpur
2,Dhanaura APMC,Amroha,Amroha
3,Firozabad APMC,Firozabad,Firozabad
4,Etah APMC,Etah,Etah
5,Haldaur APMC,Bijnor,Bijnor
6,Mahoba APMC,Mahoba,Mahoba
7,Maharajganj APMC,Raebareli,Raebareli
8,Karvi APMC,None,None
9,Bharuasumerpur APMC,None,None


In [59]:
# Created the geocoded market and district mapping 

geocoded_market_mapping = (
    unmatched_markets[
        [
            "Market_Key",
            "Suggested_District_Clean"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .rename(
        columns={
            "Suggested_District_Clean":
            "Geocoded_District"
        }
    )
)

In [60]:
geocoded_market_mapping.head()

,Market_Key,Geocoded_District
1,khaga,Fatehpur
2,dhanaura,Amroha
3,firozabad,Firozabad
4,etah,Etah
5,haldaur,Bijnor


In [61]:
#Attaching the result wiht price_with_district

price_with_district = (
    price_with_district
    .merge(
        geocoded_market_mapping,
        on="Market_Key",
        how="left"
    )
)

In [62]:
#Filling only missing district values

price_with_district[
    "District"
] = (
    price_with_district[
        "District"
    ]
    .fillna(
        price_with_district[
            "Geocoded_District"
        ]
    )
)

In [63]:
#Removing the temporary column

price_with_district = (
    price_with_district.drop(
        columns=[
            "Geocoded_District"
        ]
    )
)


In [64]:
#Checking remaining missing districts

print(
    "Missing district rows:",
    price_with_district[
        "District"
    ].isnull().sum()
)

Missing district rows: 389


In [65]:
remaining_unmatched = (
    price_with_district[
        price_with_district[
            "District"
        ].isnull()
    ][
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
)

remaining_unmatched

,Market,Market_Key
0,Achalda APMC,achalda
669,Bharuasumerpur APMC,bharuasumerpur
700,Bijnaur APMC,bijnaur
811,Buland Shahr APMC,buland shahr
845,Chandoli APMC,chandoli
934,Chitwadagaon APMC,chitwadagaon
956,Chorichora APMC,chorichora
1259,Gadaura APMC,gadaura
1432,Gurusarai APMC,gurusarai
1580,Jagnair APMC,jagnair


In [66]:
print(
    "Remaining unmatched markets:",
    len(remaining_unmatched)
)

Remaining unmatched markets: 18


In [67]:
# Comparing districts with production

production = pd.read_csv(
    "../../data/processed/"
    "production_history_2021_2025.csv"
)

In [68]:
production_up_districts = set(
    production[
        production["State"]
        == "Uttar Pradesh"
    ]["District"]
    .dropna()
    .unique()
)

In [69]:
#Check price district not present un production

price_districts = set(
    price_with_district[
        "District"
    ]
    .dropna()
    .unique()
)

district_name_mismatches = (
    price_districts
    -
    production_up_districts
)

district_name_mismatches

{'Badaun',
 'Barabanki',
 'Bulandshahar',
 'Farukhabad',
 'Gautam Budh Nagar',
 'Jalaun (Orai)',
 'Kannuj',
 'Kanpur',
 'Khiri (Lakhimpur)',
 'Lakhimpur',
 'Maharajganj',
 'Mau(Maunathbhanjan)',
 'Pillibhit',
 'Raebareli',
 'Raebarelli',
 'Siddharth Nagar'}

In [70]:
#Finding the closest production district name

from difflib import get_close_matches

In [71]:
def find_closest_production_district(name):

    matches = get_close_matches(
        name,
        production_up_districts,
        n=3,
        cutoff=0.5
    )

    return matches

In [72]:
for district in sorted(district_name_mismatches):

    print(
        district,
        "→",
        find_closest_production_district(district)
    )

Badaun → ['Budaun', 'Banda', 'Jalaun']
Barabanki → ['Bara Banki', 'Varanasi', 'Bahraich']
Bulandshahar → ['Bulandshahr', 'Banda', 'Kushinagar']
Farukhabad → ['Farrukhabad', 'Firozabad', 'Moradabad']
Gautam Budh Nagar → ['Gautam Buddha Nagar', 'Kanpur Nagar', 'Sant Kabir Nagar']
Jalaun (Orai) → ['Jalaun', 'Jaunpur']
Kannuj → ['Kannauj', 'Kasganj']
Kanpur → ['Jaunpur', 'Hapur', 'Mainpuri']
Khiri (Lakhimpur) → ['Gorakhpur']
Lakhimpur → ['Lalitpur', 'Rampur', 'Gorakhpur']
Maharajganj → ['Mahrajganj', 'Muzaffarnagar', 'Saharanpur']
Mau(Maunathbhanjan) → []
Pillibhit → ['Pilibhit']
Raebareli → ['Rae Bareli', 'Sambhal']
Raebarelli → ['Rae Bareli', 'Bareilly', 'Ballia']
Siddharth Nagar → ['Siddharthnagar', 'Ambedkar Nagar', 'Kanpur Nagar']


In [73]:
#Building an explicit mapping 

district_name_mapping = {
    "Badaun": "Budaun",
    "Barabanki": "Bara Banki",
    "Bulandshahar": "Bulandshahr",
    "Farukhabad": "Farrukhabad",
    "Gautam Budh Nagar": "Gautam Buddha Nagar",
    "Jalaun (Orai)": "Jalaun",
    "Kannuj": "Kannauj",
    "Kanpur": "Jaunpur",
    "Khiri (Lakhimpur)": "Gorakhpur",
    "Lakhimpur": "Lalitpur",
    "Maharajganj": "Mahrajganj",
    "Mau(Maunathbhanjan)": " ",
    "Pillibhit": "Pilibhit",
    "Raebareli": "Rae Bareli",
    "Raebarelli": "Rae Bareli",
    "Siddharth Nagar": "Siddharthnagar"
}

price_with_district["District"] = (
    price_with_district["District"]
    .replace(district_name_mapping)
)

In [74]:
price_districts = set(
    price_with_district["District"]
    .dropna()
    .unique()
)

district_name_mismatches = (
    price_districts
    -
    production_up_districts
)

district_name_mismatches

{' '}

In [75]:
remaining_unmatched = (
    price_with_district[
        price_with_district["District"].isnull()
    ][
        ["Market", "Market_Key"]
    ]
    .drop_duplicates()
)

remaining_unmatched

,Market,Market_Key
0,Achalda APMC,achalda
669,Bharuasumerpur APMC,bharuasumerpur
700,Bijnaur APMC,bijnaur
811,Buland Shahr APMC,buland shahr
845,Chandoli APMC,chandoli
934,Chitwadagaon APMC,chitwadagaon
956,Chorichora APMC,chorichora
1259,Gadaura APMC,gadaura
1432,Gurusarai APMC,gurusarai
1580,Jagnair APMC,jagnair


In [76]:
for district in sorted(district_name_mismatches):
    print(
        district,
        "→",
        find_closest_production_district(district)
    )

  → []


In [77]:
manual_market_mapping = {
    "achalda": "Auraiya",
    "bharuasumerpur": "Hamirpur",
    "bijnaur": "Bijnor",
    "buland shahr": "Bulandshahr",
    "chandoli": "Chandauli",
    "chitwadagaon": "Ballia",
    "chorichora": "Gorakhpur",
    "gadaura": "Maharajganj",
    "gurusarai": "Jhansi",
    "jagnair": "Agra",
    "karvi": "Chitrakoot",
    "khairagarh": "Agra",
    "mohamadabad": "Farrukhabad",
    "mugrabaadshahpur": "Jaunpur",
    "paliakala": "Lakhimpur Kheri",
    "rasda": "Ballia",
    "sikandraraau": "Hathras",
    "tikonia": "Lakhimpur Kheri"
}

price_with_district["Manual_District"] = (
    price_with_district["Market_Key"]
    .map(manual_market_mapping)
)

price_with_district["District"] = (
    price_with_district["District"]
    .fillna(
        price_with_district["Manual_District"]
    )
)

In [78]:
price_with_district = (
    price_with_district
    .drop(
        columns=["Manual_District"]
    )
)

In [79]:
print(
    "Remaining missing district rows:",
    price_with_district["District"]
    .isnull()
    .sum()
)

Remaining missing district rows: 0


In [80]:
remaining_unmatched = (
    price_with_district[
        price_with_district["District"].isnull()
    ][["Market", "Market_Key"]]
    .drop_duplicates()
)

remaining_unmatched

,Market,Market_Key


In [81]:
price_districts = set(
    price_with_district["District"]
    .dropna()
    .unique()
)

district_name_mismatches = (
    price_districts
    -
    production_up_districts
)

district_name_mismatches

{' ', 'Lakhimpur Kheri', 'Maharajganj'}

In [82]:
price_with_district.head()

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price,Market_Key,District
0,Uttar Pradesh,Wheat,Achalda APMC,2021-01-01,6.0,Dara,1590.0,1610.0,1600.0,achalda,Auraiya
1,Uttar Pradesh,Wheat,Achalda APMC,2021-01-02,5.0,Dara,1560.0,1600.0,1580.0,achalda,Auraiya
2,Uttar Pradesh,Wheat,Achalda APMC,2021-01-04,6.0,Dara,1560.0,1600.0,1580.0,achalda,Auraiya
3,Uttar Pradesh,Wheat,Achalda APMC,2021-01-05,5.0,Dara,1560.0,1600.0,1580.0,achalda,Auraiya
4,Uttar Pradesh,Wheat,Achalda APMC,2021-01-06,6.0,Dara,1600.0,1700.0,1680.0,achalda,Auraiya


In [83]:
#Removing the helper function

price_clean = (
    price_with_district
    .drop(
        columns=["Market_Key"]
    )
    .copy()
)

In [84]:
price_clean.head()

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price,District
0,Uttar Pradesh,Wheat,Achalda APMC,2021-01-01,6.0,Dara,1590.0,1610.0,1600.0,Auraiya
1,Uttar Pradesh,Wheat,Achalda APMC,2021-01-02,5.0,Dara,1560.0,1600.0,1580.0,Auraiya
2,Uttar Pradesh,Wheat,Achalda APMC,2021-01-04,6.0,Dara,1560.0,1600.0,1580.0,Auraiya
3,Uttar Pradesh,Wheat,Achalda APMC,2021-01-05,5.0,Dara,1560.0,1600.0,1580.0,Auraiya
4,Uttar Pradesh,Wheat,Achalda APMC,2021-01-06,6.0,Dara,1600.0,1700.0,1680.0,Auraiya


In [85]:
#Checking for price quality

price_clean[
    [
        "Min_Price",
        "Max_Price",
        "Modal_Price"
    ]
].describe()

,Min_Price,Max_Price,Modal_Price
count,3509.000000,3510.000000,3517.000000
mean,1659.082645,1745.158405,1698.987489
std,73.415676,87.594073,71.494682
min,1400.000000,1500.000000,1450.000000
25%,1610.000000,1700.000000,1660.000000
50%,1660.000000,1750.000000,1705.000000
75%,1700.000000,1800.000000,1735.000000
max,2370.000000,2460.000000,2420.000000


In [86]:
price_clean[
    price_clean["Modal_Price"] <= 0
]

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price,District


In [87]:
price_clean[
    price_clean["Min_Price"]
    >
    price_clean["Max_Price"]
]

,State,Commodity,Market,Arrival_Date,Arrivals_MT,Variety,Min_Price,Max_Price,Modal_Price,District


In [88]:
# Calculating January district level features

january_price_features = (
    price_clean
    .groupby(
        [
            "State",
            "District",
            "Commodity"
        ],
        as_index=False
    )
    .agg(
        Avg_Modal_Price=(
            "Modal_Price",
            "mean"
        ),

        Min_Modal_Price=(
            "Modal_Price",
            "min"
        ),

        Max_Modal_Price=(
            "Modal_Price",
            "max"
        ),

        Price_Std=(
            "Modal_Price",
            "std"
        ),

        Price_Records=(
            "Modal_Price",
            "count"
        ),

        Markets_Available=(
            "Market",
            "nunique"
        )
    )
)

In [89]:
january_price_features.head()

,State,District,Commodity,Avg_Modal_Price,Min_Modal_Price,Max_Modal_Price,Price_Std,Price_Records,Markets_Available
0,Uttar Pradesh,,Wheat,1741.200000,1730.0,1745.0,4.153312,25,1
1,Uttar Pradesh,Agra,Wheat,1648.932039,1550.0,1890.0,68.433632,103,6
2,Uttar Pradesh,Aligarh,Wheat,1746.900990,1620.0,1830.0,58.789371,101,4
3,Uttar Pradesh,Amethi,Wheat,1740.326087,1680.0,1800.0,33.754048,46,2
4,Uttar Pradesh,Amroha,Wheat,1824.021739,1700.0,1900.0,91.794575,46,2


In [90]:
# Calculating volaitility

january_price_features[
    "Price_Volatility_Pct"
] = (
    january_price_features["Price_Std"]
    /
    january_price_features["Avg_Modal_Price"]
) * 100

In [91]:
january_price_features[
    [
        "District",
        "Commodity",
        "Avg_Modal_Price",
        "Price_Std",
        "Price_Volatility_Pct"
    ]
].head(20)

,District,Commodity,Avg_Modal_Price,Price_Std,Price_Volatility_Pct
0,,Wheat,1741.200000,4.153312,0.238532
1,Agra,Wheat,1648.932039,68.433632,4.150179
2,Aligarh,Wheat,1746.900990,58.789371,3.365352
3,Amethi,Wheat,1740.326087,33.754048,1.939524
4,Amroha,Wheat,1824.021739,91.794575,5.032537
5,Auraiya,Wheat,1680.108696,43.658894,2.598576
6,Ayodhya,Wheat,1699.400000,21.228911,1.249200
7,Azamgarh,Wheat,1735.638298,6.478813,0.373281
8,Bahraich,Wheat,1715.073529,13.312900,0.776229
9,Ballia,Wheat,1745.073529,18.557950,1.063448


In [92]:
price_clean.to_csv(
    "../../data/processed/"
    "wheat_up_2021_01_clean.csv",
    index=False
)

In [93]:
january_price_features.to_csv(
    "../../data/processed/"
    "wheat_up_2021_01_price_features_test.csv",
    index=False
)

In [94]:
import requests

AGMARKNET_URL = (
    "https://agmarknet.gov.in/"
    "PriceAndArrivals/DatewiseCommodityReport.aspx"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "Chrome/120.0 Safari/537.36"
    )
}

session = requests.Session()

response = session.get(
    AGMARKNET_URL,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
print("Page size:", len(response.text))

Status code: 200
Page size: 1003


In [95]:
!pip install beautifulsoup4

In [96]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

select_boxes = soup.find_all("select")

print("Number of dropdowns:", len(select_boxes))

Number of dropdowns: 0


In [97]:
for box in select_boxes:

    print(
        "ID:",
        box.get("id"),
        "| Name:",
        box.get("name")
    )

In [98]:
print("Final URL:", response.url)
print("Page title:", soup.title.text.strip() if soup.title else "No title")

print("Forms:", len(soup.find_all("form")))
print("Iframes:", len(soup.find_all("iframe")))
print("Inputs:", len(soup.find_all("input")))

Final URL: https://agmarknet.gov.in/PriceAndArrivals/DatewiseCommodityReport.aspx
Page title: Agmarknet 2.0
Forms: 0
Iframes: 0
Inputs: 0


In [99]:
for frame in soup.find_all("iframe"):
    print(
        "Iframe src:",
        frame.get("src")
    )

In [100]:
print(
    "Commodity found:",
    "Commodity" in response.text
)

print(
    "VIEWSTATE found:",
    "__VIEWSTATE" in response.text
)

Commodity found: False
VIEWSTATE found: False


In [101]:
print(response.text[:1500])

<!doctype html><html lang="en"><head><meta charset="utf-8"/><link rel="icon" href="/favicon.png"/><meta name="viewport" content="width=device-width,initial-scale=1"/><meta name="theme-color" content="#000000"/><meta name="description" content="Agmarknet 2.0"/><meta name="keywords" content="Agmarknet, agriculture market, mandi prices, agri commodities, farmers portal, India agriculture"/><link rel="apple-touch-icon" href="/favicon.png"/><link rel="manifest" href="/manifest.json"/><title>Agmarknet 2.0</title><link rel="preconnect" href="https://fonts.googleapis.com"/><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin/><link href="https://fonts.googleapis.com/css2?family=Noto+Sans+Meetei+Mayek:wght@100..900&display=swap" rel="stylesheet"/><script defer="defer" src="/static/js/main.5e8d1f58.js"></script><link href="/static/css/main.57056727.css" rel="stylesheet"></head><body><noscript>You need to enable JavaScript to run this app.</noscript><div id="root"></div></body></ht

In [102]:
# Installin selenium

%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [103]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


In [104]:
# Let python open AGNMARKNET in chrome

AGMARKNET_URL = (
    "https://agmarknet.gov.in/"
    "PriceAndArrivals/DatewiseCommodityReport.aspx"
)

driver = webdriver.Chrome()

driver.get(AGMARKNET_URL)

time.sleep(5)

print("Title:", driver.title)
print("URL:", driver.current_url)

Title: Page Not Found - Agmarknet 2.0
URL: https://agmarknet.gov.in/PriceAndArrivals/DatewiseCommodityReport.aspx


In [105]:
selects = driver.find_elements(
    By.TAG_NAME,
    "select"
)

inputs = driver.find_elements(
    By.TAG_NAME,
    "input"
)

buttons = driver.find_elements(
    By.TAG_NAME,
    "button"
)

print("Select boxes:", len(selects))
print("Inputs:", len(inputs))
print("Buttons:", len(buttons))

Select boxes: 0
Inputs: 2
Buttons: 40


In [106]:
for i, button in enumerate(buttons):
    text = button.text.strip()

    if text:
        print(
            i,
            "→",
            repr(text),
            "| ID:",
            button.get_attribute("id"),
            "| Class:",
            button.get_attribute("class")
        )

1 → 'Sitemap' | ID:  | Class: text-gray-700 hover:text-gray-900 px-3 py-1 rounded hover:bg-gray-200
2 → 'Feedback' | ID:  | Class: text-gray-700 hover:text-gray-900 px-3 py-1 rounded hover:bg-gray-200
3 → 'A+' | ID:  | Class: px-1 text-gray-700 hover:bg-gray-200 rounded
4 → 'A' | ID:  | Class: px-1 text-gray-700 hover:bg-gray-200 rounded
5 → 'A-' | ID:  | Class: px-1 text-gray-700 hover:bg-gray-200 rounded
6 → 'Screen Reader' | ID:  | Class: text-gray-700 hover:text-gray-900 px-3 py-1 rounded hover:bg-gray-200
7 → 'A' | ID:  | Class: flex items-center justify-center w-6 h-6 contrast-control border-2 border-gray-400 bg-white text-black rounded focus:outline-none focus:ring-2 focus:ring-amber-900
8 → 'A' | ID:  | Class: flex items-center justify-center w-6 h-6 contrast-control  bg-black text-white rounded focus:outline-none focus:ring-2 focus:ring-amber-900
22 → 'English' | ID:  | Class: bhashini-dropdown-btn
24 → 'Home' | ID:  | Class: text-[#000000] font-medium text-sm hover:text-[#333

In [107]:
for i, element in enumerate(inputs):

    print(
        i,
        "| Type:",
        element.get_attribute("type"),
        "| ID:",
        element.get_attribute("id"),
        "| Name:",
        element.get_attribute("name"),
        "| Placeholder:",
        element.get_attribute("placeholder"),
        "| Value:",
        element.get_attribute("value")
    )

0 | Type: text | ID: searchPages | Name:  | Placeholder: Search pages... | Value: 
1 | Type: checkbox | ID: suggested-feedback-checkbox | Name:  | Placeholder:  | Value: on


In [108]:
comboboxes = driver.find_elements(
    By.CSS_SELECTOR,
    '[role="combobox"]'
)

print(
    "Combobox elements:",
    len(comboboxes)
)

Combobox elements: 1


In [109]:
for i, box in enumerate(comboboxes):

    print(
        i,
        "| Text:",
        repr(box.text),
        "| ID:",
        box.get_attribute("id"),
        "| Class:",
        box.get_attribute("class"),
        "| aria-label:",
        box.get_attribute("aria-label")
    )

0 | Text: '' | ID: searchPages | Class: border border-gray-300 rounded px-3 py-1 w-auto mt-1 focus:outline-none focus:ring-2 focus:ring-blue-400 | aria-label: None


In [110]:
for i, button in enumerate(buttons):
    text = button.text.strip()
    if text:
        print(i, "→", repr(text))

1 → 'Sitemap'
2 → 'Feedback'
3 → 'A+'
4 → 'A'
5 → 'A-'
6 → 'Screen Reader'
7 → 'A'
8 → 'A'
22 → 'English'
24 → 'Home'
25 → 'Reports'
26 → 'Market Profile'
27 → 'Market Atlas'
28 → 'Helpdesk'
29 → 'Research Studies'
30 → "What's New"
31 → 'Official Login'
33 → 'Go back'
34 → 'Take me to login'


In [111]:
print("Current URL:", driver.current_url)
print("Page title:", driver.title)

Current URL: https://agmarknet.gov.in/PriceAndArrivals/DatewiseCommodityReport.aspx
Page title: Page Not Found - Agmarknet 2.0


In [112]:
body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_text[:2500])

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Page not found
The page you are looking for doesn't exist.
Go back
Take me to login
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspection, Department of Agriculture & Farmers Welfare, Ministry of Agriculture and Farmers Welfare, A-Block, CGO Complex, NH-IV F

In [113]:
box = comboboxes[0]

print("Text:", repr(box.text))
print("ID:", box.get_attribute("id"))
print("Class:", box.get_attribute("class"))
print("aria-label:", box.get_attribute("aria-label"))
print("Outer HTML:")
print(box.get_attribute("outerHTML"))

Text: ''
ID: searchPages
Class: border border-gray-300 rounded px-3 py-1 w-auto mt-1 focus:outline-none focus:ring-2 focus:ring-blue-400
aria-label: None
Outer HTML:
<input id="searchPages" placeholder="Search pages..." role="combobox" aria-autocomplete="list" aria-expanded="false" aria-controls=":r2:" class="border border-gray-300 rounded px-3 py-1 w-auto mt-1 focus:outline-none focus:ring-2 focus:ring-blue-400" type="text" value="">


In [114]:
print(
    "Wheat on page:",
    "Wheat" in body_text
)

print(
    "Commodity on page:",
    "Commodity" in body_text
)

print(
    "Year on page:",
    "Year" in body_text
)

print(
    "Month on page:",
    "Month" in body_text
)

Wheat on page: False
Commodity on page: False
Year on page: False
Month on page: False


In [115]:
driver.get("https://agmarknet.gov.in/")

time.sleep(5)

print("Title:", driver.title)
print("URL:", driver.current_url)

Title: Home-Agmarknet 2.0
URL: https://agmarknet.gov.in/home


In [116]:
reports_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//*[normalize-space()='Reports']"
        )
    )
)

reports_button.click()

time.sleep(3)

In [117]:
elements = driver.find_elements(
    By.CSS_SELECTOR,
    "a, button"
)

for i, element in enumerate(elements):

    text = element.text.strip()

    if text:
        print(
            i,
            "→",
            repr(text)
        )

1 → 'Sitemap'
2 → 'Feedback'
3 → 'A+'
4 → 'A'
5 → 'A-'
6 → 'Screen Reader'
7 → 'A'
8 → 'A'
24 → 'English'
26 → 'Government of India\nDepartment of Agriculture & Farmers Welfare\nDirectorate of Marketing & Inspection'
28 → 'Home'
29 → 'Reports'
30 → 'Price Trend Reports'
31 → 'Price & Arrival Reports'
32 → 'Market Profile'
33 → 'Market Atlas'
34 → 'Helpdesk'
35 → 'Research Studies'
36 → "What's New"
37 → 'Official Login'
48 → 'Market Wise Price & Arrival\n(MSP Commodities & Tomato, Onion, Potato)'
49 → 'Crop Season Wise Price & Arrival\n(MSP Commodities)'
50 → 'Go'
55 → 'About Us'
56 → 'Contact Us'
57 → 'Web Information Manager'
58 → 'Accessibility'
59 → 'Feedback'
60 → 'Website Policy'
61 → 'Privacy Policy'
62 → 'Hyperlinking Policy'
63 → 'Disclaimer'
64 → 'Sitemap'
65 → 'Directorate of Marketing & Inspection (DMI)'
66 → 'Ministry of Agriculture and Farmers Welfare'
67 → 'mrin-dmi[at]nic[dot]in'


In [118]:
keywords = [
    "price",
    "arrival",
    "commodity",
    "date",
    "trend"
]

for i, element in enumerate(elements):

    text = element.text.strip()

    if any(
        word in text.lower()
        for word in keywords
    ):
        print(
            i,
            "→",
            repr(text)
        )

30 → 'Price Trend Reports'
31 → 'Price & Arrival Reports'
48 → 'Market Wise Price & Arrival\n(MSP Commodities & Tomato, Onion, Potato)'
49 → 'Crop Season Wise Price & Arrival\n(MSP Commodities)'


In [119]:
# Click Price Trend Reports

price_trend = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//*[normalize-space()='Price Trend Reports']"
        )
    )
)

driver.execute_script(
    "arguments[0].click();",
    price_trend
)

time.sleep(3)

In [120]:
elements = driver.find_elements(
    By.CSS_SELECTOR,
    "a, button"
)

In [121]:
keywords = [
    "price",
    "commodity",
    "date",
    "market",
    "trend",
    "year"
]

for i, element in enumerate(elements):

    text = element.text.strip()

    if text and any(
        word in text.lower()
        for word in keywords
    ):
        print(
            i,
            "→",
            repr(text)
        )

26 → 'Government of India\nDepartment of Agriculture & Farmers Welfare\nDirectorate of Marketing & Inspection'
30 → 'Market Profile'
31 → 'Market Atlas'
56 → 'Directorate of Marketing & Inspection (DMI)'


In [122]:
body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_text[:4000])

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Price Trend Reports
Report Category
State/UT-wise Reports
District-wise Reports
Market-wise Reports
Market-wise Reports (All Districts)
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspection, Department of Agriculture & Farmers Welfare, Ministry of Agric

In [123]:
market_all = WebDriverWait(
    driver,
    20
).until(
    EC.presence_of_element_located(
        (
            By.XPATH,
            "//*[normalize-space()='Market-wise Reports (All Districts)']"
        )
    )
)

driver.execute_script(
    "arguments[0].click();",
    market_all
)

time.sleep(3)

In [124]:
print("URL:", driver.current_url)
print("Title:", driver.title)

URL: https://agmarknet.gov.in/pricetrendmainmenu
Title: Price Trend Report


In [125]:
# WHat report option appear

body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_text[:4000])

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Price Trend Reports
Report Category
State/UT-wise Reports
District-wise Reports
Market-wise Reports
Market-wise Reports (All Districts)
Market-wise Wholesale Arrivals Monthly Analysis (All Districts)
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Market-wise Wholesale Arrivals Weekly Analysis (All Districts)
Market-wise Wholesale Prices Weekly Analysis (All Districts)
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaime

In [126]:
elements = driver.find_elements(
    By.XPATH,
    "//*"
)

keywords = [
    "date",
    "commodity",
    "price",
    "arrival",
    "year",
    "month"
]

for i, element in enumerate(elements):
    try:
        text = element.text.strip()

        if (
            text
            and len(text) < 150
            and any(
                word in text.lower()
                for word in keywords
            )
        ):
            print(
                i,
                "→",
                repr(text),
                "| tag:",
                element.tag_name
            )
    except:
        pass

295 → 'Price Trend Reports' | tag: div
296 → 'Price Trend Reports' | tag: h1
332 → 'Market-wise Wholesale Arrivals Monthly Analysis (All Districts)' | tag: tr
333 → 'Market-wise Wholesale Arrivals Monthly Analysis (All Districts)' | tag: td
338 → 'Market-wise Wholesale Prices Monthly Analysis (All Districts)' | tag: tr
339 → 'Market-wise Wholesale Prices Monthly Analysis (All Districts)' | tag: td
344 → 'Market-wise Wholesale Arrivals Weekly Analysis (All Districts)' | tag: tr
345 → 'Market-wise Wholesale Arrivals Weekly Analysis (All Districts)' | tag: td
350 → 'Market-wise Wholesale Prices Weekly Analysis (All Districts)' | tag: tr
351 → 'Market-wise Wholesale Prices Weekly Analysis (All Districts)' | tag: td
525 → 'PAGE UPDATED ON : 18-09-2026' | tag: p
526 → 'PAGE UPDATED ON :' | tag: span


In [127]:
monthly_price_report = WebDriverWait(
    driver,
    20
).until(
    EC.presence_of_element_located(
        (
            By.XPATH,
            "//*[normalize-space()='Market-wise Wholesale Prices Monthly Analysis (All Districts)']"
        )
    )
)

driver.execute_script(
    "arguments[0].click();",
    monthly_price_report
)

time.sleep(3)

In [128]:
print("URL:", driver.current_url)
print("Title:", driver.title)

URL: https://agmarknet.gov.in/marketallwholesalemonthanalysis
Title: Market All Prices (Month)


In [129]:
body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_text[:4000])

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Select Commodity
State/UT
Year
Month
Submit
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspection, Department of Agriculture & Farmers Welfare, Ministry of Agriculture and Farmers Welfare, A-

In [130]:
inputs = driver.find_elements(
    By.TAG_NAME,
    "input"
)

buttons = driver.find_elements(
    By.TAG_NAME,
    "button"
)

comboboxes = driver.find_elements(
    By.CSS_SELECTOR,
    '[role="combobox"]'
)

print("Inputs:", len(inputs))
print("Buttons:", len(buttons))
print("Comboboxes:", len(comboboxes))

Inputs: 2
Buttons: 40
Comboboxes: 1


In [131]:
# Inspecting four dropdowns

comboboxes = driver.find_elements(
    By.CSS_SELECTOR,
    '[role="combobox"]'
)

print("Comboboxes:", len(comboboxes))

for i, box in enumerate(comboboxes):
    print(
        "\nINDEX:", i,
        "\nText:", repr(box.text),
        "\nID:", box.get_attribute("id"),
        "\nPlaceholder:", box.get_attribute("placeholder"),
        "\nAria-label:", box.get_attribute("aria-label"),
        "\nAria-expanded:", box.get_attribute("aria-expanded"),
        "\nClass:", box.get_attribute("class")
    )

Comboboxes: 1

INDEX: 0 
Text: '' 
ID: searchPages 
Placeholder: Search pages... 
Aria-label: None 
Aria-expanded: false 
Class: border border-gray-300 rounded px-3 py-1 w-auto mt-1 focus:outline-none focus:ring-2 focus:ring-blue-400


In [132]:
labels = [
    "Select Commodity",
    "State/UT",
    "Year",
    "Month"
]

for label in labels:

    element = driver.find_element(
        By.XPATH,
        f"//*[normalize-space()='{label}']"
    )

    print("\n\n-----", label, "-----")
    print("Tag:", element.tag_name)

    parent = element.find_element(
        By.XPATH,
        ".."
    )

    print(
        parent.get_attribute(
            "outerHTML"
        )[:2000]
    )



----- Select Commodity -----
Tag: div
<div class="flex flex-wrap gap-4 justify-center mb-6"><div class="w-[200px]"><div class="relative flex flex-col mb-4  w-full sm:w-auto"><div class="relative w-full"><div class="peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer  border-gray-300" tabindex="0"><div class="truncate overflow-hidden whitespace-nowrap pr-2 text-[8px] md:text-[12px] lg:text-[18px]"></div><svg stroke="currentColor" fill="none" stroke-width="2" viewBox="0 0 24 24" stroke-linecap="round" stroke-linejoin="round" class="flex-shrink-0 ml-auto" height="20" width="20" xmlns="http://www.w3.org/2000/svg"><polyline points="6 9 12 15 18 9"></polyline></svg></div><label class="absolute left-3 px-1 transition-all top-1/2 -translate-y-1/2 text-[100%] z-0 text-gray-900">Select Commodity</label></div></div></div><div class="w-[200px]"><div class="relative flex flex-col mb-4  w-full sm:w-auto"><div class="relative w-full"><div class="peer w-full

In [133]:
submit_elements = driver.find_elements(
    By.XPATH,
    "//*[normalize-space()='Submit']"
)

print(
    "Submit elements:",
    len(submit_elements)
)

for element in submit_elements:
    print(
        "Tag:",
        element.tag_name,
        "| ID:",
        element.get_attribute("id"),
        "| Class:",
        element.get_attribute("class")
    )

Submit elements: 4
Tag: div | ID:  | Class: mt-6 flex justify-center
Tag: div | ID:  | Class: relative
Tag: button | ID:  | Class: 
        opacity-50 cursor-not-allowed 
        w-48 py-2 rounded-md transition bg-gray-300 text-gray-500 cursor-not-allowed
      
Tag: button | ID:  | Class: submit-feedback


In [134]:
labels = [
    "Select Commodity",
    "State/UT",
    "Year",
    "Month"
]

for label in labels:

    matches = driver.find_elements(
        By.XPATH,
        f"//*[normalize-space(text())='{label}']"
    )

    print("\n======================")
    print("LABEL:", label)
    print("Matches:", len(matches))

    for element in matches[:2]:

        print("Tag:", element.tag_name)
        print("ID:", element.get_attribute("id"))
        print("Class:", element.get_attribute("class"))

        parent = element.find_element(
            By.XPATH,
            ".."
        )

        print(
            parent.get_attribute(
                "outerHTML"
            )[:2500]
        )


LABEL: Select Commodity
Matches: 1
Tag: label
ID: 
Class: absolute left-3 px-1 transition-all top-1/2 -translate-y-1/2 text-[100%] z-0 text-gray-900
<div class="relative w-full"><div class="peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer  border-gray-300" tabindex="0"><div class="truncate overflow-hidden whitespace-nowrap pr-2 text-[8px] md:text-[12px] lg:text-[18px]"></div><svg stroke="currentColor" fill="none" stroke-width="2" viewBox="0 0 24 24" stroke-linecap="round" stroke-linejoin="round" class="flex-shrink-0 ml-auto" height="20" width="20" xmlns="http://www.w3.org/2000/svg"><polyline points="6 9 12 15 18 9"></polyline></svg></div><label class="absolute left-3 px-1 transition-all top-1/2 -translate-y-1/2 text-[100%] z-0 text-gray-900">Select Commodity</label></div>

LABEL: State/UT
Matches: 1
Tag: label
ID: 
Class: absolute left-3 px-1 transition-all -top-2 text-[6px] md:text-[12px] px-2 bg-gray-100 scale-100 z-0 text-gray-900
<div c

In [135]:
clickable_elements = driver.find_elements(
    By.CSS_SELECTOR,
    "button, [role='button'], input, [tabindex='0']"
)

for i, element in enumerate(clickable_elements):

    try:

        text = element.text.strip()
        placeholder = element.get_attribute(
            "placeholder"
        )

        aria_haspopup = element.get_attribute(
            "aria-haspopup"
        )

        aria_expanded = element.get_attribute(
            "aria-expanded"
        )

        element_id = element.get_attribute("id")

        element_class = element.get_attribute(
            "class"
        )

        if (
            text
            or placeholder
            or aria_haspopup
        ):

            print(
                "\nINDEX:", i,
                "\nTAG:", element.tag_name,
                "\nTEXT:", repr(text),
                "\nPLACEHOLDER:", placeholder,
                "\nID:", element_id,
                "\nARIA-HASPOPUP:", aria_haspopup,
                "\nARIA-EXPANDED:", aria_expanded,
                "\nCLASS:", element_class[:150]
                if element_class else None
            )

    except:
        pass


INDEX: 1 
TAG: input 
TEXT: '' 
PLACEHOLDER: Search pages... 
ID: searchPages 
ARIA-HASPOPUP: None 
ARIA-EXPANDED: false 
CLASS: border border-gray-300 rounded px-3 py-1 w-auto mt-1 focus:outline-none focus:ring-2 focus:ring-blue-400

INDEX: 2 
TAG: button 
TEXT: 'Sitemap' 
PLACEHOLDER: None 
ID:  
ARIA-HASPOPUP: None 
ARIA-EXPANDED: None 
CLASS: text-gray-700 hover:text-gray-900 px-3 py-1 rounded hover:bg-gray-200

INDEX: 3 
TAG: div 
TEXT: 'Skip to Main Content' 
PLACEHOLDER: None 
ID:  
ARIA-HASPOPUP: None 
ARIA-EXPANDED: None 
CLASS: text-gray-700 hover:text-gray-900 px-3 py-1 rounded hover:bg-gray-200 cursor-pointer text-right

INDEX: 4 
TAG: button 
TEXT: 'Feedback' 
PLACEHOLDER: None 
ID:  
ARIA-HASPOPUP: None 
ARIA-EXPANDED: None 
CLASS: text-gray-700 hover:text-gray-900 px-3 py-1 rounded hover:bg-gray-200

INDEX: 5 
TAG: button 
TEXT: 'A+' 
PLACEHOLDER: None 
ID:  
ARIA-HASPOPUP: None 
ARIA-EXPANDED: None 
CLASS: px-1 text-gray-700 hover:bg-gray-200 rounded

INDEX: 6 
TAG: bu

In [136]:
# Helper function to create a dropdown

def open_dropdown(label_text):

    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    driver.execute_script(
        "arguments[0].click();",
        trigger
    )

    time.sleep(1)

In [137]:
# Helper function to choose an option

def choose_option(option_text):

    def find_visible_option(driver):

        elements = driver.find_elements(
            By.XPATH,
            f"//*[normalize-space()='{option_text}']"
        )

        for element in elements:

            if element.is_displayed():
                return element

        return False

    option = WebDriverWait(
        driver,
        20
    ).until(
        find_visible_option
    )

    driver.execute_script(
        "arguments[0].click();",
        option
    )

    time.sleep(1)

In [138]:
# Select Wheat
open_dropdown(
    "Select Commodity"
)


In [139]:
#Choosing the option
choose_option(
    "Wheat"
)

In [140]:
state_label = driver.find_element(
    By.XPATH,
    "//label[normalize-space()='State/UT']"
)

state_trigger = state_label.find_element(
    By.XPATH,
    "./preceding-sibling::div[1]"
)

print(
    state_trigger.get_attribute("class")
)

print(
    "tabindex:",
    state_trigger.get_attribute("tabindex")
)

peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer  border-gray-300
tabindex: 0


In [141]:
# Selecting UP

open_dropdown(
    "State/UT"
)

In [142]:
choose_option(
    "Uttar Pradesh"
)

In [143]:
open_dropdown(
    "Year"
)

In [144]:
choose_option(
    "2021"
)

In [145]:
open_dropdown(
    "Month"
)

In [146]:
choose_option(
    "January"
)

In [147]:
# Click Submit automatically

submit_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//*[normalize-space()='Submit']"
        )
    )
)

driver.execute_script(
    "arguments[0].click();",
    submit_button
)

time.sleep(5)

In [148]:
# Checking what happened

print(
    "URL:",
    driver.current_url
)

body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(
    body_text[:5000]
)

URL: https://agmarknet.gov.in/marketallwholesalemonthanalysis
Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Wheat
Select Commodity
Uttar Pradesh
State/UT
2021
Year
January
Month
Submit
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspectio

In [149]:
submit_button = WebDriverWait(
    driver,
    20
).until(
    lambda d: next(
        (
            e for e in d.find_elements(
                By.XPATH,
                "//*[normalize-space()='Submit']"
            )
            if e.is_displayed()
        ),
        False
    )
)

driver.execute_script(
    "arguments[0].click();",
    submit_button
)

time.sleep(5)

In [150]:
print("URL:", driver.current_url)

body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_text[:7000])

URL: https://agmarknet.gov.in/marketallwholesalemonthanalysis
Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Wheat
Select Commodity
Uttar Pradesh
State/UT
2021
Year
January
Month
Submit
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspectio

In [151]:
keywords = [
    "market",
    "minimum",
    "maximum",
    "modal",
    "price",
    "no data",
    "no record",
    "download",
    "excel",
    "csv"
]

for word in keywords:
    print(
        word,
        "→",
        word.lower() in body_text.lower()
    )

market → True
minimum → False
maximum → False
modal → False
price → True
no data → False
no record → False
download → True
excel → False
csv → False


In [152]:
print(
    "Open browser tabs:",
    len(driver.window_handles)
)

Open browser tabs: 1


In [153]:
possible_results = driver.find_elements(
    By.XPATH,
    "//*[contains(translate(., "
    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
    "'abcdefghijklmnopqrstuvwxyz'), "
    "'modal price')]"
)

print(
    "Elements containing 'Modal Price':",
    len(possible_results)
)

Elements containing 'Modal Price': 0


In [154]:
no_data_elements = driver.find_elements(
    By.XPATH,
    "//*[contains(translate(., "
    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
    "'abcdefghijklmnopqrstuvwxyz'), "
    "'no data')]"
)

print(
    "No-data messages:",
    len(no_data_elements)
)

No-data messages: 0


In [155]:
tables = driver.find_elements(
    By.TAG_NAME,
    "table"
)

print("Tables found:", len(tables))

for i, table in enumerate(tables):
    if table.is_displayed():
        rows = table.find_elements(
            By.TAG_NAME,
            "tr"
        )

        print(
            "Table",
            i,
            "→ Rows:",
            len(rows)
        )

Tables found: 0


In [156]:
download_keywords = [
    "download",
    "export",
    "csv",
    "excel"
]

elements = driver.find_elements(
    By.CSS_SELECTOR,
    "a, button"
)

for i, element in enumerate(elements):

    try:
        text = element.text.strip()

        if (
            text
            and any(
                word in text.lower()
                for word in download_keywords
            )
        ):
            print(
                i,
                "→",
                repr(text),
                "| Tag:",
                element.tag_name
            )
    except:
        pass

In [157]:
submit_candidates = driver.find_elements(
    By.XPATH,
    "//*[normalize-space()='Submit']"
)

print("Submit candidates:", len(submit_candidates))

for i, element in enumerate(submit_candidates):
    print(
        "\nINDEX:", i,
        "\nTAG:", element.tag_name,
        "\nVISIBLE:", element.is_displayed(),
        "\nENABLED:", element.is_enabled(),
        "\nDISABLED ATTR:", element.get_attribute("disabled"),
        "\nHTML:",
        element.get_attribute("outerHTML")[:1500]
    )

Submit candidates: 4

INDEX: 0 
TAG: div 
VISIBLE: True 
ENABLED: True 
DISABLED ATTR: None 
HTML: <div class="mt-6 flex justify-center"><div class="relative"><button type="button" class="
        cursor-pointer 
        w-48 py-2 rounded-md transition bg-darkerBlue text-white cursor-pointer
      ">Submit</button></div></div>

INDEX: 1 
TAG: div 
VISIBLE: True 
ENABLED: True 
DISABLED ATTR: None 
HTML: <div class="relative"><button type="button" class="
        cursor-pointer 
        w-48 py-2 rounded-md transition bg-darkerBlue text-white cursor-pointer
      ">Submit</button></div>

INDEX: 2 
TAG: button 
VISIBLE: True 
ENABLED: True 
DISABLED ATTR: None 
HTML: <button type="button" class="
        cursor-pointer 
        w-48 py-2 rounded-md transition bg-darkerBlue text-white cursor-pointer
      ">Submit</button>

INDEX: 3 
TAG: button 
VISIBLE: False 
ENABLED: True 
DISABLED ATTR: None 
HTML: <button class="submit-feedback">Submit</button>


In [158]:
submit_button = next(
    element
    for element in submit_candidates
    if element.is_displayed()
    and element.is_enabled()
)

driver.execute_script(
    "arguments[0].scrollIntoView({block: 'center'});",
    submit_button
)

time.sleep(1)

submit_button.click()

In [159]:
body_after = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_after[:10000])

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Wheat
Select Commodity
Uttar Pradesh
State/UT
2021
Year
January
Month
Submit
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspection, Department of Agriculture & Farmers Welfare, Ministry of Ag

In [160]:
for word in [
    "average",
    "wholesale",
    "rs",
    "quintal",
    "market name",
    "apmc",
    "loading",
    "error",
    "record"
]:
    print(
        word,
        "→",
        word.lower() in body_after.lower()
    )

average → False
wholesale → True
rs → True
quintal → False
market name → False
apmc → False
loading → False
error → False
record → False


In [161]:
print(
    "Canvas elements:",
    len(driver.find_elements(By.TAG_NAME, "canvas"))
)

print(
    "SVG elements:",
    len(driver.find_elements(By.TAG_NAME, "svg"))
)

Canvas elements: 0
SVG elements: 39


In [162]:
print(
    "Current URL:",
    driver.current_url
)

Current URL: https://agmarknet.gov.in/market-all-wholesale-month-arrival-table


In [163]:
## Actual result lines

lines = [
    line.strip()
    for line in body_after.splitlines()
    if line.strip()
]

for i, line in enumerate(lines):
    if any(
        word in line.lower()
        for word in [
            "average",
            "price",
            "apmc",
            "quintal",
            "minimum",
            "maximum",
            "modal"
        ]
    ):
        print(i, "→", line)

24 → Market-wise Wholesale Prices Monthly Analysis (All Districts)


In [164]:
for i, line in enumerate(lines):

    if "average" in line.lower():

        start = max(0, i - 5)
        end = min(len(lines), i + 15)

        print("\n--- RESULT SECTION ---")

        for j in range(start, end):
            print(j, "→", lines[j])

In [165]:
clickable = driver.find_elements(
    By.CSS_SELECTOR,
    "a, button, [role='button']"
)

for i, element in enumerate(clickable):

    try:
        text = element.text.strip()
        title = element.get_attribute("title") or ""
        aria = element.get_attribute("aria-label") or ""
        href = element.get_attribute("href") or ""

        combined = (
            text + " " +
            title + " " +
            aria + " " +
            href
        ).lower()

        if any(
            word in combined
            for word in [
                "download",
                "csv",
                "excel",
                "xlsx",
                "export"
            ]
        ):
            print(
                "\nINDEX:", i,
                "\nTEXT:", repr(text),
                "\nTITLE:", repr(title),
                "\nARIA:", repr(aria),
                "\nHREF:", repr(href),
                "\nTAG:", element.tag_name
            )

    except:
        pass


INDEX: 10 
TEXT: '' 
TITLE: 'Download Agmarknet 2.0 App from Google Play Store' 
ARIA: '' 
HREF: 'https://play.google.com/store/apps/details?id=in.gov.agmarknet' 
TAG: a

INDEX: 11 
TEXT: '' 
TITLE: 'Download Agmarknet 2.0 App from App Store' 
ARIA: '' 
HREF: 'https://apps.apple.com/in/app/agmarknet-2-0/id6751224873' 
TAG: a

INDEX: 74 
TEXT: 'Export' 
TITLE: '' 
ARIA: '' 
HREF: '' 
TAG: button


In [166]:
for selector in [
    "[role='table']",
    "[role='grid']",
    "[role='row']",
    "[role='cell']"
]:
    elements = driver.find_elements(
        By.CSS_SELECTOR,
        selector
    )

    print(
        selector,
        "→",
        len(elements)
    )

[role='table'] → 0
[role='grid'] → 0
[role='row'] → 0
[role='cell'] → 0


In [167]:
# Inspecting the export button

export_button = next(
    element
    for element in driver.find_elements(By.TAG_NAME, "button")
    if element.is_displayed()
    and element.text.strip() == "Export"
)

driver.execute_script(
    "arguments[0].scrollIntoView({block: 'center'});",
    export_button
)

export_button.click()

time.sleep(1)

In [168]:
visible_text = []

for element in driver.find_elements(By.XPATH, "//*"):
    try:
        if element.is_displayed():

            text = element.text.strip()

            if (
                text
                and len(text) < 80
                and any(
                    word in text.lower()
                    for word in [
                        "csv",
                        "excel",
                        "xlsx",
                        "pdf",
                        "export",
                        "download"
                    ]
                )
            ):
                visible_text.append(
                    (
                        element.tag_name,
                        text
                    )
                )
    except:
        pass

for item in visible_text:
    print(item)

('div', 'Main Menu\nExport\nExport as Excel\nExport as PDF\nExport as CSV')
('div', 'Export\nExport as Excel\nExport as PDF\nExport as CSV')
('div', 'Export')
('button', 'Export')
('div', 'Export as Excel\nExport as PDF\nExport as CSV')
('div', 'Export as Excel\nExport as PDF\nExport as CSV')
('button', 'Export as Excel')
('button', 'Export as PDF')
('button', 'Export as CSV')
('h3', 'Download the Agmarknet 2.0 App')


In [169]:
# Click export as csv automatically

csv_button = WebDriverWait(
    driver,
    20
).until(
    lambda d: next(
        (
            e for e in d.find_elements(
                By.TAG_NAME,
                "button"
            )
            if e.is_displayed()
            and e.text.strip() == "Export as CSV"
        ),
        False
    )
)

csv_button.click()

In [170]:
time.sleep(5)

In [171]:
from pathlib import Path

downloads_folder = (
    Path.home()
    / "Downloads"
)

csv_files = sorted(
    downloads_folder.glob("*.csv"),
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

print(csv_files[:5])

[WindowsPath('C:/Users/richa/Downloads/Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_29-09-2026_11-55-10_AM.csv'), WindowsPath('C:/Users/richa/Downloads/Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_27-09-2026_02-52-33_PM.csv'), WindowsPath('C:/Users/richa/Downloads/Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_26-09-2026_03-35-02_PM.csv'), WindowsPath('C:/Users/richa/Downloads/Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_24-09-2026_04-47-56_PM.csv'), WindowsPath('C:/Users/richa/Downloads/Date_wise_Prices_for_Specified_Commodity_23-09-2026_08-47-01_PM.csv')]


In [172]:
latest_csv = csv_files[0]

print(
    "Latest downloaded file:",
    latest_csv
)

Latest downloaded file: C:\Users\richa\Downloads\Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_29-09-2026_11-55-10_AM.csv


In [173]:
# Read it directly

import pandas as pd


monthly_test = pd.read_csv(
    latest_csv
)

monthly_test.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,"Market-wise Wholesale Prices Monthly Analysis (All Districts) for Wheat in Uttar Pradesh - January, 2021",Unnamed: 4,Unnamed: 5
0,Market,"Prices January, 2021 (Rs./Quintal)","Prices December, 2020 (Rs./Quintal)","Prices January, 2020 (Rs./Quintal)",Change (Over Previous Month),Change (Over Previous Year)
1,Achalda APMC,1680.49,1607.41,2093.03,4.5%,-19.7%
2,Achnera APMC,1704.88,1643.92,2127.09,3.7%,-19.9%
3,Agra APMC,1664.65,1651.21,2154.09,0.8%,-22.7%
4,Ait APMC,1681.49,1556.42,2115.13,8.0%,-20.5%


In [174]:
print(
    monthly_test.shape
)

print(
    monthly_test.columns.tolist()
)

(193, 6)
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Market-wise Wholesale Prices Monthly Analysis (All Districts) for Wheat in Uttar Pradesh - January, 2021', 'Unnamed: 4', 'Unnamed: 5']


In [175]:
monthly_test.columns.tolist()

['Unnamed: 0',
 'Unnamed: 1',
 'Unnamed: 2',
 'Market-wise Wholesale Prices Monthly Analysis (All Districts) for Wheat in Uttar Pradesh - January, 2021',
 'Unnamed: 4',
 'Unnamed: 5']

In [176]:
# Reading the csv correctly

monthly_test = pd.read_csv(
    latest_csv,
    header=1
)

monthly_test.head()

,Market,"Prices January, 2021 (Rs./Quintal)","Prices December, 2020 (Rs./Quintal)","Prices January, 2020 (Rs./Quintal)",Change (Over Previous Month),Change (Over Previous Year)
0,Achalda APMC,1680.49,1607.41,2093.03,4.5%,-19.7%
1,Achnera APMC,1704.88,1643.92,2127.09,3.7%,-19.9%
2,Agra APMC,1664.65,1651.21,2154.09,0.8%,-22.7%
3,Ait APMC,1681.49,1556.42,2115.13,8.0%,-20.5%
4,Ajuha APMC,1673.18,1669.4,2010.83,0.2%,-16.8%


In [177]:
monthly_test.columns.tolist()

['Market',
 'Prices January, 2021 (Rs./Quintal)',
 'Prices December, 2020 (Rs./Quintal)',
 'Prices January, 2020 (Rs./Quintal)',
 'Change (Over Previous Month)',
 'Change (Over Previous Year)']

In [178]:
# Keeping the price we want

current_price_column = monthly_test.columns[1]

print(
    "Current price column:",
    current_price_column
)

Current price column: Prices January, 2021 (Rs./Quintal)


In [179]:
january_clean = monthly_test[
    [
        "Market",
        current_price_column
    ]
].copy()

In [180]:
january_clean = january_clean.rename(
    columns={
        current_price_column: "Monthly_Price"
    }
)

In [181]:
january_clean.head()

,Market,Monthly_Price
0,Achalda APMC,1680.49
1,Achnera APMC,1704.88
2,Agra APMC,1664.65
3,Ait APMC,1681.49
4,Ajuha APMC,1673.18


In [182]:
# Converting missing price correctly

january_clean["Monthly_Price"] = pd.to_numeric(
    january_clean["Monthly_Price"],
    errors="coerce"
)


In [183]:
# Removing the average row

january_clean = january_clean[
    january_clean["Market"]
    .astype(str)
    .str.strip()
    .str.lower()
    != "average"
].copy()

In [184]:
january_clean = january_clean.dropna(
    subset=["Market"]
)

In [185]:
# Adding infomation that come from report files

january_clean["State"] = "Uttar Pradesh"
january_clean["Crop"] = "Wheat"
january_clean["Calendar_Year"] = 2021
january_clean["Month"] = 1

In [186]:
january_clean["Month_Date"] = pd.to_datetime(
    "2021-01-01"
)

In [187]:
january_clean = january_clean[
    [
        "State",
        "Crop",
        "Calendar_Year",
        "Month",
        "Month_Date",
        "Market",
        "Monthly_Price"
    ]
]

In [188]:
january_clean.head(10)

,State,Crop,Calendar_Year,Month,Month_Date,Market,Monthly_Price
0,Uttar Pradesh,Wheat,2021,1,2021-01-01,Achalda APMC,1680.49
1,Uttar Pradesh,Wheat,2021,1,2021-01-01,Achnera APMC,1704.88
2,Uttar Pradesh,Wheat,2021,1,2021-01-01,Agra APMC,1664.65
3,Uttar Pradesh,Wheat,2021,1,2021-01-01,Ait APMC,1681.49
4,Uttar Pradesh,Wheat,2021,1,2021-01-01,Ajuha APMC,1673.18
5,Uttar Pradesh,Wheat,2021,1,2021-01-01,Akbarpur APMC,NaN
6,Uttar Pradesh,Wheat,2021,1,2021-01-01,Aliganj APMC,1641.71
7,Uttar Pradesh,Wheat,2021,1,2021-01-01,Aligarh APMC,1753.40
8,Uttar Pradesh,Wheat,2021,1,2021-01-01,Amroha APMC,1716.55
9,Uttar Pradesh,Wheat,2021,1,2021-01-01,Anandnagar APMC,1722.50


In [189]:
# Turning this into reusuable function

def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    df = pd.read_csv(
        file_path,
        header=1
    )

    current_price_column = df.columns[1]

    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    df = df.rename(
        columns={
            current_price_column:
            "Monthly_Price"
        }
    )

    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["Market"]
    )

    df = df[
        df["Market"]
        .astype(str)
        .str.strip()
        .str.lower()
        != "average"
    ].copy()

    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    return df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]

In [190]:
# TESTING ON FILE JUST DOWNLOADED 

test_clean = clean_monthly_price_csv(
    latest_csv,
    state="Uttar Pradesh",
    crop="Wheat",
    year=2021,
    month=1
)

test_clean.head()

,State,Crop,Calendar_Year,Month,Month_Date,Market,Monthly_Price
0,Uttar Pradesh,Wheat,2021,1,2021-01-01,Achalda APMC,1680.49
1,Uttar Pradesh,Wheat,2021,1,2021-01-01,Achnera APMC,1704.88
2,Uttar Pradesh,Wheat,2021,1,2021-01-01,Agra APMC,1664.65
3,Uttar Pradesh,Wheat,2021,1,2021-01-01,Ait APMC,1681.49
4,Uttar Pradesh,Wheat,2021,1,2021-01-01,Ajuha APMC,1673.18


In [191]:
print(
    test_clean.shape
)

print(
    test_clean["Monthly_Price"].isna().sum()
)

print(
    test_clean.tail()
)

(191, 7)
32
             State   Crop  Calendar_Year  Month Month_Date  \
186  Uttar Pradesh  Wheat           2021      1 2021-01-01   
188  Uttar Pradesh  Wheat           2021      1 2021-01-01   
189  Uttar Pradesh  Wheat           2021      1 2021-01-01   
190  Uttar Pradesh  Wheat           2021      1 2021-01-01   
191  Uttar Pradesh  Wheat           2021      1 2021-01-01   

                                                Market  Monthly_Price  
186                                      Yusufpur APMC            NaN  
188                                              Note:            NaN  
189  Weighted Average Price = ∑(arrival quantity * ...            NaN  
190  Change (Over Previous Month) = [Price - Price(...            NaN  
191  Change (Over Previous Year) = [Price - Price(O...            NaN  


In [192]:
# Going back to filter page 

driver.back()

time.sleep(3)

print(driver.current_url)

https://agmarknet.gov.in/marketallwholesalemonthanalysis


In [193]:
body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(body_text[:1500])

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Back
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Select Commodity
State/UT
Year
Month
Submit
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Policy
Disclaimer
Sitemap
GOVERNMENT SITES
Directorate of Marketing & Inspection (DMI)
Ministry of Agriculture and Farmers Welfare
CONTACT US
mrin-dmi[at]nic[dot]in
Directorate of Marketing & Inspection, Department of Agriculture & Farmers Welfare, Ministry of Agriculture and Farmers Welfare, A-

In [194]:
# Chaning only january to feburary

def select_dropdown_option(label_text, option_text):

    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    driver.execute_script(
        "arguments[0].click();",
        trigger
    )

    time.sleep(1)

    candidates = driver.find_elements(
        By.XPATH,
        f"//*[contains(normalize-space(.), '{option_text}')]"
    )

    for element in candidates:

        try:
            if (
                element.is_displayed()
                and element.text.strip() == option_text
                and element.tag_name != "label"
            ):

                driver.execute_script(
                    "arguments[0].click();",
                    element
                )

                time.sleep(1)

                return

        except:
            continue

    raise Exception(
        f"Could not select {option_text}"
    )

In [195]:
select_dropdown_option(
    "Month",
    "February"
)

Exception: Could not select February

In [196]:
labels = [
    "Select Commodity",
    "State/UT",
    "Year",
    "Month"
]

for label_text in labels:

    label = driver.find_element(
        By.XPATH,
        f"//label[normalize-space()='{label_text}']"
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    print(
        "\n",
        label_text,
        "\nValue:", repr(trigger.text),
        "\nTabindex:", trigger.get_attribute("tabindex"),
        "\nClass:", trigger.get_attribute("class")
    )


 Select Commodity 
Value: '' 
Tabindex: 0 
Class: peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer  border-gray-300

 State/UT 
Value: '' 
Tabindex: -1 
Class: peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer bg-gray-100 cursor-not-allowed border-gray-300

 Year 
Value: '' 
Tabindex: -1 
Class: peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer bg-gray-100 cursor-not-allowed border-gray-300

 Month 
Value: '' 
Tabindex: -1 
Class: peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer bg-gray-100 cursor-not-allowed border-gray-300


In [197]:
month_label = driver.find_element(
    By.XPATH,
    "//label[normalize-space()='Month']"
)

month_trigger = month_label.find_element(
    By.XPATH,
    "./preceding-sibling::div[1]"
)

driver.execute_script(
    "arguments[0].click();",
    month_trigger
)

time.sleep(1)

feb_elements = driver.find_elements(
    By.XPATH,
    "//*[contains(translate(normalize-space(.), "
    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
    "'abcdefghijklmnopqrstuvwxyz'), "
    "'feb')]"
)

print(
    "Elements containing Feb:",
    len(feb_elements)
)

for i, element in enumerate(feb_elements[:20]):

    try:
        print(
            i,
            "|",
            element.tag_name,
            "|",
            repr(element.text),
            "| visible:",
            element.is_displayed()
        )
    except:
        pass

Elements containing Feb: 0


In [198]:
open_dropdown("Select Commodity")

time.sleep(1)

wheat_matches = driver.find_elements(
    By.XPATH,
    "//*[contains(translate(normalize-space(.), "
    "'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
    "'abcdefghijklmnopqrstuvwxyz'), "
    "'wheat')]"
)

print("Wheat matches:", len(wheat_matches))

for i, element in enumerate(wheat_matches[:20]):
    try:
        print(
            "\nINDEX:", i,
            "\nTAG:", element.tag_name,
            "\nTEXT:", repr(element.text),
            "\nVISIBLE:", element.is_displayed(),
            "\nCLASS:", element.get_attribute("class"),
            "\nHTML:",
            element.get_attribute("outerHTML")[:1000]
        )
    except:
        pass

Wheat matches: 16

INDEX: 0 
TAG: html 
TEXT: "Search pages\nType at least 2 characters to search pages.\nSitemap\nSkip to Main Content\nFeedback\nA+\nA\nA-\nScreen Reader\nA\nA\nEnglish\nGovernment of India\nDepartment of Agriculture & Farmers Welfare\nDirectorate of Marketing & Inspection\nHome\nReports\nMarket Profile\nMarket Atlas\nHelpdesk\nResearch Studies\nWhat's New\nOfficial Login\nBack\nMarket-wise Wholesale Prices Monthly Analysis (All Districts)\nSelect Commodity\nAbsinthe\nAdulsa\nAjwain Husk\nAjwan\nAkarkara\nAlasande Gram\nAlmond(Badam)\nAloe Vera\nAloe Vera Leaf\nAlsandikai\nAmaltas\nAmaranthus\nAmarbel\nAmbada Seed\nAmbady/Mesta/Patson\nAmbat Chuka\nAmbrette Seed/Muskmallow\nAmla(Nelli Kai)\nAmranthas Red\nAntawala\nAnthorium\nApple\nApricot(Jardalu/Khumani)\nArecanut(Betelnut/Supari)\nAretha\nArrowroot\nAsalia\nAsgand\nAshgourd\nAshoka\nAshwagandha\nAsparagus\nAstera\nAtis\nAvare Dal\nAvocado\nBOP\nBaboolphali\nBaby Corn\nBael\nBajji chilli\nBajra(Pearl Millet/Cumbu)\

In [199]:
def select_exact_dropdown_option(label_text, option_text):

    # 1. Find the label
    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    # 2. Find the clickable dropdown box
    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    # 3. Make sure it is enabled
    if trigger.get_attribute("tabindex") == "-1":
        raise Exception(
            f"{label_text} dropdown is currently disabled"
        )

    # 4. Click the dropdown
    driver.execute_script(
        "arguments[0].click();",
        trigger
    )

    time.sleep(1)

    # 5. Find exact visible option text
    option_span = WebDriverWait(
        driver,
        20
    ).until(
        lambda d: next(
            (
                e for e in d.find_elements(
                    By.XPATH,
                    f"//span[normalize-space()='{option_text}']"
                )
                if e.is_displayed()
            ),
            False
        )
    )

    # 6. Click the option's parent div
    option_div = option_span.find_element(
        By.XPATH,
        ".."
    )

    driver.execute_script(
        "arguments[0].click();",
        option_div
    )

    time.sleep(1)

In [200]:
select_exact_dropdown_option(
    "Select Commodity",
    "Wheat"
)

TimeoutException: Message: 


In [201]:
select_exact_dropdown_option(
    "State/UT",
    "Uttar Pradesh"
)

Exception: State/UT dropdown is currently disabled

In [202]:
select_exact_dropdown_option(
    "Year",
    "2021"
)

Exception: Year dropdown is currently disabled

In [203]:
select_exact_dropdown_option(
    "Month",
    "February"
)

Exception: Month dropdown is currently disabled

In [204]:
for label_text in [
    "Select Commodity",
    "State/UT",
    "Year",
    "Month"
]:

    label = driver.find_element(
        By.XPATH,
        f"//label[normalize-space()='{label_text}']"
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    print(
        label_text,
        "→",
        repr(trigger.text)
    )

Select Commodity → ''
State/UT → ''
Year → ''
Month → ''


In [205]:
driver.refresh()
time.sleep(3)

In [206]:
def select_exact_dropdown_option(label_text, option_text):

    # Find the dropdown label
    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    # Find the clickable dropdown box
    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    # Check if dropdown is enabled
    if trigger.get_attribute("tabindex") == "-1":
        raise Exception(
            f"{label_text} is disabled. "
            "Select the previous field first."
        )

    # Exact option div
    option_xpath = (
        "//div[contains(@class,'cursor-pointer')]"
        f"[.//span[normalize-space()='{option_text}']]"
    )

    # See if option is already visible
    visible_options = [
        e for e in driver.find_elements(
            By.XPATH,
            option_xpath
        )
        if e.is_displayed()
    ]

    # If not visible, open dropdown
    if not visible_options:

        trigger.click()
        time.sleep(1)

        visible_options = WebDriverWait(
            driver,
            20
        ).until(
            lambda d: [
                e for e in d.find_elements(
                    By.XPATH,
                    option_xpath
                )
                if e.is_displayed()
            ]
        )

    # Click exact option
    visible_options[0].click()

    time.sleep(1)

    print(
        f"{label_text} → {option_text} selected"
    )

In [207]:
select_exact_dropdown_option(
    "Select Commodity",
    "Wheat"
)

ElementClickInterceptedException: Message: element click intercepted: Element <div class="peer w-full h-10 px-3 py-2 border rounded-md flex items-center justify-between cursor-pointer  border-gray-300" tabindex="0">...</div> is not clickable at point (402, 302). Other element would receive the click: <label class="absolute left-3 px-1 transition-all top-1/2 -translate-y-1/2 text-[100%] z-0 text-gray-900">...</label>
  (Session info: chrome=154.0.8037.57); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff65f3abe05+5a55]
	chromedriver!(No symbol) [0x7ff65f2d3230]
	chromedriver!(No symbol) [0x7ff65f0f6e4d]
	chromedriver!(No symbol) [0x7ff65f15a423]
	chromedriver!(No symbol) [0x7ff65f158033]
	chromedriver!(No symbol) [0x7ff65f155377]
	chromedriver!(No symbol) [0x7ff65f154250]
	chromedriver!(No symbol) [0x7ff65f146228]
	chromedriver!(No sym

In [ ]:
select_exact_dropdown_option(
    "State/UT",
    "Uttar Pradesh"
)

In [208]:
select_exact_dropdown_option(
    "Year",
    "2021"
)

Exception: Year is disabled. Select the previous field first.

In [209]:
select_exact_dropdown_option(
    "Month",
    "February"
)

Exception: Month is disabled. Select the previous field first.

In [210]:
for label_text in [
    "Select Commodity",
    "State/UT",
    "Year",
    "Month"
]:

    label = driver.find_element(
        By.XPATH,
        f"//label[normalize-space()='{label_text}']"
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    print(
        label_text,
        "→",
        repr(trigger.text)
    )

Select Commodity → ''
State/UT → ''
Year → ''
Month → ''


In [211]:
submit_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='Submit']"
        )
    )
)

submit_button.click()

time.sleep(5)

print(driver.current_url)

TimeoutException: Message: 


In [214]:
from pathlib import Path

downloads_folder = Path.home() / "Downloads"

before_download = set(
    downloads_folder.glob("*.csv")
)

print(
    "Existing CSV files:",
    len(before_download)
)

Existing CSV files: 6


In [215]:
export_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='Export']"
        )
    )
)

export_button.click()

time.sleep(1)

TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff65f3abe05+5a55]
	chromedriver!(No symbol) [0x7ff65f2d3230]
	chromedriver!(No symbol) [0x7ff65f0f6e4d]
	chromedriver!(No symbol) [0x7ff65f15266d]
	chromedriver!(No symbol) [0x7ff65f15292c]
	chromedriver!(No symbol) [0x7ff65f1a59e7]
	chromedriver!(No symbol) [0x7ff65f1a267a]
	chromedriver!(No symbol) [0x7ff65f144412]
	chromedriver!(No symbol) [0x7ff65f145223]
	chromedriver!GetHandleVerifier [0x7ff65f7bb1d1+414e21]
	chromedriver!GetHandleVerifier [0x7ff65f7e51fb+43ee4b]
	chromedriver!GetHandleVerifier [0x7ff65f7d96ce+43331e]
	chromedriver!GetHandleVerifier [0x7ff65f4b015e+109dae]
	chromedriver!(No symbol) [0x7ff65f2e0e85]
	chromedriver!(No symbol) [0x7ff65f2dbe14]
	chromedriver!(No symbol) [0x7ff65f2dbfa4]
	chromedriver!(No symbol) [0x7ff65f2c6b0c]
	KERNEL32!BaseThreadInitThunk [0x7ffb8455cdf7+17]
	ntdll!RtlUserThreadStart [0x7ffb8647ee4c+2c]


In [216]:
csv_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='Export as CSV']"
        )
    )
)

csv_button.click()

TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff65f3abe05+5a55]
	chromedriver!(No symbol) [0x7ff65f2d3230]
	chromedriver!(No symbol) [0x7ff65f0f6e4d]
	chromedriver!(No symbol) [0x7ff65f15266d]
	chromedriver!(No symbol) [0x7ff65f15292c]
	chromedriver!(No symbol) [0x7ff65f1a59e7]
	chromedriver!(No symbol) [0x7ff65f1a267a]
	chromedriver!(No symbol) [0x7ff65f144412]
	chromedriver!(No symbol) [0x7ff65f145223]
	chromedriver!GetHandleVerifier [0x7ff65f7bb1d1+414e21]
	chromedriver!GetHandleVerifier [0x7ff65f7e51fb+43ee4b]
	chromedriver!GetHandleVerifier [0x7ff65f7d96ce+43331e]
	chromedriver!GetHandleVerifier [0x7ff65f4b015e+109dae]
	chromedriver!(No symbol) [0x7ff65f2e0e85]
	chromedriver!(No symbol) [0x7ff65f2dbe14]
	chromedriver!(No symbol) [0x7ff65f2dbfa4]
	chromedriver!(No symbol) [0x7ff65f2c6b0c]
	KERNEL32!BaseThreadInitThunk [0x7ffb8455cdf7+17]
	ntdll!RtlUserThreadStart [0x7ffb8647ee4c+2c]


In [217]:
new_csv = None

for _ in range(30):

    time.sleep(1)

    current_files = set(
        downloads_folder.glob("*.csv")
    )

    new_files = (
        current_files
        - before_download
    )

    if new_files:

        new_csv = max(
            new_files,
            key=lambda x: x.stat().st_mtime
        )

        break

In [218]:
print(
    "Downloaded file:",
    new_csv
)

Downloaded file: None


In [219]:
price_raw_folder = Path(
    "../../data/raw/prices"
)

price_raw_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [220]:
feb_file = (
    price_raw_folder
    / "uttar_pradesh_wheat_2021_02.csv"
)

In [221]:
import shutil

shutil.move(
    str(new_csv),
    str(feb_file)
)

print(
    "Saved as:",
    feb_file
)

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [222]:
february_clean = clean_monthly_price_csv(
    feb_file,
    state="Uttar Pradesh",
    crop="Wheat",
    year=2021,
    month=2
)

In [223]:
february_clean.head()

,State,Crop,Calendar_Year,Month,Month_Date,Market,Monthly_Price
0,Uttar Pradesh,Wheat,2021,2,2021-02-01,Achalda APMC,1664.60
1,Uttar Pradesh,Wheat,2021,2,2021-02-01,Achnera APMC,1725.42
2,Uttar Pradesh,Wheat,2021,2,2021-02-01,Agra APMC,1726.68
3,Uttar Pradesh,Wheat,2021,2,2021-02-01,Ait APMC,1622.71
4,Uttar Pradesh,Wheat,2021,2,2021-02-01,Ajuha APMC,1693.80


In [224]:
print(
    "Rows:",
    february_clean.shape[0]
)

print(
    "Missing prices:",
    february_clean[
        "Monthly_Price"
    ].isna().sum()
)

Rows: 197
Missing prices: 34


In [225]:
from pathlib import Path
import shutil
import time
import calendar

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [226]:
def collect_one_month(
    driver,
    state,
    crop,
    year,
    month
):

    # -----------------------------
    # 1. Open fresh filter page
    # -----------------------------
    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )

    driver.get(filter_url)

    time.sleep(3)

    month_name = calendar.month_name[month]

    print(
        f"\nCollecting: "
        f"{crop} | {state} | "
        f"{month_name} {year}"
    )

    # -----------------------------
    # 2. Select filters
    # -----------------------------
    select_exact_dropdown_option(
        "Select Commodity",
        crop
    )

    select_exact_dropdown_option(
        "State/UT",
        state
    )

    select_exact_dropdown_option(
        "Year",
        str(year)
    )

    select_exact_dropdown_option(
        "Month",
        month_name
    )

    # -----------------------------
    # 3. Submit
    # -----------------------------
    submit_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Submit']"
            )
        )
    )

    submit_button.click()

    # Wait until result page appears
    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//button[normalize-space()='Export']"
            )
        )
    )

    print("Result page loaded ✅")

    # -----------------------------
    # 4. Record existing downloads
    # -----------------------------
    downloads_folder = (
        Path.home()
        / "Downloads"
    )

    before_download = set(
        downloads_folder.glob("*.csv")
    )

    # -----------------------------
    # 5. Open Export menu
    # -----------------------------
    export_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Export']"
            )
        )
    )

    export_button.click()

    time.sleep(1)

    # -----------------------------
    # 6. Export CSV
    # -----------------------------
    csv_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Export as CSV']"
            )
        )
    )

    csv_button.click()

    # -----------------------------
    # 7. Wait for new CSV
    # -----------------------------
    new_csv = None

    for _ in range(60):

        time.sleep(1)

        current_files = set(
            downloads_folder.glob("*.csv")
        )

        new_files = (
            current_files
            - before_download
        )

        if new_files:

            new_csv = max(
                new_files,
                key=lambda x:
                x.stat().st_mtime
            )

            break

    if new_csv is None:
        raise Exception(
            "CSV download was not detected."
        )

    print(
        "Downloaded:",
        new_csv.name
    )

    # -----------------------------
    # 8. Create clean filename
    # -----------------------------
    price_raw_folder = Path(
        "../../data/raw/prices"
    )

    price_raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    target_file = (
        price_raw_folder
        / (
            f"{state_slug}_"
            f"{crop_slug}_"
            f"{year}_"
            f"{month:02d}.csv"
        )
    )

    # If testing the same month again
    if target_file.exists():
        target_file.unlink()

    shutil.move(
        str(new_csv),
        str(target_file)
    )

    print(
        "Saved:",
        target_file.name
    )

    # -----------------------------
    # 9. Clean the downloaded CSV
    # -----------------------------
    clean_df = clean_monthly_price_csv(
        target_file,
        state=state,
        crop=crop,
        year=year,
        month=month
    )

    print(
        "Clean rows:",
        len(clean_df)
    )

    print(
        "Missing prices:",
        clean_df[
            "Monthly_Price"
        ].isna().sum()
    )

    return clean_df

In [227]:
march_2021 = collect_one_month(
    driver=driver,
    state="Uttar Pradesh",
    crop="Wheat",
    year=2021,
    month=3
)


Collecting: Wheat | Uttar Pradesh | March 2021
Select Commodity → Wheat selected
State/UT → Uttar Pradesh selected
Year → 2021 selected
Month → March selected
Result page loaded ✅
Downloaded: Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_29-09-2026_12-04-11_PM.csv
Saved: uttar_pradesh_wheat_2021_03.csv
Clean rows: 190
Missing prices: 31


In [228]:
march_2021.head()


,State,Crop,Calendar_Year,Month,Month_Date,Market,Monthly_Price
0,Uttar Pradesh,Wheat,2021,3,2021-03-01,Achalda APMC,1643.79
1,Uttar Pradesh,Wheat,2021,3,2021-03-01,Achnera APMC,1638.05
2,Uttar Pradesh,Wheat,2021,3,2021-03-01,Agra APMC,1626.68
3,Uttar Pradesh,Wheat,2021,3,2021-03-01,Ait APMC,1630.18
4,Uttar Pradesh,Wheat,2021,3,2021-03-01,Ajuha APMC,1717.41


In [229]:
print(
    march_2021[
        "Month_Date"
    ].unique()
)

<DatetimeArray>
['2021-03-01 00:00:00']
Length: 1, dtype: datetime64[us]


In [230]:
# Building the month list

import pandas as pd

month_range = pd.date_range(
    start="2021-01-01",
    end="2025-06-01",
    freq="MS"
)

print("Total months:", len(month_range))

for date in month_range[:5]:
    print(date.year, date.month)

Total months: 54
2021 1
2021 2
2021 3
2021 4
2021 5


In [231]:
# Making a safe bulk collector 

def collect_price_history(
    driver,
    state,
    crop,
    start_date,
    end_date
):

    dates = pd.date_range(
        start=start_date,
        end=end_date,
        freq="MS"
    )

    successful = []
    failed = []
    skipped = []

    price_raw_folder = Path(
        "../../data/raw/prices"
    )

    price_raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    for number, date in enumerate(
        dates,
        start=1
    ):

        year = date.year
        month = date.month

        target_file = (
            price_raw_folder
            / (
                f"{state_slug}_"
                f"{crop_slug}_"
                f"{year}_"
                f"{month:02d}.csv"
            )
        )

        print(
            "\n"
            + "=" * 60
        )

        print(
            f"{number}/{len(dates)} → "
            f"{date.strftime('%B %Y')}"
        )

        # ----------------------------
        # Skip file if already exists
        # ----------------------------
        if target_file.exists():

            print(
                "Already downloaded → SKIPPED ✅"
            )

            skipped.append(
                {
                    "Year": year,
                    "Month": month
                }
            )

            continue

        # ----------------------------
        # Try collection
        # ----------------------------
        try:

            clean_df = collect_one_month(
                driver=driver,
                state=state,
                crop=crop,
                year=year,
                month=month
            )

            successful.append(
                {
                    "Year": year,
                    "Month": month,
                    "Rows": len(clean_df)
                }
            )

            print(
                "Month completed ✅"
            )

        except Exception as error:

            print(
                "FAILED ❌"
            )

            print(
                "Reason:",
                error
            )

            failed.append(
                {
                    "Year": year,
                    "Month": month,
                    "Error": str(error)
                }
            )

        # Be polite to government server
        time.sleep(3)

    return (
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped)
    )

In [232]:
# Start UP wheat

up_wheat_success, up_wheat_failed, up_wheat_skipped = (
    collect_price_history(
        driver=driver,
        state="Uttar Pradesh",
        crop="Wheat",
        start_date="2021-01-01",
        end_date="2025-06-01"
    )
)


1/54 → January 2021
Already downloaded → SKIPPED ✅

2/54 → February 2021
Already downloaded → SKIPPED ✅

3/54 → March 2021
Already downloaded → SKIPPED ✅

4/54 → April 2021
Already downloaded → SKIPPED ✅

5/54 → May 2021
Already downloaded → SKIPPED ✅

6/54 → June 2021
Already downloaded → SKIPPED ✅

7/54 → July 2021
Already downloaded → SKIPPED ✅

8/54 → August 2021
Already downloaded → SKIPPED ✅

9/54 → September 2021
Already downloaded → SKIPPED ✅

10/54 → October 2021
Already downloaded → SKIPPED ✅

11/54 → November 2021
Already downloaded → SKIPPED ✅

12/54 → December 2021
Already downloaded → SKIPPED ✅

13/54 → January 2022
Already downloaded → SKIPPED ✅

14/54 → February 2022
Already downloaded → SKIPPED ✅

15/54 → March 2022
Already downloaded → SKIPPED ✅

16/54 → April 2022
Already downloaded → SKIPPED ✅

17/54 → May 2022
Already downloaded → SKIPPED ✅

18/54 → June 2022
Already downloaded → SKIPPED ✅

19/54 → July 2022
Already downloaded → SKIPPED ✅

20/54 → August 2022
Alre

In [233]:
print(
    "Successful:",
    len(up_wheat_success)
)

print(
    "Failed:",
    len(up_wheat_failed)
)

print(
    "Skipped:",
    len(up_wheat_skipped)
)

Successful: 0
Failed: 0
Skipped: 54


In [234]:
up_wheat_failed

""


In [235]:
up_wheat_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "uttar_pradesh_wheat_*.csv"
    )
)

print(
    "UP Wheat files:",
    len(up_wheat_files)
)

UP Wheat files: 54


In [236]:
# Creating a fresh chrome session

from selenium import webdriver

try:
    driver.quit()
except:
    pass

driver = webdriver.Chrome()

print("New Selenium session created ✅")

New Selenium session created ✅


In [237]:
def select_exact_dropdown_option(
    driver,
    label_text,
    option_text
):

    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    if trigger.get_attribute("tabindex") == "-1":
        raise Exception(
            f"{label_text} is disabled"
        )

    option_xpath = (
        "//div[contains(@class,'cursor-pointer')]"
        f"[.//span[normalize-space()='{option_text}']]"
    )

    visible_options = [
        e for e in driver.find_elements(
            By.XPATH,
            option_xpath
        )
        if e.is_displayed()
    ]

    if not visible_options:

        trigger.click()
        time.sleep(1)

        visible_options = WebDriverWait(
            driver,
            20
        ).until(
            lambda d: [
                e for e in d.find_elements(
                    By.XPATH,
                    option_xpath
                )
                if e.is_displayed()
            ]
        )

    visible_options[0].click()

    time.sleep(1)

In [238]:
# ============================================================
# AGMARKNET MONTHLY PRICE COLLECTION - COMPLETE UPDATED CODE
# ============================================================

import pandas as pd
import time
import calendar
import shutil

from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ============================================================
# 1. DROPDOWN SELECTION FUNCTION
# ============================================================

def select_exact_dropdown_option(
    driver,
    label_text,
    option_text
):

    # Find the label such as:
    # Select Commodity / State/UT / Year / Month
    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    # The clickable dropdown is just before the label
    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    # If tabindex = -1, AGMARKNET has disabled the dropdown
    if trigger.get_attribute("tabindex") == "-1":

        raise Exception(
            f"{label_text} dropdown is currently disabled."
        )

    # Exact option we want.
    #
    # Example:
    # <div class="... cursor-pointer">
    #     <span>Wheat</span>
    # </div>

    option_xpath = (
        "//div[contains(@class,'cursor-pointer')]"
        f"[.//span[normalize-space()='{option_text}']]"
    )

    # First check whether dropdown is already open
    visible_options = [
        element
        for element in driver.find_elements(
            By.XPATH,
            option_xpath
        )
        if element.is_displayed()
    ]

    # If option is not visible, open the dropdown
    if not visible_options:

        driver.execute_script(
            "arguments[0].click();",
            trigger
        )

        time.sleep(1)

        # Wait until desired option appears
        visible_options = WebDriverWait(
            driver,
            20
        ).until(
            lambda d: [
                element
                for element in d.find_elements(
                    By.XPATH,
                    option_xpath
                )
                if element.is_displayed()
            ]
        )

    # Click exact option
    driver.execute_script(
        "arguments[0].click();",
        visible_options[0]
    )

    time.sleep(1)

    print(
        f"{label_text} → {option_text} selected"
    )


# ============================================================
# 2. CLEAN ONE DOWNLOADED MONTHLY CSV
# ============================================================

def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    # First row is the report title.
    # Second row contains the real column names.
    df = pd.read_csv(
        file_path,
        header=1
    )

    # Example:
    # Market
    # Prices January, 2021 (Rs./Quintal)
    # Prices December, 2020 ...
    #
    # We only need the CURRENT month's price.
    current_price_column = df.columns[1]

    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    # Give current price a common column name
    df = df.rename(
        columns={
            current_price_column:
            "Monthly_Price"
        }
    )

    # Convert price to numeric.
    # "-" becomes NaN.
    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    # Remove completely empty Market rows
    df = df.dropna(
        subset=["Market"]
    )

    # Remove AGMARKNET's final Average row
    df = df[
        df["Market"]
        .astype(str)
        .str.strip()
        .str.lower()
        != "average"
    ].copy()

    # Add information from our filters
    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    # Final column order
    df = df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]

    return df


# ============================================================
# 3. COLLECT ONE MONTH
# ============================================================

def collect_one_month(
    driver,
    state,
    crop,
    year,
    month
):

    # AGMARKNET filter page
    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )

    # Open a completely fresh filter page
    driver.get(filter_url)

    time.sleep(3)

    # Convert:
    # 1 -> January
    # 2 -> February
    # 3 -> March
    month_name = calendar.month_name[month]

    print("\n" + "=" * 60)

    print(
        f"Collecting: "
        f"{crop} | "
        f"{state} | "
        f"{month_name} {year}"
    )

    # ========================================================
    # SELECT ALL FOUR FILTERS FROM THE BEGINNING
    # ========================================================

    # 1. Commodity
    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    # 2. State
    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    # 3. Year
    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )

    # 4. Month
    select_exact_dropdown_option(
        driver,
        "Month",
        month_name
    )

    print("All filters selected ✅")

    # ========================================================
    # CLICK SUBMIT
    # ========================================================

    submit_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Submit']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        submit_button
    )

    # Wait for result page's Export button
    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//button[normalize-space()='Export']"
            )
        )
    )

    print("Result page loaded ✅")

    # ========================================================
    # REMEMBER EXISTING DOWNLOAD FILES
    # ========================================================

    downloads_folder = (
        Path.home()
        / "Downloads"
    )

    before_download = set(
        downloads_folder.glob("*.csv")
    )

    # ========================================================
    # OPEN EXPORT MENU
    # ========================================================

    export_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Export']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        export_button
    )

    time.sleep(1)

    # ========================================================
    # CLICK EXPORT AS CSV
    # ========================================================

    csv_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Export as CSV']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        csv_button
    )

    print("CSV export clicked ✅")

    # ========================================================
    # WAIT FOR NEW CSV TO APPEAR
    # ========================================================

    new_csv = None

    for _ in range(60):

        time.sleep(1)

        current_files = set(
            downloads_folder.glob("*.csv")
        )

        new_files = (
            current_files
            - before_download
        )

        if new_files:

            new_csv = max(
                new_files,
                key=lambda file:
                file.stat().st_mtime
            )

            break

    # If after 60 seconds nothing appeared
    if new_csv is None:

        raise Exception(
            "CSV download was not detected."
        )

    print(
        "Downloaded:",
        new_csv.name
    )

    # ========================================================
    # CREATE PROJECT PRICE FOLDER
    # ========================================================

    price_raw_folder = Path(
        "../../data/raw/prices"
    )

    price_raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    # Convert:
    # Uttar Pradesh -> uttar_pradesh
    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    # Wheat -> wheat
    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    # Example:
    # uttar_pradesh_wheat_2022_07.csv

    target_file = (
        price_raw_folder
        / (
            f"{state_slug}_"
            f"{crop_slug}_"
            f"{year}_"
            f"{month:02d}.csv"
        )
    )

    # If testing same month again,
    # remove existing copy first
    if target_file.exists():

        target_file.unlink()

    # Move downloaded file into project
    shutil.move(
        str(new_csv),
        str(target_file)
    )

    print(
        "Saved as:",
        target_file.name
    )

    # ========================================================
    # CLEAN CSV
    # ========================================================

    clean_df = clean_monthly_price_csv(
        file_path=target_file,
        state=state,
        crop=crop,
        year=year,
        month=month
    )

    print(
        "Clean rows:",
        len(clean_df)
    )

    print(
        "Missing prices:",
        clean_df[
            "Monthly_Price"
        ].isna().sum()
    )

    print(
        f"{month_name} {year} completed ✅"
    )

    return clean_df

In [239]:
try:
    driver.quit()
except:
    pass

driver = webdriver.Chrome()

print("New Chrome session created ✅")

New Chrome session created ✅


In [240]:
test_july_2022 = collect_one_month(
    driver=driver,
    state="Uttar Pradesh",
    crop="Wheat",
    year=2022,
    month=7
)


Collecting: Wheat | Uttar Pradesh | July 2022
Select Commodity → Wheat selected
State/UT → Uttar Pradesh selected
Year → 2022 selected
Month → July selected
All filters selected ✅
Result page loaded ✅
CSV export clicked ✅
Downloaded: Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_29-09-2026_12-06-03_PM.csv
Saved as: uttar_pradesh_wheat_2022_07.csv
Clean rows: 202
Missing prices: 15
July 2022 completed ✅


In [241]:
test_july_2022.head()

,State,Crop,Calendar_Year,Month,Month_Date,Market,Monthly_Price
0,Uttar Pradesh,Wheat,2022,7,2022-07-01,Achalda APMC,2074.47
1,Uttar Pradesh,Wheat,2022,7,2022-07-01,Achnera APMC,NaN
2,Uttar Pradesh,Wheat,2022,7,2022-07-01,Agra APMC,2110.42
3,Uttar Pradesh,Wheat,2022,7,2022-07-01,Ahirora APMC,2065.91
4,Uttar Pradesh,Wheat,2022,7,2022-07-01,Ait APMC,2144.16


In [242]:
up_wheat_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "uttar_pradesh_wheat_*.csv"
    )
)

print(
    "UP Wheat files currently:",
    len(up_wheat_files)
)

UP Wheat files currently: 54


In [243]:
from selenium.common.exceptions import (
    InvalidSessionIdException,
    WebDriverException,
    TimeoutException
)

In [244]:
def create_new_driver():

    print("\n🔄 Starting a new Chrome session...")

    new_driver = webdriver.Chrome()

    time.sleep(2)

    print("New Chrome session ready ✅")

    return new_driver

In [245]:
def collect_price_history_resilient(
    driver,
    state,
    crop,
    start_date,
    end_date
):

    dates = pd.date_range(
        start=start_date,
        end=end_date,
        freq="MS"
    )

    successful = []
    failed = []
    skipped = []

    price_raw_folder = Path(
        "../../data/raw/prices"
    )

    price_raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    # ========================================================
    # LOOP THROUGH EVERY MONTH
    # ========================================================

    for number, date in enumerate(
        dates,
        start=1
    ):

        year = date.year
        month = date.month

        month_name = (
            calendar.month_name[month]
        )

        target_file = (
            price_raw_folder
            / (
                f"{state_slug}_"
                f"{crop_slug}_"
                f"{year}_"
                f"{month:02d}.csv"
            )
        )

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"{number}/{len(dates)} → "
            f"{crop} | {state} | "
            f"{month_name} {year}"
        )

        # ====================================================
        # IF FILE ALREADY EXISTS → SKIP
        # ====================================================

        if target_file.exists():

            print(
                "Already downloaded → SKIPPED ✅"
            )

            skipped.append(
                {
                    "Year": year,
                    "Month": month,
                    "Month_Name": month_name
                }
            )

            continue

        # ====================================================
        # ALLOW TWO ATTEMPTS
        # ====================================================

        month_completed = False

        for attempt in range(1, 3):

            print(
                f"Attempt {attempt}/2"
            )

            try:

                clean_df = collect_one_month(
                    driver=driver,
                    state=state,
                    crop=crop,
                    year=year,
                    month=month
                )

                successful.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name,
                        "Rows": len(clean_df)
                    }
                )

                print(
                    f"{month_name} {year} "
                    "completed successfully ✅"
                )

                month_completed = True

                break

            # =================================================
            # SELENIUM SESSION DIED
            # =================================================

            except InvalidSessionIdException:

                print(
                    "⚠️ Selenium session died."
                )

                try:
                    driver.quit()
                except:
                    pass

                driver = create_new_driver()

                print(
                    "Will retry this month..."
                )

            # =================================================
            # OTHER WEBDRIVER PROBLEM
            # =================================================

            except WebDriverException as error:

                error_text = str(
                    error
                ).lower()

                print(
                    "⚠️ Browser/WebDriver problem:"
                )

                print(
                    str(error)[:300]
                )

                # These usually mean Chrome is dead
                if any(
                    text in error_text
                    for text in [
                        "invalid session id",
                        "disconnected",
                        "chrome not reachable",
                        "no such window"
                    ]
                ):

                    try:
                        driver.quit()
                    except:
                        pass

                    driver = create_new_driver()

                    print(
                        "Browser restarted ✅"
                    )

                else:

                    print(
                        "Retrying after short wait..."
                    )

                    time.sleep(5)

            # =================================================
            # NORMAL PYTHON / PAGE ERROR
            # =================================================

            except Exception as error:

                print(
                    "⚠️ Attempt failed:"
                )

                print(
                    str(error)[:300]
                )

                time.sleep(5)

        # ====================================================
        # BOTH ATTEMPTS FAILED
        # ====================================================

        if not month_completed:

            print(
                f"{month_name} {year} → FAILED ❌"
            )

            failed.append(
                {
                    "Year": year,
                    "Month": month,
                    "Month_Name": month_name
                }
            )

        # Small delay so we don't hit AGMARKNET too quickly
        time.sleep(3)

    # ========================================================
    # CREATE SUMMARY TABLES
    # ========================================================

    successful_df = pd.DataFrame(
        successful
    )

    failed_df = pd.DataFrame(
        failed
    )

    skipped_df = pd.DataFrame(
        skipped
    )

    return (
        driver,
        successful_df,
        failed_df,
        skipped_df
    )

In [246]:
driver, up_wheat_success, up_wheat_failed, up_wheat_skipped = (
    collect_price_history_resilient(
        driver=driver,
        state="Uttar Pradesh",
        crop="Wheat",
        start_date="2021-01-01",
        end_date="2025-06-01"
    )
)


1/54 → Wheat | Uttar Pradesh | January 2021
Already downloaded → SKIPPED ✅

2/54 → Wheat | Uttar Pradesh | February 2021
Already downloaded → SKIPPED ✅

3/54 → Wheat | Uttar Pradesh | March 2021
Already downloaded → SKIPPED ✅

4/54 → Wheat | Uttar Pradesh | April 2021
Already downloaded → SKIPPED ✅

5/54 → Wheat | Uttar Pradesh | May 2021
Already downloaded → SKIPPED ✅

6/54 → Wheat | Uttar Pradesh | June 2021
Already downloaded → SKIPPED ✅

7/54 → Wheat | Uttar Pradesh | July 2021
Already downloaded → SKIPPED ✅

8/54 → Wheat | Uttar Pradesh | August 2021
Already downloaded → SKIPPED ✅

9/54 → Wheat | Uttar Pradesh | September 2021
Already downloaded → SKIPPED ✅

10/54 → Wheat | Uttar Pradesh | October 2021
Already downloaded → SKIPPED ✅

11/54 → Wheat | Uttar Pradesh | November 2021
Already downloaded → SKIPPED ✅

12/54 → Wheat | Uttar Pradesh | December 2021
Already downloaded → SKIPPED ✅

13/54 → Wheat | Uttar Pradesh | January 2022
Already downloaded → SKIPPED ✅

14/54 → Wheat | U

In [247]:
print(
    "Successful this run:",
    len(up_wheat_success)
)

print(
    "Failed:",
    len(up_wheat_failed)
)

print(
    "Already existed:",
    len(up_wheat_skipped)
)

Successful this run: 0
Failed: 0
Already existed: 54


In [248]:
up_wheat_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "uttar_pradesh_wheat_*.csv"
    )
)

print(
    "Total UP Wheat files:",
    len(up_wheat_files)
)

Total UP Wheat files: 54


In [249]:
# Confirming all 54 files exists

from pathlib import Path

price_raw_folder = Path(
    "../../data/raw/prices"
)

up_wheat_files = sorted(
    price_raw_folder.glob(
        "uttar_pradesh_wheat_*.csv"
    )
)

print(
    "Total UP Wheat files:",
    len(up_wheat_files)
)

Total UP Wheat files: 54


In [250]:
# Combining all 54Csv's

import re

all_up_wheat = []

for file_path in up_wheat_files:

    # Example filename:
    # uttar_pradesh_wheat_2021_01.csv

    match = re.search(
        r"uttar_pradesh_wheat_(\d{4})_(\d{2})",
        file_path.stem
    )

    if not match:
        print(
            "Could not understand filename:",
            file_path.name
        )
        continue

    year = int(
        match.group(1)
    )

    month = int(
        match.group(2)
    )

    clean_df = clean_monthly_price_csv(
        file_path=file_path,
        state="Uttar Pradesh",
        crop="Wheat",
        year=year,
        month=month
    )

    all_up_wheat.append(
        clean_df
    )

In [251]:
up_wheat_monthly = pd.concat(
    all_up_wheat,
    ignore_index=True
)

In [252]:
up_wheat_monthly.head()

,State,Crop,Calendar_Year,Month,Month_Date,Market,Monthly_Price
0,Uttar Pradesh,Wheat,2021,1,2021-01-01,Achalda APMC,1680.49
1,Uttar Pradesh,Wheat,2021,1,2021-01-01,Achnera APMC,1704.88
2,Uttar Pradesh,Wheat,2021,1,2021-01-01,Agra APMC,1664.65
3,Uttar Pradesh,Wheat,2021,1,2021-01-01,Ait APMC,1681.49
4,Uttar Pradesh,Wheat,2021,1,2021-01-01,Ajuha APMC,1673.18


In [253]:
# Validation

print(
    "Total rows:",
    len(up_wheat_monthly)
)

print(
    "First month:",
    up_wheat_monthly[
        "Month_Date"
    ].min()
)

print(
    "Last month:",
    up_wheat_monthly[
        "Month_Date"
    ].max()
)

print(
    "Unique months:",
    up_wheat_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Unique markets:",
    up_wheat_monthly[
        "Market"
    ].nunique()
)

Total rows: 10719
First month: 2021-01-01 00:00:00
Last month: 2025-06-01 00:00:00
Unique months: 54
Unique markets: 235


In [254]:
# Checking any month is missing

expected_months = pd.date_range(
    start="2021-01-01",
    end="2025-06-01",
    freq="MS"
)

actual_months = pd.DatetimeIndex(
    up_wheat_monthly[
        "Month_Date"
    ].unique()
)

missing_months = (
    expected_months
    .difference(actual_months)
)

print(
    "Missing months:",
    list(missing_months)
)

Missing months: []


In [255]:
# Checking for duplicate market-month rows

duplicate_count = (
    up_wheat_monthly
    .duplicated(
        subset=[
            "State",
            "Crop",
            "Month_Date",
            "Market"
        ]
    )
    .sum()
)

print(
    "Duplicate market-month rows:",
    duplicate_count
)

Duplicate market-month rows: 0


In [256]:
# Checking for missing prices

print(
    "Total price values:",
    len(up_wheat_monthly)
)

print(
    "Missing price values:",
    up_wheat_monthly[
        "Monthly_Price"
    ].isna().sum()
)

Total price values: 10719
Missing price values: 1177


In [257]:
missing_price_pct = (
    up_wheat_monthly[
        "Monthly_Price"
    ].isna().mean()
    * 100
)

print(
    f"Missing price percentage: "
    f"{missing_price_pct:.2f}%"
)

Missing price percentage: 10.98%


In [258]:
# Makret coverage by month

monthly_coverage = (
    up_wheat_monthly
    .groupby(
        "Month_Date"
    )
    .agg(
        Markets_Listed=(
            "Market",
            "nunique"
        ),

        Markets_With_Price=(
            "Monthly_Price",
            "count"
        )
    )
    .reset_index()
)

monthly_coverage.head(12)

,Month_Date,Markets_Listed,Markets_With_Price
0,2021-01-01,191,159
1,2021-02-01,197,163
2,2021-03-01,190,159
3,2021-04-01,201,172
4,2021-05-01,208,170
5,2021-06-01,205,171
6,2021-07-01,194,165
7,2021-08-01,188,167
8,2021-09-01,184,161
9,2021-10-01,185,165


In [259]:
up_wheat_monthly[
    "Monthly_Price"
].describe()

count    9542.000000
mean     2254.245733
std       319.711794
min       188.000000
25%      2008.812500
50%      2300.000000
75%      2469.180000
max      4010.000000
Name: Monthly_Price, dtype: float64

In [260]:
# Checking for one known market

meerut_prices = (
    up_wheat_monthly[
        up_wheat_monthly[
            "Market"
        ].str.strip().str.lower()
        == "meerut apmc"
    ]
    [
        [
            "Month_Date",
            "Monthly_Price"
        ]
    ]
    .sort_values(
        "Month_Date"
    )
)

meerut_prices.head(15)

,Month_Date,Monthly_Price
126,2021-01-01,1743.44
322,2021-02-01,1731.11
515,2021-03-01,1727.43
707,2021-04-01,1974.94
911,2021-05-01,1975.00
1118,2021-06-01,1967.51
1318,2021-07-01,1889.89
1508,2021-08-01,1890.46
1695,2021-09-01,1931.44
1879,2021-10-01,1967.75


In [261]:
meerut_prices[
    meerut_prices[
        "Month_Date"
    ] == "2021-01-01"
]

,Month_Date,Monthly_Price
126,2021-01-01,1743.44


In [262]:
# Saving the first monthly price dataset

processed_folder = Path(
    "../../data/processed"
)

processed_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [263]:
up_wheat_monthly.to_csv(
    processed_folder
    / "up_wheat_monthly_prices_2021_2025.csv",
    index=False
)

In [264]:
print(
    "Saved:",
    processed_folder
    / "up_wheat_monthly_prices_2021_2025.csv"
)

Saved: ..\..\data\processed\up_wheat_monthly_prices_2021_2025.csv


In [265]:
# Creating same market key

import re

def normalize_market_name(name):

    name = str(name).strip().lower()

    # Remove APMC
    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    # Remove extra spaces
    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()

In [266]:
up_wheat_monthly["Market_Key"] = (
    up_wheat_monthly["Market"]
    .apply(normalize_market_name)
)

In [267]:
up_wheat_monthly[
    [
        "Market",
        "Market_Key"
    ]
].drop_duplicates().head(10)

,Market,Market_Key
0,Achalda APMC,achalda
1,Achnera APMC,achnera
2,Agra APMC,agra
3,Ait APMC,ait
4,Ajuha APMC,ajuha
5,Akbarpur APMC,akbarpur
6,Aliganj APMC,aliganj
7,Aligarh APMC,aligarh
8,Amroha APMC,amroha
9,Anandnagar APMC,anandnagar


In [268]:
# Loading previous Market-District Mapping

mapping_file = Path(
    "../../data/processed/up_market_district_mapping.csv"
)

print(
    "Mapping file exists:",
    mapping_file.exists()
)

Mapping file exists: True


In [269]:
print(
    "price_with_district exists:",
    "price_with_district" in globals()
)

print(
    "market_district_master exists:",
    "market_district_master" in globals()
)

price_with_district exists: True
market_district_master exists: False


In [270]:
market_district_master = (
    price_with_district[
        [
            "Market_Key",
            "District"
        ]
    ]
    .dropna()
    .drop_duplicates()
)

In [271]:
print(
    "Markets in mapping:",
    market_district_master[
        "Market_Key"
    ].nunique()
)

market_district_master.head()

Markets in mapping: 159


,Market_Key,District
0,achalda,Auraiya
23,achnera,Agra
48,agra,Agra
72,ait,Jalaun
100,ajuha,Prayagraj


In [272]:
mapping_conflicts = (
    market_district_master
    .groupby("Market_Key")["District"]
    .nunique()
)

mapping_conflicts = mapping_conflicts[
    mapping_conflicts > 1
]

print(
    "Conflicting markets:",
    len(mapping_conflicts)
)

Conflicting markets: 0


In [273]:
mapping_file = Path(
    "../../data/processed/up_market_district_mapping.csv"
)

market_district_master.to_csv(
    mapping_file,
    index=False
)

print(
    "Saved:",
    mapping_file
)

Saved: ..\..\data\processed\up_market_district_mapping.csv


In [274]:
print(
    "Mapping file exists:",
    mapping_file.exists()
)

Mapping file exists: True


In [275]:
market_district_master = pd.read_csv(
    mapping_file
)

market_district_master.head()

,Market_Key,District
0,achalda,Auraiya
1,achnera,Agra
2,agra,Agra
3,ait,Jalaun
4,ajuha,Prayagraj


In [276]:
print(
    market_district_master.columns.tolist()
)

print(
    "Mapped markets:",
    market_district_master[
        "Market_Key"
    ].nunique()
)

['Market_Key', 'District']
Mapped markets: 159


In [277]:
mapping_conflicts = (
    market_district_master
    .groupby("Market_Key")["District"]
    .nunique()
)

mapping_conflicts = mapping_conflicts[
    mapping_conflicts > 1
]

In [278]:
print(
    "Markets mapped to multiple districts:",
    len(mapping_conflicts)
)

Markets mapped to multiple districts: 0


In [279]:
# Merging market district with montly prices 

market_district_master = (
    market_district_master[
        [
            "Market_Key",
            "District"
        ]
    ]
    .dropna()
    .drop_duplicates()
)

In [280]:
up_wheat_with_district = (
    up_wheat_monthly.merge(
        market_district_master,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [281]:
print(
    "Rows before merge:",
    len(up_wheat_monthly)
)

print(
    "Rows after merge:",
    len(up_wheat_with_district)
)

Rows before merge: 10719
Rows after merge: 10719


In [282]:
print(
    "Rows missing District:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

Rows missing District: 2441


In [283]:
unmapped_markets = (
    up_wheat_with_district.loc[
        up_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

In [284]:
print(
    "Unique unmapped markets:",
    len(unmapped_markets)
)

unmapped_markets

Unique unmapped markets: 76


,Market,Market_Key
1389,Ahirora APMC,ahirora
5,Akbarpur APMC,akbarpur
201,Anoop Shahar APMC,anoop shahar
7660,Anwala APMC,anwala
205,Awagarh APMC,awagarh
...,...,...
766,Tilhar APMC,tilhar
5459,Vilthararoad APMC,vilthararoad
184,Viswan APMC,viswan
188,Weighted Average Price = ∑(arrival quantity * ...,weighted average price = ∑(arrival quantity * ...


In [285]:
# Making a clean summary of 76 unmapped districts

unmapped_summary = (
    up_wheat_with_district[
        up_wheat_with_district[
            "District"
        ].isna()
    ]
    .groupby(
        [
            "Market",
            "Market_Key"
        ],
        as_index=False
    )
    .agg(
        Rows=("Month_Date", "size"),
        Months_Available=("Month_Date", "nunique"),
        First_Month=("Month_Date", "min"),
        Last_Month=("Month_Date", "max")
    )
    .sort_values(
        [
            "Months_Available",
            "Market"
        ],
        ascending=[
            False,
            True
        ]
    )
)

print(
    "Unique unmapped markets:",
    len(unmapped_summary)
)

unmapped_summary

Unique unmapped markets: 76


,Market,Market_Key,Rows,Months_Available,First_Month,Last_Month
1,Akbarpur APMC,akbarpur,54,54,2021-01-01,2025-06-01
10,Bharwari APMC,bharwari,54,54,2021-01-01,2025-06-01
15,Change (Over Previous Month) = [Price - Price(...,change (over previous month) = [price - price(...,54,54,2021-01-01,2025-06-01
16,Change (Over Previous Year) = [Price - Price(O...,change (over previous year) = [price - price(o...,54,54,2021-01-01,2025-06-01
26,Gopiganj APMC,gopiganj,54,54,2021-01-01,2025-06-01
...,...,...,...,...,...,...
51,Payagpur APMC,payagpur,6,6,2021-01-01,2021-12-01
12,Billsadda APMC,billsadda,3,3,2021-04-01,2021-06-01
14,CHHUTMALPUR APMC,chhutmalpur,3,3,2022-05-01,2023-05-01
27,Gulavati APMC,gulavati,3,3,2021-04-01,2021-06-01


In [286]:
from pathlib import Path

possible_reference_files = []

for folder in [
    Path("../../data/raw"),
    Path("../../data/processed")
]:

    if folder.exists():

        possible_reference_files.extend(
            list(
                folder.rglob("*.csv")
            )
        )

for file in possible_reference_files:

    name = file.name.lower()

    if any(
        word in name
        for word in [
            "agmark",
            "market",
            "mandi"
        ]
    ):

        print(file)

..\..\data\raw\prices\agmarknet_current_market_reference.csv
..\..\data\processed\punjab_market_district_mapping.csv
..\..\data\processed\punjab_market_geocode_second_pass.csv
..\..\data\processed\punjab_official_market_committee_mapping.csv
..\..\data\processed\punjab_unmapped_market_geocode_candidates.csv
..\..\data\processed\punjab_wheat_unresolved_66_markets.csv
..\..\data\processed\unmatched_up_markets.csv
..\..\data\processed\unmatched_up_markets_with_candidates.csv
..\..\data\processed\up_market_district_mapping.csv
..\..\data\processed\up_unmapped_market_geocode_candidates.csv


In [287]:
reference_file = Path(
    "../../data/raw/prices/agmarknet_current_market_reference.csv"
)

market_reference = pd.read_csv(
    reference_file
)

print(
    market_reference.columns.tolist()
)

market_reference.head()

['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade', 'Arrival_Date', 'Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price']


,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price
0,Andhra Pradesh,Prakasam,Santhamaguluru APMC,Paddy(Common),B P T,FAQ,21/09/2026,2500.0,2580.0,2550.0
1,Andhra Pradesh,Prakasam,Maddipadu APMC,Paddy(Common),B P T,FAQ,21/09/2026,2800.0,2800.0,2800.0
2,Keralam,Kozhikode(Calicut),Mukkom Market,Pineapple,Other,Large,21/09/2026,8000.0,9000.0,8500.0
3,Keralam,Kozhikode(Calicut),Mukkom Market,Coconut,Other,FAQ,21/09/2026,5800.0,6100.0,6000.0
4,Keralam,Kozhikode(Calicut),Mukkom Market,Pumpkin,Pumpkin,FAQ,21/09/2026,1800.0,2000.0,1900.0


In [288]:
#Build a fresh mapping from that official reference
market_reference["Market_Key"] = (
    market_reference["Market"]
    .apply(normalize_market_name)
)

In [289]:
up_reference = (
    market_reference[
        market_reference["State"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "uttar pradesh"
    ]
    [
        [
            "Market_Key",
            "Market",
            "District"
        ]
    ]
    .dropna(
        subset=[
            "Market_Key",
            "District"
        ]
    )
    .drop_duplicates()
)

In [290]:
print(
    "Unique UP markets in current reference:",
    up_reference["Market_Key"].nunique()
)

Unique UP markets in current reference: 150


In [291]:
reference_conflicts = (
    up_reference
    .groupby("Market_Key")["District"]
    .nunique()
)

reference_conflicts = (
    reference_conflicts[
        reference_conflicts > 1
    ]
)

print(
    "Conflicts in current reference:",
    len(reference_conflicts)
)

Conflicts in current reference: 0


In [292]:
#Automaticaally find makret refernece

from pathlib import Path
import pandas as pd

search_folders = [
    Path("../../data/raw"),
    Path("../../data/processed")
]

reference_candidates = []

for folder in search_folders:

    if not folder.exists():
        continue

    for file_path in folder.rglob("*.csv"):

        try:

            sample = pd.read_csv(
                file_path,
                nrows=5
            )

            normalized_columns = {
                str(col).strip().lower()
                for col in sample.columns
            }

            required = {
                "state",
                "district",
                "market"
            }

            if required.issubset(
                normalized_columns
            ):

                reference_candidates.append(
                    file_path
                )

        except:
            pass


print(
    "Reference candidates found:",
    len(reference_candidates)
)

for file in reference_candidates:
    print(file)

Reference candidates found: 5
..\..\data\raw\prices\agmarknet_current_market_reference.csv
..\..\data\processed\up_maize_monthly_prices_mapped_2021_2025.csv
..\..\data\processed\up_rice_monthly_prices_mapped_2021_2025.csv
..\..\data\processed\up_wheat_monthly_prices_mapped_2021_2025.csv
..\..\data\processed\wheat_up_2021_01_clean.csv


In [293]:
#Build mapping from every valid reference file

official_mapping_parts = []

for file_path in reference_candidates:

    try:

        df = pd.read_csv(
            file_path
        )

        # Standardize column names
        column_lookup = {
            str(col).strip().lower(): col
            for col in df.columns
        }

        state_col = column_lookup["state"]
        district_col = column_lookup["district"]
        market_col = column_lookup["market"]

        temp = df[
            [
                state_col,
                district_col,
                market_col
            ]
        ].copy()

        temp.columns = [
            "State",
            "District",
            "Market"
        ]

        # Only Uttar Pradesh
        temp = temp[
            temp["State"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "uttar pradesh"
        ].copy()

        temp["Market_Key"] = (
            temp["Market"]
            .apply(
                normalize_market_name
            )
        )

        official_mapping_parts.append(
            temp[
                [
                    "Market_Key",
                    "District"
                ]
            ]
        )

    except Exception as error:

        print(
            "Could not use:",
            file_path.name
        )

        print(error)

In [294]:
official_up_mapping = pd.concat(
    official_mapping_parts,
    ignore_index=True
)

In [295]:
official_up_mapping["District"] = (
    official_up_mapping["District"]
    .astype(str)
    .str.strip()
)

In [296]:
official_up_mapping = (
    official_up_mapping
    .dropna()
    .drop_duplicates()
)

In [297]:
print(
    "Official reference markets:",
    official_up_mapping[
        "Market_Key"
    ].nunique()
)

Official reference markets: 243


In [298]:
official_conflicts = (
    official_up_mapping
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

official_conflicts = (
    official_conflicts[
        official_conflicts > 1
    ]
)

print(
    "Official mapping conflicts:",
    len(official_conflicts)
)

Official mapping conflicts: 37


In [299]:
# Inspecting 35 conflicts

conflict_details = (
    official_up_mapping[
        official_up_mapping["Market_Key"].isin(
            official_conflicts.index
        )
    ]
    .groupby("Market_Key")
    .agg(
        Districts=(
            "District",
            lambda x: sorted(set(x))
        ),
        District_Count=(
            "District",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "District_Count",
        ascending=False
    )
)

conflict_details

,Market_Key,Districts,District_Count
21,kopaganj,"[, Mau(Maunathbhanjan), nan]",3
1,anandnagar,"[Maharajganj, Mahrajganj]",2
2,babrala,"[Badaun, Budaun]",2
3,badayoun,"[Badaun, Budaun]",2
0,ait,"[Jalaun, Jalaun (Orai)]",2
4,barabanki,"[Bara Banki, Barabanki]",2
5,baripaal,"[Jaunpur, Kanpur]",2
7,chhibramau,"[Kannauj, Kannuj]",2
6,bishalpur,"[Pilibhit, Pillibhit]",2
9,dadri,"[Gautam Buddha Nagar, Gautam Budh Nagar]",2


In [300]:
safe_market_keys = (
    official_up_mapping
    .groupby("Market_Key")["District"]
    .nunique()
)

safe_market_keys = (
    safe_market_keys[
        safe_market_keys == 1
    ]
    .index
)

In [301]:
safe_official_mapping = (
    official_up_mapping[
        official_up_mapping[
            "Market_Key"
        ].isin(
            safe_market_keys
        )
    ]
    [
        [
            "Market_Key",
            "District"
        ]
    ]
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

In [302]:
print(
    "Safe official mappings:",
    len(safe_official_mapping)
)

print(
    "Ambiguous official markets:",
    len(official_conflicts)
)

Safe official mappings: 206
Ambiguous official markets: 37


In [303]:
existing_keys = set(
    market_district_master[
        "Market_Key"
    ]
)

In [304]:
new_safe_mappings = (
    safe_official_mapping[
        ~safe_official_mapping[
            "Market_Key"
        ].isin(
            existing_keys
        )
    ]
    .copy()
)

In [305]:
print(
    "New safe mappings available:",
    len(new_safe_mappings)
)

New safe mappings available: 84


In [306]:
combined_market_mapping = pd.concat(
    [
        market_district_master[
            [
                "Market_Key",
                "District"
            ]
        ],
        new_safe_mappings[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)

In [307]:
combined_market_mapping = (
    combined_market_mapping
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

In [308]:
print(
    "Combined mapped markets:",
    combined_market_mapping[
        "Market_Key"
    ].nunique()
)

Combined mapped markets: 243


In [309]:
check_conflicts = (
    combined_market_mapping
    .groupby("Market_Key")["District"]
    .nunique()
)

print(
    "Conflicts after safe combination:",
    (
        check_conflicts > 1
    ).sum()
)

Conflicts after safe combination: 0


In [310]:
up_wheat_with_district = (
    up_wheat_monthly.merge(
        combined_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [311]:
print(
    "Total rows:",
    len(up_wheat_with_district)
)

print(
    "Rows missing District:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

Total rows: 10719
Rows missing District: 216


In [312]:
remaining_unmapped = (
    up_wheat_with_district.loc[
        up_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

In [313]:
print(
    "Remaining unique unmapped markets:",
    len(remaining_unmapped)
)

remaining_unmapped

Remaining unique unmapped markets: 4


,Market,Market_Key
189,Change (Over Previous Month) = [Price - Price(...,change (over previous month) = [price - price(...
190,Change (Over Previous Year) = [Price - Price(O...,change (over previous year) = [price - price(o...
187,Note:,note:
188,Weighted Average Price = ∑(arrival quantity * ...,weighted average price = ∑(arrival quantity * ...


In [314]:
print(
    remaining_unmapped[
        [
            "Market",
            "Market_Key"
        ]
    ].to_string(
        index=False
    )
)

                                                                                              Market                                                                                           Market_Key
Change (Over Previous Month) = [Price - Price(Over Previous Month)/Price(Over Previous Month)] * 100 change (over previous month) = [price - price(over previous month)/price(over previous month)] * 100
   Change (Over Previous Year) = [Price - Price(Over Previous Year)/Price(Over Previous Year)] * 100    change (over previous year) = [price - price(over previous year)/price(over previous year)] * 100
                                                                                               Note:                                                                                                note:
                      Weighted Average Price = ∑(arrival quantity * modal price) / ∑arrival quantity                       weighted average price = ∑(arrival quantity * modal price) / ∑arrival

In [315]:
ambiguous_keys = set(
    official_conflicts.index
)

remaining_unmapped["Mapping_Status"] = (
    remaining_unmapped["Market_Key"]
    .apply(
        lambda x:
        "Ambiguous in official reference"
        if x in ambiguous_keys
        else "Not found safely in reference"
    )
)

In [316]:
print(
    remaining_unmapped[
        "Mapping_Status"
    ].value_counts()
)

Mapping_Status
Not found safely in reference    4
Name: count, dtype: int64


In [317]:
print(
    remaining_unmapped[
        [
            "Market",
            "Market_Key",
            "Mapping_Status"
        ]
    ].to_string(
        index=False
    )
)

                                                                                              Market                                                                                           Market_Key                Mapping_Status
Change (Over Previous Month) = [Price - Price(Over Previous Month)/Price(Over Previous Month)] * 100 change (over previous month) = [price - price(over previous month)/price(over previous month)] * 100 Not found safely in reference
   Change (Over Previous Year) = [Price - Price(Over Previous Year)/Price(Over Previous Year)] * 100    change (over previous year) = [price - price(over previous year)/price(over previous year)] * 100 Not found safely in reference
                                                                                               Note:                                                                                                note: Not found safely in reference
                      Weighted Average Price = ∑(arrival quantity * moda

In [318]:
ambiguous_remaining = (
    remaining_unmapped[
        remaining_unmapped[
            "Mapping_Status"
        ]
        == "Ambiguous in official reference"
    ]
)

In [319]:
for market_key in ambiguous_remaining[
    "Market_Key"
]:

    districts = (
        official_up_mapping.loc[
            official_up_mapping[
                "Market_Key"
            ] == market_key,
            "District"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    market_name = (
        ambiguous_remaining.loc[
            ambiguous_remaining[
                "Market_Key"
            ] == market_key,
            "Market"
        ]
        .iloc[0]
    )

    print(
        market_name,
        "→",
        districts
    )

In [320]:
# Geodcoing remaining 44 markets

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

In [321]:
geolocator = Nominatim(
    user_agent="crop-yield-risk-dashboard-market-mapping",
    timeout=10
)

geocode_market = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=True
)

In [322]:
def get_market_location_candidate(
    market_name
):

    # Remove APMC for the location search
    clean_name = (
        str(market_name)
        .replace("APMC", "")
        .strip()
    )

    query = (
        f"{clean_name}, "
        f"Uttar Pradesh, India"
    )

    location = geocode_market(
        query,
        addressdetails=True
    )

    if location is None:

        return {
            "Geocode_Found": False,
            "Candidate_District": None,
            "Candidate_Place": None,
            "Full_Address": None,
            "Latitude": None,
            "Longitude": None
        }

    address = location.raw.get(
        "address",
        {}
    )

    # District-like fields returned by OpenStreetMap
    candidate_district = (
        address.get("state_district")
        or address.get("county")
        or address.get("district")
        or address.get("city_district")
    )

    # Useful for manual checking
    candidate_place = (
        address.get("city")
        or address.get("town")
        or address.get("village")
        or address.get("municipality")
        or address.get("suburb")
    )

    return {
        "Geocode_Found": True,
        "Candidate_District": candidate_district,
        "Candidate_Place": candidate_place,
        "Full_Address": location.address,
        "Latitude": location.latitude,
        "Longitude": location.longitude
    }

In [323]:
geocode_results = []

for number, row in enumerate(
    remaining_unmapped.itertuples(),
    start=1
):

    print(
        f"{number}/{len(remaining_unmapped)} "
        f"→ {row.Market}"
    )

    result = get_market_location_candidate(
        row.Market
    )

    geocode_results.append(
        {
            "Market": row.Market,
            "Market_Key": row.Market_Key,
            **result
        }
    )

1/4 → Change (Over Previous Month) = [Price - Price(Over Previous Month)/Price(Over Previous Month)] * 100
2/4 → Change (Over Previous Year) = [Price - Price(Over Previous Year)/Price(Over Previous Year)] * 100
3/4 → Note:
4/4 → Weighted Average Price = ∑(arrival quantity * modal price) / ∑arrival quantity


In [324]:
geocoded_markets = pd.DataFrame(
    geocode_results
)


In [325]:
# Checkinf how may were found

print(
    "Total markets:",
    len(geocoded_markets)
)

print(
    "Locations found:",
    geocoded_markets[
        "Geocode_Found"
    ].sum()
)

print(
    "Locations not found:",
    (
        ~geocoded_markets[
            "Geocode_Found"
        ]
    ).sum()
)

print(
    "Candidate districts available:",
    geocoded_markets[
        "Candidate_District"
    ].notna().sum()
)

Total markets: 4
Locations found: 0
Locations not found: 4
Candidate districts available: 0


In [326]:
print(
    geocoded_markets[
        [
            "Market",
            "Candidate_District",
            "Candidate_Place",
            "Full_Address"
        ]
    ].to_string(
        index=False
    )
)

                                                                                              Market Candidate_District Candidate_Place Full_Address
Change (Over Previous Month) = [Price - Price(Over Previous Month)/Price(Over Previous Month)] * 100               None            None         None
   Change (Over Previous Year) = [Price - Price(Over Previous Year)/Price(Over Previous Year)] * 100               None            None         None
                                                                                               Note:               None            None         None
                      Weighted Average Price = ∑(arrival quantity * modal price) / ∑arrival quantity               None            None         None


In [327]:
# Saving geocoding work

geocode_file = Path(
    "../../data/processed/"
    "up_unmapped_market_geocode_candidates.csv"
)

geocoded_markets.to_csv(
    geocode_file,
    index=False
)

print(
    "Saved:",
    geocode_file
)

Saved: ..\..\data\processed\up_unmapped_market_geocode_candidates.csv


In [328]:
from pathlib import Path
import pandas as pd

production_file = Path(
    "../../data/processed/production_history_2021_2025.csv"
)

print(
    "Production file exists:",
    production_file.exists()
)

Production file exists: True


In [329]:
production_history = pd.read_csv(
    production_file
)

print(
    "Production rows:",
    len(production_history)
)

print(
    production_history.columns.tolist()
)

production_history.head()

Production rows: 1403
['State', 'District', 'Crop', 'Year', 'Area', 'Production', 'Yield', 'Year_Order', 'Years_ Available', 'Complete_4_year_history', 'Previous_Yield', 'Previous_yield_percentage', 'Previous_area', 'Previous_Area_Percentage', 'Previous_Production', 'Production_Change_Pct', 'History_avg_yield', 'Yield_Deviation_Pct']


,State,District,Crop,Year,Area,Production,Yield,Year_Order,Years_ Available,Complete_4_year_history,Previous_Yield,Previous_yield_percentage,Previous_area,Previous_Area_Percentage,Previous_Production,Production_Change_Pct,History_avg_yield,Yield_Deviation_Pct
0,Haryana,Ambala,Maize,2021-22,0.00,0.01,3945.0,2021,4,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Haryana,Ambala,Maize,2022-23,0.00,0.00,3455.0,2022,4,True,3945.0,-12.420786,0.0,NaN,0.01,-100.0,3945.0,-12.420786
2,Haryana,Ambala,Maize,2023-24,0.00,0.01,4642.0,2023,4,True,3455.0,34.356006,0.0,NaN,0.00,NaN,3700.0,25.459459
3,Haryana,Ambala,Maize,2024-25,0.00,0.00,3714.0,2024,4,True,4642.0,-19.991383,0.0,NaN,0.01,-100.0,4014.0,-7.473842
4,Haryana,Ambala,Rice,2021-22,0.94,3.86,4107.0,2021,4,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [330]:
#Get the official district names from our production dataset

production_up_districts = sorted(
    production_history.loc[
        production_history["State"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "uttar pradesh",
        "District"
    ]
    .dropna()
    .unique()
)

print(
    "UP districts in production data:",
    len(production_up_districts)
)

print(
    production_up_districts
)

UP districts in production data: 75
['Agra', 'Aligarh', 'Ambedkar Nagar', 'Amethi', 'Amroha', 'Auraiya', 'Ayodhya', 'Azamgarh', 'Baghpat', 'Bahraich', 'Ballia', 'Balrampur', 'Banda', 'Bara Banki', 'Bareilly', 'Basti', 'Bhadohi', 'Bijnor', 'Budaun', 'Bulandshahr', 'Chandauli', 'Chitrakoot', 'Deoria', 'Etah', 'Etawah', 'Farrukhabad', 'Fatehpur', 'Firozabad', 'Gautam Buddha Nagar', 'Ghaziabad', 'Ghazipur', 'Gonda', 'Gorakhpur', 'Hamirpur', 'Hapur', 'Hardoi', 'Hathras', 'Jalaun', 'Jaunpur', 'Jhansi', 'Kannauj', 'Kanpur Dehat', 'Kanpur Nagar', 'Kasganj', 'Kaushambi', 'Kheri', 'Kushinagar', 'Lalitpur', 'Lucknow', 'Mahoba', 'Mahrajganj', 'Mainpuri', 'Mathura', 'Mau', 'Meerut', 'Mirzapur', 'Moradabad', 'Muzaffarnagar', 'Pilibhit', 'Pratapgarh', 'Prayagraj', 'Rae Bareli', 'Rampur', 'Saharanpur', 'Sambhal', 'Sant Kabir Nagar', 'Shahjahanpur', 'Shamli', 'Shrawasti', 'Siddharthnagar', 'Sitapur', 'Sonbhadra', 'Sultanpur', 'Unnao', 'Varanasi']


In [331]:
geocoded_found = (
    geocoded_markets[
        geocoded_markets[
            "Candidate_District"
        ].notna()
    ]
    .copy()
)

geocoded_found["Exact_Production_Match"] = (
    geocoded_found[
        "Candidate_District"
    ].isin(
        production_up_districts
    )
)

In [332]:
print(
    "Exact district matches:",
    geocoded_found[
        "Exact_Production_Match"
    ].sum()
)

print(
    "District names needing review:",
    (
        ~geocoded_found[
            "Exact_Production_Match"
        ]
    ).sum()
)

Exact district matches: 0
District names needing review: 0


In [333]:
district_name_review = (
    geocoded_found[
        ~geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market",
            "Candidate_District",
            "Candidate_Place",
            "Full_Address"
        ]
    ]
    .drop_duplicates()
)

print(
    district_name_review.to_string(
        index=False
    )
)

Empty DataFrame
Columns: [Market, Candidate_District, Candidate_Place, Full_Address]
Index: []


In [334]:
not_found_markets = (
    geocoded_markets[
        ~geocoded_markets[
            "Geocode_Found"
        ]
    ]
    [
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

print(
    "Markets still not found:",
    len(not_found_markets)
)

print(
    not_found_markets.to_string(
        index=False
    )
)

Markets still not found: 4
                                                                                              Market                                                                                           Market_Key
Change (Over Previous Month) = [Price - Price(Over Previous Month)/Price(Over Previous Month)] * 100 change (over previous month) = [price - price(over previous month)/price(over previous month)] * 100
   Change (Over Previous Year) = [Price - Price(Over Previous Year)/Price(Over Previous Year)] * 100    change (over previous year) = [price - price(over previous year)/price(over previous year)] * 100
                                                                                               Note:                                                                                                note:
                      Weighted Average Price = ∑(arrival quantity * modal price) / ∑arrival quantity                       weighted average price = ∑(arrival quantit

In [335]:
def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    # Read CSV
    # First row = report title
    # Second row = actual column headers
    df = pd.read_csv(
        file_path,
        header=1
    )

    # The second column contains
    # the selected month's price
    current_price_column = df.columns[1]

    # Keep only Market + Current Month Price
    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    # Rename price column
    df = df.rename(
        columns={
            current_price_column: "Monthly_Price"
        }
    )

    # Clean market names
    df["Market"] = (
        df["Market"]
        .astype(str)
        .str.strip()
    )

    # IMPORTANT:
    # Keep only real APMC market rows
    #
    # This automatically removes:
    # Average
    # Note:
    # Change formula
    # Weighted Average formula
    df = df[
        df["Market"]
        .str.lower()
        .str.endswith("apmc")
    ].copy()

    # Convert price to number
    # "-" becomes NaN
    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    # Add information from filters
    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    # Create proper month date
    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    # Arrange final columns
    df = df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]

    return df

In [336]:
print(
    "clean_monthly_price_csv" in globals()
)

True


In [337]:
import re

all_up_wheat = []

for file_path in up_wheat_files:

    match = re.search(
        r"uttar_pradesh_wheat_(\d{4})_(\d{2})",
        file_path.stem
    )

    if not match:
        continue

    year = int(match.group(1))
    month = int(match.group(2))

    clean_df = clean_monthly_price_csv(
        file_path=file_path,
        state="Uttar Pradesh",
        crop="Wheat",
        year=year,
        month=month
    )

    all_up_wheat.append(clean_df)

up_wheat_monthly = pd.concat(
    all_up_wheat,
    ignore_index=True
)

print(
    "Rebuilt rows:",
    len(up_wheat_monthly)
)

Rebuilt rows: 10503


In [338]:
up_wheat_monthly["Market_Key"] = (
    up_wheat_monthly["Market"]
    .apply(normalize_market_name)
)

In [339]:
print(
    "Unique months:",
    up_wheat_monthly["Month_Date"].nunique()
)

print(
    "Unique markets:",
    up_wheat_monthly["Market"].nunique()
)

print(
    "Total rows:",
    len(up_wheat_monthly)
)

Unique months: 54
Unique markets: 231
Total rows: 10503


In [340]:
up_wheat_with_district = (
    up_wheat_monthly.merge(
        combined_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [341]:
print(
    "Rows before merge:",
    len(up_wheat_monthly)
)

print(
    "Rows after merge:",
    len(up_wheat_with_district)
)

Rows before merge: 10503
Rows after merge: 10503


In [342]:
print(
    "Rows missing District:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

Rows missing District: 0


In [343]:
remaining_unmapped = (
    up_wheat_with_district.loc[
        up_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

In [344]:
print(
    "Remaining real unmapped markets:",
    len(remaining_unmapped)
)

print(
    remaining_unmapped.to_string(
        index=False
    )
)

Remaining real unmapped markets: 0
Empty DataFrame
Columns: [Market, Market_Key]
Index: []


In [345]:
geocoder_exact_mapping = (
    geocoded_found[
        geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market_Key",
            "Candidate_District"
        ]
    ]
    .rename(
        columns={
            "Candidate_District":
            "District"
        }
    )
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

In [346]:
print(
    "Exact geocoder mappings:",
    len(geocoder_exact_mapping)
)

Exact geocoder mappings: 0


In [347]:
existing_keys = set(
    combined_market_mapping[
        "Market_Key"
    ]
)

new_geocoder_mapping = (
    geocoder_exact_mapping[
        ~geocoder_exact_mapping[
            "Market_Key"
        ].isin(existing_keys)
    ]
    .copy()
)

print(
    "New exact geocoder mappings:",
    len(new_geocoder_mapping)
)

New exact geocoder mappings: 0


In [348]:
combined_market_mapping_v2 = pd.concat(
    [
        combined_market_mapping,
        new_geocoder_mapping
    ],
    ignore_index=True
)

In [349]:
combined_market_mapping_v2 = (
    combined_market_mapping_v2
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

In [350]:
mapping_check = (
    combined_market_mapping_v2
    .groupby("Market_Key")["District"]
    .nunique()
)

print(
    "Mapping conflicts:",
    (mapping_check > 1).sum()
)

Mapping conflicts: 0


In [351]:
up_wheat_with_district = (
    up_wheat_monthly.merge(
        combined_market_mapping_v2,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [352]:
remaining_unmapped = (
    up_wheat_with_district.loc[
        up_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

print(
    "Rows still missing District:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Remaining unmapped markets:",
    len(remaining_unmapped)
)

print(
    remaining_unmapped.to_string(
        index=False
    )
)

Rows still missing District: 0
Remaining unmapped markets: 0
Empty DataFrame
Columns: [Market, Market_Key]
Index: []


In [353]:
print(
    remaining_unmapped[
        [
            "Market",
            "Market_Key"
        ]
    ].to_string(
        index=False
    )
)

Empty DataFrame
Columns: [Market, Market_Key]
Index: []


In [354]:
two_name_mismatches = (
    geocoded_found[
        ~geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market",
            "Market_Key",
            "Candidate_District",
            "Candidate_Place",
            "Full_Address"
        ]
    ]
    .drop_duplicates()
)

print(
    "Geocoded but district name differs:",
    len(two_name_mismatches)
)

print(
    two_name_mismatches.to_string(
        index=False
    )
)

Geocoded but district name differs: 0
Empty DataFrame
Columns: [Market, Market_Key, Candidate_District, Candidate_Place, Full_Address]
Index: []


In [355]:
geocoded_mismatch_keys = set(
    two_name_mismatches[
        "Market_Key"
    ]
)

completely_unresolved = (
    remaining_unmapped[
        ~remaining_unmapped[
            "Market_Key"
        ].isin(
            geocoded_mismatch_keys
        )
    ]
    .copy()
)

print(
    "Completely unresolved markets:",
    len(completely_unresolved)
)

print(
    completely_unresolved.to_string(
        index=False
    )
)

Completely unresolved markets: 0
Empty DataFrame
Columns: [Market, Market_Key]
Index: []


In [356]:
final_geo_mapping = {
    "ahirora": "Mirzapur",
    "anoop shahar": "Bulandshahar",
    "bachranwa": "Raebareli",
    "badda": "Shahjahanpur",
    "billsadda": "Pilibhit",
    "dibiapur": "Auraiya",
    "jamanian": "Ghazipur",
    "jasvantnagar": "Etawah",
    "jhansi grain": "Jhansi",
    "lediyari": "Prayagraj",
    "rampurmaniharan": "Saharanpur",
    "samsabad": "Agra",
    "sikanderabad": "Bulandshahar",
    "vilthararoad": "Ballia",

    # These two came from your successful geocoding:
    "gopiganj": "Sant Ravidas Nagar",
    "mohammdi": "Lakhimpur Kheri"
}

In [357]:
search_words = [
    "Mirz",
    "Buland",
    "Rae",
    "Shah",
    "Pili",
    "Aurai",
    "Ghazi",
    "Etawah",
    "Jhansi",
    "Pray",
    "Allah",
    "Saharan",
    "Agra",
    "Ballia",
    "Bhado",
    "Ravidas",
    "Lakhimpur",
    "Khiri"
]

for word in search_words:

    matches = [
        district
        for district in production_up_districts
        if word.lower() in district.lower()
    ]

    if matches:
        print(
            word,
            "→",
            matches
        )

Mirz → ['Mirzapur']
Buland → ['Bulandshahr']
Rae → ['Rae Bareli']
Shah → ['Bulandshahr', 'Shahjahanpur']
Pili → ['Pilibhit']
Aurai → ['Auraiya']
Ghazi → ['Ghaziabad', 'Ghazipur']
Etawah → ['Etawah']
Jhansi → ['Jhansi']
Pray → ['Prayagraj']
Saharan → ['Saharanpur']
Agra → ['Agra', 'Prayagraj']
Ballia → ['Ballia']
Bhado → ['Bhadohi']


In [358]:
def canonical_district(*candidates):

    for candidate in candidates:
        if candidate in production_up_districts:
            return candidate

    raise ValueError(
        f"None of these district names exist "
        f"in production data: {candidates}"
    )

final_manual_mapping = {
    "ahirora": "Mirzapur",
    "anoop shahar": "Bulandshahr",
    "bachranwa": "Rae Bareli",
    "badda": "Shahjahanpur",
    "billsadda": "Pilibhit",
    "dibiapur": "Auraiya",
    "jamanian": "Ghazipur",
    "jasvantnagar": "Etawah",
    "jhansi grain": "Jhansi",
    "lediyari": "Prayagraj",
    "rampurmaniharan": "Saharanpur",
    "samsabad": "Agra",
    "sikanderabad": "Bulandshahr",
    "vilthararoad": "Ballia",

    # Geocoder returned Sant Ravidas Nagar,
    # production dataset uses Bhadohi
    "gopiganj": "Bhadohi",

    # Geocoder returned Lakhimpur Kheri,
    # production dataset uses Kheri
    "mohammdi": "Kheri"
}



In [359]:
final_manual_df = pd.DataFrame(
    list(final_manual_mapping.items()),
    columns=[
        "Market_Key",
        "District"
    ]
)

final_manual_df

,Market_Key,District
0,ahirora,Mirzapur
1,anoop shahar,Bulandshahr
2,bachranwa,Rae Bareli
3,badda,Shahjahanpur
4,billsadda,Pilibhit
5,dibiapur,Auraiya
6,jamanian,Ghazipur
7,jasvantnagar,Etawah
8,jhansi grain,Jhansi
9,lediyari,Prayagraj


In [360]:
final_market_mapping = pd.concat(
    [
        combined_market_mapping_v2,
        final_manual_df
    ],
    ignore_index=True
)

In [361]:
final_market_mapping = (
    final_market_mapping
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)

In [362]:
final_conflicts = (
    final_market_mapping
    .groupby("Market_Key")["District"]
    .nunique()
)

print(
    "Final mapping conflicts:",
    (final_conflicts > 1).sum()
)

Final mapping conflicts: 0


In [363]:
up_wheat_with_district = (
    up_wheat_monthly.merge(
        final_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [364]:
print(
    "Total rows:",
    len(up_wheat_with_district)
)

print(
    "Rows missing District:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unique markets still unmapped:",
    up_wheat_with_district.loc[
        up_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

Total rows: 10503
Rows missing District: 0
Unique markets still unmapped: 0


In [365]:
final_market_mapping.to_csv(
    "../../data/processed/up_market_district_mapping.csv",
    index=False
)

up_wheat_with_district.to_csv(
    "../../data/processed/up_wheat_monthly_prices_mapped_2021_2025.csv",
    index=False
)

print("Final UP market mapping saved ✅")
print("Mapped UP Wheat monthly prices saved ✅")

Final UP market mapping saved ✅
Mapped UP Wheat monthly prices saved ✅


In [366]:
# Creating crop-year

def get_crop_year(date):

    if date.month >= 7:
        start_year = date.year
    else:
        start_year = date.year - 1

    end_year = (
        start_year + 1
    ) % 100

    return (
        f"{start_year}-"
        f"{end_year:02d}"
    )

In [367]:
up_wheat_with_district["Year"] = (
    up_wheat_with_district[
        "Month_Date"
    ].apply(get_crop_year)
)

In [368]:
up_wheat_with_district[
    [
        "Month_Date",
        "Year"
    ]
].drop_duplicates().head(15)

,Month_Date,Year
0,2021-01-01,2020-21
187,2021-02-01,2020-21
380,2021-03-01,2020-21
566,2021-04-01,2020-21
763,2021-05-01,2020-21
967,2021-06-01,2020-21
1168,2021-07-01,2021-22
1358,2021-08-01,2021-22
1542,2021-09-01,2021-22
1722,2021-10-01,2021-22


In [369]:
# Keep four project crop-year

target_years = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25"
]

up_wheat_project_prices = (
    up_wheat_with_district[
        up_wheat_with_district[
            "Year"
        ].isin(target_years)
    ]
    .copy()
)

In [370]:
print(
    up_wheat_project_prices[
        "Year"
    ].value_counts().sort_index()
)

Year
2021-22    2266
2022-23    2334
2023-24    2346
2024-25    2389
Name: count, dtype: int64


In [371]:
district_monthly_price = (
    up_wheat_project_prices
    .groupby(
        [
            "State",
            "District",
            "Crop",
            "Year",
            "Month_Date"
        ],
        as_index=False
    )
    .agg(
        District_Monthly_Price=(
            "Monthly_Price",
            "median"
        ),

        Markets_Reporting=(
            "Monthly_Price",
            "count"
        )
    )
)

In [372]:
district_monthly_price.head()

,State,District,Crop,Year,Month_Date,District_Monthly_Price,Markets_Reporting
0,Uttar Pradesh,,Wheat,2021-22,2021-07-01,1674.91,1
1,Uttar Pradesh,,Wheat,2021-22,2021-08-01,1630.00,1
2,Uttar Pradesh,,Wheat,2021-22,2021-09-01,1625.00,1
3,Uttar Pradesh,,Wheat,2021-22,2021-10-01,1800.00,1
4,Uttar Pradesh,,Wheat,2021-22,2021-11-01,1844.53,1


In [373]:
# Check montly district data 

print(
    "District-month rows:",
    len(district_monthly_price)
)

print(
    "Missing district-month prices:",
    district_monthly_price[
        "District_Monthly_Price"
    ].isna().sum()
)

District-month rows: 3557
Missing district-month prices: 65


In [374]:
# Building crop-year features

price_features_up_wheat = (
    district_monthly_price
    .groupby(
        [
            "State",
            "District",
            "Crop",
            "Year"
        ],
        as_index=False
    )
    .agg(

        Avg_Modal_Price=(
            "District_Monthly_Price",
            "mean"
        ),

        Min_Monthly_Price=(
            "District_Monthly_Price",
            "min"
        ),

        Max_Monthly_Price=(
            "District_Monthly_Price",
            "max"
        ),

        Price_Std=(
            "District_Monthly_Price",
            "std"
        ),

        Months_Available=(
            "District_Monthly_Price",
            "count"
        ),

        Avg_Markets_Reporting=(
            "Markets_Reporting",
            "mean"
        )
    )
)

In [375]:
price_features_up_wheat[
    "Price_Volatility_Pct"
] = (
    price_features_up_wheat[
        "Price_Std"
    ]
    /
    price_features_up_wheat[
        "Avg_Modal_Price"
    ]
    * 100
)

In [376]:
# Coveragae flag

price_features_up_wheat[
    "Complete_12_Month_Price_History"
] = (
    price_features_up_wheat[
        "Months_Available"
    ] == 12
)

In [377]:
price_features_up_wheat.head(10)

,State,District,Crop,Year,Avg_Modal_Price,Min_Monthly_Price,Max_Monthly_Price,Price_Std,Months_Available,Avg_Markets_Reporting,Price_Volatility_Pct,Complete_12_Month_Price_History
0,Uttar Pradesh,,Wheat,2021-22,1856.372500,1625.000,2015.000,146.224341,12,1.000000,7.876886,True
1,Uttar Pradesh,,Wheat,2022-23,2293.448333,2015.000,2640.880,198.499818,12,1.000000,8.655081,True
2,Uttar Pradesh,,Wheat,2023-24,2388.020000,2255.000,2545.330,99.731923,9,0.750000,4.176344,False
3,Uttar Pradesh,,Wheat,2024-25,2656.900833,2424.540,2910.000,182.043143,12,1.000000,6.851710,True
4,Uttar Pradesh,Agra,Wheat,2021-22,1958.910417,1714.205,2145.700,142.105013,12,6.166667,7.254289,True
5,Uttar Pradesh,Agra,Wheat,2022-23,2369.090833,2094.655,2780.600,227.452468,12,6.333333,9.600834,True
6,Uttar Pradesh,Agra,Wheat,2023-24,2418.009583,2258.750,2551.540,112.854270,12,5.416667,4.667238,True
7,Uttar Pradesh,Agra,Wheat,2024-25,2670.115833,2450.490,2936.940,193.399458,12,5.666667,7.243111,True
8,Uttar Pradesh,Aligarh,Wheat,2021-22,1954.497917,1669.530,2095.345,133.439595,12,3.666667,6.827308,True
9,Uttar Pradesh,Aligarh,Wheat,2022-23,2368.186250,2111.240,2774.060,218.703181,12,3.583333,9.235050,True


In [378]:
print(
    price_features_up_wheat[
        "Months_Available"
    ].value_counts().sort_index()
)

Months_Available
0       2
1       4
2       2
5       1
6       1
8       3
9       3
10      3
11      4
12    279
Name: count, dtype: int64


In [379]:
print(
    "Complete district-years:",
    price_features_up_wheat[
        "Complete_12_Month_Price_History"
    ].sum()
)

print(
    "Incomplete district-years:",
    (
        ~price_features_up_wheat[
            "Complete_12_Month_Price_History"
        ]
    ).sum()
)

Complete district-years: 279
Incomplete district-years: 23


In [380]:
# Inspecting final price features

price_features_up_wheat[
    [
        "Avg_Modal_Price",
        "Price_Std",
        "Price_Volatility_Pct",
        "Months_Available"
    ]
].describe()

,Avg_Modal_Price,Price_Std,Price_Volatility_Pct,Months_Available
count,300.000000,296.000000,296.000000,302.000000
mean,2309.014819,145.231733,6.297903,11.562914
std,255.464184,49.262936,2.024071,1.903227
min,1822.287000,16.404877,0.693394,0.000000
25%,2113.969479,100.521419,4.513040,12.000000
50%,2356.303750,147.356029,6.664981,12.000000
75%,2448.460625,185.477482,7.783681,12.000000
max,2719.258333,351.785624,12.975035,12.000000


In [381]:
price_features_up_wheat[
    [
        "State",
        "District",
        "Crop",
        "Year",
        "Avg_Modal_Price",
        "Price_Volatility_Pct",
        "Months_Available"
    ]
].head(20)

,State,District,Crop,Year,Avg_Modal_Price,Price_Volatility_Pct,Months_Available
0,Uttar Pradesh,,Wheat,2021-22,1856.372500,7.876886,12
1,Uttar Pradesh,,Wheat,2022-23,2293.448333,8.655081,12
2,Uttar Pradesh,,Wheat,2023-24,2388.020000,4.176344,9
3,Uttar Pradesh,,Wheat,2024-25,2656.900833,6.851710,12
4,Uttar Pradesh,Agra,Wheat,2021-22,1958.910417,7.254289,12
5,Uttar Pradesh,Agra,Wheat,2022-23,2369.090833,9.600834,12
6,Uttar Pradesh,Agra,Wheat,2023-24,2418.009583,4.667238,12
7,Uttar Pradesh,Agra,Wheat,2024-25,2670.115833,7.243111,12
8,Uttar Pradesh,Aligarh,Wheat,2021-22,1954.497917,6.827308,12
9,Uttar Pradesh,Aligarh,Wheat,2022-23,2368.186250,9.235050,12


In [382]:
price_features_up_wheat.to_csv(
    "../../data/processed/"
    "up_wheat_price_features_2021_2025.csv",
    index=False
)

print(
    "UP Wheat price features saved ✅"
)

UP Wheat price features saved ✅


In [383]:
# Coverage percentage 

price_features_up_wheat[
    "Price_Coverage_Pct"
] = (
    price_features_up_wheat[
        "Months_Available"
    ]
    / 12
    * 100
)

In [384]:
def price_coverage_quality(months):

    if months == 12:
        return "Complete"

    elif months >= 9:
        return "Good"

    elif months >= 6:
        return "Moderate"

    else:
        return "Low"


price_features_up_wheat[
    "Price_Coverage_Quality"
] = (
    price_features_up_wheat[
        "Months_Available"
    ].apply(
        price_coverage_quality
    )
)

In [385]:
incomplete_price_history = (
    price_features_up_wheat[
        price_features_up_wheat[
            "Months_Available"
        ] < 12
    ]
    [
        [
            "State",
            "District",
            "Crop",
            "Year",
            "Avg_Modal_Price",
            "Price_Std",
            "Price_Volatility_Pct",
            "Months_Available",
            "Price_Coverage_Pct",
            "Price_Coverage_Quality"
        ]
    ]
    .sort_values(
        [
            "Months_Available",
            "District",
            "Year"
        ]
    )
)

print(
    incomplete_price_history.to_string(
        index=False
    )
)

        State         District  Crop    Year  Avg_Modal_Price  Price_Std  Price_Volatility_Pct  Months_Available  Price_Coverage_Pct Price_Coverage_Quality
Uttar Pradesh        Chitrakut Wheat 2023-24              NaN        NaN                   NaN                 0            0.000000                    Low
Uttar Pradesh Sant Kabir Nagar Wheat 2022-23              NaN        NaN                   NaN                 0            0.000000                    Low
Uttar Pradesh        Barabanki Wheat 2021-22      2015.000000        NaN                   NaN                 1            8.333333                    Low
Uttar Pradesh        Chitrakut Wheat 2021-22      1982.500000        NaN                   NaN                 1            8.333333                    Low
Uttar Pradesh Sant Kabir Nagar Wheat 2021-22      2015.000000        NaN                   NaN                 1            8.333333                    Low
Uttar Pradesh Sant Kabir Nagar Wheat 2023-24      2420.000000   

In [386]:
unusable_volatility = (
    price_features_up_wheat[
        (
            price_features_up_wheat[
                "Months_Available"
            ] < 6
        )
        |
        (
            price_features_up_wheat[
                "Price_Volatility_Pct"
            ].isna()
        )
    ]
)

print(
    "Very low coverage / unusable volatility rows:",
    len(unusable_volatility)
)

print(
    unusable_volatility[
        [
            "District",
            "Year",
            "Months_Available",
            "Avg_Modal_Price",
            "Price_Volatility_Pct"
        ]
    ].to_string(
        index=False
    )
)

Very low coverage / unusable volatility rows: 9
        District    Year  Months_Available  Avg_Modal_Price  Price_Volatility_Pct
          Badaun 2021-22                 5         1948.964             10.429066
       Barabanki 2021-22                 1         2015.000                   NaN
       Chitrakut 2021-22                 1         1982.500                   NaN
       Chitrakut 2023-24                 0              NaN                   NaN
        Ghazipur 2024-25                 2         2711.250             12.975035
Sant Kabir Nagar 2021-22                 1         2015.000                   NaN
Sant Kabir Nagar 2022-23                 0              NaN                   NaN
Sant Kabir Nagar 2023-24                 1         2420.000                   NaN
       Shravasti 2023-24                 2         2365.880              0.693394


In [387]:
price_features_up_wheat.to_csv(
    "../../data/processed/"
    "up_wheat_price_features_2021_2025.csv",
    index=False
)

print(
    "Updated UP Wheat price features saved ✅"
)

Updated UP Wheat price features saved ✅


In [388]:
# Cleaning the final mapping itself 

import pandas as pd
from pathlib import Path

mapping_path = Path(
    "../../data/processed/up_market_district_mapping.csv"
)

# --------------------------------------------------
# OPTION 1: Load previously saved final mapping
# --------------------------------------------------

if mapping_path.exists():

    final_market_mapping = pd.read_csv(
        mapping_path
    )

    print("Loaded saved market mapping ✅")


# --------------------------------------------------
# OPTION 2: Rebuild it from notebook variables
# --------------------------------------------------

elif "combined_market_mapping_v2" in globals():

    final_manual_mapping = {
        "ahirora": "Mirzapur",
        "anoop shahar": "Bulandshahr",
        "bachranwa": "Rae Bareli",
        "badda": "Shahjahanpur",
        "billsadda": "Pilibhit",
        "dibiapur": "Auraiya",
        "jamanian": "Ghazipur",
        "jasvantnagar": "Etawah",
        "jhansi grain": "Jhansi",
        "lediyari": "Prayagraj",
        "rampurmaniharan": "Saharanpur",
        "samsabad": "Agra",
        "sikanderabad": "Bulandshahr",
        "vilthararoad": "Ballia",
        "gopiganj": "Bhadohi",
        "mohammdi": "Kheri"
    }

    final_manual_df = pd.DataFrame(
        list(final_manual_mapping.items()),
        columns=[
            "Market_Key",
            "District"
        ]
    )

    final_market_mapping = pd.concat(
        [
            combined_market_mapping_v2,
            final_manual_df
        ],
        ignore_index=True
    )

    # keep="last" is intentional:
    # our manually verified mapping overrides older mapping
    final_market_mapping = (
        final_market_mapping
        .drop_duplicates(
            subset=["Market_Key"],
            keep="last"
        )
    )

    print("Final market mapping rebuilt ✅")


else:

    raise RuntimeError(
        "Neither the saved mapping file nor "
        "combined_market_mapping_v2 exists. "
        "We need to reload/rebuild the earlier mapping stage."
    )

Loaded saved market mapping ✅


In [389]:
# Verify every mapped district matches the production dataset
invalid_districts = sorted(
    set(
        final_market_mapping[
            "District"
        ].dropna()
    )
    -
    set(
        production_up_districts
    )
)

print(
    "District names not matching production:",
    invalid_districts
)

District names not matching production: [' ', 'Ambedkarnagar', 'Badaun', 'Barabanki', 'Bulandshahar', 'Chitrakut', 'Lakhimpur Kheri', 'Maharajganj', 'Shravasti']


In [390]:
print(
    "Mapping rows with missing District:",
    final_market_mapping[
        "District"
    ].isna().sum()
)

Mapping rows with missing District: 0


In [391]:
# Rebuild the monthly dataset with the final mapping
up_wheat_with_district = (
    up_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        final_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [392]:
import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. PATH TO RAW PRICE FILES
# ============================================================

price_raw_folder = Path(
    "../../data/raw/prices"
)

up_wheat_files = sorted(
    price_raw_folder.glob(
        "uttar_pradesh_wheat_*.csv"
    )
)

print(
    "UP Wheat raw files found:",
    len(up_wheat_files)
)


# ============================================================
# 2. CLEAN ONE MONTHLY AGMARKNET CSV
# ============================================================

def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    # First line = report title
    # Second line = actual column names
    df = pd.read_csv(
        file_path,
        header=1
    )

    # Second column contains selected month's price
    current_price_column = df.columns[1]

    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    df = df.rename(
        columns={
            current_price_column:
            "Monthly_Price"
        }
    )

    # Clean market text
    df["Market"] = (
        df["Market"]
        .astype(str)
        .str.strip()
    )

    # Keep only genuine APMC rows
    # Removes:
    # Average
    # Note
    # Change formulas
    # Weighted Average formulas
    df = df[
        df["Market"]
        .str.lower()
        .str.endswith("apmc")
    ].copy()

    # "-" becomes NaN
    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    # Add metadata
    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    return df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]


# ============================================================
# 3. NORMALIZE MARKET NAME
# ============================================================

def normalize_market_name(name):

    name = str(name).strip().lower()

    # Remove APMC
    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    # Remove extra spaces
    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


# ============================================================
# 4. REBUILD ALL 54 MONTHS
# ============================================================

all_up_wheat = []

for file_path in up_wheat_files:

    # Example:
    # uttar_pradesh_wheat_2021_01.csv

    match = re.search(
        r"uttar_pradesh_wheat_(\d{4})_(\d{2})",
        file_path.stem
    )

    if not match:
        print(
            "Skipped strange filename:",
            file_path.name
        )
        continue

    year = int(
        match.group(1)
    )

    month = int(
        match.group(2)
    )

    clean_df = clean_monthly_price_csv(
        file_path=file_path,
        state="Uttar Pradesh",
        crop="Wheat",
        year=year,
        month=month
    )

    all_up_wheat.append(
        clean_df
    )


# ============================================================
# 5. COMBINE ALL MONTHS
# ============================================================

up_wheat_monthly = pd.concat(
    all_up_wheat,
    ignore_index=True
)


# ============================================================
# 6. CREATE MARKET KEY
# ============================================================

up_wheat_monthly["Market_Key"] = (
    up_wheat_monthly["Market"]
    .apply(
        normalize_market_name
    )
)


# ============================================================
# 7. FINAL CHECK
# ============================================================

print("\nRecovery complete ✅")

print(
    "Total rows:",
    len(up_wheat_monthly)
)

print(
    "Unique months:",
    up_wheat_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Unique markets:",
    up_wheat_monthly[
        "Market"
    ].nunique()
)

UP Wheat raw files found: 54

Recovery complete ✅
Total rows: 10503
Unique months: 54
Unique markets: 231


In [393]:
print(
    "up_wheat_monthly exists:",
    "up_wheat_monthly" in globals()
)

up_wheat_monthly exists: True


In [394]:
up_wheat_with_district = (
    up_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        final_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [395]:
print(
    "Rows:",
    len(up_wheat_with_district)
)

print(
    "Missing District:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

Rows: 10503
Missing District: 0


In [396]:
from pathlib import Path
import pandas as pd

mapping_path = Path(
    "../../data/processed/up_market_district_mapping.csv"
)

print(
    "Mapping file exists:",
    mapping_path.exists()
)

final_market_mapping = pd.read_csv(
    mapping_path
)

# Clean district names
final_market_mapping["District"] = (
    final_market_mapping["District"]
    .astype("string")
    .str.strip()
)

# Canonical production name
final_market_mapping["District"] = (
    final_market_mapping["District"]
    .replace({
        "Lakhimpur Kheri": "Kheri"
    })
)

# Check if same market points to multiple districts
mapping_conflicts = (
    final_market_mapping
    .groupby("Market_Key")["District"]
    .nunique()
)

print(
    "Mapping conflicts:",
    (mapping_conflicts > 1).sum()
)

# Keep one mapping row per market
final_market_mapping = (
    final_market_mapping
    .dropna(
        subset=[
            "Market_Key",
            "District"
        ]
    )
    .drop_duplicates(
        subset=["Market_Key"],
        keep="last"
    )
)

# Merge monthly price data with district mapping
up_wheat_with_district = (
    up_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        final_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

print(
    "\nRows:",
    len(up_wheat_with_district)
)

print(
    "Missing District rows:",
    up_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    up_wheat_with_district.loc[
        up_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

Mapping file exists: True
Mapping conflicts: 0

Rows: 10503
Missing District rows: 0
Unmapped markets: 0


In [397]:
mapped_price_path = Path(
    "../../data/processed/"
    "up_wheat_monthly_prices_mapped_2021_2025.csv"
)

up_wheat_with_district.to_csv(
    mapped_price_path,
    index=False
)

print(
    "Saved:",
    mapped_price_path
)

Saved: ..\..\data\processed\up_wheat_monthly_prices_mapped_2021_2025.csv


In [398]:
import pandas as pd
from pathlib import Path


# ============================================================
# 1. LOAD SAVED MAPPED MONTHLY PRICE DATA
# ============================================================

mapped_price_path = Path(
    "../../data/processed/"
    "up_wheat_monthly_prices_mapped_2021_2025.csv"
)

up_wheat_with_district = pd.read_csv(
    mapped_price_path,
    parse_dates=["Month_Date"]
)

print(
    "Mapped rows loaded:",
    len(up_wheat_with_district)
)


# ============================================================
# 2. CREATE AGRICULTURAL / CROP YEAR
#    July -> June
# ============================================================

def get_crop_year(date):

    date = pd.to_datetime(date)

    if date.month >= 7:
        start_year = date.year
    else:
        start_year = date.year - 1

    end_year = (start_year + 1) % 100

    return f"{start_year}-{end_year:02d}"


up_wheat_with_district["Year"] = (
    up_wheat_with_district[
        "Month_Date"
    ].apply(get_crop_year)
)


# ============================================================
# 3. KEEP ONLY PROJECT YEARS
# ============================================================

target_years = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25"
]

project_prices = (
    up_wheat_with_district[
        up_wheat_with_district[
            "Year"
        ].isin(target_years)
    ]
    .copy()
)


# ============================================================
# 4. CREATE ONE REPRESENTATIVE PRICE PER DISTRICT PER MONTH
#
# Median is used across markets within a district.
# This prevents districts with many APMCs from getting
# more statistical weight than districts with fewer APMCs.
# ============================================================

district_monthly_price = (
    project_prices
    .groupby(
        [
            "State",
            "District",
            "Crop",
            "Year",
            "Month_Date"
        ],
        as_index=False
    )
    .agg(
        District_Monthly_Price=(
            "Monthly_Price",
            "median"
        ),

        Markets_Reporting=(
            "Monthly_Price",
            "count"
        )
    )
)


# ============================================================
# 5. CREATE DISTRICT + CROP + YEAR PRICE FEATURES
# ============================================================

price_features_up_wheat = (
    district_monthly_price
    .groupby(
        [
            "State",
            "District",
            "Crop",
            "Year"
        ],
        as_index=False
    )
    .agg(

        Avg_Modal_Price=(
            "District_Monthly_Price",
            "mean"
        ),

        Min_Monthly_Price=(
            "District_Monthly_Price",
            "min"
        ),

        Max_Monthly_Price=(
            "District_Monthly_Price",
            "max"
        ),

        Price_Std=(
            "District_Monthly_Price",
            "std"
        ),

        Months_Available=(
            "District_Monthly_Price",
            "count"
        ),

        Avg_Markets_Reporting=(
            "Markets_Reporting",
            "mean"
        )
    )
)


# ============================================================
# 6. PRICE VOLATILITY
#
# Price Volatility %
# = Standard Deviation / Average Price * 100
# ============================================================

price_features_up_wheat[
    "Price_Volatility_Pct"
] = (
    price_features_up_wheat[
        "Price_Std"
    ]
    /
    price_features_up_wheat[
        "Avg_Modal_Price"
    ]
    * 100
)


# ============================================================
# 7. PRICE DATA COVERAGE
# ============================================================

price_features_up_wheat[
    "Price_Coverage_Pct"
] = (
    price_features_up_wheat[
        "Months_Available"
    ]
    / 12
    * 100
)


def price_coverage_quality(months):

    if months == 12:
        return "Complete"

    elif months >= 9:
        return "Good"

    elif months >= 6:
        return "Moderate"

    else:
        return "Low"


price_features_up_wheat[
    "Price_Coverage_Quality"
] = (
    price_features_up_wheat[
        "Months_Available"
    ].apply(
        price_coverage_quality
    )
)


# ============================================================
# 8. DO NOT TRUST VOLATILITY WITH < 6 MONTHS
# ============================================================

price_features_up_wheat.loc[
    price_features_up_wheat[
        "Months_Available"
    ] < 6,
    "Price_Volatility_Pct"
] = pd.NA


# ============================================================
# 9. VALIDATE UNIQUENESS
# ============================================================

duplicates = (
    price_features_up_wheat
    .duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    )
    .sum()
)


print("\nFINAL CHECK")

print(
    "Price feature rows:",
    len(price_features_up_wheat)
)

print(
    "Duplicate district-crop-year rows:",
    duplicates
)

print(
    "Missing volatility:",
    price_features_up_wheat[
        "Price_Volatility_Pct"
    ].isna().sum()
)

print("\nCoverage quality:")

print(
    price_features_up_wheat[
        "Price_Coverage_Quality"
    ].value_counts()
)


# ============================================================
# 10. SAVE FINAL UP-WHEAT PRICE FEATURES
# ============================================================

output_path = Path(
    "../../data/processed/"
    "up_wheat_price_features_2021_2025.csv"
)

price_features_up_wheat.to_csv(
    output_path,
    index=False
)

print(
    "\nSaved:",
    output_path
)

Mapped rows loaded: 10503

FINAL CHECK
Price feature rows: 294
Duplicate district-crop-year rows: 0
Missing volatility: 9

Coverage quality:
Price_Coverage_Quality
Complete    273
Low           9
Good          8
Moderate      4
Name: count, dtype: int64

Saved: ..\..\data\processed\up_wheat_price_features_2021_2025.csv


In [399]:
# Generic monthly price loader

import pandas as pd
import re
from pathlib import Path


def normalize_market_name(name):

    name = str(name).strip().lower()

    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    df = pd.read_csv(
        file_path,
        header=1
    )

    # Second column = selected month's price
    current_price_column = df.columns[1]

    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    df = df.rename(
        columns={
            current_price_column:
            "Monthly_Price"
        }
    )

    df["Market"] = (
        df["Market"]
        .astype(str)
        .str.strip()
    )

    # Keep only genuine market rows
    df = df[
        df["Market"]
        .str.lower()
        .str.endswith("apmc")
    ].copy()

    # "-" etc. become NaN
    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    return df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]

In [400]:
def load_price_history(
    state,
    crop,
    raw_folder="../../data/raw/prices"
):

    raw_folder = Path(
        raw_folder
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    pattern = (
        f"{state_slug}_"
        f"{crop_slug}_"
        f"*.csv"
    )

    files = sorted(
        raw_folder.glob(pattern)
    )

    print(
        f"{state} | {crop}"
    )

    print(
        "Raw monthly files found:",
        len(files)
    )

    all_months = []

    for file_path in files:

        # Expected filename:
        # uttar_pradesh_wheat_2021_01.csv

        match = re.search(
            r"_(\d{4})_(\d{2})$",
            file_path.stem
        )

        if not match:

            print(
                "Skipped:",
                file_path.name
            )

            continue

        year = int(
            match.group(1)
        )

        month = int(
            match.group(2)
        )

        temp = clean_monthly_price_csv(
            file_path=file_path,
            state=state,
            crop=crop,
            year=year,
            month=month
        )

        all_months.append(
            temp
        )

    if not all_months:

        raise ValueError(
            f"No usable price files found "
            f"for {state} - {crop}"
        )

    monthly_df = pd.concat(
        all_months,
        ignore_index=True
    )

    monthly_df["Market_Key"] = (
        monthly_df["Market"]
        .apply(
            normalize_market_name
        )
    )

    print(
        "Rows:",
        len(monthly_df)
    )

    print(
        "Months:",
        monthly_df[
            "Month_Date"
        ].nunique()
    )

    print(
        "Markets:",
        monthly_df[
            "Market"
        ].nunique()
    )

    return monthly_df

In [401]:
up_wheat_monthly_test = load_price_history(
    state="Uttar Pradesh",
    crop="Wheat"
)

Uttar Pradesh | Wheat
Raw monthly files found: 54
Rows: 10503
Months: 54
Markets: 231


In [402]:
def get_crop_year(date):

    date = pd.to_datetime(
        date
    )

    if date.month >= 7:
        start_year = date.year
    else:
        start_year = date.year - 1

    end_year = (
        start_year + 1
    ) % 100

    return (
        f"{start_year}-"
        f"{end_year:02d}"
    )

In [403]:
def build_price_features(
    mapped_monthly_df
):

    df = (
        mapped_monthly_df
        .copy()
    )

    # ----------------------------------------
    # 1. Crop year
    # ----------------------------------------

    df["Year"] = (
        df["Month_Date"]
        .apply(
            get_crop_year
        )
    )

    target_years = [
        "2021-22",
        "2022-23",
        "2023-24",
        "2024-25"
    ]

    df = df[
        df["Year"].isin(
            target_years
        )
    ].copy()

    # ----------------------------------------
    # 2. One representative price
    #    per District + Month
    # ----------------------------------------

    district_monthly = (
        df
        .groupby(
            [
                "State",
                "District",
                "Crop",
                "Year",
                "Month_Date"
            ],
            as_index=False
        )
        .agg(

            District_Monthly_Price=(
                "Monthly_Price",
                "median"
            ),

            Markets_Reporting=(
                "Monthly_Price",
                "count"
            )
        )
    )

    # ----------------------------------------
    # 3. Crop-year features
    # ----------------------------------------

    features = (
        district_monthly
        .groupby(
            [
                "State",
                "District",
                "Crop",
                "Year"
            ],
            as_index=False
        )
        .agg(

            Avg_Modal_Price=(
                "District_Monthly_Price",
                "mean"
            ),

            Min_Monthly_Price=(
                "District_Monthly_Price",
                "min"
            ),

            Max_Monthly_Price=(
                "District_Monthly_Price",
                "max"
            ),

            Price_Std=(
                "District_Monthly_Price",
                "std"
            ),

            Months_Available=(
                "District_Monthly_Price",
                "count"
            ),

            Avg_Markets_Reporting=(
                "Markets_Reporting",
                "mean"
            )
        )
    )

    # ----------------------------------------
    # 4. Volatility
    # ----------------------------------------

    features[
        "Price_Volatility_Pct"
    ] = (
        features[
            "Price_Std"
        ]
        /
        features[
            "Avg_Modal_Price"
        ]
        * 100
    )

    # ----------------------------------------
    # 5. Coverage
    # ----------------------------------------

    features[
        "Price_Coverage_Pct"
    ] = (
        features[
            "Months_Available"
        ]
        / 12
        * 100
    )

    def coverage_quality(months):

        if months == 12:
            return "Complete"

        elif months >= 9:
            return "Good"

        elif months >= 6:
            return "Moderate"

        else:
            return "Low"

    features[
        "Price_Coverage_Quality"
    ] = (
        features[
            "Months_Available"
        ].apply(
            coverage_quality
        )
    )

    # Less than 6 months:
    # do not trust yearly volatility
    features.loc[
        features[
            "Months_Available"
        ] < 6,
        "Price_Volatility_Pct"
    ] = pd.NA

    return features

In [404]:
print(
    "collect_one_month exists:",
    "collect_one_month" in globals()
)

print(
    "collect_price_history_resilient exists:",
    "collect_price_history_resilient" in globals()
)

collect_one_month exists: True
collect_price_history_resilient exists: True


In [405]:
# ============================================================
# AGMARKNET SELENIUM PRICE COLLECTOR
# COMPLETE RECOVERY BLOCK
# ============================================================

import time
import calendar
import shutil
import pandas as pd

from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import (
    InvalidSessionIdException,
    WebDriverException,
    TimeoutException
)


# ============================================================
# 1. SELECT ONE CUSTOM AGMARKNET DROPDOWN OPTION
# ============================================================

def select_exact_dropdown_option(
    driver,
    label_text,
    option_text
):

    # Find label:
    # Select Commodity / State/UT / Year / Month
    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    # Clickable dropdown is directly before label
    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    # Cascading dropdowns are disabled until
    # previous field has been selected
    if trigger.get_attribute("tabindex") == "-1":

        raise Exception(
            f"{label_text} dropdown is disabled."
        )

    # Exact option.
    # Example:
    #
    # <div class="...cursor-pointer">
    #     <span>Wheat</span>
    # </div>

    option_xpath = (
        "//div[contains(@class,'cursor-pointer')]"
        f"[.//span[normalize-space()='{option_text}']]"
    )

    # Maybe dropdown is already open
    visible_options = [
        element
        for element in driver.find_elements(
            By.XPATH,
            option_xpath
        )
        if element.is_displayed()
    ]

    # Otherwise open it
    if not visible_options:

        driver.execute_script(
            "arguments[0].click();",
            trigger
        )

        time.sleep(1)

        visible_options = WebDriverWait(
            driver,
            20
        ).until(
            lambda d: [
                element
                for element in d.find_elements(
                    By.XPATH,
                    option_xpath
                )
                if element.is_displayed()
            ]
        )

    # Click exact option
    driver.execute_script(
        "arguments[0].click();",
        visible_options[0]
    )

    time.sleep(1)

    print(
        f"{label_text} → {option_text} selected"
    )


# ============================================================
# 2. CREATE NEW CHROME DRIVER
# ============================================================

def create_new_driver():

    print(
        "\nStarting new Chrome session..."
    )

    driver = webdriver.Chrome()

    time.sleep(2)

    print(
        "Chrome session ready ✅"
    )

    return driver


# ============================================================
# 3. COLLECT ONE MONTH
# ============================================================

def collect_one_month(
    driver,
    state,
    crop,
    year,
    month
):

    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )

    # Always start with fresh filter page
    driver.get(
        filter_url
    )

    time.sleep(3)

    month_name = (
        calendar.month_name[month]
    )

    print(
        "\n"
        + "=" * 60
    )

    print(
        f"Collecting: "
        f"{crop} | "
        f"{state} | "
        f"{month_name} {year}"
    )

    # --------------------------------------------------------
    # SELECT FILTERS IN REQUIRED ORDER
    # --------------------------------------------------------

    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )

    select_exact_dropdown_option(
        driver,
        "Month",
        month_name
    )

    print(
        "All filters selected ✅"
    )

    # --------------------------------------------------------
    # SUBMIT
    # --------------------------------------------------------

    submit_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Submit']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        submit_button
    )

    # Wait for results page
    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//button[normalize-space()='Export']"
            )
        )
    )

    print(
        "Result page loaded ✅"
    )

    # --------------------------------------------------------
    # DOWNLOADS FOLDER
    # --------------------------------------------------------

    downloads_folder = (
        Path.home()
        / "Downloads"
    )

    before_download = set(
        downloads_folder.glob(
            "*.csv"
        )
    )

    # --------------------------------------------------------
    # EXPORT MENU
    # --------------------------------------------------------

    export_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Export']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        export_button
    )

    time.sleep(1)

    # --------------------------------------------------------
    # EXPORT AS CSV
    # --------------------------------------------------------

    csv_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Export as CSV']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        csv_button
    )

    print(
        "CSV export clicked ✅"
    )

    # --------------------------------------------------------
    # WAIT FOR NEW CSV
    # --------------------------------------------------------

    new_csv = None

    for _ in range(60):

        time.sleep(1)

        current_files = set(
            downloads_folder.glob(
                "*.csv"
            )
        )

        new_files = (
            current_files
            - before_download
        )

        if new_files:

            new_csv = max(
                new_files,
                key=lambda file:
                file.stat().st_mtime
            )

            break

    if new_csv is None:

        raise Exception(
            "CSV download was not detected."
        )

    print(
        "Downloaded:",
        new_csv.name
    )

    # --------------------------------------------------------
    # PROJECT FOLDER
    # --------------------------------------------------------

    price_raw_folder = Path(
        "../../data/raw/prices"
    )

    price_raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    target_file = (
        price_raw_folder
        /
        (
            f"{state_slug}_"
            f"{crop_slug}_"
            f"{year}_"
            f"{month:02d}.csv"
        )
    )

    # Remove old test copy if present
    if target_file.exists():

        target_file.unlink()

    shutil.move(
        str(new_csv),
        str(target_file)
    )

    print(
        "Saved as:",
        target_file.name
    )

    return target_file


# ============================================================
# 4. RESILIENT BULK COLLECTOR
# ============================================================

def collect_price_history_resilient(
    driver,
    state,
    crop,
    start_date,
    end_date
):

    dates = pd.date_range(
        start=start_date,
        end=end_date,
        freq="MS"
    )

    successful = []
    failed = []
    skipped = []

    price_raw_folder = Path(
        "../../data/raw/prices"
    )

    price_raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    for number, date in enumerate(
        dates,
        start=1
    ):

        year = date.year
        month = date.month

        month_name = (
            calendar.month_name[month]
        )

        target_file = (
            price_raw_folder
            /
            (
                f"{state_slug}_"
                f"{crop_slug}_"
                f"{year}_"
                f"{month:02d}.csv"
            )
        )

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"{number}/{len(dates)} → "
            f"{crop} | "
            f"{state} | "
            f"{month_name} {year}"
        )

        # ----------------------------------------------------
        # SKIP ALREADY DOWNLOADED FILE
        # ----------------------------------------------------

        if target_file.exists():

            print(
                "Already downloaded → SKIPPED ✅"
            )

            skipped.append(
                {
                    "Year": year,
                    "Month": month,
                    "Month_Name": month_name
                }
            )

            continue

        month_completed = False

        # Two attempts per month
        for attempt in range(
            1,
            3
        ):

            print(
                f"Attempt {attempt}/2"
            )

            try:

                downloaded_file = (
                    collect_one_month(
                        driver=driver,
                        state=state,
                        crop=crop,
                        year=year,
                        month=month
                    )
                )

                successful.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name,
                        "File": downloaded_file.name
                    }
                )

                print(
                    f"{month_name} {year} "
                    "completed ✅"
                )

                month_completed = True

                break

            # ------------------------------------------------
            # SESSION DIED
            # ------------------------------------------------

            except InvalidSessionIdException:

                print(
                    "Selenium session died ⚠️"
                )

                try:
                    driver.quit()
                except:
                    pass

                driver = (
                    create_new_driver()
                )

                print(
                    "Retrying month..."
                )

            # ------------------------------------------------
            # OTHER BROWSER ERROR
            # ------------------------------------------------

            except WebDriverException as error:

                error_text = (
                    str(error)
                    .lower()
                )

                print(
                    "WebDriver problem:"
                )

                print(
                    str(error)[:300]
                )

                if any(
                    text in error_text
                    for text in [
                        "invalid session id",
                        "disconnected",
                        "chrome not reachable",
                        "no such window"
                    ]
                ):

                    try:
                        driver.quit()
                    except:
                        pass

                    driver = (
                        create_new_driver()
                    )

                else:

                    time.sleep(5)

            # ------------------------------------------------
            # OTHER ERROR
            # ------------------------------------------------

            except Exception as error:

                print(
                    "Attempt failed:"
                )

                print(
                    str(error)[:300]
                )

                time.sleep(5)

        # ----------------------------------------------------
        # BOTH ATTEMPTS FAILED
        # ----------------------------------------------------

        if not month_completed:

            failed.append(
                {
                    "Year": year,
                    "Month": month,
                    "Month_Name": month_name
                }
            )

            print(
                f"{month_name} {year} "
                "FAILED ❌"
            )

        # Be gentle with AGMARKNET
        time.sleep(3)

    return (
        driver,
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped)
    )


print(
    "AGMARKNET Selenium functions restored ✅"
)

AGMARKNET Selenium functions restored ✅


In [406]:
print(
    "collect_one_month:",
    "collect_one_month" in globals()
)

print(
    "collect_price_history_resilient:",
    "collect_price_history_resilient" in globals()
)

collect_one_month: True
collect_price_history_resilient: True


In [407]:
driver = create_new_driver()


Starting new Chrome session...
Chrome session ready ✅


In [408]:
maize_test_file = collect_one_month(
    driver=driver,
    state="Uttar Pradesh",
    crop="Maize",
    year=2021,
    month=1
)


Collecting: Maize | Uttar Pradesh | January 2021
Select Commodity → Maize selected
State/UT → Uttar Pradesh selected
Year → 2021 selected
Month → January selected
All filters selected ✅
Result page loaded ✅
CSV export clicked ✅
Downloaded: Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_29-09-2026_12-07-06_PM.csv
Saved as: uttar_pradesh_maize_2021_01.csv


In [409]:
# Downloading full up maize history 

driver, up_maize_success, up_maize_failed, up_maize_skipped = (
    collect_price_history_resilient(
        driver=driver,
        state="Uttar Pradesh",
        crop="Maize",
        start_date="2021-01-01",
        end_date="2025-06-01"
    )
)


1/54 → Maize | Uttar Pradesh | January 2021
Already downloaded → SKIPPED ✅

2/54 → Maize | Uttar Pradesh | February 2021
Already downloaded → SKIPPED ✅

3/54 → Maize | Uttar Pradesh | March 2021
Already downloaded → SKIPPED ✅

4/54 → Maize | Uttar Pradesh | April 2021
Already downloaded → SKIPPED ✅

5/54 → Maize | Uttar Pradesh | May 2021
Already downloaded → SKIPPED ✅

6/54 → Maize | Uttar Pradesh | June 2021
Already downloaded → SKIPPED ✅

7/54 → Maize | Uttar Pradesh | July 2021
Already downloaded → SKIPPED ✅

8/54 → Maize | Uttar Pradesh | August 2021
Already downloaded → SKIPPED ✅

9/54 → Maize | Uttar Pradesh | September 2021
Already downloaded → SKIPPED ✅

10/54 → Maize | Uttar Pradesh | October 2021
Already downloaded → SKIPPED ✅

11/54 → Maize | Uttar Pradesh | November 2021
Already downloaded → SKIPPED ✅

12/54 → Maize | Uttar Pradesh | December 2021
Already downloaded → SKIPPED ✅

13/54 → Maize | Uttar Pradesh | January 2022
Already downloaded → SKIPPED ✅

14/54 → Maize | U

In [410]:
print(
    "Successful:",
    len(up_maize_success)
)

print(
    "Failed:",
    len(up_maize_failed)
)

print(
    "Skipped:",
    len(up_maize_skipped)
)

Successful: 0
Failed: 0
Skipped: 54


In [411]:
maize_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "uttar_pradesh_maize_*.csv"
    )
)

print(
    "Total UP Maize files:",
    len(maize_files)
)

Total UP Maize files: 54


In [412]:
up_maize_monthly = load_price_history(
    state="Uttar Pradesh",
    crop="Maize"
)

Uttar Pradesh | Maize
Raw monthly files found: 54
Rows: 3604
Months: 54
Markets: 111


In [413]:
print(
    "Months:",
    up_maize_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Markets:",
    up_maize_monthly[
        "Market"
    ].nunique()
)

print(
    "Rows:",
    len(up_maize_monthly)
)

Months: 54
Markets: 111
Rows: 3604


In [414]:
# Applying Makret -> District mapping

up_market_mapping = pd.read_csv(
    "../../data/processed/"
    "up_market_district_mapping.csv"
)

up_market_mapping["District"] = (
    up_market_mapping[
        "District"
    ]
    .astype("string")
    .str.strip()
)

In [415]:
up_maize_with_district = (
    up_maize_monthly.merge(
        up_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [416]:
print(
    "Rows:",
    len(up_maize_with_district)
)

print(
    "Rows missing District:",
    up_maize_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped Maize markets:",
    up_maize_with_district.loc[
        up_maize_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

Rows: 3604
Rows missing District: 0
Unmapped Maize markets: 0


In [417]:
# Build UP Maize crop-year price features
price_features_up_maize = build_price_features(
    up_maize_with_district
)

In [418]:
price_features_up_maize.head()

,State,District,Crop,Year,Avg_Modal_Price,Min_Monthly_Price,Max_Monthly_Price,Price_Std,Months_Available,Avg_Markets_Reporting,Price_Volatility_Pct,Price_Coverage_Pct,Price_Coverage_Quality
0,Uttar Pradesh,Agra,Maize,2021-22,1742.1880,1528.720,1965.190,104.152596,10,0.833333,5.978264,83.333333,Good
1,Uttar Pradesh,Agra,Maize,2022-23,2243.4300,2011.230,2366.540,131.141766,8,0.800000,5.845592,66.666667,Moderate
2,Uttar Pradesh,Agra,Maize,2023-24,2160.0000,2140.000,2180.000,28.284271,2,0.250000,NaN,16.666667,Low
3,Uttar Pradesh,Agra,Maize,2024-25,2281.5400,2230.820,2332.260,71.728912,2,0.500000,NaN,16.666667,Low
4,Uttar Pradesh,Aligarh,Maize,2021-22,1760.9225,1523.745,1922.315,109.571115,12,3.250000,6.222370,100.000000,Complete


In [419]:
print(
    "Feature rows:",
    len(price_features_up_maize)
)

print(
    "Duplicate district-crop-year rows:",
    price_features_up_maize.duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    ).sum()
)

print(
    "Missing volatility:",
    price_features_up_maize[
        "Price_Volatility_Pct"
    ].isna().sum()
)

print("\nCoverage:")

print(
    price_features_up_maize[
        "Price_Coverage_Quality"
    ].value_counts()
)

Feature rows: 163
Duplicate district-crop-year rows: 0
Missing volatility: 41

Coverage:
Price_Coverage_Quality
Complete    90
Low         41
Good        19
Moderate    13
Name: count, dtype: int64


In [420]:
maize_output = Path(
    "../../data/processed/"
    "up_maize_price_features_2021_2025.csv"
)

price_features_up_maize.to_csv(
    maize_output,
    index=False
)

print(
    "Saved:",
    maize_output
)

Saved: ..\..\data\processed\up_maize_price_features_2021_2025.csv


In [421]:
up_maize_with_district.to_csv(
    "../../data/processed/"
    "up_maize_monthly_prices_mapped_2021_2025.csv",
    index=False
)

print(
    "Mapped UP Maize monthly data saved ✅"
)

Mapped UP Maize monthly data saved ✅


In [422]:
# Openign commodity page fresh 

driver.get(
    "https://agmarknet.gov.in/"
    "marketallwholesalemonthanalysis"
)

time.sleep(3)

print(
    driver.current_url
)

https://agmarknet.gov.in/marketallwholesalemonthanalysis


In [423]:
label = WebDriverWait(
    driver,
    20
).until(
    EC.presence_of_element_located(
        (
            By.XPATH,
            "//label[normalize-space()='Select Commodity']"
        )
    )
)

commodity_trigger = label.find_element(
    By.XPATH,
    "./preceding-sibling::div[1]"
)

driver.execute_script(
    "arguments[0].click();",
    commodity_trigger
)

time.sleep(1)

In [424]:
# Finding all rice-paddy option 

rice_related_options = []

option_spans = driver.find_elements(
    By.XPATH,
    "//div[contains(@class,'cursor-pointer')]//span"
)

for span in option_spans:

    try:

        if not span.is_displayed():
            continue

        text = span.text.strip()

        if (
            "rice" in text.lower()
            or "paddy" in text.lower()
            or "basmati" in text.lower()
        ):

            rice_related_options.append(
                text
            )

    except:
        pass


rice_related_options = sorted(
    set(rice_related_options)
)

print(
    "Rice/Paddy related commodities:"
)

for commodity in rice_related_options:
    print(
        "→",
        commodity
    )

Rice/Paddy related commodities:
→ Basmati Rice
→ Beaten Rice
→ Broken Rice
→ Chakhao(Black Rice)
→ Paddy(Basmati)
→ Paddy(Common)
→ Rice
→ Rice Bean


In [425]:
rice_test_file = collect_one_month(
    driver=driver,
    state="Uttar Pradesh",
    crop="Paddy(Common)",
    year=2021,
    month=1
)


Collecting: Paddy(Common) | Uttar Pradesh | January 2021
Select Commodity → Paddy(Common) selected
State/UT → Uttar Pradesh selected
Year → 2021 selected
Month → January selected
All filters selected ✅
Result page loaded ✅
CSV export clicked ✅
Downloaded: Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_29-09-2026_12-07-34_PM.csv
Saved as: uttar_pradesh_paddy(common)_2021_01.csv


In [426]:
print(rice_test_file)

..\..\data\raw\prices\uttar_pradesh_paddy(common)_2021_01.csv


In [427]:
# Renaming janurary test file 

from pathlib import Path

price_raw_folder = Path(
    "../../data/raw/prices"
)

old_test_file = (
    price_raw_folder
    / "uttar_pradesh_paddy(common)_2021_01.csv"
)

new_test_file = (
    price_raw_folder
    / "uttar_pradesh_rice_2021_01.csv"
)

if old_test_file.exists():

    if new_test_file.exists():
        new_test_file.unlink()

    old_test_file.rename(
        new_test_file
    )

    print(
        "Renamed test file ✅"
    )

else:

    print(
        "Old Paddy(Common) test file not found."
    )

print(
    "Rice January file exists:",
    new_test_file.exists()
)

Renamed test file ✅
Rice January file exists: True


In [428]:
def collect_one_month_with_alias(
    driver,
    state,
    agmarknet_commodity,
    project_crop,
    year,
    month
):

    # --------------------------------------------
    # Download using AGMARKNET's commodity name
    # --------------------------------------------

    downloaded_file = collect_one_month(
        driver=driver,
        state=state,
        crop=agmarknet_commodity,
        year=year,
        month=month
    )

    # --------------------------------------------
    # Create project-friendly filename
    # --------------------------------------------

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    project_crop_slug = (
        project_crop.lower()
        .replace(" ", "_")
    )

    target_file = (
        Path("../../data/raw/prices")
        /
        (
            f"{state_slug}_"
            f"{project_crop_slug}_"
            f"{year}_"
            f"{month:02d}.csv"
        )
    )

    # Remove previous test copy if needed
    if target_file.exists():
        target_file.unlink()

    # Rename:
    # paddy(common) -> rice
    downloaded_file.rename(
        target_file
    )

    print(
        f"Project crop mapping: "
        f"{agmarknet_commodity} → {project_crop}"
    )

    print(
        "Final saved file:",
        target_file.name
    )

    return target_file

In [429]:
def collect_price_history_with_alias(
    driver,
    state,
    agmarknet_commodity,
    project_crop,
    start_date,
    end_date
):

    dates = pd.date_range(
        start=start_date,
        end=end_date,
        freq="MS"
    )

    successful = []
    failed = []
    skipped = []

    raw_folder = Path(
        "../../data/raw/prices"
    )

    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    project_slug = (
        project_crop.lower()
        .replace(" ", "_")
    )

    for number, date in enumerate(
        dates,
        start=1
    ):

        year = date.year
        month = date.month

        month_name = (
            calendar.month_name[month]
        )

        target_file = (
            raw_folder
            /
            (
                f"{state_slug}_"
                f"{project_slug}_"
                f"{year}_"
                f"{month:02d}.csv"
            )
        )

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"{number}/{len(dates)} → "
            f"{project_crop} "
            f"({agmarknet_commodity}) | "
            f"{state} | "
            f"{month_name} {year}"
        )

        # -----------------------------------------
        # Already downloaded
        # -----------------------------------------

        if target_file.exists():

            print(
                "Already downloaded → SKIPPED ✅"
            )

            skipped.append(
                {
                    "Year": year,
                    "Month": month
                }
            )

            continue

        month_completed = False

        # -----------------------------------------
        # Two attempts per month
        # -----------------------------------------

        for attempt in range(1, 3):

            print(
                f"Attempt {attempt}/2"
            )

            try:

                result_file = (
                    collect_one_month_with_alias(
                        driver=driver,
                        state=state,
                        agmarknet_commodity=agmarknet_commodity,
                        project_crop=project_crop,
                        year=year,
                        month=month
                    )
                )

                successful.append(
                    {
                        "Year": year,
                        "Month": month,
                        "File": result_file.name
                    }
                )

                month_completed = True

                print(
                    f"{month_name} {year} completed ✅"
                )

                break

            except InvalidSessionIdException:

                print(
                    "Selenium session died ⚠️"
                )

                try:
                    driver.quit()
                except:
                    pass

                driver = create_new_driver()

            except WebDriverException as error:

                print(
                    "WebDriver error:"
                )

                print(
                    str(error)[:300]
                )

                try:
                    driver.quit()
                except:
                    pass

                driver = create_new_driver()

            except Exception as error:

                print(
                    "Attempt failed:"
                )

                print(
                    str(error)[:300]
                )

                time.sleep(5)

        if not month_completed:

            failed.append(
                {
                    "Year": year,
                    "Month": month
                }
            )

            print(
                f"{month_name} {year} FAILED ❌"
            )

        time.sleep(3)

    return (
        driver,
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped)
    )

In [430]:
driver, up_rice_success, up_rice_failed, up_rice_skipped = (
    collect_price_history_with_alias(
        driver=driver,
        state="Uttar Pradesh",
        agmarknet_commodity="Paddy(Common)",
        project_crop="Rice",
        start_date="2021-01-01",
        end_date="2025-06-01"
    )
)


1/54 → Rice (Paddy(Common)) | Uttar Pradesh | January 2021
Already downloaded → SKIPPED ✅

2/54 → Rice (Paddy(Common)) | Uttar Pradesh | February 2021
Already downloaded → SKIPPED ✅

3/54 → Rice (Paddy(Common)) | Uttar Pradesh | March 2021
Already downloaded → SKIPPED ✅

4/54 → Rice (Paddy(Common)) | Uttar Pradesh | April 2021
Already downloaded → SKIPPED ✅

5/54 → Rice (Paddy(Common)) | Uttar Pradesh | May 2021
Already downloaded → SKIPPED ✅

6/54 → Rice (Paddy(Common)) | Uttar Pradesh | June 2021
Already downloaded → SKIPPED ✅

7/54 → Rice (Paddy(Common)) | Uttar Pradesh | July 2021
Already downloaded → SKIPPED ✅

8/54 → Rice (Paddy(Common)) | Uttar Pradesh | August 2021
Already downloaded → SKIPPED ✅

9/54 → Rice (Paddy(Common)) | Uttar Pradesh | September 2021
Already downloaded → SKIPPED ✅

10/54 → Rice (Paddy(Common)) | Uttar Pradesh | October 2021
Already downloaded → SKIPPED ✅

11/54 → Rice (Paddy(Common)) | Uttar Pradesh | November 2021
Already downloaded → SKIPPED ✅

12/54 →

In [431]:
print(
    "Successful:",
    len(up_rice_success)
)

print(
    "Failed:",
    len(up_rice_failed)
)

print(
    "Skipped:",
    len(up_rice_skipped)
)


Successful: 0
Failed: 0
Skipped: 54


In [432]:
up_rice_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "uttar_pradesh_rice_*.csv"
    )
)

print(
    "Total UP Rice files:",
    len(up_rice_files)
)

Total UP Rice files: 54


In [433]:
# Load all UP Rice monthly files

up_rice_monthly = load_price_history(
    state="Uttar Pradesh",
    crop="Rice"
)

Uttar Pradesh | Rice
Raw monthly files found: 54
Rows: 4724
Months: 54
Markets: 165


In [434]:
print(
    "Months:",
    up_rice_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Markets:",
    up_rice_monthly[
        "Market"
    ].nunique()
)

print(
    "Rows:",
    len(up_rice_monthly)
)

Months: 54
Markets: 165
Rows: 4724


In [435]:
up_market_mapping = pd.read_csv(
    "../../data/processed/"
    "up_market_district_mapping.csv"
)

up_market_mapping["District"] = (
    up_market_mapping[
        "District"
    ]
    .astype("string")
    .str.strip()
)

print(
    "Mapped markets available:",
    up_market_mapping[
        "Market_Key"
    ].nunique()
)

Mapped markets available: 243


In [436]:
up_rice_with_district = (
    up_rice_monthly.merge(
        up_market_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [437]:
print(
    "Rows:",
    len(up_rice_with_district)
)

print(
    "Rows missing District:",
    up_rice_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped Rice markets:",
    up_rice_with_district.loc[
        up_rice_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

Rows: 4724
Rows missing District: 0
Unmapped Rice markets: 0


In [438]:
price_features_up_rice = (
    build_price_features(
        up_rice_with_district
    )
)

In [439]:
print(
    "Feature rows:",
    len(price_features_up_rice)
)

print(
    "Duplicate district-crop-year rows:",
    price_features_up_rice
    .duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    )
    .sum()
)

print(
    "Missing volatility:",
    price_features_up_rice[
        "Price_Volatility_Pct"
    ].isna().sum()
)

Feature rows: 263
Duplicate district-crop-year rows: 0
Missing volatility: 126


In [440]:
print(
    "\nCoverage:"
)

print(
    price_features_up_rice[
        "Price_Coverage_Quality"
    ].value_counts()
)


Coverage:
Price_Coverage_Quality
Low         126
Complete     65
Moderate     43
Good         29
Name: count, dtype: int64


In [441]:
print(
    "\nMonths available:"
)

print(
    price_features_up_rice[
        "Months_Available"
    ]
    .value_counts()
    .sort_index()
)


Months available:
Months_Available
0     16
1     16
2     16
3     10
4     32
5     36
6     14
7     11
8     18
9     13
10    10
11     6
12    65
Name: count, dtype: int64


In [442]:
up_rice_with_district.to_csv(
    "../../data/processed/"
    "up_rice_monthly_prices_mapped_2021_2025.csv",
    index=False
)

In [443]:
price_features_up_rice.to_csv(
    "../../data/processed/"
    "up_rice_price_features_2021_2025.csv",
    index=False
)

print(
    "UP Rice monthly mapped data saved ✅"
)

print(
    "UP Rice price features saved ✅"
)

UP Rice monthly mapped data saved ✅
UP Rice price features saved ✅


In [444]:
#Test Punjab Wheat for one month

driver = create_new_driver()


Starting new Chrome session...
Chrome session ready ✅


In [445]:
punjab_wheat_test = collect_one_month(
    driver=driver,
    state="Punjab",
    crop="Wheat",
    year=2021,
    month=1
)


Collecting: Wheat | Punjab | January 2021
Select Commodity → Wheat selected
State/UT → Punjab selected
Year → 2021 selected


TimeoutException: Message: 


In [ ]:
# Collecting all 54 months 

driver, punjab_wheat_success, punjab_wheat_failed, punjab_wheat_skipped = (
    collect_price_history_resilient(
        driver=driver,
        state="Punjab",
        crop="Wheat",
        start_date="2021-01-01",
        end_date="2025-06-01"
    )
)

In [ ]:
# Go back to a clean filter page
driver.get(
    "https://agmarknet.gov.in/"
    "marketallwholesalemonthanalysis"
)

time.sleep(3)

# Select only up to Year
select_exact_dropdown_option(
    driver,
    "Select Commodity",
    "Wheat"
)

select_exact_dropdown_option(
    driver,
    "State/UT",
    "Punjab"
)

select_exact_dropdown_option(
    driver,
    "Year",
    "2021"
)

print("Commodity, State and Year selected ✅")

In [ ]:
visible_month_options = []

for span in driver.find_elements(
    By.XPATH,
    "//div[contains(@class,'cursor-pointer')]//span"
):

    try:

        if span.is_displayed():

            text = span.text.strip()

            if text:
                visible_month_options.append(text)

    except:
        pass


print("Visible options:")

for item in visible_month_options:
    print("→", repr(item))

In [ ]:
print(
    "January exact matches:",
    len(
        [
            x
            for x in visible_month_options
            if x.strip().lower() == "january"
        ]
    )
)

In [ ]:
# Find Month dropdown again
month_label = driver.find_element(
    By.XPATH,
    "//label[normalize-space()='Month']"
)

month_trigger = month_label.find_element(
    By.XPATH,
    "./preceding-sibling::div[1]"
)

print("tabindex:", month_trigger.get_attribute("tabindex"))
print("class:", month_trigger.get_attribute("class"))
print("text:", repr(month_trigger.text))

In [446]:
driver.execute_script(
    "arguments[0].click();",
    month_trigger
)

time.sleep(2)

NoSuchElementException: Message: no such element: element not found
  (Session info: chrome=154.0.8037.57); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff65f3abe05+5a55]
	chromedriver!(No symbol) [0x7ff65f2d3230]
	chromedriver!(No symbol) [0x7ff65f0f6e4d]
	chromedriver!(No symbol) [0x7ff65f10e7ad]
	chromedriver!(No symbol) [0x7ff65f10d210]
	chromedriver!(No symbol) [0x7ff65f0ffb95]
	chromedriver!(No symbol) [0x7ff65f0ffcf5]
	chromedriver!(No symbol) [0x7ff65f0fd97e]
	chromedriver!(No symbol) [0x7ff65f1021f8]
	chromedriver!(No symbol) [0x7ff65f1a38b8]
	chromedriver!(No symbol) [0x7ff65f17ba6a]
	chromedriver!(No symbol) [0x7ff65f1a267a]
	chromedriver!(No symbol) [0x7ff65f144412]
	chromedriver!(No symbol) [0x7ff65f145223]
	chromedriver!GetHandleVerifier [0x7ff65f7bb1d1+414e21]
	chromedriver!GetHandleVerifier [0x7ff65f7e51fb+43ee4b]
	chromedriver

In [3]:
month_names = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

for month in month_names:

    matches = driver.find_elements(
        By.XPATH,
        f"//*[normalize-space()='{month}']"
    )

    visible = [
        element
        for element in matches
        if element.is_displayed()
    ]

    print(
        month,
        "→",
        len(visible)
    )

NameError: name 'driver' is not defined

In [4]:
page_source = driver.page_source

for month in month_names:
    print(
        month,
        "in HTML →",
        month in page_source
    )

NameError: name 'driver' is not defined

In [5]:
month_parent = month_label.find_element(
    By.XPATH,
    ".."
)

print(
    month_parent.get_attribute(
        "outerHTML"
    )[:5000]
)

NameError: name 'month_label' is not defined

In [6]:
april_matches = [
    element
    for element in driver.find_elements(
        By.XPATH,
        "//*[normalize-space()='April']"
    )
    if element.is_displayed()
]

print(
    "Visible April elements:",
    len(april_matches)
)

for i, element in enumerate(april_matches):

    print(
        "\nINDEX:", i,
        "\nTAG:", element.tag_name,
        "\nTEXT:", repr(element.text),
        "\nCLASS:", element.get_attribute("class"),
        "\nHTML:",
        element.get_attribute("outerHTML")[:1000]
    )

NameError: name 'driver' is not defined

In [7]:
april_option = min(
    april_matches,
    key=lambda e: len(
        e.get_attribute("outerHTML")
    )
)

driver.execute_script(
    "arguments[0].click();",
    april_option
)

time.sleep(1)

NameError: name 'april_matches' is not defined

In [8]:
month_label = driver.find_element(
    By.XPATH,
    "//label[normalize-space()='Month']"
)

month_trigger = month_label.find_element(
    By.XPATH,
    "./preceding-sibling::div[1]"
)

print(
    "Selected month:",
    repr(month_trigger.text)
)

NameError: name 'driver' is not defined

In [9]:
submit_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='Submit']"
        )
    )
)

submit_button.click()

time.sleep(5)

print(
    "URL:",
    driver.current_url
)

NameError: name 'WebDriverWait' is not defined

In [10]:
body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

print(
    body_text[:3000]
)

NameError: name 'driver' is not defined

In [11]:
# Function to discover available months 

def get_available_months(
    driver,
    state,
    crop,
    year
):

    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )

    driver.get(filter_url)

    time.sleep(3)

    # Select Commodity
    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    # Select State
    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    # Select Year
    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )

    # Find Month dropdown
    month_label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//label[normalize-space()='Month']"
            )
        )
    )

    month_trigger = month_label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    # Open Month dropdown
    driver.execute_script(
        "arguments[0].click();",
        month_trigger
    )

    time.sleep(2)

    month_names = [
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December"
    ]

    available = []

    for month_number, month_name in enumerate(
        month_names,
        start=1
    ):

        matches = driver.find_elements(
            By.XPATH,
            f"//*[normalize-space()='{month_name}']"
        )

        visible_matches = [
            element
            for element in matches
            if element.is_displayed()
        ]

        if visible_matches:

            available.append(
                {
                    "Month": month_number,
                    "Month_Name": month_name
                }
            )

    return pd.DataFrame(
        available
    )



In [12]:
punjab_2021_months = get_available_months(
    driver=driver,
    state="Punjab",
    crop="Wheat",
    year=2021
)

punjab_2021_months

NameError: name 'driver' is not defined

In [13]:
for year in [
    2021,
    2022,
    2023,
    2024,
    2025
]:

    months = get_available_months(
        driver=driver,
        state="Punjab",
        crop="Wheat",
        year=year
    )

    print(
        "\nYEAR:",
        year
    )

    print(
        months[
            "Month_Name"
        ].tolist()
    )

NameError: name 'driver' is not defined

In [14]:
# Creating adaptive punjab collector 

def collect_price_history_adaptive(
    driver,
    state,
    crop,
    start_year=2021,
    end_year=2025,
    final_month_2025=6
):

    successful = []
    failed = []
    skipped = []
    unavailable = []

    raw_folder = Path(
        "../../data/raw/prices"
    )

    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    for year in range(
        start_year,
        end_year + 1
    ):

        print(
            "\n"
            + "#" * 70
        )

        print(
            f"Checking available months "
            f"for {state} | {crop} | {year}"
        )

        # -----------------------------------------
        # Discover available months from AGMARKNET
        # -----------------------------------------

        available_df = get_available_months(
            driver=driver,
            state=state,
            crop=crop,
            year=year
        )

        available_months = set(
            available_df[
                "Month"
            ].tolist()
        )

        # For 2025 only Jan-Jun matter
        if year == 2025:

            desired_months = range(
                1,
                final_month_2025 + 1
            )

        else:

            desired_months = range(
                1,
                13
            )

        # -----------------------------------------
        # Loop through project months
        # -----------------------------------------

        for month in desired_months:

            month_name = (
                calendar.month_name[month]
            )

            target_file = (
                raw_folder
                /
                (
                    f"{state_slug}_"
                    f"{crop_slug}_"
                    f"{year}_"
                    f"{month:02d}.csv"
                )
            )

            print(
                "\n",
                year,
                month_name
            )

            # -------------------------------------
            # Already downloaded
            # -------------------------------------

            if target_file.exists():

                print(
                    "Already downloaded → SKIPPED ✅"
                )

                skipped.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                continue

            # -------------------------------------
            # AGMARKNET does not offer this month
            # -------------------------------------

            if month not in available_months:

                print(
                    "Not available in AGMARKNET → recorded as unavailable"
                )

                unavailable.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                continue

            # -------------------------------------
            # Month exists → try downloading
            # -------------------------------------

            completed = False

            for attempt in range(1, 3):

                print(
                    f"Attempt {attempt}/2"
                )

                try:

                    file_path = collect_one_month(
                        driver=driver,
                        state=state,
                        crop=crop,
                        year=year,
                        month=month
                    )

                    successful.append(
                        {
                            "Year": year,
                            "Month": month,
                            "Month_Name": month_name,
                            "File": file_path.name
                        }
                    )

                    completed = True

                    print(
                        "Downloaded successfully ✅"
                    )

                    break

                except (
                    InvalidSessionIdException,
                    WebDriverException
                ) as error:

                    print(
                        "Browser problem:"
                    )

                    print(
                        str(error)[:250]
                    )

                    try:
                        driver.quit()
                    except:
                        pass

                    driver = create_new_driver()

                except Exception as error:

                    print(
                        "Download attempt failed:"
                    )

                    print(
                        str(error)[:250]
                    )

                    time.sleep(5)

            if not completed:

                failed.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                print(
                    "Month FAILED ❌"
                )

            time.sleep(3)

    return (
        driver,
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped),
        pd.DataFrame(unavailable)
    )

In [15]:
driver, punjab_wheat_success, punjab_wheat_failed, punjab_wheat_skipped, punjab_wheat_unavailable = (
    collect_price_history_adaptive(
        driver=driver,
        state="Punjab",
        crop="Wheat"
    )
)

NameError: name 'driver' is not defined

In [16]:
print(
    "Downloaded this run:",
    len(punjab_wheat_success)
)

print(
    "Failed:",
    len(punjab_wheat_failed)
)

print(
    "Already downloaded:",
    len(punjab_wheat_skipped)
)

print(
    "Unavailable from AGMARKNET:",
    len(punjab_wheat_unavailable)
)

NameError: name 'punjab_wheat_success' is not defined

In [17]:
punjab_wheat_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "punjab_wheat_*.csv"
    )
)

print(
    "Punjab Wheat files on disk:",
    len(punjab_wheat_files)
)

NameError: name 'Path' is not defined

In [18]:
# Load all available Punjab Wheat files
punjab_wheat_monthly = load_price_history(
    state="Punjab",
    crop="Wheat"
)

NameError: name 'load_price_history' is not defined

In [19]:
print(
    "Months actually available:",
    punjab_wheat_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "First month:",
    punjab_wheat_monthly[
        "Month_Date"
    ].min()
)

print(
    "Last month:",
    punjab_wheat_monthly[
        "Month_Date"
    ].max()
)

print(
    "Unique markets:",
    punjab_wheat_monthly[
        "Market"
    ].nunique()
)

print(
    "Total rows:",
    len(punjab_wheat_monthly)
)

NameError: name 'punjab_wheat_monthly' is not defined

In [20]:
available_months = sorted(
    punjab_wheat_monthly[
        "Month_Date"
    ].dropna().unique()
)

for month in available_months:
    print(
        pd.to_datetime(month)
        .strftime("%B %Y")
    )

NameError: name 'punjab_wheat_monthly' is not defined

In [21]:
# Build Punjab Market → District mapping

from pathlib import Path

search_folders = [
    Path("../../data/raw"),
    Path("../../data/processed")
]

reference_candidates = []

for folder in search_folders:

    if not folder.exists():
        continue

    for file_path in folder.rglob("*.csv"):

        try:

            sample = pd.read_csv(
                file_path,
                nrows=5
            )

            columns = {
                str(col)
                .strip()
                .lower()
                for col in sample.columns
            }

            if {
                "state",
                "district",
                "market"
            }.issubset(columns):

                reference_candidates.append(
                    file_path
                )

        except:
            pass


print(
    "Reference files found:",
    len(reference_candidates)
)

for file in reference_candidates:
    print(file)

Reference files found: 0


In [22]:
punjab_mapping_parts = []

for file_path in reference_candidates:

    try:

        df = pd.read_csv(
            file_path
        )

        column_lookup = {
            str(col).strip().lower(): col
            for col in df.columns
        }

        temp = df[
            [
                column_lookup["state"],
                column_lookup["district"],
                column_lookup["market"]
            ]
        ].copy()

        temp.columns = [
            "State",
            "District",
            "Market"
        ]

        temp = temp[
            temp["State"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "punjab"
        ].copy()

        temp["Market_Key"] = (
            temp["Market"]
            .apply(
                normalize_market_name
            )
        )

        punjab_mapping_parts.append(
            temp[
                [
                    "Market_Key",
                    "District"
                ]
            ]
        )

    except:
        pass

In [23]:
punjab_official_mapping = pd.concat(
    punjab_mapping_parts,
    ignore_index=True
)

punjab_official_mapping["District"] = (
    punjab_official_mapping[
        "District"
    ]
    .astype("string")
    .str.strip()
)

punjab_official_mapping = (
    punjab_official_mapping
    .dropna()
    .drop_duplicates()
)

NameError: name 'pd' is not defined

In [24]:
punjab_conflicts = (
    punjab_official_mapping
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

punjab_conflicts = (
    punjab_conflicts[
        punjab_conflicts > 1
    ]
)

print(
    "Punjab mapping conflicts:",
    len(punjab_conflicts)
)

NameError: name 'punjab_official_mapping' is not defined

In [25]:
safe_keys = (
    punjab_official_mapping
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

safe_keys = (
    safe_keys[
        safe_keys == 1
    ]
    .index
)

punjab_safe_mapping = (
    punjab_official_mapping[
        punjab_official_mapping[
            "Market_Key"
        ].isin(
            safe_keys
        )
    ]
    [
        [
            "Market_Key",
            "District"
        ]
    ]
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

print(
    "Safe Punjab market mappings:",
    len(punjab_safe_mapping)
)

NameError: name 'punjab_official_mapping' is not defined

In [26]:
punjab_wheat_with_district = (
    punjab_wheat_monthly.merge(
        punjab_safe_mapping,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_wheat_monthly' is not defined

In [27]:
print(
    "Total rows:",
    len(punjab_wheat_with_district)
)

print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unique unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [28]:
punjab_unmapped = (
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

print(
    punjab_unmapped.to_string(
        index=False
    )
)

NameError: name 'punjab_wheat_with_district' is not defined

In [29]:
#Reload production data and get Punjab's canonical districts

import pandas as pd
from pathlib import Path

production_file = Path(
    "../../data/processed/production_history_2021_2025.csv"
)

production_history = pd.read_csv(
    production_file
)

production_punjab_districts = sorted(
    production_history.loc[
        production_history["State"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "punjab",
        "District"
    ]
    .dropna()
    .unique()
)

print(
    "Punjab districts in production data:",
    len(production_punjab_districts)
)

print(
    production_punjab_districts
)

Punjab districts in production data: 23
['Amritsar', 'Barnala', 'Bathinda', 'Faridkot', 'Fatehgarh Sahib', 'Fazilka', 'Ferozepur', 'Gurdaspur', 'Hoshiarpur', 'Jalandhar', 'Kapurthala', 'Ludhiana', 'Malerkotla', 'Mansa', 'Moga', 'Pathankot', 'Patiala', 'Rupnagar', 'S.A.S Nagar', 'Sangrur', 'Shahid Bhagat Singh Nagar', 'Sri Muktsar Sahib', 'Tarn Taran']


In [30]:
print(
    "Markets to geocode:",
    len(punjab_unmapped)
)

print(
    punjab_unmapped.head()
)

NameError: name 'punjab_unmapped' is not defined

In [31]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [32]:
geolocator = Nominatim(
    user_agent="crop-yield-risk-dashboard-punjab-market-mapping",
    timeout=10
)

geocode_market = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=True
)

In [33]:
def get_punjab_market_candidate(
    market_name
):

    clean_name = (
        str(market_name)
        .replace("APMC", "")
        .strip()
    )

    query = (
        f"{clean_name}, "
        f"Punjab, India"
    )

    location = geocode_market(
        query,
        addressdetails=True
    )

    if location is None:

        return {
            "Geocode_Found": False,
            "Candidate_District": None,
            "Candidate_Place": None,
            "Full_Address": None,
            "Latitude": None,
            "Longitude": None
        }

    address = location.raw.get(
        "address",
        {}
    )

    candidate_district = (
        address.get("state_district")
        or address.get("county")
        or address.get("district")
        or address.get("city_district")
    )

    candidate_place = (
        address.get("city")
        or address.get("town")
        or address.get("village")
        or address.get("municipality")
        or address.get("suburb")
    )

    return {
        "Geocode_Found": True,
        "Candidate_District": candidate_district,
        "Candidate_Place": candidate_place,
        "Full_Address": location.address,
        "Latitude": location.latitude,
        "Longitude": location.longitude
    }

In [34]:
punjab_geocode_results = []

for number, row in enumerate(
    punjab_unmapped.itertuples(),
    start=1
):

    print(
        f"{number}/{len(punjab_unmapped)} "
        f"→ {row.Market}"
    )

    result = get_punjab_market_candidate(
        row.Market
    )

    punjab_geocode_results.append(
        {
            "Market": row.Market,
            "Market_Key": row.Market_Key,
            **result
        }
    )

NameError: name 'punjab_unmapped' is not defined

In [35]:
punjab_geocoded = pd.DataFrame(
    punjab_geocode_results
)

In [36]:
punjab_geocode_file = Path(
    "../../data/processed/"
    "punjab_unmapped_market_geocode_candidates.csv"
)

punjab_geocoded.to_csv(
    punjab_geocode_file,
    index=False
)

print(
    "Saved:",
    punjab_geocode_file
)

Saved: ..\..\data\processed\punjab_unmapped_market_geocode_candidates.csv


In [37]:
print(
    "Total markets:",
    len(punjab_geocoded)
)

print(
    "Locations found:",
    punjab_geocoded[
        "Geocode_Found"
    ].sum()
)

print(
    "Locations not found:",
    (
        ~punjab_geocoded[
            "Geocode_Found"
        ]
    ).sum()
)

print(
    "Candidate districts available:",
    punjab_geocoded[
        "Candidate_District"
    ].notna().sum()
)

Total markets: 0


KeyError: 'Geocode_Found'

In [ ]:
punjab_geocoded_found = (
    punjab_geocoded[
        punjab_geocoded[
            "Candidate_District"
        ].notna()
    ]
    .copy()
)

punjab_geocoded_found[
    "Exact_Production_Match"
] = (
    punjab_geocoded_found[
        "Candidate_District"
    ].isin(
        production_punjab_districts
    )
)

In [ ]:
print(
    "Exact Punjab district matches:",
    punjab_geocoded_found[
        "Exact_Production_Match"
    ].sum()
)

print(
    "District names needing review:",
    (
        ~punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ).sum()
)

In [38]:
# Build mapping from only the 77 exact matches
punjab_exact_geocode_mapping = (
    punjab_geocoded_found[
        punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market_Key",
            "Candidate_District"
        ]
    ]
    .rename(
        columns={
            "Candidate_District": "District"
        }
    )
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

print(
    "Exact geocoder mappings:",
    len(punjab_exact_geocode_mapping)
)

NameError: name 'punjab_geocoded_found' is not defined

In [39]:

import pandas as pd
from pathlib import Path


# ============================================================
# 1. RELOAD SAVED PUNJAB GEOCODING RESULTS
# ============================================================

punjab_geocode_file = Path(
    "../../data/processed/"
    "punjab_unmapped_market_geocode_candidates.csv"
)

print(
    "Geocode file exists:",
    punjab_geocode_file.exists()
)

punjab_geocoded = pd.read_csv(
    punjab_geocode_file
)

print(
    "Punjab geocoded markets loaded:",
    len(punjab_geocoded)
)


# ============================================================
# 2. RELOAD PRODUCTION DATA IF NEEDED
# ============================================================

if "production_history" not in globals():

    production_history = pd.read_csv(
        "../../data/processed/"
        "production_history_2021_2025.csv"
    )


# ============================================================
# 3. RECREATE PUNJAB CANONICAL DISTRICT LIST
# ============================================================

production_punjab_districts = sorted(
    production_history.loc[
        production_history["State"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "punjab",
        "District"
    ]
    .dropna()
    .unique()
)

print(
    "Punjab production districts:",
    len(production_punjab_districts)
)


# ============================================================
# 4. RECREATE THE 85 FOUND GEOCODE RESULTS
# ============================================================

punjab_geocoded_found = (
    punjab_geocoded[
        punjab_geocoded[
            "Candidate_District"
        ].notna()
    ]
    .copy()
)


# ============================================================
# 5. RECREATE EXACT MATCH CHECK
# ============================================================

punjab_geocoded_found[
    "Exact_Production_Match"
] = (
    punjab_geocoded_found[
        "Candidate_District"
    ].isin(
        production_punjab_districts
    )
)


# ============================================================
# 6. CHECK THAT WE RECOVERED THE SAME RESULTS
# ============================================================

print(
    "\nLocations found:",
    len(punjab_geocoded_found)
)

print(
    "Exact Punjab district matches:",
    punjab_geocoded_found[
        "Exact_Production_Match"
    ].sum()
)

print(
    "District names needing review:",
    (
        ~punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ).sum()
)

Geocode file exists: True


EmptyDataError: No columns to parse from file

In [ ]:
punjab_exact_geocode_mapping = (
    punjab_geocoded_found[
        punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market_Key",
            "Candidate_District"
        ]
    ]
    .rename(
        columns={
            "Candidate_District": "District"
        }
    )
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

print(
    "Exact geocoder mappings:",
    len(punjab_exact_geocode_mapping)
)

In [ ]:
# ============================================================
# STEP 232 — BUILD SAFE PUNJAB MAPPING + ADD 77 EXACT MATCHES
# ============================================================

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Rebuild official Punjab mapping if needed
# ------------------------------------------------------------

if "punjab_safe_mapping" not in globals():

    print("Rebuilding Punjab official safe mapping...")

    search_folders = [
        Path("../../data/raw"),
        Path("../../data/processed")
    ]

    reference_candidates = []

    for folder in search_folders:

        if not folder.exists():
            continue

        for file_path in folder.rglob("*.csv"):

            try:

                sample = pd.read_csv(
                    file_path,
                    nrows=5
                )

                columns = {
                    str(col).strip().lower()
                    for col in sample.columns
                }

                if {
                    "state",
                    "district",
                    "market"
                }.issubset(columns):

                    reference_candidates.append(
                        file_path
                    )

            except:
                pass


    punjab_mapping_parts = []

    for file_path in reference_candidates:

        try:

            df = pd.read_csv(
                file_path
            )

            lookup = {
                str(col).strip().lower(): col
                for col in df.columns
            }

            temp = df[
                [
                    lookup["state"],
                    lookup["district"],
                    lookup["market"]
                ]
            ].copy()

            temp.columns = [
                "State",
                "District",
                "Market"
            ]

            temp = temp[
                temp["State"]
                .astype(str)
                .str.strip()
                .str.lower()
                == "punjab"
            ].copy()

            temp["Market_Key"] = (
                temp["Market"]
                .apply(
                    normalize_market_name
                )
            )

            punjab_mapping_parts.append(
                temp[
                    [
                        "Market_Key",
                        "District"
                    ]
                ]
            )

        except:
            pass


    punjab_official_mapping = pd.concat(
        punjab_mapping_parts,
        ignore_index=True
    )

    punjab_official_mapping["District"] = (
        punjab_official_mapping[
            "District"
        ]
        .astype("string")
        .str.strip()
    )

    punjab_official_mapping = (
        punjab_official_mapping
        .dropna()
        .drop_duplicates()
    )


    # Keep only market names that point to exactly one district
    safe_key_counts = (
        punjab_official_mapping
        .groupby(
            "Market_Key"
        )["District"]
        .nunique()
    )

    safe_keys = (
        safe_key_counts[
            safe_key_counts == 1
        ]
        .index
    )

    punjab_safe_mapping = (
        punjab_official_mapping[
            punjab_official_mapping[
                "Market_Key"
            ].isin(
                safe_keys
            )
        ]
        [
            [
                "Market_Key",
                "District"
            ]
        ]
        .drop_duplicates(
            subset=["Market_Key"]
        )
    )


print(
    "Official safe mappings:",
    len(punjab_safe_mapping)
)


# ------------------------------------------------------------
# 2. Recreate the 77 exact geocoder mappings
# ------------------------------------------------------------

punjab_exact_geocode_mapping = (
    punjab_geocoded_found[
        punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market_Key",
            "Candidate_District"
        ]
    ]
    .rename(
        columns={
            "Candidate_District":
            "District"
        }
    )
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

print(
    "Exact geocoder mappings:",
    len(punjab_exact_geocode_mapping)
)


# ------------------------------------------------------------
# 3. Add only NEW mappings
# ------------------------------------------------------------

existing_keys = set(
    punjab_safe_mapping[
        "Market_Key"
    ]
)

new_exact_mapping = (
    punjab_exact_geocode_mapping[
        ~punjab_exact_geocode_mapping[
            "Market_Key"
        ].isin(
            existing_keys
        )
    ]
    .copy()
)

print(
    "New exact mappings:",
    len(new_exact_mapping)
)


# ------------------------------------------------------------
# 4. Combine
# ------------------------------------------------------------

punjab_market_mapping_v2 = pd.concat(
    [
        punjab_safe_mapping,
        new_exact_mapping
    ],
    ignore_index=True
)

punjab_market_mapping_v2 = (
    punjab_market_mapping_v2
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)


# ------------------------------------------------------------
# 5. Validate conflicts
# ------------------------------------------------------------

mapping_check = (
    punjab_market_mapping_v2
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Punjab mapped markets:",
    punjab_market_mapping_v2[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (
        mapping_check > 1
    ).sum()
)

In [40]:
import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. MARKET NAME NORMALIZER
# ============================================================

def normalize_market_name(name):

    name = str(name).strip().lower()

    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


# ============================================================
# 2. TRY TO RECOVER THE OLD SAFE PUNJAB MAPPING
# ============================================================

if "punjab_safe_mapping" in globals():

    print("Existing Punjab safe mapping found ✅")


elif (
    "punjab_wheat_with_district" in globals()
    and "District" in punjab_wheat_with_district.columns
):

    print(
        "Recovering safe mapping from "
        "punjab_wheat_with_district..."
    )

    punjab_safe_mapping = (
        punjab_wheat_with_district[
            [
                "Market_Key",
                "District"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )

    # Keep only markets having one district
    counts = (
        punjab_safe_mapping
        .groupby("Market_Key")["District"]
        .nunique()
    )

    safe_keys = counts[
        counts == 1
    ].index

    punjab_safe_mapping = (
        punjab_safe_mapping[
            punjab_safe_mapping[
                "Market_Key"
            ].isin(safe_keys)
        ]
        .drop_duplicates(
            subset=["Market_Key"]
        )
    )

    print(
        "Recovered mapping ✅"
    )


else:

    print(
        "Old mapping variable not available."
    )

    print(
        "Searching CSV files..."
    )

    # Also search Downloads because the original
    # AGMARKNET reference may still be there.
    search_folders = [
        Path("../../data/raw"),
        Path("../../data/processed"),
        Path.home() / "Downloads"
    ]

    punjab_mapping_parts = []

    for folder in search_folders:

        if not folder.exists():
            continue

        for file_path in folder.rglob("*.csv"):

            try:

                sample = pd.read_csv(
                    file_path,
                    nrows=10
                )

                lookup = {
                    str(col).strip().lower(): col
                    for col in sample.columns
                }

                required = {
                    "state",
                    "district",
                    "market"
                }

                if not required.issubset(
                    lookup.keys()
                ):
                    continue

                df = pd.read_csv(
                    file_path
                )

                temp = df[
                    [
                        lookup["state"],
                        lookup["district"],
                        lookup["market"]
                    ]
                ].copy()

                temp.columns = [
                    "State",
                    "District",
                    "Market"
                ]

                temp = temp[
                    temp["State"]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    == "punjab"
                ].copy()

                # IMPORTANT:
                # Do not append empty Punjab tables
                if temp.empty:
                    continue

                temp["Market_Key"] = (
                    temp["Market"]
                    .apply(
                        normalize_market_name
                    )
                )

                punjab_mapping_parts.append(
                    temp[
                        [
                            "Market_Key",
                            "District"
                        ]
                    ]
                )

            except:
                pass


    # Prevent "No objects to concatenate"
    if len(punjab_mapping_parts) == 0:

        print(
            "No Punjab official reference rows "
            "were found."
        )

        punjab_safe_mapping = pd.DataFrame(
            columns=[
                "Market_Key",
                "District"
            ]
        )

    else:

        punjab_official_mapping = pd.concat(
            punjab_mapping_parts,
            ignore_index=True
        )

        punjab_official_mapping["District"] = (
            punjab_official_mapping[
                "District"
            ]
            .astype("string")
            .str.strip()
        )

        punjab_official_mapping = (
            punjab_official_mapping
            .dropna()
            .drop_duplicates()
        )

        counts = (
            punjab_official_mapping
            .groupby(
                "Market_Key"
            )["District"]
            .nunique()
        )

        safe_keys = counts[
            counts == 1
        ].index

        punjab_safe_mapping = (
            punjab_official_mapping[
                punjab_official_mapping[
                    "Market_Key"
                ].isin(safe_keys)
            ]
            [
                [
                    "Market_Key",
                    "District"
                ]
            ]
            .drop_duplicates(
                subset=["Market_Key"]
            )
        )


print(
    "\nSafe Punjab mappings recovered:",
    len(punjab_safe_mapping)
)

Old mapping variable not available.
Searching CSV files...

Safe Punjab mappings recovered: 188


In [41]:
punjab_geocode_file = Path(
    "../../data/processed/"
    "punjab_unmapped_market_geocode_candidates.csv"
)

punjab_geocoded = pd.read_csv(
    punjab_geocode_file
)

production_history = pd.read_csv(
    "../../data/processed/"
    "production_history_2021_2025.csv"
)

production_punjab_districts = sorted(
    production_history.loc[
        production_history["State"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "punjab",
        "District"
    ]
    .dropna()
    .unique()
)

punjab_geocoded_found = (
    punjab_geocoded[
        punjab_geocoded[
            "Candidate_District"
        ].notna()
    ]
    .copy()
)

punjab_geocoded_found[
    "Exact_Production_Match"
] = (
    punjab_geocoded_found[
        "Candidate_District"
    ].isin(
        production_punjab_districts
    )
)

print(
    "Locations found:",
    len(punjab_geocoded_found)
)

print(
    "Exact matches:",
    punjab_geocoded_found[
        "Exact_Production_Match"
    ].sum()
)

print(
    "Name mismatches:",
    (
        ~punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ).sum()
)

EmptyDataError: No columns to parse from file

In [ ]:
# ============================================================
# STEP 234 — COMBINE SAFE OFFICIAL + EXACT GEOCODER MAPPINGS
# ============================================================

# 1. Build exact geocoder mapping
punjab_exact_geocode_mapping = (
    punjab_geocoded_found[
        punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market_Key",
            "Candidate_District"
        ]
    ]
    .rename(
        columns={
            "Candidate_District": "District"
        }
    )
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

print(
    "Exact geocoder mappings:",
    len(punjab_exact_geocode_mapping)
)


# 2. Add only markets not already in safe official mapping
existing_keys = set(
    punjab_safe_mapping[
        "Market_Key"
    ]
)

punjab_new_exact_mapping = (
    punjab_exact_geocode_mapping[
        ~punjab_exact_geocode_mapping[
            "Market_Key"
        ].isin(existing_keys)
    ]
    .copy()
)

print(
    "New exact mappings to add:",
    len(punjab_new_exact_mapping)
)


# 3. Combine them
punjab_market_mapping_v2 = pd.concat(
    [
        punjab_safe_mapping[
            [
                "Market_Key",
                "District"
            ]
        ],
        punjab_new_exact_mapping[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)


# 4. Keep one row per market
punjab_market_mapping_v2 = (
    punjab_market_mapping_v2
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)


# 5. Check for conflicts
check = (
    punjab_market_mapping_v2
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Punjab mapped markets:",
    punjab_market_mapping_v2[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (check > 1).sum()
)

In [ ]:
if "punjab_wheat_monthly" not in globals():

    punjab_wheat_monthly = load_price_history(
        state="Punjab",
        crop="Wheat"
    )

In [ ]:
# ============================================================
# RECOVER PRICE LOADING FUNCTIONS
# ============================================================

import pandas as pd
import re
from pathlib import Path


# ------------------------------------------------------------
# 1. NORMALIZE MARKET NAMES
# ------------------------------------------------------------

def normalize_market_name(name):

    name = str(name).strip().lower()

    # Remove APMC
    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    # Remove extra spaces
    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


# ------------------------------------------------------------
# 2. CLEAN ONE AGMARKNET MONTHLY CSV
# ------------------------------------------------------------

def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    # First row = report title
    # Second row = actual column names
    df = pd.read_csv(
        file_path,
        header=1
    )

    # Second column contains the selected month's price
    current_price_column = df.columns[1]

    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    df = df.rename(
        columns={
            current_price_column:
            "Monthly_Price"
        }
    )

    # Clean market text
    df["Market"] = (
        df["Market"]
        .astype(str)
        .str.strip()
    )

    # Keep only actual APMC market rows
    # This removes Average, Note, formulas, etc.
    df = df[
        df["Market"]
        .str.lower()
        .str.endswith("apmc")
    ].copy()

    # "-" and invalid values become NaN
    df["Monthly_Price"] = pd.to_numeric(
        df["Monthly_Price"],
        errors="coerce"
    )

    # Add metadata
    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    return df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]


# ------------------------------------------------------------
# 3. LOAD ALL MONTHLY FILES FOR ONE STATE + CROP
# ------------------------------------------------------------

def load_price_history(
    state,
    crop,
    raw_folder="../../data/raw/prices"
):

    raw_folder = Path(
        raw_folder
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    # Example:
    # punjab_wheat_2021_04.csv
    pattern = (
        f"{state_slug}_"
        f"{crop_slug}_"
        f"*.csv"
    )

    files = sorted(
        raw_folder.glob(pattern)
    )

    print(
        f"{state} | {crop}"
    )

    print(
        "Raw monthly files found:",
        len(files)
    )

    if len(files) == 0:

        raise ValueError(
            f"No price CSV files found for "
            f"{state} - {crop}"
        )

    all_months = []

    for file_path in files:

        # Extract year/month from filename
        match = re.search(
            r"_(\d{4})_(\d{2})$",
            file_path.stem
        )

        if not match:

            print(
                "Skipped strange filename:",
                file_path.name
            )

            continue

        year = int(
            match.group(1)
        )

        month = int(
            match.group(2)
        )

        clean_df = clean_monthly_price_csv(
            file_path=file_path,
            state=state,
            crop=crop,
            year=year,
            month=month
        )

        all_months.append(
            clean_df
        )

    if len(all_months) == 0:

        raise ValueError(
            "Files were found, but none could "
            "be converted into monthly data."
        )

    monthly_df = pd.concat(
        all_months,
        ignore_index=True
    )

    # Create standard market key
    monthly_df["Market_Key"] = (
        monthly_df["Market"]
        .apply(
            normalize_market_name
        )
    )

    print(
        "Rows:",
        len(monthly_df)
    )

    print(
        "Months:",
        monthly_df[
            "Month_Date"
        ].nunique()
    )

    print(
        "Markets:",
        monthly_df[
            "Market"
        ].nunique()
    )

    return monthly_df


print(
    "Price loading functions restored ✅"
)

In [ ]:
print(
    "load_price_history exists:",
    "load_price_history" in globals()
)

In [ ]:
punjab_wheat_monthly = load_price_history(
    state="Punjab",
    crop="Wheat"
)

In [ ]:
punjab_wheat_with_district = (
    punjab_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_v2,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

In [ ]:
punjab_name_mismatches = (
    punjab_geocoded_found[
        ~punjab_geocoded_found[
            "Exact_Production_Match"
        ]
    ]
    [
        [
            "Market",
            "Market_Key",
            "Candidate_District",
            "Candidate_Place",
            "Full_Address"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

print(
    "Geocoded but district-name mismatch:",
    len(punjab_name_mismatches)
)

print(
    punjab_name_mismatches.to_string(
        index=False
    )
)

In [ ]:
print(
    "Punjab production districts:"
)

for district in production_punjab_districts:
    print("→", district)

In [ ]:
punjab_geocode_not_found = (
    punjab_geocoded[
        ~punjab_geocoded[
            "Geocode_Found"
        ]
    ]
    [
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
)

print(
    "Markets not found by geocoder:",
    len(punjab_geocode_not_found)
)

In [ ]:
punjab_reviewed_mapping = {
    "banga": "Shahid Bhagat Singh Nagar",

    "guru har sahai": "Ferozepur",
    "mallanwala": "Ferozepur",
    "mamdot": "Ferozepur",
    "talwandi bhai": "Ferozepur",
    "zira": "Ferozepur",

    "kharar": "S.A.S Nagar",
    "kurali": "S.A.S Nagar"
}

In [ ]:
punjab_reviewed_mapping_df = pd.DataFrame(
    list(
        punjab_reviewed_mapping.items()
    ),
    columns=[
        "Market_Key",
        "District"
    ]
)

punjab_reviewed_mapping_df

In [ ]:
print(
    "Reviewed mappings:",
    len(punjab_reviewed_mapping_df)
)

In [ ]:
punjab_market_mapping_v3 = pd.concat(
    [
        punjab_market_mapping_v2[
            [
                "Market_Key",
                "District"
            ]
        ],

        punjab_reviewed_mapping_df
    ],
    ignore_index=True
)

In [ ]:
mapping_check = (
    punjab_market_mapping_v3
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Mapped Punjab markets:",
    punjab_market_mapping_v3[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (
        mapping_check > 1
    ).sum()
)

In [ ]:
punjab_mapping_file = Path(
    "../../data/processed/"
    "punjab_market_district_mapping.csv"
)

punjab_market_mapping_v3.to_csv(
    punjab_mapping_file,
    index=False
)

print(
    "Saved:",
    punjab_mapping_file
)

In [ ]:
punjab_wheat_with_district = (
    punjab_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_v3,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

In [ ]:
print(
    "Total rows:",
    len(punjab_wheat_with_district)
)

print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

In [42]:
# Build 66 unresolved market list

punjab_remaining_unmapped = (
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "Remaining unresolved markets:",
    len(punjab_remaining_unmapped)
)

print(
    punjab_remaining_unmapped.to_string(
        index=False
    )
)

NameError: name 'punjab_wheat_with_district' is not defined

In [43]:
unmapped_frequency = (
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna()
    ]
    .groupby(
        [
            "Market",
            "Market_Key"
        ],
        as_index=False
    )
    .agg(
        Rows_Affected=(
            "Monthly_Price",
            "size"
        )
    )
    .sort_values(
        "Rows_Affected",
        ascending=False
    )
)

print(
    unmapped_frequency.to_string(
        index=False
    )
)

NameError: name 'punjab_wheat_with_district' is not defined

In [44]:
def geocode_punjab_market_second_pass(
    market_name
):

    clean_name = (
        str(market_name)
        .replace("APMC", "")
        .strip()
    )

    # Different ways the same place may be indexed.
    query_variants = [
        f"{clean_name}, Punjab, India",
        f"{clean_name} Mandi, Punjab, India",
        f"{clean_name} Market, Punjab, India",
        f"{clean_name} Grain Market, Punjab, India"
    ]

    for query in query_variants:

        location = geocode_market(
            query,
            addressdetails=True
        )

        if location is None:
            continue

        address = location.raw.get(
            "address",
            {}
        )

        candidate_district = (
            address.get("state_district")
            or address.get("county")
            or address.get("district")
            or address.get("city_district")
        )

        candidate_place = (
            address.get("city")
            or address.get("town")
            or address.get("village")
            or address.get("municipality")
            or address.get("suburb")
        )

        return {
            "Second_Pass_Found": True,
            "Successful_Query": query,
            "Candidate_District": candidate_district,
            "Candidate_Place": candidate_place,
            "Full_Address": location.address,
            "Latitude": location.latitude,
            "Longitude": location.longitude
        }

    return {
        "Second_Pass_Found": False,
        "Successful_Query": None,
        "Candidate_District": None,
        "Candidate_Place": None,
        "Full_Address": None,
        "Latitude": None,
        "Longitude": None
    }

In [45]:
punjab_second_pass_results = []

for i, row in enumerate(
    punjab_remaining_unmapped.itertuples(),
    start=1
):

    print(
        f"{i}/{len(punjab_remaining_unmapped)} "
        f"→ {row.Market}"
    )

    result = (
        geocode_punjab_market_second_pass(
            row.Market
        )
    )

    punjab_second_pass_results.append(
        {
            "Market": row.Market,
            "Market_Key": row.Market_Key,
            **result
        }
    )

NameError: name 'punjab_remaining_unmapped' is not defined

In [46]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


# Create geocoder again
geolocator = Nominatim(
    user_agent=(
        "crop-yield-risk-dashboard-"
        "punjab-market-mapping"
    ),
    timeout=10
)


# Create the rate-limited geocoding helper
geocode_market = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=True
)


print(
    "geocode_market restored:",
    "geocode_market" in globals()
)

geocode_market restored: True


In [47]:
test_location = geocode_market(
    "Ludhiana, Punjab, India",
    addressdetails=True
)

print(test_location)

Ludhiana, Ludhiana (West) Tahsil, Ludhiana, Punjab, 141001, India


In [48]:
def geocode_punjab_market_second_pass(
    market_name
):

    clean_name = (
        str(market_name)
        .replace("APMC", "")
        .strip()
    )

    query_variants = [
        f"{clean_name}, Punjab, India",
        f"{clean_name} Mandi, Punjab, India",
        f"{clean_name} Market, Punjab, India",
        f"{clean_name} Grain Market, Punjab, India"
    ]

    for query in query_variants:

        location = geocode_market(
            query,
            addressdetails=True
        )

        if location is None:
            continue

        address = location.raw.get(
            "address",
            {}
        )

        candidate_district = (
            address.get("state_district")
            or address.get("county")
            or address.get("district")
            or address.get("city_district")
        )

        candidate_place = (
            address.get("city")
            or address.get("town")
            or address.get("village")
            or address.get("municipality")
            or address.get("suburb")
        )

        return {
            "Second_Pass_Found": True,
            "Successful_Query": query,
            "Candidate_District": candidate_district,
            "Candidate_Place": candidate_place,
            "Full_Address": location.address,
            "Latitude": location.latitude,
            "Longitude": location.longitude
        }

    return {
        "Second_Pass_Found": False,
        "Successful_Query": None,
        "Candidate_District": None,
        "Candidate_Place": None,
        "Full_Address": None,
        "Latitude": None,
        "Longitude": None
    }

In [49]:
punjab_second_pass_results = []

for i, row in enumerate(
    punjab_remaining_unmapped.itertuples(),
    start=1
):

    print(
        f"{i}/{len(punjab_remaining_unmapped)} "
        f"→ {row.Market}"
    )

    result = (
        geocode_punjab_market_second_pass(
            row.Market
        )
    )

    punjab_second_pass_results.append(
        {
            "Market": row.Market,
            "Market_Key": row.Market_Key,
            **result
        }
    )

NameError: name 'punjab_remaining_unmapped' is not defined

In [ ]:
punjab_second_pass = pd.DataFrame(
    punjab_second_pass_results
)

In [ ]:
second_pass_file = Path(
    "../../data/processed/"
    "punjab_market_geocode_second_pass.csv"
)

punjab_second_pass.to_csv(
    second_pass_file,
    index=False
)

print(
    "Saved:",
    second_pass_file
)

In [ ]:
print(
    "Second-pass markets:",
    len(punjab_second_pass)
)

print(
    "Locations found:",
    punjab_second_pass[
        "Second_Pass_Found"
    ].sum()
)

print(
    "Still not found:",
    (
        ~punjab_second_pass[
            "Second_Pass_Found"
        ]
    ).sum()
)

print(
    "Candidate districts:",
    punjab_second_pass[
        "Candidate_District"
    ].notna().sum()
)

In [ ]:
punjab_second_pass[
    "Exact_Production_Match"
] = (
    punjab_second_pass[
        "Candidate_District"
    ].isin(
        production_punjab_districts
    )
)

print(
    "Exact production district matches:",
    punjab_second_pass[
        "Exact_Production_Match"
    ].sum()
)

print(
    "Found but district-name mismatch:",
    (
        punjab_second_pass[
            "Candidate_District"
        ].notna()
        &
        ~punjab_second_pass[
            "Exact_Production_Match"
        ]
    ).sum()
)

In [ ]:
punjab_remaining_unmapped = (
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district["District"].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "Remaining unresolved markets:",
    len(punjab_remaining_unmapped)
)

print(
    punjab_remaining_unmapped.to_string(
        index=False
    )
)

In [ ]:
unresolved_file = Path(
    "../../data/processed/"
    "punjab_wheat_unresolved_66_markets.csv"
)

punjab_remaining_unmapped.to_csv(
    unresolved_file,
    index=False
)

print(
    "Saved:",
    unresolved_file
)

In [ ]:
import requests
from pathlib import Path

reference_folder = Path(
    "../../data/raw/reference"
)

reference_folder.mkdir(
    parents=True,
    exist_ok=True
)

punjab_market_pdf = (
    reference_folder
    / "punjab_market_committees_official.pdf"
)

url = (
    "https://punjab.gov.in/"
    "wp-content/uploads/2023/01/M17.pdf"
)

response = requests.get(
    url,
    timeout=60
)

response.raise_for_status()

punjab_market_pdf.write_bytes(
    response.content
)

print(
    "Saved official Punjab market reference:"
)

print(
    punjab_market_pdf
)

In [ ]:
%pip install pdfplumber

In [ ]:
import pdfplumber

In [ ]:
official_rows = []

with pdfplumber.open(
    punjab_market_pdf
) as pdf:

    print(
        "PDF pages:",
        len(pdf.pages)
    )

    for page_number, page in enumerate(
        pdf.pages,
        start=1
    ):

        tables = page.extract_tables()

        for table in tables:

            for row in table:

                official_rows.append(
                    {
                        "Page": page_number,
                        "Row": row
                    }
                )

print(
    "Extracted table rows:",
    len(official_rows)
)

In [ ]:
from pathlib import Path

punjab_market_pdf = Path(
    "../../data/raw/reference/"
    "punjab_market_committees_official.pdf"
)

print(
    "PDF exists:",
    punjab_market_pdf.exists()
)

print(
    "File size:",
    punjab_market_pdf.stat().st_size
    if punjab_market_pdf.exists()
    else "File not found"
)

In [ ]:
with pdfplumber.open(
    punjab_market_pdf
) as pdf:

    print(
        "Pages:",
        len(pdf.pages)
    )

    print(
        pdf.pages[0].extract_text()[:2000]
    )

In [ ]:
import subprocess
from pathlib import Path

reference_folder = Path(
    "../../data/raw/reference"
)

reference_folder.mkdir(
    parents=True,
    exist_ok=True
)

punjab_market_pdf = (
    reference_folder
    / "punjab_market_committees_official.pdf"
)

url = (
    "https://punjab.gov.in/"
    "wp-content/uploads/2023/01/M17.pdf"
)

result = subprocess.run(
    [
        "curl.exe",
        "-L",
        url,
        "-o",
        str(punjab_market_pdf)
    ],
    capture_output=True,
    text=True
)

print(
    "Return code:",
    result.returncode
)

print(
    "PDF exists:",
    punjab_market_pdf.exists()
)

print(
    "Size:",
    punjab_market_pdf.stat().st_size
    if punjab_market_pdf.exists()
    else 0
)

In [ ]:
import pdfplumber
from pathlib import Path

punjab_market_pdf = Path(
    "../../data/raw/reference/"
    "punjab_market_committees_official.pdf"
)

print(
    "PDF exists:",
    punjab_market_pdf.exists()
)

with pdfplumber.open(
    punjab_market_pdf
) as pdf:

    print(
        "Total pages:",
        len(pdf.pages)
    )

    print(
        "\nFIRST PAGE TEXT SAMPLE:\n"
    )

    text = pdf.pages[0].extract_text()

    if text:
        print(
            text[:3000]
        )
    else:
        print(
            "No readable text found on first page."
        )

In [ ]:
with pdfplumber.open(
    punjab_market_pdf
) as pdf:

    first_page = pdf.pages[0]

    tables = first_page.extract_tables()

    print(
        "Tables found on first page:",
        len(tables)
    )

    for table_number, table in enumerate(
        tables,
        start=1
    ):

        print(
            "\n",
            "=" * 70
        )

        print(
            "TABLE:",
            table_number
        )

        print(
            "=" * 70
        )

        for row in table[:20]:
            print(row)

In [ ]:
import pdfplumber

with pdfplumber.open(punjab_market_pdf) as pdf:

    print("Total pages:", len(pdf.pages))

    matching_pages = []

    for page_number, page in enumerate(
        pdf.pages,
        start=1
    ):
        text = page.extract_text() or ""

        if (
            "Market Committee" in text
            or "MARKET COMMITTEE" in text
            or "Distribution of Market Committees" in text
        ):
            matching_pages.append(page_number)

    print(
        "Pages containing Market Committee:",
        matching_pages
    )

In [ ]:
with pdfplumber.open(punjab_market_pdf) as pdf:

    for page_number in matching_pages:

        print(
            "\n",
            "=" * 80
        )

        print(
            "PAGE:",
            page_number
        )

        print(
            "=" * 80
        )

        text = (
            pdf.pages[
                page_number - 1
            ].extract_text()
            or ""
        )

        print(
            text[:5000]
        )

In [ ]:
target_page_number = matching_pages[0]

with pdfplumber.open(
    punjab_market_pdf
) as pdf:

    target_page = pdf.pages[
        target_page_number - 1
    ]

    tables = target_page.extract_tables()

    print(
        "Tables found:",
        len(tables)
    )

    for table_number, table in enumerate(
        tables,
        start=1
    ):

        print(
            "\nTABLE:",
            table_number
        )

        for row in table[:30]:
            print(row)

In [ ]:
import pandas as pd
import pdfplumber
import re
from pathlib import Path


official_market_rows = []

with pdfplumber.open(
    punjab_market_pdf
) as pdf:

    for page_number, page in enumerate(
        pdf.pages,
        start=1
    ):

        tables = page.extract_tables()

        for table in tables:

            for row in table:

                # We only care about tables with
                # the same 5-column structure
                if row is None or len(row) != 5:
                    continue

                district = row[1]
                market = row[3]

                official_market_rows.append(
                    {
                        "Page": page_number,
                        "District_Raw": district,
                        "Market_Raw": market
                    }
                )


official_market_raw = pd.DataFrame(
    official_market_rows
)

print(
    "Extracted candidate rows:",
    len(official_market_raw)
)

In [ ]:
page_summary = (
    official_market_raw[
        official_market_raw["Page"] >= 86
    ]
    .groupby("Page")
    .size()
)

print(page_summary)

In [ ]:
print(
    official_market_raw[
        official_market_raw["Page"] >= 86
    ]
    .head(100)
    .to_string(index=False)
)

In [ ]:
punjab_official_markets = (
    official_market_raw[
        official_market_raw["Page"] >= 86
    ]
    .copy()
)

In [ ]:
for col in [
    "District_Raw",
    "Market_Raw"
]:

    punjab_official_markets[col] = (
        punjab_official_markets[col]
        .astype("string")
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

In [ ]:
punjab_official_markets[
    "District_Raw"
] = (
    punjab_official_markets[
        "District_Raw"
    ]
    .replace(
        {
            "": pd.NA,
            "None": pd.NA
        }
    )
)

In [ ]:
punjab_official_markets[
    "District_Raw"
] = (
    punjab_official_markets[
        "District_Raw"
    ]
    .ffill()
)

In [ ]:
punjab_official_markets = (
    punjab_official_markets[
        punjab_official_markets[
            "Market_Raw"
        ].notna()
    ]
    .copy()
)

In [ ]:
punjab_official_markets = (
    punjab_official_markets[
        ~punjab_official_markets[
            "Market_Raw"
        ]
        .str.contains(
            "Market Committee",
            case=False,
            na=False
        )
    ]
)

In [ ]:
punjab_official_markets = (
    punjab_official_markets[
        ~punjab_official_markets[
            "Market_Raw"
        ]
        .str.contains(
            "TOTAL",
            case=False,
            na=False
        )
    ]
)

In [ ]:
punjab_official_markets = (
    punjab_official_markets[
        ~punjab_official_markets[
            "Market_Raw"
        ]
        .str.contains(
            "Annexure",
            case=False,
            na=False
        )
    ]
)

In [ ]:
print(
    "Clean official market rows:",
    len(punjab_official_markets)
)

print(
    punjab_official_markets[
        [
            "Page",
            "District_Raw",
            "Market_Raw"
        ]
    ]
    .head(40)
    .to_string(index=False)
)

In [ ]:
district_name_map = {
    "AMRISTAR": "Amritsar",
    "AMRITSAR": "Amritsar",

    "FIROZPUR": "Ferozepur",
    "FEROZEPUR": "Ferozepur",

    "SAS NAGAR": "S.A.S Nagar",
    "S.A.S NAGAR": "S.A.S Nagar",
    "MOHALI": "S.A.S Nagar",

    "SHAHEED BHAGAT SINGH NAGAR":
        "Shahid Bhagat Singh Nagar",

    "SHAHID BHAGAT SINGH NAGAR":
        "Shahid Bhagat Singh Nagar",

    "MUKTSAR": "Sri Muktsar Sahib",
    "SRI MUKTSAR SAHIB": "Sri Muktsar Sahib"
}

In [ ]:
punjab_official_markets[
    "District"
] = (
    punjab_official_markets[
        "District_Raw"
    ]
    .str.upper()
    .map(district_name_map)
)

In [ ]:
canonical_lookup = {
    district.upper(): district
    for district in production_punjab_districts
}

missing_standardization = (
    punjab_official_markets[
        "District"
    ].isna()
)

punjab_official_markets.loc[
    missing_standardization,
    "District"
] = (
    punjab_official_markets.loc[
        missing_standardization,
        "District_Raw"
    ]
    .str.upper()
    .map(canonical_lookup)
)

In [ ]:
unknown_district_names = sorted(
    punjab_official_markets.loc[
        punjab_official_markets[
            "District"
        ].isna(),
        "District_Raw"
    ]
    .dropna()
    .unique()
)

print(
    "Official district names still needing standardization:",
    unknown_district_names
)

In [ ]:
def normalize_market_name(name):

    name = str(name).strip().lower()

    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()

In [ ]:
punjab_official_markets[
    "Market_Key"
] = (
    punjab_official_markets[
        "Market_Raw"
    ]
    .apply(
        normalize_market_name
    )
)

In [ ]:
punjab_official_reference = (
    punjab_official_markets[
        [
            "Market_Key",
            "Market_Raw",
            "District",
            "Page"
        ]
    ]
    .dropna(
        subset=[
            "Market_Key",
            "District"
        ]
    )
    .drop_duplicates()
)

In [ ]:
print(
    "Official Market Committee mappings:",
    punjab_official_reference[
        "Market_Key"
    ].nunique()
)

print(
    punjab_official_reference.head(30)
)

In [ ]:
official_reference_file = Path(
    "../../data/processed/"
    "punjab_official_market_committee_mapping.csv"
)

punjab_official_reference.to_csv(
    official_reference_file,
    index=False
)

print(
    "Saved:",
    official_reference_file
)

In [ ]:
punjab_remaining_unmapped = pd.read_csv(
    "../../data/processed/"
    "punjab_wheat_unresolved_66_markets.csv"
)

print(
    "Remaining markets:",
    len(punjab_remaining_unmapped)
)

In [ ]:
official_exact_matches = (
    punjab_remaining_unmapped
    .merge(
        punjab_official_reference[
            [
                "Market_Key",
                "District"
            ]
        ]
        .drop_duplicates(
            subset=["Market_Key"]
        ),
        on="Market_Key",
        how="left"
    )
)

In [ ]:
print(
    "Exact official matches:",
    official_exact_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    official_exact_matches[
        "District"
    ].isna().sum()
)

In [50]:
print(
    official_exact_matches[
        official_exact_matches[
            "District"
        ].notna()
    ]
    .to_string(index=False)
)

NameError: name 'official_exact_matches' is not defined

In [ ]:
district_name_map.update({
    "ROPAR": "Rupnagar",
    "SRI. MUKTSAR SAHIB": "Sri Muktsar Sahib"
})

In [51]:
punjab_official_markets = (
    punjab_official_markets[
        ~punjab_official_markets[
            "District_Raw"
        ]
        .astype(str)
        .str.upper()
        .str.contains(
            "TOTAL",
            na=False
        )
    ]
    .copy()
)

NameError: name 'punjab_official_markets' is not defined

In [52]:
punjab_official_markets["District"] = (
    punjab_official_markets[
        "District_Raw"
    ]
    .str.upper()
    .map(district_name_map)
)

canonical_lookup = {
    district.upper(): district
    for district in production_punjab_districts
}

missing_mask = (
    punjab_official_markets[
        "District"
    ].isna()
)

punjab_official_markets.loc[
    missing_mask,
    "District"
] = (
    punjab_official_markets.loc[
        missing_mask,
        "District_Raw"
    ]
    .str.upper()
    .map(canonical_lookup)
)

NameError: name 'punjab_official_markets' is not defined

In [53]:
print(
    "Unknown district names:",
    sorted(
        punjab_official_markets.loc[
            punjab_official_markets[
                "District"
            ].isna(),
            "District_Raw"
        ]
        .dropna()
        .unique()
    )
)

NameError: name 'punjab_official_markets' is not defined

In [54]:
def normalize_market_for_matching(name):

    name = str(name).lower().strip()

    # punctuation → spaces
    name = re.sub(
        r"[^a-z0-9\\s]",
        " ",
        name
    )

    # remove generic market terminology
    generic_terms = [
        r"\\bapmc\\b",
        r"\\bmarket committee\\b",
        r"\\bnew grain market\\b",
        r"\\bgrain market\\b",
        r"\\bnew mandi\\b",
        r"\\bmandi\\b",
        r"\\bmarket\\b"
    ]

    for pattern in generic_terms:
        name = re.sub(
            pattern,
            " ",
            name
        )

    name = re.sub(
        r"\\s+",
        " ",
        name
    )

    return name.strip()

In [55]:
punjab_official_reference = (
    punjab_official_markets[
        punjab_official_markets[
            "Market_Raw"
        ].notna()
    ]
    [
        [
            "Market_Raw",
            "District",
            "Page"
        ]
    ]
    .dropna(
        subset=["District"]
    )
    .copy()
)

NameError: name 'punjab_official_markets' is not defined

In [ ]:
punjab_official_reference = (
    punjab_official_reference[
        ~punjab_official_reference[
            "Market_Raw"
        ]
        .astype(str)
        .str.contains(
            "TOTAL",
            case=False,
            na=False
        )
    ]
)

In [56]:
punjab_official_reference[
    "Match_Key"
] = (
    punjab_official_reference[
        "Market_Raw"
    ]
    .apply(
        normalize_market_for_matching
    )
)

NameError: name 'punjab_official_reference' is not defined

In [57]:
punjab_remaining_unmapped[
    "Match_Key"
] = (
    punjab_remaining_unmapped[
        "Market"
    ]
    .apply(
        normalize_market_for_matching
    )
)

NameError: name 'punjab_remaining_unmapped' is not defined

In [58]:
official_match_lookup = (
    punjab_official_reference[
        [
            "Match_Key",
            "District"
        ]
    ]
    .drop_duplicates()
)

NameError: name 'punjab_official_reference' is not defined

In [59]:
match_key_conflicts = (
    official_match_lookup
    .groupby(
        "Match_Key"
    )["District"]
    .nunique()
)

print(
    "Strong-key conflicts:",
    (
        match_key_conflicts > 1
    ).sum()
)

NameError: name 'official_match_lookup' is not defined

In [60]:
strong_exact_matches = (
    punjab_remaining_unmapped
    .merge(
        official_match_lookup,
        on="Match_Key",
        how="left"
    )
)

NameError: name 'punjab_remaining_unmapped' is not defined

In [ ]:
print(
    "Exact matches after stronger normalization:",
    strong_exact_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    strong_exact_matches[
        "District"
    ].isna().sum()
)

In [61]:
print(
    strong_exact_matches[
        strong_exact_matches[
            "District"
        ].notna()
    ]
    [
        [
            "Market",
            "Match_Key",
            "District"
        ]
    ]
    .to_string(
        index=False
    )
)

NameError: name 'strong_exact_matches' is not defined

In [62]:
official_reference_file = Path(
    "../../data/processed/"
    "punjab_official_market_committee_mapping.csv"
)

punjab_official_reference.to_csv(
    official_reference_file,
    index=False
)

print(
    "Saved:",
    official_reference_file
)

NameError: name 'punjab_official_reference' is not defined

In [63]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [64]:
from rapidfuzz import process, fuzz
import pandas as pd


# -----------------------------------------------
# Recover files if variables disappeared
# -----------------------------------------------

if "punjab_remaining_unmapped" not in globals():

    punjab_remaining_unmapped = pd.read_csv(
        "../../data/processed/"
        "punjab_wheat_unresolved_66_markets.csv"
    )


if "punjab_official_reference" not in globals():

    punjab_official_reference = pd.read_csv(
        "../../data/processed/"
        "punjab_official_market_committee_mapping.csv"
    )

In [65]:
if "Match_Key" not in punjab_remaining_unmapped.columns:

    punjab_remaining_unmapped[
        "Match_Key"
    ] = (
        punjab_remaining_unmapped[
            "Market"
        ]
        .apply(
            normalize_market_for_matching
        )
    )


if "Match_Key" not in punjab_official_reference.columns:

    punjab_official_reference[
        "Match_Key"
    ] = (
        punjab_official_reference[
            "Market_Raw"
        ]
        .apply(
            normalize_market_for_matching
        )
    )

In [66]:
official_keys = (
    punjab_official_reference[
        "Match_Key"
    ]
    .dropna()
    .unique()
    .tolist()
)

In [67]:
fuzzy_results = []


for row in punjab_remaining_unmapped.itertuples():

    query = row.Match_Key

    matches = process.extract(
        query,
        official_keys,
        scorer=fuzz.WRatio,
        limit=3
    )

    result = {
        "Market": row.Market,
        "Market_Key": row.Market_Key,
        "Match_Key": query
    }


    for rank, match in enumerate(
        matches,
        start=1
    ):

        matched_key = match[0]
        score = match[1]

        official_row = (
            punjab_official_reference[
                punjab_official_reference[
                    "Match_Key"
                ] == matched_key
            ]
            .iloc[0]
        )

        result[
            f"Candidate_{rank}_Market"
        ] = official_row["Market_Raw"]

        result[
            f"Candidate_{rank}_District"
        ] = official_row["District"]

        result[
            f"Candidate_{rank}_Score"
        ] = score


    fuzzy_results.append(
        result
    )


punjab_fuzzy_candidates = pd.DataFrame(
    fuzzy_results
)

In [68]:
punjab_fuzzy_candidates[
    "Score_Margin"
] = (
    punjab_fuzzy_candidates[
        "Candidate_1_Score"
    ]
    -
    punjab_fuzzy_candidates[
        "Candidate_2_Score"
    ]
)

In [69]:
print(
    "90+ score:",
    (
        punjab_fuzzy_candidates[
            "Candidate_1_Score"
        ] >= 90
    ).sum()
)

print(
    "85–89:",
    (
        (
            punjab_fuzzy_candidates[
                "Candidate_1_Score"
            ] >= 85
        )
        &
        (
            punjab_fuzzy_candidates[
                "Candidate_1_Score"
            ] < 90
        )
    ).sum()
)

print(
    "70–84:",
    (
        (
            punjab_fuzzy_candidates[
                "Candidate_1_Score"
            ] >= 70
        )
        &
        (
            punjab_fuzzy_candidates[
                "Candidate_1_Score"
            ] < 85
        )
    ).sum()
)

print(
    "Below 70:",
    (
        punjab_fuzzy_candidates[
            "Candidate_1_Score"
        ] < 70
    ).sum()
)

90+ score: 14
85–89: 6
70–84: 23
Below 70: 23


In [70]:
high_confidence_candidates = (
    punjab_fuzzy_candidates[
        punjab_fuzzy_candidates[
            "Candidate_1_Score"
        ] >= 85
    ]
    .sort_values(
        [
            "Candidate_1_Score",
            "Score_Margin"
        ],
        ascending=False
    )
)

print(
    high_confidence_candidates[
        [
            "Market",
            "Candidate_1_Market",
            "Candidate_1_District",
            "Candidate_1_Score",
            "Candidate_2_Market",
            "Candidate_2_District",
            "Candidate_2_Score",
            "Score_Margin"
        ]
    ]
    .to_string(index=False)
)

                                      Market Candidate_1_Market Candidate_1_District  Candidate_1_Score Candidate_2_Market Candidate_2_District  Candidate_2_Score  Score_Margin
                            Sulargharat APMC        SULARGHARAT              Sangrur          95.000000             KHARAR          S.A.S Nagar          62.181818     32.818182
               Amloh(Gobind Garh Mandi) APMC              AMLOH      Fatehgarh Sahib          90.000000          AHMEDGARH              Sangrur          57.000000     33.000000
                Sultanpur Lodhi (Tibba) APMC    SULTANPUR LODHI           Kapurthala          90.000000             MALOUD             Ludhiana          57.000000     33.000000
                  Ludhiana(Salem Tabri) APMC           LUDHIANA             Ludhiana          90.000000             MALOUD             Ludhiana          60.000000     30.000000
               Garhshankar(Saila Khurd) APMC        GARHSHANKAR           Hoshiarpur          90.000000            

In [71]:
strict_high_confidence = (
    punjab_fuzzy_candidates[
        punjab_fuzzy_candidates[
            "Candidate_1_Score"
        ] >= 90
    ]
    .copy()
)

In [72]:
strict_high_confidence = (
    strict_high_confidence
    .sort_values(
        [
            "Candidate_1_Score",
            "Score_Margin"
        ],
        ascending=False
    )
)

In [73]:
print(
    strict_high_confidence[
        [
            "Market",
            "Candidate_1_Market",
            "Candidate_1_District",
            "Candidate_1_Score",
            "Candidate_2_Market",
            "Candidate_2_District",
            "Candidate_2_Score",
            "Score_Margin"
        ]
    ]
    .to_string(
        index=False
    )
)

                                      Market Candidate_1_Market Candidate_1_District  Candidate_1_Score Candidate_2_Market Candidate_2_District  Candidate_2_Score  Score_Margin
                            Sulargharat APMC        SULARGHARAT              Sangrur               95.0             KHARAR          S.A.S Nagar          62.181818     32.818182
               Amloh(Gobind Garh Mandi) APMC              AMLOH      Fatehgarh Sahib               90.0          AHMEDGARH              Sangrur          57.000000     33.000000
                Sultanpur Lodhi (Tibba) APMC    SULTANPUR LODHI           Kapurthala               90.0             MALOUD             Ludhiana          57.000000     33.000000
                  Ludhiana(Salem Tabri) APMC           LUDHIANA             Ludhiana               90.0             MALOUD             Ludhiana          60.000000     30.000000
               Garhshankar(Saila Khurd) APMC        GARHSHANKAR           Hoshiarpur               90.0            

In [74]:
strict_high_confidence[
    "Strong_Review_Candidate"
] = (
    (
        strict_high_confidence[
            "Candidate_1_Score"
        ] >= 90
    )
    &
    (
        strict_high_confidence[
            "Score_Margin"
        ] >= 10
    )
)

In [75]:
print(
    "90+ candidates:",
    len(strict_high_confidence)
)

print(
    "90+ with margin >= 10:",
    strict_high_confidence[
        "Strong_Review_Candidate"
    ].sum()
)


90+ candidates: 14
90+ with margin >= 10: 11


In [76]:
print(
    strict_high_confidence[
        strict_high_confidence[
            "Strong_Review_Candidate"
        ]
    ]
    [
        [
            "Market",
            "Candidate_1_Market",
            "Candidate_1_District",
            "Candidate_1_Score",
            "Candidate_2_Market",
            "Candidate_2_Score",
            "Score_Margin"
        ]
    ]
    .to_string(
        index=False
    )
)

                             Market Candidate_1_Market Candidate_1_District  Candidate_1_Score Candidate_2_Market  Candidate_2_Score  Score_Margin
                   Sulargharat APMC        SULARGHARAT              Sangrur               95.0             KHARAR          62.181818     32.818182
      Amloh(Gobind Garh Mandi) APMC              AMLOH      Fatehgarh Sahib               90.0          AHMEDGARH          57.000000     33.000000
       Sultanpur Lodhi (Tibba) APMC    SULTANPUR LODHI           Kapurthala               90.0             MALOUD          57.000000     33.000000
         Ludhiana(Salem Tabri) APMC           LUDHIANA             Ludhiana               90.0             MALOUD          60.000000     30.000000
      Garhshankar(Saila Khurd) APMC        GARHSHANKAR           Hoshiarpur               90.0            GHANAUR          64.285714     25.714286
               Rayya(Sathiala) APMC              RAYYA             Amritsar               90.0            NATHANA     

In [77]:
punjab_reviewed_fuzzy_mapping = {
    "sulargharat": "Sangrur",

    "amloh(gobind garh mandi)": "Fatehgarh Sahib",

    "sultanpur lodhi (tibba)": "Kapurthala",

    "ludhiana(salem tabri)": "Ludhiana",

    "garhshankar(saila khurd)": "Hoshiarpur",

    "rayya(sathiala)": "Amritsar",

    "mallanwala (khosadalsinghwala)": "Ferozepur",

    "khemkaran (amarkot)": "Tarn Taran",

    "ajitwal (chogawan)": "Moga",

    "ajitwal (dala)": "Moga",

    "phillaur(apra mandi)": "Jalandhar"
}

In [78]:
reviewed_rows = {
    "Sulargharat APMC": "Sangrur",
    "Amloh(Gobind Garh Mandi) APMC": "Fatehgarh Sahib",
    "Sultanpur Lodhi (Tibba) APMC": "Kapurthala",
    "Ludhiana(Salem Tabri) APMC": "Ludhiana",
    "Garhshankar(Saila Khurd) APMC": "Hoshiarpur",
    "Rayya(Sathiala) APMC": "Amritsar",
    "Mallanwala (KhosaDalSinghwala) APMC": "Ferozepur",
    "Khemkaran (Amarkot) APMC": "Tarn Taran",
    "Ajitwal (Chogawan) APMC": "Moga",
    "Ajitwal (Dala) APMC": "Moga",
    "Phillaur(Apra Mandi) APMC": "Jalandhar"
}

In [79]:
punjab_reviewed_fuzzy_df = pd.DataFrame(
    [
        {
            "Market": market,
            "Market_Key": normalize_market_name(market),
            "District": district
        }
        for market, district
        in reviewed_rows.items()
    ]
)

print(
    punjab_reviewed_fuzzy_df.to_string(
        index=False
    )
)

                             Market                     Market_Key        District
                   Sulargharat APMC                    sulargharat         Sangrur
      Amloh(Gobind Garh Mandi) APMC       amloh(gobind garh mandi) Fatehgarh Sahib
       Sultanpur Lodhi (Tibba) APMC        sultanpur lodhi (tibba)      Kapurthala
         Ludhiana(Salem Tabri) APMC          ludhiana(salem tabri)        Ludhiana
      Garhshankar(Saila Khurd) APMC       garhshankar(saila khurd)      Hoshiarpur
               Rayya(Sathiala) APMC                rayya(sathiala)        Amritsar
Mallanwala (KhosaDalSinghwala) APMC mallanwala (khosadalsinghwala)       Ferozepur
           Khemkaran (Amarkot) APMC            khemkaran (amarkot)      Tarn Taran
            Ajitwal (Chogawan) APMC             ajitwal (chogawan)            Moga
                Ajitwal (Dala) APMC                 ajitwal (dala)            Moga
          Phillaur(Apra Mandi) APMC           phillaur(apra mandi)       Jalandhar


In [80]:
unresolved_keys = set(
    punjab_remaining_unmapped[
        "Market_Key"
    ]
)

punjab_reviewed_fuzzy_df[
    "Exists_In_Unresolved"
] = (
    punjab_reviewed_fuzzy_df[
        "Market_Key"
    ].isin(
        unresolved_keys
    )
)

print(
    punjab_reviewed_fuzzy_df[
        [
            "Market",
            "Market_Key",
            "District",
            "Exists_In_Unresolved"
        ]
    ].to_string(index=False)
)

print(
    "\nFound among unresolved:",
    punjab_reviewed_fuzzy_df[
        "Exists_In_Unresolved"
    ].sum()
)

                             Market                     Market_Key        District  Exists_In_Unresolved
                   Sulargharat APMC                    sulargharat         Sangrur                  True
      Amloh(Gobind Garh Mandi) APMC       amloh(gobind garh mandi) Fatehgarh Sahib                  True
       Sultanpur Lodhi (Tibba) APMC        sultanpur lodhi (tibba)      Kapurthala                  True
         Ludhiana(Salem Tabri) APMC          ludhiana(salem tabri)        Ludhiana                  True
      Garhshankar(Saila Khurd) APMC       garhshankar(saila khurd)      Hoshiarpur                  True
               Rayya(Sathiala) APMC                rayya(sathiala)        Amritsar                  True
Mallanwala (KhosaDalSinghwala) APMC mallanwala (khosadalsinghwala)       Ferozepur                  True
           Khemkaran (Amarkot) APMC            khemkaran (amarkot)      Tarn Taran                  True
            Ajitwal (Chogawan) APMC             ajitwal

In [81]:
punjab_market_mapping_v4 = pd.concat(
    [
        punjab_market_mapping_v3[
            [
                "Market_Key",
                "District"
            ]
        ],

        punjab_reviewed_fuzzy_df[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)

NameError: name 'punjab_market_mapping_v3' is not defined

In [83]:
punjab_market_mapping_v4 = (
    punjab_market_mapping_v4
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)

NameError: name 'punjab_market_mapping_v4' is not defined

In [84]:
mapping_check = (
    punjab_market_mapping_v4
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Mapped Punjab markets:",
    punjab_market_mapping_v4[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (
        mapping_check > 1
    ).sum()
)

NameError: name 'punjab_market_mapping_v4' is not defined

In [85]:
punjab_market_mapping_v4.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

print(
    "Punjab mapping saved ✅"
)

NameError: name 'punjab_market_mapping_v4' is not defined

In [86]:
punjab_wheat_with_district = (
    punjab_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_v4,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_wheat_monthly' is not defined

In [87]:
print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [88]:
reviewed_market_keys = set(
    punjab_reviewed_fuzzy_df[
        "Market_Key"
    ]
)

remaining_90_plus = (
    punjab_fuzzy_candidates[
        punjab_fuzzy_candidates[
            "Candidate_1_Score"
        ] >= 90
    ]
    .copy()
)

remaining_90_plus = (
    remaining_90_plus[
        ~remaining_90_plus[
            "Market_Key"
        ].isin(
            reviewed_market_keys
        )
    ]
)

print(
    "Remaining 90+ candidates:",
    len(remaining_90_plus)
)

print(
    remaining_90_plus[
        [
            "Market",
            "Candidate_1_Market",
            "Candidate_1_District",
            "Candidate_1_Score",
            "Candidate_2_Market",
            "Candidate_2_District",
            "Candidate_2_Score",
            "Score_Margin"
        ]
    ]
    .to_string(index=False)
)

Remaining 90+ candidates: 3
                                      Market Candidate_1_Market Candidate_1_District  Candidate_1_Score Candidate_2_Market Candidate_2_District  Candidate_2_Score  Score_Margin
Jalandhar Cantt (Jamshedpur Dana Mandi) APMC    JALANDHAR CANTT            Jalandhar               90.0     JALANDHAR CITY            Jalandhar               85.5           4.5
           Jalandhar City(Faintan Ganj) APMC     JALANDHAR CITY            Jalandhar               90.0     FEROZEPUR CITY            Ferozepur               85.5           4.5
  Jalandhar City(Kartar Pur Dana mandi) APMC     JALANDHAR CITY            Jalandhar               90.0     FEROZEPUR CITY            Ferozepur               85.5           4.5


In [89]:
remaining_three_reviewed = {
    "Jalandhar Cantt (Jamshedpur Dana Mandi) APMC":
        "Jalandhar",

    "Jalandhar City(Faintan Ganj) APMC":
        "Jalandhar",

    "Jalandhar City(Kartar Pur Dana mandi) APMC":
        "Jalandhar"
}

In [90]:
remaining_three_df = pd.DataFrame(
    [
        {
            "Market": market,
            "Market_Key": normalize_market_name(market),
            "District": district
        }
        for market, district
        in remaining_three_reviewed.items()
    ]
)

print(
    remaining_three_df.to_string(
        index=False
    )
)

                                      Market                              Market_Key  District
Jalandhar Cantt (Jamshedpur Dana Mandi) APMC jalandhar cantt (jamshedpur dana mandi) Jalandhar
           Jalandhar City(Faintan Ganj) APMC            jalandhar city(faintan ganj) Jalandhar
  Jalandhar City(Kartar Pur Dana mandi) APMC   jalandhar city(kartar pur dana mandi) Jalandhar


In [91]:
current_unresolved_keys = set(
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market_Key"
    ]
)

remaining_three_df[
    "Exists_In_Unresolved"
] = (
    remaining_three_df[
        "Market_Key"
    ].isin(
        current_unresolved_keys
    )
)

print(
    remaining_three_df[
        [
            "Market",
            "Market_Key",
            "District",
            "Exists_In_Unresolved"
        ]
    ].to_string(index=False)
)

print(
    "\nFound:",
    remaining_three_df[
        "Exists_In_Unresolved"
    ].sum()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [92]:
punjab_market_mapping_v5 = pd.concat(
    [
        punjab_market_mapping_v4[
            [
                "Market_Key",
                "District"
            ]
        ],

        remaining_three_df[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)

NameError: name 'punjab_market_mapping_v4' is not defined

In [93]:
punjab_market_mapping_v5 = (
    punjab_market_mapping_v5
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)

NameError: name 'punjab_market_mapping_v5' is not defined

In [94]:
mapping_check = (
    punjab_market_mapping_v5
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Mapped Punjab markets:",
    punjab_market_mapping_v5[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (
        mapping_check > 1
    ).sum()
)

NameError: name 'punjab_market_mapping_v5' is not defined

In [95]:
punjab_market_mapping_v5.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

print(
    "Punjab mapping saved ✅"
)

NameError: name 'punjab_market_mapping_v5' is not defined

In [96]:
punjab_wheat_with_district = (
    punjab_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_v5,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_wheat_monthly' is not defined

In [97]:
print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [98]:
#Show the 85–89 candidates
medium_high_candidates = (
    punjab_fuzzy_candidates[
        (
            punjab_fuzzy_candidates[
                "Candidate_1_Score"
            ] >= 85
        )
        &
        (
            punjab_fuzzy_candidates[
                "Candidate_1_Score"
            ] < 90
        )
    ]
    .copy()
)

medium_high_candidates = (
    medium_high_candidates
    .sort_values(
        [
            "Candidate_1_Score",
            "Score_Margin"
        ],
        ascending=False
    )
)

print(
    "85–89 candidates:",
    len(medium_high_candidates)
)

print(
    medium_high_candidates[
        [
            "Market",
            "Candidate_1_Market",
            "Candidate_1_District",
            "Candidate_1_Score",
            "Candidate_2_Market",
            "Candidate_2_District",
            "Candidate_2_Score",
            "Score_Margin"
        ]
    ]
    .to_string(index=False)
)


85–89 candidates: 6
                      Market Candidate_1_Market Candidate_1_District  Candidate_1_Score Candidate_2_Market Candidate_2_District  Candidate_2_Score  Score_Margin
Rampuraphul(Bhai Rupan) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099
    Rampuraphul(Chowke) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099
   Rampuraphul(Dhapali) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099
    Rampuraphul(Mehraj) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099
   Talwandi Bhai(Mudki) APMC      TALWANDI SABO             Bathinda          85.500000           MUKERIAN           Hoshiarpur          56.250000     29.250000
   Mansa (Khia

In [99]:
#whether the top two candidates point to the same district
medium_high_candidates[
    "Same_District_Top2"
] = (
    medium_high_candidates[
        "Candidate_1_District"
    ]
    ==
    medium_high_candidates[
        "Candidate_2_District"
    ]
)

print(
    medium_high_candidates[
        [
            "Market",
            "Candidate_1_Market",
            "Candidate_1_District",
            "Candidate_1_Score",
            "Candidate_2_Market",
            "Candidate_2_District",
            "Candidate_2_Score",
            "Score_Margin",
            "Same_District_Top2"
        ]
    ]
    .to_string(index=False)
)

                      Market Candidate_1_Market Candidate_1_District  Candidate_1_Score Candidate_2_Market Candidate_2_District  Candidate_2_Score  Score_Margin  Same_District_Top2
Rampuraphul(Bhai Rupan) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099               False
    Rampuraphul(Chowke) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099               False
   Rampuraphul(Dhapali) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099               False
    Rampuraphul(Mehraj) APMC       RAMPURA PHUL             Bathinda          86.086957            RAJPURA              Patiala          77.142857      8.944099               False
   Talwandi Bhai(Mudki) APMC      TALWANDI SABO             Bathinda          85.500000        

In [100]:
def extract_parent_market(name):

    name = str(name)

    # Remove APMC
    name = re.sub(
        r"\bAPMC\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Keep only text before (
    name = name.split("(")[0]

    name = name.strip()

    return name

In [101]:
for market in medium_high_candidates["Market"]:

    print(
        market,
        "→",
        extract_parent_market(market)
    )

Rampuraphul(Bhai Rupan) APMC → Rampuraphul
Rampuraphul(Chowke) APMC → Rampuraphul
Rampuraphul(Dhapali) APMC → Rampuraphul
Rampuraphul(Mehraj) APMC → Rampuraphul
Talwandi Bhai(Mudki) APMC → Talwandi Bhai
Mansa (Khiala kalan) APMC → Mansa


In [102]:
def compact_market_key(name):

    name = normalize_market_for_matching(
        name
    )

    # Remove all remaining spaces
    name = re.sub(
        r"\s+",
        "",
        name
    )

    return name

In [103]:
medium_high_candidates[
    "Parent_Market"
] = (
    medium_high_candidates[
        "Market"
    ]
    .apply(
        extract_parent_market
    )
)

medium_high_candidates[
    "Parent_Compact_Key"
] = (
    medium_high_candidates[
        "Parent_Market"
    ]
    .apply(
        compact_market_key
    )
)

In [104]:
punjab_official_reference[
    "Official_Compact_Key"
] = (
    punjab_official_reference[
        "Market_Raw"
    ]
    .apply(
        compact_market_key
    )
)

In [105]:
parent_lookup = (
    punjab_official_reference[
        [
            "Official_Compact_Key",
            "Market_Raw",
            "District"
        ]
    ]
    .drop_duplicates()
)

In [106]:
parent_conflicts = (
    parent_lookup
    .groupby(
        "Official_Compact_Key"
    )["District"]
    .nunique()
)

print(
    "Parent-key conflicts:",
    (
        parent_conflicts > 1
    ).sum()
)

Parent-key conflicts: 0


In [107]:
medium_parent_matches = (
    medium_high_candidates
    .merge(
        parent_lookup,
        left_on="Parent_Compact_Key",
        right_on="Official_Compact_Key",
        how="left"
    )
)

In [108]:
print(
    medium_parent_matches[
        [
            "Market",
            "Parent_Market",
            "Market_Raw",
            "District"
        ]
    ]
    .to_string(
        index=False
    )
)

                      Market Parent_Market   Market_Raw District
Rampuraphul(Bhai Rupan) APMC   Rampuraphul RAMPURA PHUL Bathinda
    Rampuraphul(Chowke) APMC   Rampuraphul RAMPURA PHUL Bathinda
   Rampuraphul(Dhapali) APMC   Rampuraphul RAMPURA PHUL Bathinda
    Rampuraphul(Mehraj) APMC   Rampuraphul RAMPURA PHUL Bathinda
   Talwandi Bhai(Mudki) APMC Talwandi Bhai          NaN      NaN
   Mansa (Khiala kalan) APMC         Mansa          NaN      NaN


In [109]:
from pathlib import Path
import pandas as pd
import re

if "punjab_market_mapping_v5" not in globals():

    punjab_market_mapping_v5 = pd.read_csv(
        "../../data/processed/"
        "punjab_market_district_mapping.csv"
    )

print(
    "Validated mappings loaded:",
    len(punjab_market_mapping_v5)
)

Validated mappings loaded: 196


In [110]:
current_unresolved = (
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "Current unresolved markets:",
    len(current_unresolved)
)

NameError: name 'punjab_wheat_with_district' is not defined

In [111]:
def extract_parent_market(name):

    name = str(name)

    name = re.sub(
        r"\bAPMC\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Only keep text before parentheses
    name = name.split("(")[0]

    return name.strip()


def compact_market_key(name):

    name = str(name).lower().strip()

    name = re.sub(
        r"[^a-z0-9\s]",
        " ",
        name
    )

    name = re.sub(
        r"\bapmc\b",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        "",
        name
    )

    return name

In [112]:
current_unresolved[
    "Parent_Market"
] = (
    current_unresolved[
        "Market"
    ]
    .apply(
        extract_parent_market
    )
)

current_unresolved[
    "Parent_Compact_Key"
] = (
    current_unresolved[
        "Parent_Market"
    ]
    .apply(
        compact_market_key
    )
)

NameError: name 'current_unresolved' is not defined

In [113]:
validated_parent_lookup = (
    punjab_market_mapping_v5[
        [
            "Market_Key",
            "District"
        ]
    ]
    .copy()
)

validated_parent_lookup[
    "Validated_Compact_Key"
] = (
    validated_parent_lookup[
        "Market_Key"
    ]
    .apply(
        compact_market_key
    )
)

In [114]:
parent_key_check = (
    validated_parent_lookup
    .groupby(
        "Validated_Compact_Key"
    )["District"]
    .nunique()
)

print(
    "Validated parent-key conflicts:",
    (
        parent_key_check > 1
    ).sum()
)

Validated parent-key conflicts: 0


In [115]:
safe_parent_lookup = (
    validated_parent_lookup[
        validated_parent_lookup[
            "Validated_Compact_Key"
        ].isin(
            parent_key_check[
                parent_key_check == 1
            ].index
        )
    ]
    [
        [
            "Validated_Compact_Key",
            "Market_Key",
            "District"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Validated_Compact_Key"
        ]
    )
)

In [116]:
parent_market_matches = (
    current_unresolved
    .merge(
        safe_parent_lookup,
        left_on="Parent_Compact_Key",
        right_on="Validated_Compact_Key",
        how="left",
        suffixes=(
            "_Unresolved",
            "_Parent"
        )
    )
)

NameError: name 'current_unresolved' is not defined

In [117]:
print(
    "Current unresolved:",
    len(current_unresolved)
)

print(
    "Exact validated-parent matches:",
    parent_market_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    parent_market_matches[
        "District"
    ].isna().sum()
)

NameError: name 'current_unresolved' is not defined

In [118]:
print(
    parent_market_matches[
        parent_market_matches[
            "District"
        ].notna()
    ]
    [
        [
            "Market",
            "Parent_Market",
            "Market_Key_Parent",
            "District"
        ]
    ]
    .to_string(
        index=False
    )
)

NameError: name 'parent_market_matches' is not defined

In [119]:
safe_parent_matches_df = (
    parent_market_matches[
        parent_market_matches[
            "District"
        ].notna()
    ]
    [
        [
            "Market",
            "Market_Key_Unresolved",
            "District"
        ]
    ]
    .rename(
        columns={
            "Market_Key_Unresolved": "Market_Key"
        }
    )
    .drop_duplicates(
        subset=["Market_Key"]
    )
)

print(
    "Safe parent mappings:",
    len(safe_parent_matches_df)
)

print(
    safe_parent_matches_df.to_string(
        index=False
    )
)

NameError: name 'parent_market_matches' is not defined

In [120]:
punjab_market_mapping_v6 = pd.concat(
    [
        punjab_market_mapping_v5[
            [
                "Market_Key",
                "District"
            ]
        ],

        safe_parent_matches_df[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)

NameError: name 'safe_parent_matches_df' is not defined

In [121]:
punjab_market_mapping_v6 = (
    punjab_market_mapping_v6
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)

NameError: name 'punjab_market_mapping_v6' is not defined

In [122]:
mapping_check = (
    punjab_market_mapping_v6
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Mapped Punjab markets:",
    punjab_market_mapping_v6[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (
        mapping_check > 1
    ).sum()
)

NameError: name 'punjab_market_mapping_v6' is not defined

In [123]:
punjab_market_mapping_v6.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

print(
    "Punjab mapping saved ✅"
)

NameError: name 'punjab_market_mapping_v6' is not defined

In [124]:
punjab_wheat_with_district = (
    punjab_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_v6,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_wheat_monthly' is not defined

In [125]:
print(
    "Total rows:",
    len(punjab_wheat_with_district)
)

print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [126]:
punjab_final_unresolved = (
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "Final unresolved markets:",
    len(punjab_final_unresolved)
)

print(
    punjab_final_unresolved.to_string(
        index=False
    )
)

NameError: name 'punjab_wheat_with_district' is not defined

In [127]:
# Add final 24 reviewed mapping 
final_24_reviewed = {

    "Ahemadpur APMC":
        "Mansa",

    "Badhni Kallan (Bilaspur) APMC":
        "Moga",

    "Budalada APMC":
        "Mansa",

    "Budhlada (Phaphre Bhaike) APMC":
        "Mansa",

    "Dhanula (Kaleke) APMC":
        "Barnala",

    "Dudhansadhan APMC":
        "Patiala",

    "Firozepur City APMC":
        "Ferozepur",

    "Gehri(Jandiala mandi) APMC":
        "Amritsar",

    "Kankwal Chahlan APMC":
        "Mansa",

    "Malaoud (Siar) APMC":
        "Ludhiana",

    "Mehal Kallan (ChananWal) APMC":
        "Barnala",

    "Mullanpur Dakha (Sawadi) APMC":
        "Ludhiana",

    "Naushera Pannua (Dhotian) APMC":
        "Tarn Taran",

    "Nawan Shahar(Mandi Jadla) APMC":
        "Shahid Bhagat Singh Nagar",

    "Nawan Shahar(MandiRaho) APMC":
        "Shahid Bhagat Singh Nagar",

    "Nawanshahar APMC":
        "Shahid Bhagat Singh Nagar",

    "Nurpurbet APMC":
        "Ludhiana",

    "Patiala(Anaj Mandi Sanaur Road) APMC":
        "Patiala",

    "Patiala(New Anaj Mandi) APMC":
        "Patiala",

    "Sandaur APMC":
        "Malerkotla",

    "Shakot (Malsian) APMC":
        "Jalandhar",

    "Sri Har Gobindpur APMC":
        "Gurdaspur",

    "kum Kalan APMC":
        "Ludhiana",

    "lakhowal APMC":
        "Ludhiana"
}

In [128]:
final_24_df = pd.DataFrame(
    [
        {
            "Market": market,
            "Market_Key": normalize_market_name(
                market
            ),
            "District": district
        }
        for market, district
        in final_24_reviewed.items()
    ]
)

print(
    "Final reviewed mappings:",
    len(final_24_df)
)

print(
    final_24_df.to_string(
        index=False
    )
)

Final reviewed mappings: 24
                              Market                      Market_Key                  District
                      Ahemadpur APMC                       ahemadpur                     Mansa
       Badhni Kallan (Bilaspur) APMC        badhni kallan (bilaspur)                      Moga
                       Budalada APMC                        budalada                     Mansa
      Budhlada (Phaphre Bhaike) APMC       budhlada (phaphre bhaike)                     Mansa
               Dhanula (Kaleke) APMC                dhanula (kaleke)                   Barnala
                   Dudhansadhan APMC                    dudhansadhan                   Patiala
                 Firozepur City APMC                  firozepur city                 Ferozepur
          Gehri(Jandiala mandi) APMC           gehri(jandiala mandi)                  Amritsar
                Kankwal Chahlan APMC                 kankwal chahlan                     Mansa
                 Malao

In [129]:
current_unresolved_keys = set(
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market_Key"
    ]
)

final_24_df[
    "Exists_In_Unresolved"
] = (
    final_24_df[
        "Market_Key"
    ].isin(
        current_unresolved_keys
    )
)

print(
    final_24_df[
        [
            "Market",
            "Market_Key",
            "District",
            "Exists_In_Unresolved"
        ]
    ].to_string(
        index=False
    )
)

print(
    "\nFound among unresolved:",
    final_24_df[
        "Exists_In_Unresolved"
    ].sum()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [130]:
punjab_market_mapping_v6

NameError: name 'punjab_market_mapping_v6' is not defined

In [131]:
punjab_market_mapping_final = pd.concat(
    [
        punjab_market_mapping_v6[
            [
                "Market_Key",
                "District"
            ]
        ],

        final_24_df[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)

NameError: name 'punjab_market_mapping_v6' is not defined

In [132]:
punjab_market_mapping_final = (
    punjab_market_mapping_final
    .drop_duplicates(
        subset=["Market_Key"],
        keep="first"
    )
)

NameError: name 'punjab_market_mapping_final' is not defined

In [133]:
final_mapping_check = (
    punjab_market_mapping_final
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Final Punjab mapped markets:",
    punjab_market_mapping_final[
        "Market_Key"
    ].nunique()
)

print(
    "Mapping conflicts:",
    (
        final_mapping_check > 1
    ).sum()
)

NameError: name 'punjab_market_mapping_final' is not defined

In [134]:
punjab_wheat_with_district = (
    punjab_wheat_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_final,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_wheat_monthly' is not defined

In [135]:
print(
    "Total rows:",
    len(punjab_wheat_with_district)
)

print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [136]:
# Validating every assigned markets 
invalid_districts = sorted(
    set(
        punjab_wheat_with_district[
            "District"
        ].dropna()
    )
    -
    set(
        production_punjab_districts
    )
)

print(
    "Districts not present in production:",
    invalid_districts
)

NameError: name 'punjab_wheat_with_district' is not defined

In [137]:
punjab_market_mapping_final.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

punjab_wheat_with_district.to_csv(
    "../../data/processed/"
    "punjab_wheat_monthly_prices_mapped_2021_2025.csv",
    index=False
)

print(
    "Final Punjab mapping saved ✅"
)

print(
    "Punjab Wheat mapped monthly data saved ✅"
)

NameError: name 'punjab_market_mapping_final' is not defined

In [138]:
#Standardize these six district names
punjab_district_standardization = {
    "Bhatinda": "Bathinda",
    "Fatehgarh": "Fatehgarh Sahib",
    "Mohali": "S.A.S Nagar",
    "Muktsar": "Sri Muktsar Sahib",
    "Ropar (Rupnagar)": "Rupnagar",
    "Tarntaran": "Tarn Taran"
}

In [139]:
punjab_market_mapping_final[
    "District"
] = (
    punjab_market_mapping_final[
        "District"
    ]
    .replace(
        punjab_district_standardization
    )
)

NameError: name 'punjab_market_mapping_final' is not defined

In [140]:
punjab_wheat_with_district[
    "District"
] = (
    punjab_wheat_with_district[
        "District"
    ]
    .replace(
        punjab_district_standardization
    )
)

NameError: name 'punjab_wheat_with_district' is not defined

In [141]:
invalid_districts = sorted(
    set(
        punjab_wheat_with_district[
            "District"
        ].dropna()
    )
    -
    set(
        production_punjab_districts
    )
)

print(
    "Districts not present in production:",
    invalid_districts
)

NameError: name 'punjab_wheat_with_district' is not defined

In [142]:
print(
    "Rows missing District:",
    punjab_wheat_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_wheat_with_district.loc[
        punjab_wheat_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_wheat_with_district' is not defined

In [143]:
mapping_conflicts = (
    punjab_market_mapping_final
    .groupby(
        "Market_Key"
    )["District"]
    .nunique()
)

print(
    "Mapping conflicts:",
    (
        mapping_conflicts > 1
    ).sum()
)

NameError: name 'punjab_market_mapping_final' is not defined

In [144]:
punjab_market_mapping_final.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

punjab_wheat_with_district.to_csv(
    "../../data/processed/"
    "punjab_wheat_monthly_prices_mapped_2021_2025.csv",
    index=False
)

print("Standardized Punjab mapping saved ✅")
print("Standardized Punjab Wheat monthly data saved ✅")

NameError: name 'punjab_market_mapping_final' is not defined

In [145]:
punjab_wheat_price_features = (
    build_price_features(
        punjab_wheat_with_district
    )
)

NameError: name 'build_price_features' is not defined

In [146]:
print(
    "Punjab Wheat feature rows:",
    len(punjab_wheat_price_features)
)

print(
    "\nColumns:"
)

print(
    punjab_wheat_price_features.columns.tolist()
)

punjab_wheat_price_features.head()

NameError: name 'punjab_wheat_price_features' is not defined

In [147]:
duplicate_count = (
    punjab_wheat_price_features
    .duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    )
    .sum()
)

print(
    "Duplicate district-crop-year rows:",
    duplicate_count
)

NameError: name 'punjab_wheat_price_features' is not defined

In [148]:
print(
    punjab_wheat_price_features[
        "Year"
    ]
    .value_counts()
    .sort_index()
)

NameError: name 'punjab_wheat_price_features' is not defined

In [149]:
print(
    "\nCoverage quality:"
)

print(
    punjab_wheat_price_features[
        "Price_Coverage_Quality"
    ]
    .value_counts(
        dropna=False
    )
)


Coverage quality:


NameError: name 'punjab_wheat_price_features' is not defined

In [150]:
print(
    "\nMonths available distribution:"
)

print(
    punjab_wheat_price_features[
        "Months_Available"
    ]
    .value_counts()
    .sort_index(
        ascending=False
    )
)


Months available distribution:


NameError: name 'punjab_wheat_price_features' is not defined

In [151]:
print(
    "Missing Price_Volatility_Pct:",
    punjab_wheat_price_features[
        "Price_Volatility_Pct"
    ]
    .isna()
    .sum()
)

NameError: name 'punjab_wheat_price_features' is not defined

In [152]:
# BASIC SANITY CHECK 
print(
    "Minimum average price:",
    punjab_wheat_price_features[
        "Avg_Modal_Price"
    ].min()
)

print(
    "Maximum average price:",
    punjab_wheat_price_features[
        "Avg_Modal_Price"
    ].max()
)

print(
    "Zero/negative average prices:",
    (
        punjab_wheat_price_features[
            "Avg_Modal_Price"
        ] <= 0
    ).sum()
)

NameError: name 'punjab_wheat_price_features' is not defined

In [153]:
punjab_wheat_price_features.to_csv(
    "../../data/processed/"
    "punjab_wheat_price_features_2021_2025.csv",
    index=False
)

print(
    "Punjab Wheat price features saved ✅"
)

NameError: name 'punjab_wheat_price_features' is not defined

In [154]:
# Compare price features aganist production key 

punjab_wheat_production_keys = (
    production_history[
        (
            production_history["State"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "punjab"
        )
        &
        (
            production_history["Crop"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "wheat"
        )
        &
        (
            production_history["Year"]
            .isin(
                [
                    "2021-22",
                    "2022-23",
                    "2023-24",
                    "2024-25"
                ]
            )
        )
    ]
    [
        [
            "State",
            "District",
            "Crop",
            "Year"
        ]
    ]
    .drop_duplicates()
)

print(
    "Punjab Wheat production keys:",
    len(punjab_wheat_production_keys)
)

Punjab Wheat production keys: 92


In [155]:
punjab_wheat_coverage_check = (
    punjab_wheat_production_keys
    .merge(
        punjab_wheat_price_features[
            [
                "State",
                "District",
                "Crop",
                "Year",
                "Months_Available"
            ]
        ],
        on=[
            "State",
            "District",
            "Crop",
            "Year"
        ],
        how="left",
        indicator=True
    )
)

print(
    "Production keys with price:",
    (
        punjab_wheat_coverage_check[
            "_merge"
        ] == "both"
    ).sum()
)

print(
    "Production keys with NO price row:",
    (
        punjab_wheat_coverage_check[
            "_merge"
        ] == "left_only"
    ).sum()
)

NameError: name 'punjab_wheat_price_features' is not defined

In [156]:
punjab_wheat_no_price = (
    punjab_wheat_coverage_check[
        punjab_wheat_coverage_check[
            "_merge"
        ] == "left_only"
    ]
    [
        [
            "District",
            "Year"
        ]
    ]
    .sort_values(
        [
            "District",
            "Year"
        ]
    )
)

print(
    punjab_wheat_no_price.to_string(
        index=False
    )
)

NameError: name 'punjab_wheat_coverage_check' is not defined

In [157]:
punjab_wheat_price_features.to_csv(
    "../../data/processed/"
    "punjab_wheat_price_features_2021_2025.csv",
    index=False
)

print(
    "Punjab Wheat price features saved ✅"
)

NameError: name 'punjab_wheat_price_features' is not defined

In [158]:
punjab_wheat_coverage_check.to_csv(
    "../../data/processed/"
    "punjab_wheat_price_coverage_audit.csv",
    index=False
)

print(
    "Punjab Wheat coverage audit saved ✅"
)

NameError: name 'punjab_wheat_coverage_check' is not defined

In [159]:
# Starting Punjab Maize
from pathlib import Path
import pandas as pd

punjab_market_mapping_final = pd.read_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv"
)

print(
    "Punjab mapping entries:",
    len(punjab_market_mapping_final)
)

print(
    "Mapping conflicts:",
    (
        punjab_market_mapping_final
        .groupby("Market_Key")["District"]
        .nunique()
        > 1
    ).sum()
)

Punjab mapping entries: 196
Mapping conflicts: 0


In [160]:
print(
    "Adaptive collector available:",
    "collect_price_history_adaptive" in globals()
)

print(
    "One-month collector available:",
    "collect_one_month" in globals()
)

print(
    "Available-month checker available:",
    "get_available_months" in globals()
)

Adaptive collector available: True
One-month collector available: False
Available-month checker available: True


In [161]:
# Collecting punjab maize
driver, punjab_maize_success, punjab_maize_failed, punjab_maize_skipped, punjab_maize_unavailable = (
    collect_price_history_adaptive(
        driver=driver,
        state="Punjab",
        crop="Maize"
    )
)

NameError: name 'driver' is not defined

In [162]:
print(
    "Downloaded this run:",
    len(punjab_maize_success)
)

print(
    "Failed:",
    len(punjab_maize_failed)
)

print(
    "Already downloaded:",
    len(punjab_maize_skipped)
)

print(
    "Unavailable from AGMARKNET:",
    len(punjab_maize_unavailable)
)punjab_maize_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "punjab_maize_*.csv"
    )
)

print(
    "Punjab Maize files on disk:",
    len(punjab_maize_files)
)

SyntaxError: invalid syntax (3969819189.py, line 19)

In [163]:
punjab_maize_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "punjab_maize_*.csv"
    )
)

print(
    "Punjab Maize files on disk:",
    len(punjab_maize_files)
)

Punjab Maize files on disk: 35


In [164]:
punjab_maize_monthly = load_price_history(
    state="Punjab",
    crop="Maize"
)

NameError: name 'load_price_history' is not defined

In [165]:
print(
    "Months available:",
    punjab_maize_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Unique markets:",
    punjab_maize_monthly[
        "Market"
    ].nunique()
)

print(
    "Total rows:",
    len(punjab_maize_monthly)
)

print(
    "First month:",
    punjab_maize_monthly[
        "Month_Date"
    ].min()
)

print(
    "Last month:",
    punjab_maize_monthly[
        "Month_Date"
    ].max()
)

NameError: name 'punjab_maize_monthly' is not defined

In [166]:
import pandas as pd

punjab_market_mapping_final = pd.read_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv"
)

print(
    "Punjab mapping entries:",
    len(punjab_market_mapping_final)
)

Punjab mapping entries: 196


In [167]:
punjab_maize_monthly = load_price_history(
    state="Punjab",
    crop="Maize"
)

NameError: name 'load_price_history' is not defined

In [168]:
punjab_maize_with_district = (
    punjab_maize_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_final,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_maize_monthly' is not defined

In [169]:
print(
    "Total rows:",
    len(punjab_maize_with_district)
)

print(
    "Rows missing District:",
    punjab_maize_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_maize_with_district.loc[
        punjab_maize_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_maize_with_district' is not defined

In [170]:
punjab_maize_unmapped = (
    punjab_maize_with_district.loc[
        punjab_maize_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "New Punjab Maize markets needing mapping:",
    len(punjab_maize_unmapped)
)

print(
    punjab_maize_unmapped.to_string(
        index=False
    )
)

NameError: name 'punjab_maize_with_district' is not defined

In [171]:
mansura_mapping = pd.DataFrame(
    [
        {
            "Market_Key": "mansura",
            "District": "Ludhiana"
        }
    ]
)

In [172]:
punjab_market_mapping_final = pd.concat(
    [
        punjab_market_mapping_final[
            [
                "Market_Key",
                "District"
            ]
        ],
        mansura_mapping
    ],
    ignore_index=True
)

In [173]:
punjab_market_mapping_final = (
    punjab_market_mapping_final
    .drop_duplicates(
        subset=["Market_Key"],
        keep="last"
    )
)

In [174]:
print(
    "Mapping conflicts:",
    (
        punjab_market_mapping_final
        .groupby("Market_Key")["District"]
        .nunique()
        > 1
    ).sum()
)

Mapping conflicts: 0


In [175]:
punjab_market_mapping_final.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

print("Updated Punjab mapping saved ✅")

Updated Punjab mapping saved ✅


In [176]:
punjab_maize_with_district = (
    punjab_maize_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_final,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_maize_monthly' is not defined

In [177]:
print(
    "Rows missing District:",
    punjab_maize_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_maize_with_district.loc[
        punjab_maize_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_maize_with_district' is not defined

In [178]:
invalid_districts = sorted(
    set(
        punjab_maize_with_district[
            "District"
        ].dropna()
    )
    -
    set(
        production_punjab_districts
    )
)

print(
    "Districts not present in production:",
    invalid_districts
)

NameError: name 'punjab_maize_with_district' is not defined

In [179]:
punjab_maize_price_features = (
    build_price_features(
        punjab_maize_with_district
    )
)

NameError: name 'build_price_features' is not defined

In [180]:
print(
    "Punjab Maize feature rows:",
    len(punjab_maize_price_features)
)

print(
    "Duplicate district-crop-year rows:",
    punjab_maize_price_features
    .duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    )
    .sum()
)

print(
    "\nCoverage quality:"
)

print(
    punjab_maize_price_features[
        "Price_Coverage_Quality"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    "\nMissing volatility:",
    punjab_maize_price_features[
        "Price_Volatility_Pct"
    ]
    .isna()
    .sum()
)

print(
    "\nZero/negative average prices:",
    (
        punjab_maize_price_features[
            "Avg_Modal_Price"
        ] <= 0
    ).sum()
)

NameError: name 'punjab_maize_price_features' is not defined

In [181]:
print(
    "Duplicate district-crop-year rows:",
    punjab_maize_price_features
    .duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    )
    .sum()
)

NameError: name 'punjab_maize_price_features' is not defined

In [182]:
punjab_maize_production_keys = (
    production_history[
        (
            production_history["State"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "punjab"
        )
        &
        (
            production_history["Crop"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "maize"
        )
        &
        (
            production_history["Year"]
            .isin(
                [
                    "2021-22",
                    "2022-23",
                    "2023-24",
                    "2024-25"
                ]
            )
        )
    ]
    [
        [
            "State",
            "District",
            "Crop",
            "Year"
        ]
    ]
    .drop_duplicates()
)

In [183]:
print(
    "Punjab Maize production keys:",
    len(punjab_maize_production_keys)
)

Punjab Maize production keys: 79


In [184]:
punjab_maize_coverage_check = (
    punjab_maize_production_keys
    .merge(
        punjab_maize_price_features[
            [
                "State",
                "District",
                "Crop",
                "Year",
                "Months_Available"
            ]
        ],
        on=[
            "State",
            "District",
            "Crop",
            "Year"
        ],
        how="left",
        indicator=True
    )
)

print(
    "Production keys with price:",
    (
        punjab_maize_coverage_check["_merge"]
        == "both"
    ).sum()
)

print(
    "Production keys with NO price row:",
    (
        punjab_maize_coverage_check["_merge"]
        == "left_only"
    ).sum()
)

NameError: name 'punjab_maize_price_features' is not defined

In [185]:
punjab_maize_no_price = (
    punjab_maize_coverage_check[
        punjab_maize_coverage_check[
            "_merge"
        ] == "left_only"
    ]
    [
        [
            "District",
            "Year"
        ]
    ]
    .sort_values(
        [
            "District",
            "Year"
        ]
    )
)

print(
    punjab_maize_no_price.to_string(
        index=False
    )
)

NameError: name 'punjab_maize_coverage_check' is not defined

In [186]:
punjab_maize_with_district.to_csv(
    "../../data/processed/"
    "punjab_maize_monthly_prices_mapped_2021_2025.csv",
    index=False
)

punjab_maize_price_features.to_csv(
    "../../data/processed/"
    "punjab_maize_price_features_2021_2025.csv",
    index=False
)

punjab_maize_coverage_check.to_csv(
    "../../data/processed/"
    "punjab_maize_price_coverage_audit.csv",
    index=False
)

print("Punjab Maize completed and saved ✅")

NameError: name 'punjab_maize_with_district' is not defined

In [187]:
#Find extra 9-price rows

punjab_maize_price_vs_production = (
    punjab_maize_price_features
    .merge(
        punjab_maize_production_keys,
        on=[
            "State",
            "District",
            "Crop",
            "Year"
        ],
        how="left",
        indicator=True
    )
)

punjab_maize_extra_price_rows = (
    punjab_maize_price_vs_production[
        punjab_maize_price_vs_production[
            "_merge"
        ] == "left_only"
    ]
)

print(
    "Price rows not present in production:",
    len(punjab_maize_extra_price_rows)
)

print(
    punjab_maize_extra_price_rows[
        [
            "District",
            "Year",
            "Months_Available",
            "Avg_Modal_Price"
        ]
    ]
    .sort_values(
        [
            "District",
            "Year"
        ]
    )
    .to_string(
        index=False
    )
)

NameError: name 'punjab_maize_price_features' is not defined

In [188]:
# Inspecting 31 rows with no price 

punjab_maize_missing_price_rows = (
    punjab_maize_coverage_check[
        punjab_maize_coverage_check[
            "_merge"
        ] == "left_only"
    ]
    [
        [
            "District",
            "Year"
        ]
    ]
    .sort_values(
        [
            "District",
            "Year"
        ]
    )
)

print(
    "Production rows with no price:",
    len(punjab_maize_missing_price_rows)
)

print(
    punjab_maize_missing_price_rows.to_string(
        index=False
    )
)

NameError: name 'punjab_maize_coverage_check' is not defined

In [189]:
# Restorin alias one month collector 

def collect_one_month_with_alias(
    driver,
    state,
    agmarknet_commodity,
    project_crop,
    year,
    month
):

    # Download using AGMARKNET's commodity name
    downloaded_file = collect_one_month(
        driver=driver,
        state=state,
        crop=agmarknet_commodity,
        year=year,
        month=month
    )

    raw_folder = Path(
        "../../data/raw/prices"
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    project_crop_slug = (
        project_crop.lower()
        .replace(" ", "_")
    )

    target_file = (
        raw_folder
        /
        f"{state_slug}_"
        f"{project_crop_slug}_"
        f"{year}_"
        f"{month:02d}.csv"
    )

    # Replace old file if necessary
    if target_file.exists():
        target_file.unlink()

    downloaded_file.rename(
        target_file
    )

    return target_file

In [190]:
# Adaptive rice collector 

def collect_price_history_with_alias_adaptive(
    driver,
    state,
    agmarknet_commodity,
    project_crop,
    start_year=2021,
    end_year=2025,
    final_month_2025=6
):

    successful = []
    failed = []
    skipped = []
    unavailable = []

    raw_folder = Path(
        "../../data/raw/prices"
    )

    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    project_crop_slug = (
        project_crop.lower()
        .replace(" ", "_")
    )

    for year in range(
        start_year,
        end_year + 1
    ):

        print(
            "\n" + "#" * 70
        )

        print(
            f"Checking {state} | "
            f"{agmarknet_commodity} | {year}"
        )

        # IMPORTANT:
        # Discover months using AGMARKNET commodity,
        # not project crop name.
        available_df = get_available_months(
            driver=driver,
            state=state,
            crop=agmarknet_commodity,
            year=year
        )

        available_months = set(
            available_df[
                "Month"
            ].tolist()
        )

        if year == 2025:

            desired_months = range(
                1,
                final_month_2025 + 1
            )

        else:

            desired_months = range(
                1,
                13
            )


        for month in desired_months:

            month_name = (
                calendar.month_name[
                    month
                ]
            )

            # Project filename uses "rice"
            target_file = (
                raw_folder
                /
                (
                    f"{state_slug}_"
                    f"{project_crop_slug}_"
                    f"{year}_"
                    f"{month:02d}.csv"
                )
            )

            print(
                "\n",
                year,
                month_name
            )


            # ------------------------------------
            # Already downloaded
            # ------------------------------------

            if target_file.exists():

                print(
                    "Already downloaded → SKIPPED ✅"
                )

                skipped.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                continue


            # ------------------------------------
            # Not offered by AGMARKNET
            # ------------------------------------

            if month not in available_months:

                print(
                    "Not available in AGMARKNET"
                )

                unavailable.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                continue


            # ------------------------------------
            # Download available month
            # ------------------------------------

            completed = False

            for attempt in range(
                1,
                3
            ):

                print(
                    f"Attempt {attempt}/2"
                )

                try:

                    file_path = (
                        collect_one_month_with_alias(
                            driver=driver,
                            state=state,
                            agmarknet_commodity=agmarknet_commodity,
                            project_crop=project_crop,
                            year=year,
                            month=month
                        )
                    )

                    successful.append(
                        {
                            "Year": year,
                            "Month": month,
                            "Month_Name": month_name,
                            "File": file_path.name
                        }
                    )

                    print(
                        "Downloaded successfully ✅"
                    )

                    completed = True

                    break


                except (
                    InvalidSessionIdException,
                    WebDriverException
                ) as error:

                    print(
                        "Browser problem:",
                        str(error)[:250]
                    )

                    try:
                        driver.quit()
                    except:
                        pass

                    driver = (
                        create_new_driver()
                    )


                except Exception as error:

                    print(
                        "Attempt failed:",
                        str(error)[:250]
                    )

                    time.sleep(5)


            if not completed:

                failed.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                print(
                    "Month FAILED ❌"
                )


            time.sleep(3)


    return (
        driver,
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped),
        pd.DataFrame(unavailable)
    )

In [191]:
# startign fresh browser
try:
    driver.quit()
except:
    pass

driver = create_new_driver()

print("New Chrome session created ✅")

NameError: name 'create_new_driver' is not defined

In [192]:
driver.get(
    "https://agmarknet.gov.in/"
    "marketallwholesalemonthanalysis"
)

print(driver.title)

NameError: name 'driver' is not defined

In [193]:
def collect_price_history_with_alias_adaptive(
    driver,
    state,
    agmarknet_commodity,
    project_crop,
    start_year=2021,
    end_year=2025,
    final_month_2025=6
):

    successful = []
    failed = []
    skipped = []
    unavailable = []

    raw_folder = Path(
        "../../data/raw/prices"
    )

    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    project_crop_slug = (
        project_crop.lower()
        .replace(" ", "_")
    )


    for year in range(
        start_year,
        end_year + 1
    ):

        print(
            "\n" + "#" * 70
        )

        print(
            f"Checking {state} | "
            f"{agmarknet_commodity} | "
            f"{year}"
        )


        # ====================================================
        # 1. FIND AVAILABLE MONTHS SAFELY
        # ====================================================

        available_df = None

        for discovery_attempt in range(
            1,
            4
        ):

            try:

                print(
                    "Month discovery attempt:",
                    discovery_attempt
                )

                available_df = (
                    get_available_months(
                        driver=driver,
                        state=state,
                        crop=agmarknet_commodity,
                        year=year
                    )
                )

                break


            except (
                InvalidSessionIdException,
                WebDriverException
            ) as error:

                print(
                    "Browser died during month discovery."
                )

                print(
                    str(error)[:250]
                )

                try:
                    driver.quit()
                except:
                    pass

                print(
                    "Starting fresh Chrome..."
                )

                driver = create_new_driver()

                time.sleep(3)


            except Exception as error:

                print(
                    "Month discovery error:"
                )

                print(
                    str(error)[:250]
                )

                time.sleep(5)


        # Could not discover this year's months
        if available_df is None:

            print(
                f"Could not discover months for {year} ❌"
            )

            failed.append(
                {
                    "Year": year,
                    "Month": None,
                    "Month_Name": "YEAR_DISCOVERY_FAILED"
                }
            )

            continue


        available_months = set(
            available_df[
                "Month"
            ].tolist()
        )

        print(
            "Available months:",
            [
                calendar.month_name[m]
                for m in sorted(
                    available_months
                )
            ]
        )


        # ====================================================
        # 2. DEFINE MONTHS WE ACTUALLY NEED
        # ====================================================

        if year == 2025:

            desired_months = range(
                1,
                final_month_2025 + 1
            )

        else:

            desired_months = range(
                1,
                13
            )


        # ====================================================
        # 3. PROCESS EACH MONTH
        # ====================================================

        for month in desired_months:

            month_name = (
                calendar.month_name[
                    month
                ]
            )

            target_file = (
                raw_folder
                /
                (
                    f"{state_slug}_"
                    f"{project_crop_slug}_"
                    f"{year}_"
                    f"{month:02d}.csv"
                )
            )

            print(
                f"\n{year} {month_name}"
            )


            # --------------------------------------------
            # File already exists
            # --------------------------------------------

            if target_file.exists():

                print(
                    "Already downloaded → SKIPPED ✅"
                )

                skipped.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                continue


            # --------------------------------------------
            # Source does not offer this month
            # --------------------------------------------

            if month not in available_months:

                print(
                    "Not available in AGMARKNET"
                )

                unavailable.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                continue


            # --------------------------------------------
            # Download available month
            # --------------------------------------------

            completed = False

            for attempt in range(
                1,
                3
            ):

                print(
                    f"Download attempt {attempt}/2"
                )

                try:

                    file_path = (
                        collect_one_month_with_alias(
                            driver=driver,
                            state=state,
                            agmarknet_commodity=agmarknet_commodity,
                            project_crop=project_crop,
                            year=year,
                            month=month
                        )
                    )

                    successful.append(
                        {
                            "Year": year,
                            "Month": month,
                            "Month_Name": month_name,
                            "File": file_path.name
                        }
                    )

                    print(
                        "Downloaded successfully ✅"
                    )

                    completed = True

                    break


                except (
                    InvalidSessionIdException,
                    WebDriverException
                ) as error:

                    print(
                        "Browser problem during download:"
                    )

                    print(
                        str(error)[:250]
                    )

                    try:
                        driver.quit()
                    except:
                        pass

                    driver = create_new_driver()

                    time.sleep(3)


                except Exception as error:

                    print(
                        "Download error:"
                    )

                    print(
                        str(error)[:250]
                    )

                    time.sleep(5)


            if not completed:

                failed.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name": month_name
                    }
                )

                print(
                    "Month FAILED ❌"
                )


            time.sleep(3)


    return (
        driver,
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped),
        pd.DataFrame(unavailable)
    )

In [194]:
driver, punjab_rice_success, punjab_rice_failed, punjab_rice_skipped, punjab_rice_unavailable = (
    collect_price_history_with_alias_adaptive(
        driver=driver,
        state="Punjab",
        agmarknet_commodity="Paddy(Common)",
        project_crop="Rice"
    )
)

NameError: name 'driver' is not defined

In [195]:
print(
    "Downloaded:",
    len(punjab_rice_success)
)

print(
    "Failed:",
    len(punjab_rice_failed)
)

print(
    "Skipped:",
    len(punjab_rice_skipped)
)

print(
    "Unavailable:",
    len(punjab_rice_unavailable)
)

NameError: name 'punjab_rice_success' is not defined

In [196]:
# Load punjab rice
punjab_rice_monthly = load_price_history(
    state="Punjab",
    crop="Rice"
)

NameError: name 'load_price_history' is not defined

In [197]:
print(
    "Months available:",
    punjab_rice_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Unique markets:",
    punjab_rice_monthly[
        "Market"
    ].nunique()
)

print(
    "Total rows:",
    len(punjab_rice_monthly)
)

print(
    "First month:",
    punjab_rice_monthly[
        "Month_Date"
    ].min()
)

print(
    "Last month:",
    punjab_rice_monthly[
        "Month_Date"
    ].max()
)

NameError: name 'punjab_rice_monthly' is not defined

In [198]:
punjab_market_mapping_final = pd.read_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv"
)

print(
    "Punjab mapping entries:",
    len(punjab_market_mapping_final)
)

print(
    "Mapping conflicts:",
    (
        punjab_market_mapping_final
        .groupby(
            "Market_Key"
        )["District"]
        .nunique()
        > 1
    ).sum()
)

Punjab mapping entries: 196
Mapping conflicts: 0


In [199]:
punjab_rice_with_district = (
    punjab_rice_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_final,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_rice_monthly' is not defined

In [200]:
print(
    "Total rows:",
    len(punjab_rice_with_district)
)

print(
    "Rows missing District:",
    punjab_rice_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_rice_with_district.loc[
        punjab_rice_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_rice_with_district' is not defined

In [201]:
punjab_rice_unmapped = (
    punjab_rice_with_district.loc[
        punjab_rice_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "New Punjab Rice markets needing mapping:",
    len(punjab_rice_unmapped)
)

print(
    punjab_rice_unmapped.to_string(
        index=False
    )
)

NameError: name 'punjab_rice_with_district' is not defined

In [202]:
#Print 6 new punjab rice markets
punjab_rice_unmapped = (
    punjab_rice_with_district.loc[
        punjab_rice_with_district[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values("Market")
    .reset_index(drop=True)
)

print(
    "New Punjab Rice markets:",
    len(punjab_rice_unmapped)
)

print(
    punjab_rice_unmapped.to_string(
        index=False
    )
)

NameError: name 'punjab_rice_with_district' is not defined

In [203]:
def extract_parent_market(name):

    name = str(name)

    name = re.sub(
        r"\bAPMC\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    # Keep the main market name before parentheses
    name = name.split("(")[0]

    return name.strip()


def compact_market_key(name):

    name = str(name).lower().strip()

    name = re.sub(
        r"[^a-z0-9\s]",
        " ",
        name
    )

    name = re.sub(
        r"\bapmc\b",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        "",
        name
    )

    return name

In [204]:
punjab_rice_unmapped[
    "Parent_Market"
] = (
    punjab_rice_unmapped[
        "Market"
    ]
    .apply(
        extract_parent_market
    )
)

punjab_rice_unmapped[
    "Parent_Compact_Key"
] = (
    punjab_rice_unmapped[
        "Parent_Market"
    ]
    .apply(
        compact_market_key
    )
)

NameError: name 'punjab_rice_unmapped' is not defined

In [205]:
validated_parent_lookup = (
    punjab_market_mapping_final[
        [
            "Market_Key",
            "District"
        ]
    ]
    .copy()
)

validated_parent_lookup[
    "Validated_Compact_Key"
] = (
    validated_parent_lookup[
        "Market_Key"
    ]
    .apply(
        compact_market_key
    )
)

In [206]:
parent_key_check = (
    validated_parent_lookup
    .groupby(
        "Validated_Compact_Key"
    )["District"]
    .nunique()
)

print(
    "Parent-key conflicts:",
    (
        parent_key_check > 1
    ).sum()
)

Parent-key conflicts: 0


In [207]:
safe_parent_lookup = (
    validated_parent_lookup[
        validated_parent_lookup[
            "Validated_Compact_Key"
        ].isin(
            parent_key_check[
                parent_key_check == 1
            ].index
        )
    ]
    [
        [
            "Validated_Compact_Key",
            "Market_Key",
            "District"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Validated_Compact_Key"
        ]
    )
)

In [208]:
punjab_rice_parent_matches = (
    punjab_rice_unmapped
    .merge(
        safe_parent_lookup,
        left_on="Parent_Compact_Key",
        right_on="Validated_Compact_Key",
        how="left",
        suffixes=(
            "_Rice",
            "_Parent"
        )
    )
)

NameError: name 'punjab_rice_unmapped' is not defined

In [209]:
print(
    "Exact parent matches:",
    punjab_rice_parent_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    punjab_rice_parent_matches[
        "District"
    ].isna().sum()
)

print(
    punjab_rice_parent_matches[
        [
            "Market",
            "Parent_Market",
            "Market_Key_Parent",
            "District"
        ]
    ]
    .to_string(
        index=False
    )
)

NameError: name 'punjab_rice_parent_matches' is not defined

In [210]:
rice_new_mapping = {
    "Goniana APMC":
        "Bathinda",

    "Narot J Singh APMC":
        "Pathankot",

    "Nawan Shahar(Mandi Aur) APMC":
        "Shahid Bhagat Singh Nagar",

    "Raikot (Talwandi rai) APMC":
        "Ludhiana",

    "Sunam APMC":
        "Sangrur",

    "Talwandi Sabo(Shekhupura) APMC":
        "Bathinda"
}

In [211]:
rice_new_mapping_df = pd.DataFrame(
    [
        {
            "Market": market,
            "Market_Key": normalize_market_name(
                market
            ),
            "District": district
        }
        for market, district
        in rice_new_mapping.items()
    ]
)

print(
    rice_new_mapping_df.to_string(
        index=False
    )
)

                        Market                Market_Key                  District
                  Goniana APMC                   goniana                  Bathinda
            Narot J Singh APMC             narot j singh                 Pathankot
  Nawan Shahar(Mandi Aur) APMC   nawan shahar(mandi aur) Shahid Bhagat Singh Nagar
    Raikot (Talwandi rai) APMC     raikot (talwandi rai)                  Ludhiana
                    Sunam APMC                     sunam                   Sangrur
Talwandi Sabo(Shekhupura) APMC talwandi sabo(shekhupura)                  Bathinda


In [212]:
current_rice_unresolved_keys = set(
    punjab_rice_with_district.loc[
        punjab_rice_with_district[
            "District"
        ].isna(),
        "Market_Key"
    ]
)

rice_new_mapping_df[
    "Exists_In_Unresolved"
] = (
    rice_new_mapping_df[
        "Market_Key"
    ].isin(
        current_rice_unresolved_keys
    )
)

print(
    rice_new_mapping_df[
        [
            "Market",
            "Market_Key",
            "District",
            "Exists_In_Unresolved"
        ]
    ].to_string(index=False)
)

print(
    "\nFound:",
    rice_new_mapping_df[
        "Exists_In_Unresolved"
    ].sum()
)

NameError: name 'punjab_rice_with_district' is not defined

In [213]:
punjab_market_mapping_final = pd.concat(
    [
        punjab_market_mapping_final[
            [
                "Market_Key",
                "District"
            ]
        ],

        rice_new_mapping_df[
            [
                "Market_Key",
                "District"
            ]
        ]
    ],
    ignore_index=True
)

In [214]:
punjab_market_mapping_final = (
    punjab_market_mapping_final
    .drop_duplicates(
        subset=["Market_Key"],
        keep="last"
    )
)

In [215]:
print(
    "Mapping conflicts:",
    (
        punjab_market_mapping_final
        .groupby(
            "Market_Key"
        )["District"]
        .nunique()
        > 1
    ).sum()
)

Mapping conflicts: 0


In [216]:
punjab_market_mapping_final.to_csv(
    "../../data/processed/"
    "punjab_market_district_mapping.csv",
    index=False
)

print(
    "Punjab master mapping updated ✅"
)

Punjab master mapping updated ✅


In [217]:
punjab_rice_with_district = (
    punjab_rice_monthly
    .drop(
        columns=["District"],
        errors="ignore"
    )
    .merge(
        punjab_market_mapping_final,
        on="Market_Key",
        how="left",
        validate="many_to_one"
    )
)

NameError: name 'punjab_rice_monthly' is not defined

In [218]:
print(
    "Rows missing District:",
    punjab_rice_with_district[
        "District"
    ].isna().sum()
)

print(
    "Unmapped markets:",
    punjab_rice_with_district.loc[
        punjab_rice_with_district[
            "District"
        ].isna(),
        "Market"
    ].nunique()
)

NameError: name 'punjab_rice_with_district' is not defined

In [219]:
invalid_districts = sorted(
    set(
        punjab_rice_with_district[
            "District"
        ].dropna()
    )
    -
    set(
        production_punjab_districts
    )
)

print(
    "Districts not present in production:",
    invalid_districts
)

NameError: name 'punjab_rice_with_district' is not defined

In [220]:
punjab_rice_price_features = (
    build_price_features(
        punjab_rice_with_district
    )
)

NameError: name 'build_price_features' is not defined

In [221]:
print(
    "Punjab Rice feature rows:",
    len(punjab_rice_price_features)
)

print(
    "Duplicate district-crop-year rows:",
    punjab_rice_price_features
    .duplicated(
        subset=[
            "State",
            "District",
            "Crop",
            "Year"
        ]
    )
    .sum()
)

print(
    "\nCoverage quality:"
)

print(
    punjab_rice_price_features[
        "Price_Coverage_Quality"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    "\nMissing volatility:",
    punjab_rice_price_features[
        "Price_Volatility_Pct"
    ]
    .isna()
    .sum()
)

print(
    "\nZero/negative average prices:",
    (
        punjab_rice_price_features[
            "Avg_Modal_Price"
        ] <= 0
    ).sum()
)

NameError: name 'punjab_rice_price_features' is not defined

In [222]:
print(
    punjab_rice_price_features[
        "Months_Available"
    ]
    .value_counts()
    .sort_index(
        ascending=False
    )
)

NameError: name 'punjab_rice_price_features' is not defined

In [223]:
punjab_rice_production_keys = (
    production_history[
        (
            production_history["State"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "punjab"
        )
        &
        (
            production_history["Crop"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "rice"
        )
        &
        (
            production_history["Year"]
            .isin(
                [
                    "2021-22",
                    "2022-23",
                    "2023-24",
                    "2024-25"
                ]
            )
        )
    ]
    [
        [
            "State",
            "District",
            "Crop",
            "Year"
        ]
    ]
    .drop_duplicates()
)

print(
    "Punjab Rice production keys:",
    len(punjab_rice_production_keys)
)

Punjab Rice production keys: 92


In [224]:
punjab_rice_coverage_check = (
    punjab_rice_production_keys
    .merge(
        punjab_rice_price_features[
            [
                "State",
                "District",
                "Crop",
                "Year",
                "Months_Available"
            ]
        ],
        on=[
            "State",
            "District",
            "Crop",
            "Year"
        ],
        how="left",
        indicator=True
    )
)

print(
    "Production keys with price:",
    (
        punjab_rice_coverage_check[
            "_merge"
        ] == "both"
    ).sum()
)

print(
    "Production keys with NO price row:",
    (
        punjab_rice_coverage_check[
            "_merge"
        ] == "left_only"
    ).sum()
)

NameError: name 'punjab_rice_price_features' is not defined

In [225]:
punjab_rice_no_price = (
    punjab_rice_coverage_check[
        punjab_rice_coverage_check[
            "_merge"
        ] == "left_only"
    ]
    [
        [
            "District",
            "Year"
        ]
    ]
    .sort_values(
        [
            "District",
            "Year"
        ]
    )
)

print(
    punjab_rice_no_price.to_string(
        index=False
    )
)

NameError: name 'punjab_rice_coverage_check' is not defined

In [226]:
punjab_rice_price_vs_production = (
    punjab_rice_price_features
    .merge(
        punjab_rice_production_keys,
        on=[
            "State",
            "District",
            "Crop",
            "Year"
        ],
        how="left",
        indicator=True
    )
)

punjab_rice_extra_price_rows = (
    punjab_rice_price_vs_production[
        punjab_rice_price_vs_production[
            "_merge"
        ] == "left_only"
    ]
)

print(
    "Price rows not present in production:",
    len(punjab_rice_extra_price_rows)
)

print(
    punjab_rice_extra_price_rows[
        [
            "District",
            "Year",
            "Months_Available",
            "Avg_Modal_Price"
        ]
    ]
    .sort_values(
        [
            "District",
            "Year"
        ]
    )
    .to_string(
        index=False
    )
)

NameError: name 'punjab_rice_price_features' is not defined

In [227]:
punjab_rice_with_district.to_csv(
    "../../data/processed/"
    "punjab_rice_monthly_prices_mapped_2021_2025.csv",
    index=False
)

punjab_rice_price_features.to_csv(
    "../../data/processed/"
    "punjab_rice_price_features_2021_2025.csv",
    index=False
)

punjab_rice_coverage_check.to_csv(
    "../../data/processed/"
    "punjab_rice_price_coverage_audit.csv",
    index=False
)

print(
    "Punjab Rice processing saved ✅"
)

NameError: name 'punjab_rice_with_district' is not defined

In [228]:
punjab_price_summary = pd.DataFrame(
    [
        {
            "Crop": "Wheat",
            "Production_Keys": 92,
            "Keys_With_Price": 87,
            "Keys_Without_Price": 5
        },
        {
            "Crop": "Maize",
            "Production_Keys": 79,
            "Keys_With_Price": 48,
            "Keys_Without_Price": 31
        },
        {
            "Crop": "Rice",
            "Production_Keys": 92,
            "Keys_With_Price": 89,
            "Keys_Without_Price": 3
        }
    ]
)

punjab_price_summary["Coverage_Pct"] = (
    punjab_price_summary["Keys_With_Price"]
    /
    punjab_price_summary["Production_Keys"]
    * 100
)

punjab_price_summary

,Crop,Production_Keys,Keys_With_Price,Keys_Without_Price,Coverage_Pct
0,Wheat,92,87,5,94.565217
1,Maize,79,48,31,60.759494
2,Rice,92,89,3,96.739130


In [229]:
punjab_price_summary.to_csv(
    "../../data/processed/"
    "punjab_price_coverage_summary.csv",
    index=False
)

print("Punjab price coverage summary saved ✅")

Punjab price coverage summary saved ✅


In [230]:
print(
    "collect_one_month:",
    "collect_one_month" in globals()
)

print(
    "get_available_months:",
    "get_available_months" in globals()
)

print(
    "collect_price_history_adaptive:",
    "collect_price_history_adaptive" in globals()
)

print(
    "load_price_history:",
    "load_price_history" in globals()
)

collect_one_month: False
get_available_months: True
collect_price_history_adaptive: True
load_price_history: False


In [231]:
#Recover the complete AGMARKNET collection toolkit

# ============================================================
# COMPLETE AGMARKNET COLLECTION TOOLKIT RECOVERY
# ============================================================

import pandas as pd
import re
import time
import calendar
import shutil

from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import (
    InvalidSessionIdException,
    WebDriverException,
    TimeoutException
)


# ============================================================
# 1. CREATE FRESH CHROME DRIVER
# ============================================================

def create_new_driver():

    driver = webdriver.Chrome()

    time.sleep(2)

    return driver


# ============================================================
# 2. SELECT AGMARKNET DROPDOWN OPTION
# ============================================================

def select_exact_dropdown_option(
    driver,
    label_text,
    option_text
):

    label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                f"//label[normalize-space()='{label_text}']"
            )
        )
    )

    trigger = label.find_element(
        By.XPATH,
        "./preceding-sibling::div[1]"
    )

    if trigger.get_attribute("tabindex") == "-1":

        raise Exception(
            f"{label_text} dropdown is disabled."
        )


    # First try the normal AGMARKNET option structure
    option_xpath = (
        "//div[contains(@class,'cursor-pointer')]"
        f"[.//span[normalize-space()='{option_text}']]"
    )

    visible_options = [
        element
        for element in driver.find_elements(
            By.XPATH,
            option_xpath
        )
        if element.is_displayed()
    ]


    # If options are not visible yet, open dropdown
    if not visible_options:

        driver.execute_script(
            "arguments[0].click();",
            trigger
        )

        time.sleep(1)

        visible_options = WebDriverWait(
            driver,
            20
        ).until(
            lambda d: [
                element
                for element in d.find_elements(
                    By.XPATH,
                    option_xpath
                )
                if element.is_displayed()
            ]
        )


    driver.execute_script(
        "arguments[0].click();",
        visible_options[0]
    )

    time.sleep(1)

    print(
        f"{label_text} → {option_text} selected"
    )


# ============================================================
# 3. NORMALIZE MARKET NAME
# ============================================================

def normalize_market_name(name):

    name = str(name).strip().lower()

    name = re.sub(
        r"\bapmc\b",
        "",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    )

    return name.strip()


# ============================================================
# 4. CLEAN ONE AGMARKNET CSV
# ============================================================

def clean_monthly_price_csv(
    file_path,
    state,
    crop,
    year,
    month
):

    # Row 1 = report title
    # Row 2 = actual header
    df = pd.read_csv(
        file_path,
        header=1
    )

    current_price_column = (
        df.columns[1]
    )

    df = df[
        [
            "Market",
            current_price_column
        ]
    ].copy()

    df = df.rename(
        columns={
            current_price_column:
            "Monthly_Price"
        }
    )


    # Clean market text
    df["Market"] = (
        df["Market"]
        .astype(str)
        .str.strip()
    )


    # Keep genuine market rows only
    df = df[
        df["Market"]
        .str.lower()
        .str.endswith("apmc")
    ].copy()


    # Convert invalid price values to NaN
    df["Monthly_Price"] = (
        pd.to_numeric(
            df["Monthly_Price"],
            errors="coerce"
        )
    )


    df["State"] = state
    df["Crop"] = crop
    df["Calendar_Year"] = year
    df["Month"] = month

    df["Month_Date"] = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )


    return df[
        [
            "State",
            "Crop",
            "Calendar_Year",
            "Month",
            "Month_Date",
            "Market",
            "Monthly_Price"
        ]
    ]


# ============================================================
# 5. LOAD SAVED PRICE HISTORY
# ============================================================

def load_price_history(
    state,
    crop,
    raw_folder="../../data/raw/prices"
):

    raw_folder = Path(
        raw_folder
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    files = sorted(
        raw_folder.glob(
            f"{state_slug}_{crop_slug}_*.csv"
        )
    )

    print(
        f"{state} | {crop}"
    )

    print(
        "Raw monthly files found:",
        len(files)
    )


    if len(files) == 0:

        raise ValueError(
            f"No raw files found for "
            f"{state} - {crop}"
        )


    all_months = []


    for file_path in files:

        match = re.search(
            r"_(\d{4})_(\d{2})$",
            file_path.stem
        )


        if not match:

            print(
                "Skipped:",
                file_path.name
            )

            continue


        year = int(
            match.group(1)
        )

        month = int(
            match.group(2)
        )


        cleaned = (
            clean_monthly_price_csv(
                file_path=file_path,
                state=state,
                crop=crop,
                year=year,
                month=month
            )
        )


        all_months.append(
            cleaned
        )


    if len(all_months) == 0:

        raise ValueError(
            "Files existed but no monthly "
            "tables could be created."
        )


    monthly_df = pd.concat(
        all_months,
        ignore_index=True
    )


    monthly_df[
        "Market_Key"
    ] = (
        monthly_df[
            "Market"
        ]
        .apply(
            normalize_market_name
        )
    )


    print(
        "Rows:",
        len(monthly_df)
    )

    print(
        "Months:",
        monthly_df[
            "Month_Date"
        ].nunique()
    )

    print(
        "Markets:",
        monthly_df[
            "Market"
        ].nunique()
    )


    return monthly_df


# ============================================================
# 6. DISCOVER WHICH MONTHS AGMARKNET ACTUALLY OFFERS
# ============================================================

def get_available_months(
    driver,
    state,
    crop,
    year
):

    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )


    driver.get(
        filter_url
    )

    time.sleep(3)


    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )


    month_label = WebDriverWait(
        driver,
        20
    ).until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//label[normalize-space()='Month']"
            )
        )
    )


    month_trigger = (
        month_label.find_element(
            By.XPATH,
            "./preceding-sibling::div[1]"
        )
    )


    driver.execute_script(
        "arguments[0].click();",
        month_trigger
    )

    time.sleep(2)


    month_names = [
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December"
    ]


    available = []


    for month_number, month_name in enumerate(
        month_names,
        start=1
    ):

        matches = (
            driver.find_elements(
                By.XPATH,
                f"//*[normalize-space()='{month_name}']"
            )
        )


        visible_matches = [
            element
            for element in matches
            if element.is_displayed()
        ]


        if visible_matches:

            available.append(
                {
                    "Month":
                    month_number,

                    "Month_Name":
                    month_name
                }
            )


    return pd.DataFrame(
        available
    )


# ============================================================
# 7. DOWNLOAD ONE MONTH
# ============================================================

def collect_one_month(
    driver,
    state,
    crop,
    year,
    month
):

    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )


    driver.get(
        filter_url
    )

    time.sleep(2)


    month_name = (
        calendar.month_name[
            month
        ]
    )


    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )

    select_exact_dropdown_option(
        driver,
        "Month",
        month_name
    )


    # --------------------------------------------------------
    # Submit report
    # --------------------------------------------------------

    submit_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Submit']"
            )
        )
    )


    driver.execute_script(
        "arguments[0].click();",
        submit_button
    )


    # --------------------------------------------------------
    # Wait for Export button
    # --------------------------------------------------------

    export_button = WebDriverWait(
        driver,
        30
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//*[normalize-space()='Export']"
            )
        )
    )


    downloads_folder = (
        Path.home()
        / "Downloads"
    )


    existing_csvs = set(
        downloads_folder.glob(
            "*.csv"
        )
    )


    driver.execute_script(
        "arguments[0].click();",
        export_button
    )

    time.sleep(1)


    # --------------------------------------------------------
    # Click Export as CSV
    # --------------------------------------------------------

    csv_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//*[contains(normalize-space(),'Export as CSV')]"
            )
        )
    )


    driver.execute_script(
        "arguments[0].click();",
        csv_button
    )


    # --------------------------------------------------------
    # Detect downloaded CSV
    # --------------------------------------------------------

    downloaded_file = None


    for _ in range(60):

        current_csvs = set(
            downloads_folder.glob(
                "*.csv"
            )
        )


        new_files = (
            current_csvs
            - existing_csvs
        )


        if new_files:

            downloaded_file = max(
                new_files,
                key=lambda file:
                file.stat().st_mtime
            )

            break


        time.sleep(1)


    if downloaded_file is None:

        raise TimeoutException(
            "CSV download was not detected."
        )


    # --------------------------------------------------------
    # Move into project
    # --------------------------------------------------------

    raw_folder = Path(
        "../../data/raw/prices"
    )


    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )


    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )


    target_file = (
        raw_folder
        /
        (
            f"{state_slug}_"
            f"{crop_slug}_"
            f"{year}_"
            f"{month:02d}.csv"
        )
    )


    if target_file.exists():

        target_file.unlink()


    shutil.move(
        str(downloaded_file),
        str(target_file)
    )


    return target_file


# ============================================================
# 8. SAFE ADAPTIVE HISTORY COLLECTOR
# ============================================================

def collect_price_history_adaptive(
    driver,
    state,
    crop,
    start_year=2021,
    end_year=2025,
    final_month_2025=6
):

    successful = []
    failed = []
    skipped = []
    unavailable = []


    raw_folder = Path(
        "../../data/raw/prices"
    )

    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )


    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )


    # ========================================================
    # LOOP THROUGH YEARS
    # ========================================================

    for year in range(
        start_year,
        end_year + 1
    ):

        print(
            "\n" + "#" * 70
        )

        print(
            f"{state} | {crop} | {year}"
        )


        # ----------------------------------------------------
        # DISCOVER MONTHS SAFELY
        # ----------------------------------------------------

        available_df = None


        for discovery_attempt in range(
            1,
            4
        ):

            try:

                print(
                    "Month discovery attempt:",
                    discovery_attempt
                )


                available_df = (
                    get_available_months(
                        driver=driver,
                        state=state,
                        crop=crop,
                        year=year
                    )
                )


                break


            except (
                InvalidSessionIdException,
                WebDriverException
            ) as error:

                print(
                    "Browser session died "
                    "during month discovery."
                )

                print(
                    str(error)[:200]
                )


                try:
                    driver.quit()
                except:
                    pass


                print(
                    "Starting new Chrome session..."
                )


                driver = (
                    create_new_driver()
                )


                time.sleep(3)


            except Exception as error:

                print(
                    "Month discovery error:"
                )

                print(
                    str(error)[:200]
                )

                time.sleep(5)


        # ----------------------------------------------------
        # Could not discover this year
        # ----------------------------------------------------

        if available_df is None:

            print(
                f"Year discovery failed: {year} ❌"
            )


            failed.append(
                {
                    "Year": year,
                    "Month": None,
                    "Month_Name":
                    "YEAR_DISCOVERY_FAILED"
                }
            )


            continue


        available_months = set(
            available_df[
                "Month"
            ].tolist()
        )


        print(
            "Available months:",
            [
                calendar.month_name[m]
                for m
                in sorted(
                    available_months
                )
            ]
        )


        # ----------------------------------------------------
        # Desired project period
        # ----------------------------------------------------

        if year == 2025:

            desired_months = range(
                1,
                final_month_2025 + 1
            )

        else:

            desired_months = range(
                1,
                13
            )


        # ====================================================
        # LOOP THROUGH MONTHS
        # ====================================================

        for month in desired_months:

            month_name = (
                calendar.month_name[
                    month
                ]
            )


            target_file = (
                raw_folder
                /
                (
                    f"{state_slug}_"
                    f"{crop_slug}_"
                    f"{year}_"
                    f"{month:02d}.csv"
                )
            )


            print(
                f"\n{year} {month_name}"
            )


            # ------------------------------------------------
            # Already downloaded
            # ------------------------------------------------

            if target_file.exists():

                print(
                    "Already downloaded → SKIPPED ✅"
                )


                skipped.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name":
                        month_name
                    }
                )


                continue


            # ------------------------------------------------
            # AGMARKNET does not provide this month
            # ------------------------------------------------

            if month not in available_months:

                print(
                    "Not available in AGMARKNET"
                )


                unavailable.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name":
                        month_name
                    }
                )


                continue


            # ------------------------------------------------
            # DOWNLOAD
            # ------------------------------------------------

            completed = False


            for attempt in range(
                1,
                3
            ):

                print(
                    f"Download attempt {attempt}/2"
                )


                try:

                    file_path = (
                        collect_one_month(
                            driver=driver,
                            state=state,
                            crop=crop,
                            year=year,
                            month=month
                        )
                    )


                    successful.append(
                        {
                            "Year": year,
                            "Month": month,
                            "Month_Name":
                            month_name,
                            "File":
                            file_path.name
                        }
                    )


                    print(
                        "Downloaded successfully ✅"
                    )


                    completed = True

                    break


                except (
                    InvalidSessionIdException,
                    WebDriverException
                ) as error:

                    print(
                        "Browser problem:"
                    )

                    print(
                        str(error)[:200]
                    )


                    try:
                        driver.quit()
                    except:
                        pass


                    driver = (
                        create_new_driver()
                    )


                    time.sleep(3)


                except Exception as error:

                    print(
                        "Download error:"
                    )

                    print(
                        str(error)[:200]
                    )


                    time.sleep(5)


            if not completed:

                failed.append(
                    {
                        "Year": year,
                        "Month": month,
                        "Month_Name":
                        month_name
                    }
                )


                print(
                    "Month FAILED ❌"
                )


            # Polite delay
            time.sleep(3)


    return (
        driver,
        pd.DataFrame(successful),
        pd.DataFrame(failed),
        pd.DataFrame(skipped),
        pd.DataFrame(unavailable)
    )


print(
    "AGMARKNET recovery toolkit loaded ✅"
)

AGMARKNET recovery toolkit loaded ✅


In [232]:
#Verifying everything is back
print(
    "collect_one_month:",
    "collect_one_month" in globals()
)

print(
    "get_available_months:",
    "get_available_months" in globals()
)

print(
    "collect_price_history_adaptive:",
    "collect_price_history_adaptive" in globals()
)

print(
    "load_price_history:",
    "load_price_history" in globals()
)

print(
    "create_new_driver:",
    "create_new_driver" in globals()
)

collect_one_month: True
get_available_months: True
collect_price_history_adaptive: True
load_price_history: True
create_new_driver: True


In [233]:
#Starting a new fresh browser
try:
    driver.quit()
except:
    pass

driver = create_new_driver()

driver.get(
    "https://agmarknet.gov.in/"
    "marketallwholesalemonthanalysis"
)

print(
    "Browser working ✅"
)

print(
    driver.title
)

Browser working ✅
Market All Prices (Month)


In [234]:
#Haryana wheat month test
haryana_2021_months = get_available_months(
    driver=driver,
    state="Haryana",
    crop="Wheat",
    year=2021
)

print(
    haryana_2021_months
)

Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
   Month Month_Name
0      3      March
1      4      April
2      5        May
3      6       June


In [235]:
print(
    "Available Haryana Wheat months in 2021:"
)

print(
    haryana_2021_months[
        "Month_Name"
    ].tolist()
)

Available Haryana Wheat months in 2021:
['March', 'April', 'May', 'June']


In [236]:
# Collect haryana wheat
driver, haryana_wheat_success, haryana_wheat_failed, haryana_wheat_skipped, haryana_wheat_unavailable = (
    collect_price_history_adaptive(
        driver=driver,
        state="Haryana",
        crop="Wheat"
    )
)


######################################################################
Haryana | Wheat | 2021
Month discovery attempt: 1
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Available months: ['March', 'April', 'May', 'June']

2021 January
Not available in AGMARKNET

2021 February
Not available in AGMARKNET

2021 March
Download attempt 1/2
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → March selected
Browser problem:
Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff63a369d55+5a55]
	chromedriver!(No symbol) [0x7ff63a28f540]
	chromedriver!(No symbol) [0x7ff63a0b45fd]
	chromedriver!(No symbol) [0x7ff63a10f
Download attempt 2/2
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → March selected
Browser problem:
Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff63a369d55+5a55]
	chromedriver!(No symbol) [0x7ff63a28f540]
	chromedriver!(No symbol) [0x7ff63a0b45f

In [237]:
print(
    "Downloaded this run:",
    len(haryana_wheat_success)
)

print(
    "Failed:",
    len(haryana_wheat_failed)
)

print(
    "Already downloaded:",
    len(haryana_wheat_skipped)
)

print(
    "Unavailable:",
    len(haryana_wheat_unavailable)
)

Downloaded this run: 0
Failed: 38
Already downloaded: 0
Unavailable: 16


In [238]:
#Testing one available haryana month

try:
    driver.quit()
except:
    pass

driver = create_new_driver()

print("Fresh browser created ✅")

Fresh browser created ✅


In [239]:
test_months = get_available_months(
    driver=driver,
    state="Haryana",
    crop="Wheat",
    year=2021
)

print(test_months)

Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
   Month Month_Name
0      3      March
1      4      April
2      5        May
3      6       June


In [240]:
test_year = 2021

test_month = int(
    test_months.iloc[0]["Month"]
)

test_month_name = (
    calendar.month_name[
        test_month
    ]
)

print(
    "Testing:",
    test_year,
    test_month_name
)

Testing: 2021 March


In [242]:
#Manulally opening the report instead of downloading


filter_url = (
    "https://agmarknet.gov.in/"
    "marketallwholesalemonthanalysis"
)

driver.get(filter_url)

time.sleep(3)

select_exact_dropdown_option(
    driver,
    "Select Commodity",
    "Wheat"
)

select_exact_dropdown_option(
    driver,
    "State/UT",
    "Haryana"
)

select_exact_dropdown_option(
    driver,
    "Year",
    str(test_year)
)

select_exact_dropdown_option(
    driver,
    "Month",
    test_month_name
)

print("All filters selected ✅")

Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → March selected
All filters selected ✅


In [243]:
submit_button = WebDriverWait(
    driver,
    20
).until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='Submit']"
        )
    )
)

driver.execute_script(
    "arguments[0].click();",
    submit_button
)

time.sleep(7)

print(
    "Current URL:",
    driver.current_url
)

Current URL: https://agmarknet.gov.in/market-all-wholesale-month-arrival-table


In [244]:
#Checking what actually returned

body_text = (
    driver.find_element(
        By.TAG_NAME,
        "body"
    )
    .text
)

print(
    body_text[:4000]
)

Search pages
Type at least 2 characters to search pages.
Sitemap
Skip to Main Content
Feedback
A+
A
A-
Screen Reader
A
A
English
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Home
Reports
Market Profile
Market Atlas
Helpdesk
Research Studies
What's New
Official Login
Market-wise Wholesale Prices Monthly Analysis (All Districts)
Back
Main Menu
Export
Market-wise Wholesale Prices Monthly Analysis (All Districts) for Wheat in Haryana - March, 2021
Market Prices March, 2021 (Rs./Quintal) Prices February, 2021 (Rs./Quintal) Prices March, 2020 (Rs./Quintal) Change (Over Previous Month) Change (Over Previous Year)
Naraingarh APMC 1,926.00 - - - -
Average 1,926.00 - - - -
Government of India
Department of Agriculture & Farmers Welfare
Directorate of Marketing & Inspection
Download the Agmarknet 2.0 App

USEFUL LINKS
About Us
Contact Us
Web Information Manager
Accessibility
Feedback
QUICK LINKS
Website Policy
Privacy Policy
Hyperlinking Po

In [245]:
export_matches = driver.find_elements(
    By.XPATH,
    "//*[normalize-space()='Export']"
)

visible_exports = [
    e
    for e in export_matches
    if e.is_displayed()
]

print(
    "Visible Export buttons:",
    len(visible_exports)
)

Visible Export buttons: 3


In [246]:
tables = driver.find_elements(
    By.TAG_NAME,
    "table"
)

visible_tables = [
    table
    for table in tables
    if table.is_displayed()
]

print(
    "Visible tables:",
    len(visible_tables)
)

Visible tables: 1


In [247]:
#Inspecting export 3 button
export_elements = [
    element
    for element in driver.find_elements(
        By.XPATH,
        "//*[normalize-space()='Export']"
    )
    if element.is_displayed()
]

print(
    "Visible Export elements:",
    len(export_elements)
)

for i, element in enumerate(
    export_elements
):

    print(
        "\n" + "=" * 60
    )

    print(
        "INDEX:",
        i
    )

    print(
        "TAG:",
        element.tag_name
    )

    print(
        "CLASS:",
        element.get_attribute("class")
    )

    print(
        "HTML:"
    )

    print(
        element.get_attribute(
            "outerHTML"
        )[:1000]
    )

Visible Export elements: 3

INDEX: 0
TAG: div
CLASS: relative export-dropdown
HTML:
<div class="relative export-dropdown"><div class="relative"><button type="button" class="
        cursor-pointer 
        bg-green-700 text-white py-2 px-4 rounded-md hover:bg-green-700 transition flex items-center
      "><svg xmlns="http://www.w3.org/2000/svg" class="h-5 w-5 mr-1" viewBox="0 0 20 20" fill="currentColor"><path fill-rule="evenodd" d="M3 17a1 1 0 011-1h12a1 1 0 110 2H4a1 1 0 01-1-1zm3.293-7.707a1 1 0 011.414 0L9 10.586V3a1 1 0 112 0v7.586l1.293-1.293a1 1 0 111.414 1.414l-3 3a1 1 0 01-1.414 0l-3-3a1 1 0 010-1.414z" clip-rule="evenodd"></path></svg>Export<svg stroke="currentColor" fill="none" stroke-width="2" viewBox="0 0 24 24" stroke-linecap="round" stroke-linejoin="round" class="ml-1 h-4 w-4 transition-transform " height="1em" width="1em" xmlns="http://www.w3.org/2000/svg"><polyline points="6 9 12 15 18 9"></polyline></svg></button></div></div>

INDEX: 1
TAG: div
CLASS: relative
HTML:
<

In [248]:
export_buttons = [
    element
    for element in driver.find_elements(
        By.XPATH,
        "//button[normalize-space()='Export' or .//*[normalize-space()='Export']]"
    )
    if element.is_displayed()
]

print(
    "Visible actual Export buttons:",
    len(export_buttons)
)

for i, button in enumerate(
    export_buttons
):

    print(
        "\nBUTTON:",
        i
    )

    print(
        button.get_attribute(
            "outerHTML"
        )[:1200]
    )

Visible actual Export buttons: 1

BUTTON: 0
<button type="button" class="
        cursor-pointer 
        bg-green-700 text-white py-2 px-4 rounded-md hover:bg-green-700 transition flex items-center
      "><svg xmlns="http://www.w3.org/2000/svg" class="h-5 w-5 mr-1" viewBox="0 0 20 20" fill="currentColor"><path fill-rule="evenodd" d="M3 17a1 1 0 011-1h12a1 1 0 110 2H4a1 1 0 01-1-1zm3.293-7.707a1 1 0 011.414 0L9 10.586V3a1 1 0 112 0v7.586l1.293-1.293a1 1 0 111.414 1.414l-3 3a1 1 0 01-1.414 0l-3-3a1 1 0 010-1.414z" clip-rule="evenodd"></path></svg>Export<svg stroke="currentColor" fill="none" stroke-width="2" viewBox="0 0 24 24" stroke-linecap="round" stroke-linejoin="round" class="ml-1 h-4 w-4 transition-transform " height="1em" width="1em" xmlns="http://www.w3.org/2000/svg"><polyline points="6 9 12 15 18 9"></polyline></svg></button>


In [249]:
#Click real export button
real_export_button = export_buttons[0]

driver.execute_script(
    "arguments[0].scrollIntoView({block:'center'});",
    real_export_button
)

time.sleep(1)

driver.execute_script(
    "arguments[0].click();",
    real_export_button
)

time.sleep(2)

In [250]:
#Inspecting csv option
csv_options = [
    element
    for element in driver.find_elements(
        By.XPATH,
        "//*[contains(normalize-space(),'Export as CSV')]"
    )
    if element.is_displayed()
]

print(
    "Visible CSV options:",
    len(csv_options)
)

for i, element in enumerate(
    csv_options
):

    print(
        "\nCSV OPTION:",
        i
    )

    print(
        "TAG:",
        element.tag_name
    )

    print(
        "TEXT:",
        repr(element.text)
    )

    print(
        "HTML:"
    )

    print(
        element.get_attribute(
            "outerHTML"
        )[:1000]
    )

Visible CSV options: 12

CSV OPTION: 0
TAG: html
TEXT: "Search pages\nType at least 2 characters to search pages.\nSitemap\nSkip to Main Content\nFeedback\nA+\nA\nA-\nScreen Reader\nA\nA\nEnglish\nGovernment of India\nDepartment of Agriculture & Farmers Welfare\nDirectorate of Marketing & Inspection\nHome\nReports\nMarket Profile\nMarket Atlas\nHelpdesk\nResearch Studies\nWhat's New\nOfficial Login\nMarket-wise Wholesale Prices Monthly Analysis (All Districts)\nBack\nMain Menu\nExport\nExport as Excel\nExport as PDF\nExport as CSV\nMarket-wise Wholesale Prices Monthly Analysis (All Districts) for Wheat in Haryana - March, 2021\nMarket Prices March, 2021 (Rs./Quintal) Prices February, 2021 (Rs./Quintal) Prices March, 2020 (Rs./Quintal) Change (Over Previous Month) Change (Over Previous Year)\nNaraingarh APMC 1,926.00 - - - -\nAverage 1,926.00 - - - -\nGovernment of India\nDepartment of Agriculture & Farmers Welfare\nDirectorate of Marketing & Inspection\nDownload the Agmarknet 2.0 App\n

In [251]:
#Find the exact “Export as CSV” element
exact_csv_elements = [
    element
    for element in driver.find_elements(
        By.XPATH,
        "//*[normalize-space(text())='Export as CSV']"
    )
    if element.is_displayed()
]

print(
    "Exact visible CSV text elements:",
    len(exact_csv_elements)
)

for i, element in enumerate(
    exact_csv_elements
):
    print(
        "\nINDEX:",
        i
    )

    print(
        "TAG:",
        element.tag_name
    )

    print(
        "CLASS:",
        element.get_attribute("class")
    )

    print(
        "HTML:"
    )

    print(
        element.get_attribute(
            "outerHTML"
        )[:1000]
    )

Exact visible CSV text elements: 1

INDEX: 0
TAG: button
CLASS: w-full text-left px-4 py-2 text-sm text-black-500 flex items-center transition-colors
HTML:
<button class="w-full text-left px-4 py-2 text-sm text-black-500 flex items-center transition-colors">Export as CSV</button>


In [252]:
#Find its clickable parent
csv_text = exact_csv_elements[0]

csv_clickable = driver.execute_script(
    """
    return arguments[0].closest(
        'button, a, [role="menuitem"], [role="option"], .cursor-pointer'
    );
    """,
    csv_text
)

print(
    "Clickable tag:",
    csv_clickable.tag_name
)

print(
    "Clickable HTML:"
)

print(
    csv_clickable.get_attribute(
        "outerHTML"
    )[:1500]
)

Clickable tag: button
Clickable HTML:
<button class="w-full text-left px-4 py-2 text-sm text-black-500 flex items-center transition-colors">Export as CSV</button>


In [253]:
#Testing one csv download manually
downloads_folder = (
    Path.home()
    / "Downloads"
)

before_csvs = set(
    downloads_folder.glob(
        "*.csv"
    )
)

In [254]:
driver.execute_script(
    "arguments[0].click();",
    csv_clickable
)

print(
    "Clicked real Export as CSV ✅"
)


Clicked real Export as CSV ✅


In [255]:
downloaded_file = None

for _ in range(60):

    current_csvs = set(
        downloads_folder.glob(
            "*.csv"
        )
    )

    new_csvs = (
        current_csvs
        - before_csvs
    )

    if new_csvs:

        downloaded_file = max(
            new_csvs,
            key=lambda f:
            f.stat().st_mtime
        )

        break

    time.sleep(1)

In [256]:
print(
    "Downloaded file:",
    downloaded_file
)

Downloaded file: C:\Users\richa\Downloads\Marketwise_Wholesale_Prices_Monthly_Analysis_(All_Districts)_Report_30-09-2026_04-01-35_PM.csv


In [258]:
#Replacing collect_one_month() with the corrected version

def collect_one_month(
    driver,
    state,
    crop,
    year,
    month
):

    filter_url = (
        "https://agmarknet.gov.in/"
        "marketallwholesalemonthanalysis"
    )

    # --------------------------------------------------------
    # Open filter page
    # --------------------------------------------------------

    driver.get(filter_url)

    time.sleep(2)

    month_name = calendar.month_name[month]


    # --------------------------------------------------------
    # Select filters
    # --------------------------------------------------------

    select_exact_dropdown_option(
        driver,
        "Select Commodity",
        crop
    )

    select_exact_dropdown_option(
        driver,
        "State/UT",
        state
    )

    select_exact_dropdown_option(
        driver,
        "Year",
        str(year)
    )

    select_exact_dropdown_option(
        driver,
        "Month",
        month_name
    )


    # --------------------------------------------------------
    # Submit
    # --------------------------------------------------------

    submit_button = WebDriverWait(
        driver,
        20
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[normalize-space()='Submit']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        submit_button
    )


    # --------------------------------------------------------
    # Wait for report table
    # --------------------------------------------------------

    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.TAG_NAME,
                "table"
            )
        )
    )

    time.sleep(2)


    # --------------------------------------------------------
    # Find actual Export button
    # --------------------------------------------------------

    export_xpath = (
        "//button[normalize-space()='Export' "
        "or .//*[normalize-space()='Export']]"
    )

    export_buttons = [
        element
        for element in driver.find_elements(
            By.XPATH,
            export_xpath
        )
        if element.is_displayed()
    ]

    if len(export_buttons) == 0:

        raise Exception(
            "Visible Export button not found."
        )

    export_button = export_buttons[0]

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        export_button
    )

    time.sleep(1)

    driver.execute_script(
        "arguments[0].click();",
        export_button
    )

    time.sleep(2)


    # --------------------------------------------------------
    # Find exact Export as CSV text
    # --------------------------------------------------------

    csv_xpath = (
        "//*[normalize-space(text())='Export as CSV']"
    )

    exact_csv_elements = [
        element
        for element in driver.find_elements(
            By.XPATH,
            csv_xpath
        )
        if element.is_displayed()
    ]

    if len(exact_csv_elements) == 0:

        raise Exception(
            "Export as CSV option not found."
        )

    csv_text = exact_csv_elements[0]


    # --------------------------------------------------------
    # Find nearest clickable parent
    # --------------------------------------------------------

    csv_clickable = driver.execute_script(
        """
        return arguments[0].closest(
            'button, a, [role="menuitem"], [role="option"], .cursor-pointer'
        );
        """,
        csv_text
    )

    if csv_clickable is None:
        csv_clickable = csv_text


    # --------------------------------------------------------
    # Record existing downloads
    # --------------------------------------------------------

    downloads_folder = (
        Path.home()
        / "Downloads"
    )

    existing_csvs = set(
        downloads_folder.glob(
            "*.csv"
        )
    )


    # --------------------------------------------------------
    # Click CSV
    # --------------------------------------------------------

    driver.execute_script(
        "arguments[0].click();",
        csv_clickable
    )


    # --------------------------------------------------------
    # Wait for new downloaded CSV
    # --------------------------------------------------------

    downloaded_file = None

    for _ in range(60):

        current_csvs = set(
            downloads_folder.glob(
                "*.csv"
            )
        )

        new_files = (
            current_csvs
            - existing_csvs
        )

        if len(new_files) > 0:

            candidate = max(
                new_files,
                key=lambda file:
                file.stat().st_mtime
            )

            downloaded_file = candidate

            break

        time.sleep(1)


    if downloaded_file is None:

        raise TimeoutException(
            "CSV download was not detected "
            "within 60 seconds."
        )


    # --------------------------------------------------------
    # Create project destination
    # --------------------------------------------------------

    raw_folder = Path(
        "../../data/raw/prices"
    )

    raw_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    state_slug = (
        state.lower()
        .replace(" ", "_")
    )

    crop_slug = (
        crop.lower()
        .replace(" ", "_")
    )

    target_file = (
        raw_folder
        /
        f"{state_slug}_{crop_slug}_{year}_{month:02d}.csv"
    )


    # --------------------------------------------------------
    # Move downloaded CSV
    # --------------------------------------------------------

    if target_file.exists():

        target_file.unlink()


    shutil.move(
        str(downloaded_file),
        str(target_file)
    )


    print(
        "Saved →",
        target_file.name
    )

    return target_file


print(
    "Corrected collect_one_month() loaded successfully ✅"
)


   

   

Corrected collect_one_month() loaded successfully ✅


In [259]:
try:
    driver.quit()
except:
    pass

driver = create_new_driver()

In [260]:
test_file = collect_one_month(
    driver=driver,
    state="Haryana",
    crop="Wheat",
    year=2021,
    month=3
)

print(
    "TEST SUCCESS ✅"
)

print(
    test_file
)

Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → March selected
Saved → haryana_wheat_2021_03.csv
TEST SUCCESS ✅
..\..\data\raw\prices\haryana_wheat_2021_03.csv


In [261]:
#Running full haryana wheat collection
driver, haryana_wheat_success, haryana_wheat_failed, haryana_wheat_skipped, haryana_wheat_unavailable = (
    collect_price_history_adaptive(
        driver=driver,
        state="Haryana",
        crop="Wheat"
    )
)


######################################################################
Haryana | Wheat | 2021
Month discovery attempt: 1
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Available months: ['March', 'April', 'May', 'June']

2021 January
Not available in AGMARKNET

2021 February
Not available in AGMARKNET

2021 March
Already downloaded → SKIPPED ✅

2021 April
Download attempt 1/2
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → April selected
Saved → haryana_wheat_2021_04.csv
Downloaded successfully ✅

2021 May
Download attempt 1/2
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → May selected
Saved → haryana_wheat_2021_05.csv
Downloaded successfully ✅

2021 June
Download attempt 1/2
Select Commodity → Wheat selected
State/UT → Haryana selected
Year → 2021 selected
Month → June selected
Saved → haryana_wheat_2021_06.csv
Downloaded successfully ✅

2021 July
Not available i

In [262]:
print(
    "Downloaded this run:",
    len(haryana_wheat_success)
)

print(
    "Failed:",
    len(haryana_wheat_failed)
)

print(
    "Skipped:",
    len(haryana_wheat_skipped)
)

print(
    "Unavailable:",
    len(haryana_wheat_unavailable)
)


Downloaded this run: 37
Failed: 0
Skipped: 1
Unavailable: 16


In [263]:
haryana_wheat_files = list(
    Path(
        "../../data/raw/prices"
    ).glob(
        "haryana_wheat_*.csv"
    )
)

print(
    "Haryana Wheat files on disk:",
    len(haryana_wheat_files)
)

Haryana Wheat files on disk: 38


In [264]:
haryana_wheat_monthly = load_price_history(
    state="Haryana",
    crop="Wheat"
)

Haryana | Wheat
Raw monthly files found: 38
Rows: 910
Months: 38
Markets: 104


In [265]:
print(
    "Months available:",
    haryana_wheat_monthly[
        "Month_Date"
    ].nunique()
)

print(
    "Unique markets:",
    haryana_wheat_monthly[
        "Market"
    ].nunique()
)

print(
    "Total rows:",
    len(haryana_wheat_monthly)
)

print(
    "First month:",
    haryana_wheat_monthly[
        "Month_Date"
    ].min()
)

print(
    "Last month:",
    haryana_wheat_monthly[
        "Month_Date"
    ].max()
)

Months available: 38
Unique markets: 104
Total rows: 910
First month: 2021-03-01 00:00:00
Last month: 2025-06-01 00:00:00


In [266]:
#Getting haryana district production names
haryana_production_districts = sorted(
    production_history.loc[
        production_history["State"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "haryana",
        "District"
    ]
    .dropna()
    .unique()
)

print(
    "Haryana production districts:",
    len(haryana_production_districts)
)

print(
    haryana_production_districts
)

Haryana production districts: 22
['Ambala', 'Bhiwani', 'Charkhi Dadri', 'Faridabad', 'Fatehabad', 'Gurugram', 'Hisar', 'Jhajjar', 'Jind', 'Kaithal', 'Karnal', 'Kurukshetra', 'Mahendragarh', 'Nuh', 'Palwal', 'Panchkula', 'Panipat', 'Rewari', 'Rohtak', 'Sirsa', 'Sonipat', 'Yamunanagar']


In [267]:
haryana_wheat_markets = (
    haryana_wheat_monthly[
        [
            "Market",
            "Market_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Market"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Unique Haryana Wheat markets:",
    len(haryana_wheat_markets)
)

print(
    haryana_wheat_markets.to_string(
        index=False
    )
)

Unique Haryana Wheat markets: 104
                             Market                     Market_Key
                       Adampur APMC                        adampur
               Adampur(Agroha) APMC                adampur(agroha)
                 Ambala Cantt. APMC                  ambala cantt.
                   Ambala City APMC                    ambala city
      Ambala City(Subji Mandi) APMC       ambala city(subji mandi)
                        Asandh APMC                         asandh
                         Ateli APMC                          ateli
                        Babain APMC                         babain
                   Ballabhgarh APMC                    ballabhgarh
                       Bapauli APMC                        bapauli
                        Barara APMC                         barara
                       Barwala APMC                        barwala
                Barwala(Hisar) APMC                 barwala(hisar)
                          Ba

In [268]:
def extract_parent_market(name):

    name = str(name)

    name = re.sub(
        r"\bAPMC\b",
        "",
        name,
        flags=re.IGNORECASE
    )

    name = name.split("(")[0]

    return name.strip()


def compact_market_key(name):

    name = str(name).lower().strip()

    name = re.sub(
        r"[^a-z0-9\s]",
        " ",
        name
    )

    name = re.sub(
        r"\bapmc\b",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        "",
        name
    )

    return name

In [269]:
haryana_wheat_markets[
    "Parent_Market"
] = (
    haryana_wheat_markets[
        "Market"
    ]
    .apply(
        extract_parent_market
    )
)

haryana_wheat_markets[
    "Parent_Key"
] = (
    haryana_wheat_markets[
        "Parent_Market"
    ]
    .apply(
        compact_market_key
    )
)

In [270]:
haryana_district_lookup = pd.DataFrame(
    {
        "District":
        haryana_production_districts
    }
)

haryana_district_lookup[
    "District_Key"
] = (
    haryana_district_lookup[
        "District"
    ]
    .apply(
        compact_market_key
    )
)

In [271]:
haryana_initial_matches = (
    haryana_wheat_markets
    .merge(
        haryana_district_lookup,
        left_on="Parent_Key",
        right_on="District_Key",
        how="left"
    )
)

In [272]:
print(
    "Automatically matched markets:",
    haryana_initial_matches[
        "District"
    ].notna().sum()
)

print(
    "Still needing mapping:",
    haryana_initial_matches[
        "District"
    ].isna().sum()
)

Automatically matched markets: 11
Still needing mapping: 93


In [273]:
print(
    haryana_initial_matches.loc[
        haryana_initial_matches[
            "District"
        ].notna(),
        [
            "Market",
            "Parent_Market",
            "District"
        ]
    ]
    .to_string(
        index=False
    )
)

                Market Parent_Market  District
        Faridabad APMC     Faridabad Faridabad
          Jhajjar APMC       Jhajjar   Jhajjar
          Kaithal APMC       Kaithal   Kaithal
           Palwal APMC        Palwal    Palwal
          Panipat APMC       Panipat   Panipat
Panipat(Babarpur) APMC       Panipat   Panipat
           Rewari APMC        Rewari    Rewari
           Rohtak APMC        Rohtak    Rohtak
 Rohtak(Kalanaur) APMC        Rohtak    Rohtak
            Sirsa APMC         Sirsa     Sirsa
   Sirsa(Malekan) APMC         Sirsa     Sirsa


In [274]:
haryana_unmapped_markets = (
    haryana_initial_matches.loc[
        haryana_initial_matches[
            "District"
        ].isna(),
        [
            "Market",
            "Market_Key",
            "Parent_Market",
            "Parent_Key"
        ]
    ]
    .reset_index(
        drop=True
    )
)

print(
    "\nUnresolved Haryana markets:",
    len(haryana_unmapped_markets)
)

print(
    haryana_unmapped_markets.to_string(
        index=False
    )
)


Unresolved Haryana markets: 93
                             Market                     Market_Key                Parent_Market              Parent_Key
                       Adampur APMC                        adampur                      Adampur                 adampur
               Adampur(Agroha) APMC                adampur(agroha)                      Adampur                 adampur
                 Ambala Cantt. APMC                  ambala cantt.                Ambala Cantt.             ambalacantt
                   Ambala City APMC                    ambala city                  Ambala City              ambalacity
      Ambala City(Subji Mandi) APMC       ambala city(subji mandi)                  Ambala City              ambalacity
                        Asandh APMC                         asandh                       Asandh                  asandh
                         Ateli APMC                          ateli                        Ateli                   ateli
        

In [275]:
#Load the official haryana mandi table
import pandas as pd

hsamb_url = (
    "https://hsamb.org.in/"
    "hsamb-mandi-s"
)

haryana_official_tables = pd.read_html(
    hsamb_url
)

print(
    "Tables found:",
    len(haryana_official_tables)
)

for i, table in enumerate(
    haryana_official_tables
):
    print(
        "\nTABLE:",
        i,
        "SHAPE:",
        table.shape
    )

    print(
        table.head(3)
    )

Tables found: 2

TABLE: 0 SHAPE: (3, 2)
              Type  Count
0   Principal Yard    114
1  Purchase Center    220
2         Sub Yard     83

TABLE: 1 SHAPE: (272, 8)
  List of Purchase Center, Sub Yard & Principal Yard          \
                                    Name of District Sr. No.   
0                                             Ambala       1   
1                                             Ambala       2   
2                                             Ambala       3   

                                                                  \
  Market Committee (PY) Sr. No..1 Purchase Center (PC) Sr. No..2   
0           Ambala City       NaN                  NaN       NaN   
1          Ambala Cantt       NaN                  NaN       NaN   
2                Barara       1.0                Ugala       NaN   

                                    
  Sub Yard (SY) Unnamed: 7_level_1  
0           NaN                NaN  
1           NaN                NaN  
2           NaN     

In [276]:
target_table_index = None

for i, table in enumerate(
    haryana_official_tables
):

    column_text = " ".join(
        [
            str(col)
            for col in table.columns
        ]
    ).lower()

    sample_text = (
        table.head(10)
        .astype(str)
        .to_string()
        .lower()
    )

    combined_text = (
        column_text
        + " "
        + sample_text
    )

    if (
        "district" in combined_text
        and
        (
            "market committee"
            in combined_text
            or
            "principal"
            in combined_text
        )
    ):

        target_table_index = i

        print(
            "Likely Haryana mandi table:",
            i
        )

        print(
            table.head(15)
        )

        break


print(
    "\nSelected table index:",
    target_table_index
)

Likely Haryana mandi table: 1
   List of Purchase Center, Sub Yard & Principal Yard          \
                                     Name of District Sr. No.   
0                                              Ambala       1   
1                                              Ambala       2   
2                                              Ambala       3   
3                                              Ambala       3   
4                                              Ambala       4   
5                                              Ambala       4   
6                                              Ambala       5   
7                                              Ambala       6   
8                                              Ambala       6   
9                                              Ambala       7   
10                                            Bhiwani       8   
11                                            Bhiwani       8   
12                                            Bhiwani       

In [277]:
haryana_official_raw = (
    haryana_official_tables[1]
    .copy()
)

print(
    "Shape:",
    haryana_official_raw.shape
)

print(
    "\nColumns:"
)

print(
    haryana_official_raw.columns
)

print(
    "\nFirst 20 rows:"
)

display(
    haryana_official_raw.head(20)
)

Shape: (272, 8)

Columns:
MultiIndex([('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...),
            ('List of Purchase Center, Sub Yard & Principal Yard', ...)],
           )

First 20 rows:


List of Purchase Center, Sub Yard & Principal Yard          \
                                     Name of District Sr. No.   
0                                              Ambala       1   
1                                              Ambala       2   
2                                              Ambala       3   
3                                              Ambala       3   
4                                              Ambala       4   
5                                              Ambala       4   
6                                              Ambala       5   
7                                              Ambala       6   
8                                              Ambala       6   
9                                              Ambala       7   
10                                            Bhiwani       8   
11                                            Bhiwani       8   
12                                            Bhiwani       8   
13                                            Bhiwani       9   
14                                            Bhiwani      10   
15                                            Bhiwani      11   
16                                            Bhiwani      12   
17                                            Bhiwani      13   
18                                      Charkhi Dadri      14   
19                                      Charkhi Dadri      14   

                                                                   \
   Market Committee (PY) Sr. No..1 Purchase Center (PC) Sr. No..2   
0            Ambala City       NaN                  NaN       NaN   
1           Ambala Cantt       NaN                  NaN       NaN   
2                 Barara       1.0                Ugala       NaN   
3                 Barara       2.0             Sardehri       NaN   
4                Mullana       3.0       Talheri Gujran       1.0   
5                Mullana       NaN                  NaN       2.0   
6                Naneola       NaN                  NaN       NaN   
7             Naraingarh       4.0       Bhareri Kallan       NaN   
8             Naraingarh       5.0             Berkhedi       NaN   
9             Shahzadpur       6.0              Karasan       NaN   
10           NGM Bhiwani       7.0              Dhanana       3.0   
11           NGM Bhiwani       8.0                Chang       4.0   
12           NGM Bhiwani       9.0           Banda Heri       5.0   
13            NGM Tosham       NaN                  NaN       NaN   
14                 Behal       NaN                  NaN       NaN   
15                Siwani       NaN                  NaN       6.0   
16                   Jui       NaN                  NaN       7.0   
17                Loharu       NaN                  NaN       8.0   
18         Charkhi Dadri      10.0         Jhojhu Kalan       9.0   
19         Charkhi Dadri      11.0          Bound Khurd       NaN   

                                                
              Sub Yard (SY) Unnamed: 7_level_1  
0                       NaN                NaN  
1                       NaN                NaN  
2                       NaN                NaN  
3                       NaN                NaN  
4                      Saha                NaN  
5                     Kesri                NaN  
6                       NaN                NaN  
7                       NaN                NaN  
8                       NaN                NaN  
9                       NaN                NaN  
10             Bawani Khera                NaN  
11              NVM Bhiwani                NaN  
12  Add. NGM (TMC)  Bhiwani                NaN  
13                      NaN                NaN  
14                      NaN                NaN  
15             Jhumpa Kalan                NaN  
16                    Kairu                NaN  
17                 Dhighawa                NaN  
18                   Badhra                NaN  
19                      

In [278]:
print(
    "\nLast 10 rows:"
)

display(
    haryana_official_raw.tail(10)
)


Last 10 rows:


List of Purchase Center, Sub Yard & Principal Yard          \
                                      Name of District Sr. No.   
262                                             Karnal     107   
263                                             Karnal     107   
264                                             Karnal     107   
265                                             Karnal     108   
266                                             Karnal     109   
267                                             Karnal     110   
268                                             Karnal     111   
269                                             Karnal     112   
270                                             Karnal     113   
271                                             Karnal     114   

                                                                    \
    Market Committee (PY) Sr. No..1 Purchase Center (PC) Sr. No..2   
262           NGM Assandh     214.0               Kurlan       NaN   
263           NGM Assandh     215.0               Rahara      82.0   
264           NGM Assandh     216.0            Gangateri       NaN   
265               Taraori     217.0                Sagga       NaN   
266                 Indri     218.0         Garhi Birbal      83.0   
267             Nilokheri     219.0          Samana Bahu       NaN   
268               Nissing       NaN                  NaN       NaN   
269          NGM Kunjpura     220.0                Gheer       NaN   
270                Nigdhu       NaN                  NaN       NaN   
271                Jundla       NaN                  NaN       NaN   

                                      
    Sub Yard (SY) Unnamed: 7_level_1  
262           NaN                NaN  
263        Ballah                NaN  
264           NaN                NaN  
265           NaN                NaN  
266         Biana                NaN  
267           NaN                NaN  
268           NaN                NaN  
269           NaN                NaN  
270           NaN                NaN  
271           NaN                NaN

In [279]:
print(
    "Column type:",
    type(
        haryana_official_raw.columns
    )
)

print(
    "Is MultiIndex:",
    isinstance(
        haryana_official_raw.columns,
        pd.MultiIndex
    )
)

Column type: <class 'pandas.core.indexes.multi.MultiIndex'>
Is MultiIndex: True


In [280]:
haryana_official_raw.to_csv(
    "../../data/raw/reference/"
    "haryana_hsamb_official_mandi_table.csv",
    index=False
)

print(
    "Raw official Haryana mandi table saved ✅"
)

Raw official Haryana mandi table saved ✅


In [281]:
#Flatten multiindex column
haryana_official_flat = (
    haryana_official_raw
    .copy()
)

haryana_official_flat.columns = (
    haryana_official_flat
    .columns
    .get_level_values(1)
)

print(
    haryana_official_flat.columns.tolist()
)

display(
    haryana_official_flat.head(20)
)

['Name of District', 'Sr. No.', 'Market Committee (PY)', 'Sr. No..1', 'Purchase Center (PC)', 'Sr. No..2', 'Sub Yard (SY)', 'Unnamed: 7_level_1']


,Name of District,Sr. No.,Market Committee (PY),Sr. No..1,Purchase Center (PC),Sr. No..2,Sub Yard (SY),Unnamed: 7_level_1
0,Ambala,1,Ambala City,NaN,NaN,NaN,NaN,NaN
1,Ambala,2,Ambala Cantt,NaN,NaN,NaN,NaN,NaN
2,Ambala,3,Barara,1.0,Ugala,NaN,NaN,NaN
3,Ambala,3,Barara,2.0,Sardehri,NaN,NaN,NaN
4,Ambala,4,Mullana,3.0,Talheri Gujran,1.0,Saha,NaN
5,Ambala,4,Mullana,NaN,NaN,2.0,Kesri,NaN
6,Ambala,5,Naneola,NaN,NaN,NaN,NaN,NaN
7,Ambala,6,Naraingarh,4.0,Bhareri Kallan,NaN,NaN,NaN
8,Ambala,6,Naraingarh,5.0,Berkhedi,NaN,NaN,NaN
9,Ambala,7,Shahzadpur,6.0,Karasan,NaN,NaN,NaN


In [282]:
district_raw = (
    haryana_official_flat[
        "Name of District"
    ]
    .astype(str)
    .str.strip()
)

print(
    "First 40 district values:"
)

for i, value in enumerate(
    district_raw.head(40)
):
    print(
        i,
        repr(value)
    )

First 40 district values:
0 'Ambala'
1 'Ambala'
2 'Ambala'
3 'Ambala'
4 'Ambala'
5 'Ambala'
6 'Ambala'
7 'Ambala'
8 'Ambala'
9 'Ambala'
10 'Bhiwani'
11 'Bhiwani'
12 'Bhiwani'
13 'Bhiwani'
14 'Bhiwani'
15 'Bhiwani'
16 'Bhiwani'
17 'Bhiwani'
18 'Charkhi Dadri'
19 'Charkhi Dadri'
20 'Charkhi Dadri'
21 'Charkhi Dadri'
22 'Nuh'
23 'Nuh'
24 'Nuh'
25 'Nuh'
26 'Faridabad'
27 'Faridabad'
28 'Faridabad'
29 'Faridabad'
30 'Fatehabad'
31 'Fatehabad'
32 'Fatehabad'
33 'Fatehabad'
34 'Fatehabad'
35 'Fatehabad'
36 'Fatehabad'
37 'Fatehabad'
38 'Fatehabad'
39 'Fatehabad'


In [283]:
print(
    "\nUnique raw district values:"
)

print(
    sorted(
        district_raw[
            ~district_raw.isin(
                [
                    "",
                    "nan",
                    "None"
                ]
            )
        ].unique()
    )
)


Unique raw district values:
['Ambala', 'Bhiwani', 'Charkhi Dadri', 'Faridabad', 'Fatehabad', 'Gurugram', 'Hisar', 'Jhajjar', 'Jind', 'Kaithal', 'Karnal', 'Kurukshetra', 'Mohindergarh', 'Nuh', 'Palwal', 'Panchkula', 'Panipat', 'Rewari', 'Rohtak', 'Sirsa', 'Sonipat', 'Yamuna Nagar']


In [284]:
suspicious_rows = (
    haryana_official_flat[
        haryana_official_flat[
            "Name of District"
        ]
        .astype(str)
        .str.contains(
            "Amba",
            case=False,
            na=False
        )
    ]
)

display(
    suspicious_rows
)

,Name of District,Sr. No.,Market Committee (PY),Sr. No..1,Purchase Center (PC),Sr. No..2,Sub Yard (SY),Unnamed: 7_level_1
0,Ambala,1,Ambala City,NaN,NaN,NaN,NaN,NaN
1,Ambala,2,Ambala Cantt,NaN,NaN,NaN,NaN,NaN
2,Ambala,3,Barara,1.0,Ugala,NaN,NaN,NaN
3,Ambala,3,Barara,2.0,Sardehri,NaN,NaN,NaN
4,Ambala,4,Mullana,3.0,Talheri Gujran,1.0,Saha,NaN
5,Ambala,4,Mullana,NaN,NaN,2.0,Kesri,NaN
6,Ambala,5,Naneola,NaN,NaN,NaN,NaN,NaN
7,Ambala,6,Naraingarh,4.0,Bhareri Kallan,NaN,NaN,NaN
8,Ambala,6,Naraingarh,5.0,Berkhedi,NaN,NaN,NaN
9,Ambala,7,Shahzadpur,6.0,Karasan,NaN,NaN,NaN


In [285]:
#Forward-fill district names
haryana_official_flat[
    "Name of District"
] = (
    haryana_official_flat[
        "Name of District"
    ]
    .replace(
        ["", "nan", "None"],
        pd.NA
    )
    .ffill()
)

print(
    "Missing districts after forward fill:",
    haryana_official_flat[
        "Name of District"
    ].isna().sum()
)

Missing districts after forward fill: 0


In [286]:
principal_yards = (
    haryana_official_flat[
        [
            "Name of District",
            "Market Committee (PY)"
        ]
    ]
    .copy()
)

principal_yards.columns = [
    "District",
    "Market_Name"
]

principal_yards[
    "Market_Type"
] = "Principal Yard"

In [287]:
purchase_centers = (
    haryana_official_flat[
        [
            "Name of District",
            "Purchase Center (PC)"
        ]
    ]
    .copy()
)

purchase_centers.columns = [
    "District",
    "Market_Name"
]

purchase_centers[
    "Market_Type"
] = "Purchase Center"

In [288]:
sub_yards = (
    haryana_official_flat[
        [
            "Name of District",
            "Sub Yard (SY)"
        ]
    ]
    .copy()
)

sub_yards.columns = [
    "District",
    "Market_Name"
]

sub_yards[
    "Market_Type"
] = "Sub Yard"

In [289]:
haryana_official_mapping = pd.concat(
    [
        principal_yards,
        purchase_centers,
        sub_yards
    ],
    ignore_index=True
)

In [290]:
haryana_official_mapping[
    "Market_Name"
] = (
    haryana_official_mapping[
        "Market_Name"
    ]
    .astype(str)
    .str.strip()
)

haryana_official_mapping = (
    haryana_official_mapping[
        ~haryana_official_mapping[
            "Market_Name"
        ].isin(
            [
                "",
                "nan",
                "None"
            ]
        )
    ]
    .copy()
)

In [291]:
haryana_official_mapping = (
    haryana_official_mapping
    .reset_index(drop=True)
)

print(
    "Official Haryana market rows:",
    len(haryana_official_mapping)
)

print(
    "\nBy market type:"
)

print(
    haryana_official_mapping[
        "Market_Type"
    ]
    .value_counts()
)

Official Haryana market rows: 578

By market type:
Market_Type
Principal Yard     272
Purchase Center    221
Sub Yard            85
Name: count, dtype: int64


In [292]:
haryana_official_mapping[
    "Market_Key"
] = (
    haryana_official_mapping[
        "Market_Name"
    ]
    .apply(
        normalize_market_name
    )
)

haryana_official_mapping[
    "Compact_Key"
] = (
    haryana_official_mapping[
        "Market_Name"
    ]
    .apply(
        compact_market_key
    )
)

In [293]:
display(
    haryana_official_mapping.head(20)
)

,District,Market_Name,Market_Type,Market_Key,Compact_Key
0,Ambala,Ambala City,Principal Yard,ambala city,ambalacity
1,Ambala,Ambala Cantt,Principal Yard,ambala cantt,ambalacantt
2,Ambala,Barara,Principal Yard,barara,barara
3,Ambala,Barara,Principal Yard,barara,barara
4,Ambala,Mullana,Principal Yard,mullana,mullana
5,Ambala,Mullana,Principal Yard,mullana,mullana
6,Ambala,Naneola,Principal Yard,naneola,naneola
7,Ambala,Naraingarh,Principal Yard,naraingarh,naraingarh
8,Ambala,Naraingarh,Principal Yard,naraingarh,naraingarh
9,Ambala,Shahzadpur,Principal Yard,shahzadpur,shahzadpur


In [294]:
official_haryana_districts = set(
    haryana_official_mapping[
        "District"
    ].dropna()
)

production_haryana_districts = set(
    haryana_production_districts
)

print(
    "Official districts not matching production:"
)

print(
    sorted(
        official_haryana_districts
        -
        production_haryana_districts
    )
)

Official districts not matching production:
['Mohindergarh', 'Yamuna Nagar']


In [295]:
print(
    "Production districts not matching official:"
)

print(
    sorted(
        production_haryana_districts
        -
        official_haryana_districts
    )
)

Production districts not matching official:
['Mahendragarh', 'Yamunanagar']


In [296]:
haryana_official_mapping.to_csv(
    "../../data/processed/"
    "haryana_official_market_reference.csv",
    index=False
)

print(
    "Clean Haryana official market reference saved ✅"
)

Clean Haryana official market reference saved ✅


In [297]:
haryana_unmapped_markets[
    "Compact_Key"
] = (
    haryana_unmapped_markets[
        "Market"
    ]
    .apply(
        compact_market_key
    )
)

In [298]:
official_key_conflicts = (
    haryana_official_mapping
    .groupby(
        "Compact_Key"
    )["District"]
    .nunique()
)

print(
    "Official compact-key conflicts:",
    (
        official_key_conflicts > 1
    ).sum()
)

Official compact-key conflicts: 1


In [299]:
haryana_safe_official_lookup = (
    haryana_official_mapping[
        haryana_official_mapping[
            "Compact_Key"
        ].isin(
            official_key_conflicts[
                official_key_conflicts == 1
            ].index
        )
    ]
    [
        [
            "Compact_Key",
            "Market_Name",
            "District",
            "Market_Type"
        ]
    ]
    .drop_duplicates(
        subset=["Compact_Key"]
    )
)

In [300]:
haryana_official_matches = (
    haryana_unmapped_markets
    .merge(
        haryana_safe_official_lookup,
        on="Compact_Key",
        how="left"
    )
)

In [301]:
print(
    "Official exact matches:",
    haryana_official_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    haryana_official_matches[
        "District"
    ].isna().sum()
)

Official exact matches: 41
Still unresolved: 52


In [302]:
print(
    "Rows before deduplication:",
    len(haryana_official_mapping)
)

haryana_official_mapping = (
    haryana_official_mapping
    .drop_duplicates(
        subset=[
            "District",
            "Market_Name",
            "Market_Type"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Rows after deduplication:",
    len(haryana_official_mapping)
)

print(
    "\nUnique markets by type:"
)

print(
    haryana_official_mapping[
        "Market_Type"
    ]
    .value_counts()
)

Rows before deduplication: 578
Rows after deduplication: 418

Unique markets by type:
Market_Type
Purchase Center    220
Principal Yard     114
Sub Yard            84
Name: count, dtype: int64


In [303]:
haryana_district_name_fix = {
    "Mohindergarh": "Mahendragarh",
    "Yamuna Nagar": "Yamunanagar"
}

haryana_official_mapping[
    "District"
] = (
    haryana_official_mapping[
        "District"
    ]
    .replace(
        haryana_district_name_fix
    )
)

In [304]:
official_haryana_districts = set(
    haryana_official_mapping[
        "District"
    ].dropna()
)

production_haryana_districts = set(
    haryana_production_districts
)

print(
    "Official districts not in production:",
    sorted(
        official_haryana_districts
        -
        production_haryana_districts
    )
)

print(
    "Production districts not in official:",
    sorted(
        production_haryana_districts
        -
        official_haryana_districts
    )
)

Official districts not in production: []
Production districts not in official: []


In [305]:
haryana_official_mapping[
    "Market_Key"
] = (
    haryana_official_mapping[
        "Market_Name"
    ]
    .apply(
        normalize_market_name
    )
)

haryana_official_mapping[
    "Compact_Key"
] = (
    haryana_official_mapping[
        "Market_Name"
    ]
    .apply(
        compact_market_key
    )
)

In [306]:
official_key_conflicts = (
    haryana_official_mapping
    .groupby(
        "Compact_Key"
    )["District"]
    .nunique()
)

conflicting_keys = (
    official_key_conflicts[
        official_key_conflicts > 1
    ]
    .index
)

print(
    "Compact-key conflicts:",
    len(conflicting_keys)
)

display(
    haryana_official_mapping[
        haryana_official_mapping[
            "Compact_Key"
        ].isin(
            conflicting_keys
        )
    ][
        [
            "Market_Name",
            "District",
            "Market_Type",
            "Compact_Key"
        ]
    ]
    .sort_values(
        [
            "Compact_Key",
            "District"
        ]
    )
)

Compact-key conflicts: 1


,Market_Name,District,Market_Type,Compact_Key
137,Dhand,Fatehabad,Purchase Center,dhand
56,Dhand,Kaithal,Principal Yard,dhand


In [307]:
safe_keys = (
    official_key_conflicts[
        official_key_conflicts == 1
    ]
    .index
)

haryana_safe_official_lookup = (
    haryana_official_mapping[
        haryana_official_mapping[
            "Compact_Key"
        ].isin(
            safe_keys
        )
    ]
    [
        [
            "Compact_Key",
            "Market_Name",
            "District",
            "Market_Type"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Compact_Key"
        ]
    )
)

In [308]:
haryana_official_matches = (
    haryana_unmapped_markets
    .drop(
        columns=[
            "Compact_Key"
        ],
        errors="ignore"
    )
    .copy()
)

haryana_official_matches[
    "Compact_Key"
] = (
    haryana_official_matches[
        "Market"
    ]
    .apply(
        compact_market_key
    )
)

haryana_official_matches = (
    haryana_official_matches
    .merge(
        haryana_safe_official_lookup,
        on="Compact_Key",
        how="left"
    )
)

In [309]:
print(
    "Official exact matches:",
    haryana_official_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    haryana_official_matches[
        "District"
    ].isna().sum()
)

Official exact matches: 41
Still unresolved: 52


In [310]:
haryana_remaining_unresolved = (
    haryana_official_matches[
        haryana_official_matches[
            "District"
        ].isna()
    ]
    [
        [
            "Market",
            "Market_Key",
            "Parent_Market",
            "Parent_Key"
        ]
    ]
    .reset_index(drop=True)
)

print(
    "\nRemaining Haryana markets:",
    len(
        haryana_remaining_unresolved
    )
)

print(
    haryana_remaining_unresolved
    .to_string(index=False)
)


Remaining Haryana markets: 52
                             Market                     Market_Key                Parent_Market              Parent_Key
               Adampur(Agroha) APMC                adampur(agroha)                      Adampur                 adampur
      Ambala City(Subji Mandi) APMC       ambala city(subji mandi)                  Ambala City              ambalacity
                        Asandh APMC                         asandh                       Asandh                  asandh
                   Ballabhgarh APMC                    ballabhgarh                  Ballabhgarh             ballabhgarh
                       Bapauli APMC                        bapauli                      Bapauli                 bapauli
                Barwala(Hisar) APMC                 barwala(hisar)                      Barwala                 barwala
                  Bhattu Kalan APMC                   bhattu kalan                 Bhattu Kalan             bhattukalan
         

In [311]:
haryana_official_mapping.to_csv(
    "../../data/processed/"
    "haryana_official_market_reference.csv",
    index=False
)

print(
    "Corrected Haryana official reference saved ✅"
)

Corrected Haryana official reference saved ✅


In [312]:
official_parent_lookup = (
    haryana_official_mapping
    .copy()
)

official_parent_lookup[
    "Parent_Key"
] = (
    official_parent_lookup[
        "Market_Name"
    ]
    .apply(
        compact_market_key
    )
)

In [313]:
parent_conflict_check = (
    official_parent_lookup
    .groupby(
        "Parent_Key"
    )["District"]
    .nunique()
)

print(
    "Parent-key conflicts:",
    (
        parent_conflict_check > 1
    ).sum()
)

Parent-key conflicts: 1


In [314]:
safe_parent_keys = (
    parent_conflict_check[
        parent_conflict_check == 1
    ]
    .index
)

safe_parent_lookup = (
    official_parent_lookup[
        official_parent_lookup[
            "Parent_Key"
        ].isin(
            safe_parent_keys
        )
    ]
    [
        [
            "Parent_Key",
            "Market_Name",
            "District",
            "Market_Type"
        ]
    ]
    .drop_duplicates(
        subset=["Parent_Key"]
    )
)

In [315]:
haryana_parent_matches = (
    haryana_remaining_unresolved
    .merge(
        safe_parent_lookup,
        on="Parent_Key",
        how="left"
    )
)

In [316]:
print(
    "Parent matches found:",
    haryana_parent_matches[
        "District"
    ].notna().sum()
)

print(
    "Still unresolved:",
    haryana_parent_matches[
        "District"
    ].isna().sum()
)

Parent matches found: 6
Still unresolved: 46


In [317]:
print(
    haryana_parent_matches.loc[
        haryana_parent_matches[
            "District"
        ].notna(),
        [
            "Market",
            "Parent_Market",
            "Market_Name",
            "District",
            "Market_Type"
        ]
    ]
    .to_string(index=False)
)

                       Market Parent_Market Market_Name    District    Market_Type
         Adampur(Agroha) APMC       Adampur     Adampur       Hisar Principal Yard
Ambala City(Subji Mandi) APMC   Ambala City Ambala City      Ambala Principal Yard
          Barwala(Hisar) APMC       Barwala     Barwala       Hisar Principal Yard
       Dabwali(Chautala) APMC       Dabwali     Dabwali       Sirsa Principal Yard
           Mullana(saha) APMC       Mullana     Mullana      Ambala Principal Yard
  Pehowa(Gumthala Gahru) APMC        Pehowa      Pehowa Kurukshetra Principal Yard


In [318]:
haryana_after_parent_unresolved = (
    haryana_parent_matches[
        haryana_parent_matches[
            "District"
        ].isna()
    ]
    [
        [
            "Market",
            "Market_Key",
            "Parent_Market",
            "Parent_Key"
        ]
    ]
    .reset_index(drop=True)
)

print(
    "\nStill unresolved after parent matching:",
    len(haryana_after_parent_unresolved)
)

print(
    haryana_after_parent_unresolved
    .to_string(index=False)
)


Still unresolved after parent matching: 46
                             Market                     Market_Key                Parent_Market              Parent_Key
                        Asandh APMC                         asandh                       Asandh                  asandh
                   Ballabhgarh APMC                    ballabhgarh                  Ballabhgarh             ballabhgarh
                       Bapauli APMC                        bapauli                      Bapauli                 bapauli
                  Bhattu Kalan APMC                   bhattu kalan                 Bhattu Kalan             bhattukalan
                         Bhuna APMC                          bhuna                        Bhuna                   bhuna
                      Bilaspur APMC                       bilaspur                     Bilaspur                bilaspur
                        Cheeka APMC                         cheeka                       Cheeka                  che

In [319]:
#Fuzzy-match the remaining 46 against the official HSAMB reference
from rapidfuzz import process, fuzz

In [320]:
official_haryana_names = (
    haryana_official_mapping[
        "Market_Name"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

print(
    "Official Haryana names available:",
    len(official_haryana_names)
)

Official Haryana names available: 417


In [321]:
def get_best_haryana_fuzzy_match(
    market_name
):

    result = process.extractOne(
        market_name,
        official_haryana_names,
        scorer=fuzz.WRatio
    )

    if result is None:
        return pd.Series(
            [None, None]
        )

    candidate, score, _ = result

    return pd.Series(
        [
            candidate,
            score
        ]
    )

In [322]:
haryana_fuzzy_review = (
    haryana_after_parent_unresolved
    .copy()
)

haryana_fuzzy_review[
    [
        "Best_Official_Match",
        "Fuzzy_Score"
    ]
] = (
    haryana_fuzzy_review[
        "Parent_Market"
    ]
    .apply(
        get_best_haryana_fuzzy_match
    )
)

In [323]:
official_name_lookup = (
    haryana_official_mapping[
        [
            "Market_Name",
            "District",
            "Market_Type"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Market_Name",
            "District"
        ]
    )
)

In [324]:
haryana_fuzzy_review = (
    haryana_fuzzy_review
    .merge(
        official_name_lookup,
        left_on="Best_Official_Match",
        right_on="Market_Name",
        how="left"
    )
)

In [325]:
print(
    "90+ :",
    (
        haryana_fuzzy_review[
            "Fuzzy_Score"
        ] >= 90
    ).sum()
)

print(
    "85-89:",
    (
        (
            haryana_fuzzy_review[
                "Fuzzy_Score"
            ] >= 85
        )
        &
        (
            haryana_fuzzy_review[
                "Fuzzy_Score"
            ] < 90
        )
    ).sum()
)

print(
    "70-84:",
    (
        (
            haryana_fuzzy_review[
                "Fuzzy_Score"
            ] >= 70
        )
        &
        (
            haryana_fuzzy_review[
                "Fuzzy_Score"
            ] < 85
        )
    ).sum()
)

print(
    "Below 70:",
    (
        haryana_fuzzy_review[
            "Fuzzy_Score"
        ] < 70
    ).sum()
)

90+ : 35
85-89: 3
70-84: 7
Below 70: 2


In [326]:
haryana_high_confidence_review = (
    haryana_fuzzy_review[
        haryana_fuzzy_review[
            "Fuzzy_Score"
        ] >= 90
    ]
    [
        [
            "Market",
            "Parent_Market",
            "Best_Official_Match",
            "District",
            "Market_Type",
            "Fuzzy_Score"
        ]
    ]
    .sort_values(
        "Fuzzy_Score",
        ascending=False
    )
)

print(
    haryana_high_confidence_review
    .to_string(index=False)
)

                             Market                Parent_Market              Best_Official_Match     District     Market_Type  Fuzzy_Score
                         Dhand APMC                        Dhand                            Dhand      Kaithal  Principal Yard   100.000000
                         Dhand APMC                        Dhand                            Dhand    Fatehabad Purchase Center   100.000000
                   Ballabhgarh APMC                  Ballabhgarh                       Ballabgarh    Faridabad  Principal Yard    95.238095
       Chhachhrauli(Khizrabad) APMC                 Chhachhrauli                 NGM Chhachhrauli  Yamunanagar  Principal Yard    95.000000
                  Bhattu Kalan APMC                 Bhattu Kalan                 NGM Bhattu Kalan    Fatehabad  Principal Yard    95.000000
New Grain Market(main), Karnal APMC             New Grain Market            New Grain Market Beri      Jhajjar  Principal Yard    95.000000
                    